# Multilingual Song Data Collection Pipeline
### English, Hindi and Chinese human-song dataset

This notebook builds a balanced human-song research dataset across three languages and five genres.

| Dimension | Target |
|---|---:|
| Languages | English, Hindi, Chinese |
| Genres | Pop, Rock, Hip-Hop/Rap, R&B/Soul, Folk/Acoustic |
| Songs per genre | 60 |
| Songs per language | 300 |
| Total human songs | 900 |




## 0. Install dependencies

Run the next cell once in a new environment. `ffmpeg` must also be installed on the computer for MP3 conversion.


In [ ]:
%pip install -q -U pandas numpy requests tqdm matplotlib mutagen yt-dlp rapidfuzz


## 1. Environment and project folders

This cell imports libraries, creates directories and requests API credentials. Do not hard-code credentials in a notebook that will be shared.


In [ ]:
# ============================================================
# CELL 1: ENVIRONMENT SETUP, IMPORTS & MULTI-API CONFIGURATION
# ============================================================

import os
import re
import json
import time
import random
import requests

from pathlib import Path
from getpass import getpass
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from urllib.parse import urlparse

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


# ============================================================
# OPTIONAL LIBRARIES
# ============================================================

try:
    import yt_dlp

    YTDLP_AVAILABLE = True

except ImportError:
    YTDLP_AVAILABLE = False

    print(
        "yt_dlp not installed. "
        "Download cells will not work until installed."
    )


try:
    from mutagen import File as MutagenFile

    MUTAGEN_AVAILABLE = True

except ImportError:
    MUTAGEN_AVAILABLE = False

    print(
        "mutagen not installed. "
        "Audio duration checking may not work."
    )


# ============================================================
# GLOBAL SETTINGS
# ============================================================

RANDOM_STATE = 42

random.seed(
    RANDOM_STATE
)

pd.set_option(
    "display.max_columns",
    120,
)


# ============================================================
# DIRECTORY SETUP
# ============================================================

DATASET_ROOT = Path(
    "dataset_collection_clean"
)

MANIFEST_DIR = (
    DATASET_ROOT
    / "manifests"
)

REPORT_DIR = (
    DATASET_ROOT
    / "reports"
)

CACHE_DIR = (
    DATASET_ROOT
    / "api_cache"
)

AI_ROOT = (
    DATASET_ROOT
    / "ai"
)


AUDIO_DIRS = {
    "English": (
        DATASET_ROOT
        / "human/english/audio"
    ),
    "Hindi": (
        DATASET_ROOT
        / "human/hindi/audio"
    ),
    "Chinese": (
        DATASET_ROOT
        / "human/chinese/audio"
    ),
}


AI_AUDIO_DIRS = {
    "English": (
        AI_ROOT
        / "english/audio"
    ),
    "Hindi": (
        AI_ROOT
        / "hindi/audio"
    ),
    "Chinese": (
        AI_ROOT
        / "chinese/audio"
    ),
}


folders_to_create = [
    MANIFEST_DIR,
    REPORT_DIR,
    CACHE_DIR,
    AI_ROOT,
    AI_ROOT / "prompts",
    AI_ROOT / "reports",
    AI_ROOT / "raw_sources",
    *AUDIO_DIRS.values(),
    *AI_AUDIO_DIRS.values(),
]


for folder in folders_to_create:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


print(
    "Project folders initialized successfully."
)


# ============================================================
# GENERAL API-KEY HELPERS
# ============================================================

def parse_api_keys(value):
    """
    Parse comma, semicolon or newline-separated API keys.
    """
    if not value:
        return []

    values = re.split(
        r"[,;\n]+",
        str(value),
    )

    return [
        item.strip()
        for item in values
        if item.strip()
    ]


def remove_duplicate_keys(keys):
    """
    Remove duplicates while preserving original order.
    """
    return list(
        dict.fromkeys(
            key
            for key in keys
            if str(key).strip()
        )
    )


def mask_api_key(key):
    """
    Display only a small safe portion of a credential.
    """
    key = str(
        key or ""
    ).strip()

    if not key:
        return "missing"

    if len(key) <= 10:
        return "********"

    return (
        key[:4]
        + "..."
        + key[-4:]
    )


# ============================================================
# YOUTUBE API KEYS — FIVE-KEY SUPPORT
# ============================================================

YOUTUBE_API_KEYS = []


# Option 1:
# YOUTUBE_API_KEYS=key1,key2,key3,key4,key5
YOUTUBE_API_KEYS.extend(
    parse_api_keys(
        os.getenv(
            "YOUTUBE_API_KEYS",
            "",
        )
    )
)


# Option 2:
# YOUTUBE_API_KEY_1 through YOUTUBE_API_KEY_5
for key_number in range(
    1,
    6,
):
    environment_key = os.getenv(
        f"YOUTUBE_API_KEY_{key_number}",
        "",
    ).strip()

    if environment_key:
        YOUTUBE_API_KEYS.append(
            environment_key
        )


# Backward-compatible single environment key.
single_environment_key = os.getenv(
    "YOUTUBE_API_KEY",
    "",
).strip()

if single_environment_key:
    YOUTUBE_API_KEYS.append(
        single_environment_key
    )


YOUTUBE_API_KEYS = remove_duplicate_keys(
    YOUTUBE_API_KEYS
)


# Ask for any remaining key slots, up to a total of five.
MAX_YOUTUBE_KEYS = 5

YOUTUBE_API_KEYS = remove_duplicate_keys(
    YOUTUBE_API_KEYS
)

remaining_slots = (
    MAX_YOUTUBE_KEYS
    - len(YOUTUBE_API_KEYS)
)

if remaining_slots > 0:
    print(
        f"\n{len(YOUTUBE_API_KEYS)} YouTube key(s) "
        "already detected."
    )

    print(
        f"You can enter up to "
        f"{remaining_slots} additional key(s)."
    )

    print(
        "Press Enter to skip any unused slot."
    )

    start_number = (
        len(YOUTUBE_API_KEYS)
        + 1
    )

    for key_number in range(
        start_number,
        MAX_YOUTUBE_KEYS + 1,
    ):
        entered_key = getpass(
            f"YouTube API key {key_number} "
            "(press Enter to skip): "
        ).strip()

        if entered_key:
            YOUTUBE_API_KEYS.append(
                entered_key
            )

else:
    print(
        "\nFive YouTube API keys are already configured."
    )


YOUTUBE_API_KEYS = remove_duplicate_keys(
    YOUTUBE_API_KEYS
)[:MAX_YOUTUBE_KEYS]


# Current active-key index.
YOUTUBE_KEY_INDEX = 0


# Track keys that failed during the current notebook session.
YOUTUBE_DISABLED_KEY_INDEXES = set()


# Backward compatibility:
# Older cells can continue reading YOUTUBE_API_KEY.
YOUTUBE_API_KEY = (
    YOUTUBE_API_KEYS[0]
    if YOUTUBE_API_KEYS
    else ""
)


def get_current_youtube_key():
    """
    Return the active YouTube key and update the
    backward-compatible YOUTUBE_API_KEY variable.
    """
    global YOUTUBE_API_KEY

    if not YOUTUBE_API_KEYS:
        raise ValueError(
            "No YouTube API keys are configured."
        )

    if (
        YOUTUBE_KEY_INDEX
        >= len(YOUTUBE_API_KEYS)
    ):
        raise IndexError(
            "YOUTUBE_KEY_INDEX is outside "
            "the configured key list."
        )

    YOUTUBE_API_KEY = (
        YOUTUBE_API_KEYS[
            YOUTUBE_KEY_INDEX
        ]
    )

    return YOUTUBE_API_KEY


def active_youtube_key_number():
    """
    Return a one-based key number for progress messages.
    """
    return (
        YOUTUBE_KEY_INDEX
        + 1
    )


def rotate_youtube_key(
    reason="",
    disable_current=False,
):
    """
    Move to the next available YouTube key.

    Set disable_current=True only when the current key is
    invalid, forbidden, rate-limited or quota-exhausted.
    """
    global YOUTUBE_KEY_INDEX
    global YOUTUBE_API_KEY

    if not YOUTUBE_API_KEYS:
        raise ValueError(
            "No YouTube API keys are configured."
        )

    old_index = (
        YOUTUBE_KEY_INDEX
    )

    if disable_current:
        YOUTUBE_DISABLED_KEY_INDEXES.add(
            old_index
        )

    available_indexes = [
        index
        for index in range(
            len(YOUTUBE_API_KEYS)
        )
        if index
        not in YOUTUBE_DISABLED_KEY_INDEXES
    ]

    if not available_indexes:
        raise RuntimeError(
            "All configured YouTube API keys "
            "are currently unavailable."
        )

    for offset in range(
        1,
        len(YOUTUBE_API_KEYS) + 1,
    ):
        candidate_index = (
            old_index + offset
        ) % len(YOUTUBE_API_KEYS)

        if (
            candidate_index
            in available_indexes
        ):
            YOUTUBE_KEY_INDEX = (
                candidate_index
            )

            YOUTUBE_API_KEY = (
                YOUTUBE_API_KEYS[
                    candidate_index
                ]
            )

            print(
                "YouTube key changed: "
                f"{old_index + 1} → "
                f"{candidate_index + 1}"
                + (
                    f" | reason: {reason}"
                    if reason
                    else ""
                )
            )

            return YOUTUBE_API_KEY

    raise RuntimeError(
        "No usable YouTube API key remains."
    )


def reset_youtube_key_status():
    """
    Re-enable all keys and start again from key 1.
    Useful after quota reset or after correcting credentials.
    """
    global YOUTUBE_KEY_INDEX
    global YOUTUBE_API_KEY

    YOUTUBE_DISABLED_KEY_INDEXES.clear()

    YOUTUBE_KEY_INDEX = 0

    YOUTUBE_API_KEY = (
        YOUTUBE_API_KEYS[0]
        if YOUTUBE_API_KEYS
        else ""
    )

    print(
        "YouTube key status reset."
    )


# ============================================================
# OTHER API KEYS
# ============================================================

LASTFM_API_KEY = (
    os.getenv(
        "LASTFM_API_KEY"
    )
    or getpass(
        "Enter Last.fm API key "
        "(or press Enter to skip): "
    ).strip()
)


DISCOGS_TOKEN = (
    os.getenv(
        "DISCOGS_TOKEN"
    )
    or getpass(
        "Enter Discogs token "
        "(or press Enter to skip): "
    ).strip()
)


# Spotify metadata only.
SPOTIFY_CLIENT_ID = (
    os.getenv(
        "SPOTIFY_CLIENT_ID"
    )
    or os.getenv(
        "SPOTIPY_CLIENT_ID"
    )
    or input(
        "Enter Spotify Client ID "
        "(or press Enter to skip): "
    ).strip()
)


SPOTIFY_CLIENT_SECRET = (
    os.getenv(
        "SPOTIFY_CLIENT_SECRET"
    )
    or os.getenv(
        "SPOTIPY_CLIENT_SECRET"
    )
    or getpass(
        "Enter Spotify Client Secret "
        "(or press Enter to skip): "
    ).strip()
)


# ============================================================
# EXPORT KEYS FOR LATER NOTEBOOK CELLS
# ============================================================

if YOUTUBE_API_KEYS:
    os.environ[
        "YOUTUBE_API_KEYS"
    ] = ",".join(
        YOUTUBE_API_KEYS
    )

    # Backward compatibility for old notebook cells.
    os.environ[
        "YOUTUBE_API_KEY"
    ] = YOUTUBE_API_KEYS[0]

    for key_number, key in enumerate(
        YOUTUBE_API_KEYS,
        start=1,
    ):
        os.environ[
            f"YOUTUBE_API_KEY_{key_number}"
        ] = key


if LASTFM_API_KEY:
    os.environ[
        "LASTFM_API_KEY"
    ] = LASTFM_API_KEY


if DISCOGS_TOKEN:
    os.environ[
        "DISCOGS_TOKEN"
    ] = DISCOGS_TOKEN


if SPOTIFY_CLIENT_ID:
    os.environ[
        "SPOTIFY_CLIENT_ID"
    ] = SPOTIFY_CLIENT_ID

    os.environ[
        "SPOTIPY_CLIENT_ID"
    ] = SPOTIFY_CLIENT_ID


if SPOTIFY_CLIENT_SECRET:
    os.environ[
        "SPOTIFY_CLIENT_SECRET"
    ] = SPOTIFY_CLIENT_SECRET

    os.environ[
        "SPOTIPY_CLIENT_SECRET"
    ] = SPOTIFY_CLIENT_SECRET


# ============================================================
# SHARED REQUEST SESSION
# ============================================================

API_SESSION = requests.Session()

API_SESSION.headers.update(
    {
        "User-Agent": (
            "MusicDatasetResearch/1.0"
        )
    }
)


# ============================================================
# YOUTUBE REQUEST HELPER WITH SAFE KEY SWITCHING
# ============================================================

YOUTUBE_API_BASE_URL = (
    "https://www.googleapis.com/youtube/v3"
)


YOUTUBE_PERMANENT_KEY_ERROR_REASONS = {
    "quotaExceeded",
    "dailyLimitExceeded",
    "keyInvalid",
    "accessNotConfigured",
    "forbidden",
}

YOUTUBE_TEMPORARY_RATE_LIMIT_REASONS = {
    "rateLimitExceeded",
    "userRateLimitExceeded",
}


def extract_youtube_error_reasons(
    response,
):
    """
    Extract Google/YouTube API error reason strings.
    """
    try:
        payload = response.json()

    except Exception:
        return []

    error_data = payload.get(
        "error",
        {},
    )

    errors = error_data.get(
        "errors",
        [],
    )

    reasons = []

    for item in errors:
        if not isinstance(
            item,
            dict,
        ):
            continue

        reason = str(
            item.get(
                "reason",
                "",
            )
        ).strip()

        if reason:
            reasons.append(
                reason
            )

    return reasons


def youtube_api_get(
    endpoint,
    params=None,
    timeout=30,
    max_network_retries_per_key=2,
    max_rate_limit_retries_per_key=5,
):
    """
    Perform a YouTube Data API GET request.

    Behaviour:
      - quota/auth/key errors: disable current key and rotate
      - temporary rateLimitExceeded: wait and retry same key
      - after repeated temporary throttling: try another key for
        this request without permanently disabling the key
      - network and 5xx failures: exponential retry
    """
    if not YOUTUBE_API_KEYS:
        raise ValueError(
            "No YouTube API keys are configured."
        )

    endpoint = str(
        endpoint
    ).strip().lstrip("/")

    if not endpoint:
        raise ValueError(
            "A YouTube API endpoint is required."
        )

    request_url = (
        f"{YOUTUBE_API_BASE_URL}/"
        f"{endpoint}"
    )

    base_params = dict(
        params or {}
    )

    base_params.pop(
        "key",
        None,
    )

    attempted_key_indexes = set()
    last_error = None

    while (
        len(attempted_key_indexes)
        < len(YOUTUBE_API_KEYS)
    ):
        current_index = (
            YOUTUBE_KEY_INDEX
        )

        if (
            current_index
            in attempted_key_indexes
            or current_index
            in YOUTUBE_DISABLED_KEY_INDEXES
        ):
            rotate_youtube_key(
                reason=(
                    "current key already attempted"
                ),
                disable_current=False,
            )
            continue

        attempted_key_indexes.add(
            current_index
        )

        current_key = (
            get_current_youtube_key()
        )

        request_params = (
            base_params.copy()
        )

        request_params[
            "key"
        ] = current_key

        network_attempt = 0
        rate_limit_attempt = 0

        while True:
            try:
                response = API_SESSION.get(
                    request_url,
                    params=request_params,
                    timeout=timeout,
                )

            except requests.RequestException as error:
                network_attempt += 1
                last_error = error

                if (
                    network_attempt
                    <= max_network_retries_per_key
                ):
                    sleep_seconds = min(
                        20.0,
                        (
                            1.5
                            * (2 ** (
                                network_attempt - 1
                            ))
                            + random.uniform(
                                0.1,
                                0.5,
                            )
                        ),
                    )

                    print(
                        "Temporary YouTube network error "
                        f"on key {current_index + 1}; "
                        f"retrying in "
                        f"{sleep_seconds:.1f}s."
                    )

                    time.sleep(
                        sleep_seconds
                    )
                    continue

                print(
                    "YouTube network error on "
                    f"key {current_index + 1}: "
                    f"{type(error).__name__}"
                )

                rotate_youtube_key(
                    reason="network retries exhausted",
                    disable_current=False,
                )
                break

            if response.status_code == 200:
                try:
                    return response.json()

                except ValueError as error:
                    raise RuntimeError(
                        "YouTube returned HTTP 200 "
                        "but the response was not valid JSON."
                    ) from error

            reasons = (
                extract_youtube_error_reasons(
                    response
                )
            )

            reason_text = (
                ", ".join(reasons)
                if reasons
                else f"HTTP {response.status_code}"
            )

            last_error = RuntimeError(
                reason_text
            )

            temporary_rate_limit = (
                response.status_code == 429
                or any(
                    reason
                    in YOUTUBE_TEMPORARY_RATE_LIMIT_REASONS
                    for reason in reasons
                )
            )

            if temporary_rate_limit:
                rate_limit_attempt += 1

                if (
                    rate_limit_attempt
                    <= max_rate_limit_retries_per_key
                ):
                    retry_after_header = (
                        response.headers.get(
                            "Retry-After",
                            "",
                        )
                    )

                    try:
                        retry_after = float(
                            retry_after_header
                        )
                    except Exception:
                        retry_after = 0.0

                    exponential_wait = min(
                        60.0,
                        (
                            2.0
                            * (
                                2 ** (
                                    rate_limit_attempt - 1
                                )
                            )
                            + random.uniform(
                                0.25,
                                0.75,
                            )
                        ),
                    )

                    sleep_seconds = max(
                        retry_after,
                        exponential_wait,
                    )

                    print(
                        "YouTube temporary rate limit "
                        f"on key {current_index + 1}; "
                        f"retry {rate_limit_attempt}/"
                        f"{max_rate_limit_retries_per_key} "
                        f"in {sleep_seconds:.1f}s."
                    )

                    time.sleep(
                        sleep_seconds
                    )
                    continue

                print(
                    "YouTube key "
                    f"{current_index + 1} remains "
                    "temporarily rate-limited; "
                    "trying another key for this request."
                )

                rotate_youtube_key(
                    reason=(
                        "temporary rate-limit retries exhausted"
                    ),
                    disable_current=False,
                )
                break

            permanent_key_error = (
                response.status_code
                in {
                    401,
                    403,
                }
                and any(
                    reason
                    in YOUTUBE_PERMANENT_KEY_ERROR_REASONS
                    for reason in reasons
                )
            )

            if permanent_key_error:
                print(
                    "YouTube key "
                    f"{current_index + 1} unavailable: "
                    f"{reason_text}"
                )

                rotate_youtube_key(
                    reason=reason_text,
                    disable_current=True,
                )
                break

            if (
                response.status_code
                in {
                    500,
                    502,
                    503,
                    504,
                }
            ):
                network_attempt += 1

                if (
                    network_attempt
                    <= max_network_retries_per_key
                ):
                    sleep_seconds = min(
                        20.0,
                        (
                            2.0
                            * (
                                2 ** (
                                    network_attempt - 1
                                )
                            )
                            + random.uniform(
                                0.1,
                                0.5,
                            )
                        ),
                    )

                    print(
                        "Temporary YouTube server error "
                        f"{response.status_code}; "
                        f"retrying in "
                        f"{sleep_seconds:.1f}s."
                    )

                    time.sleep(
                        sleep_seconds
                    )
                    continue

                rotate_youtube_key(
                    reason=(
                        f"HTTP {response.status_code} "
                        "retries exhausted"
                    ),
                    disable_current=False,
                )
                break

            try:
                error_payload = (
                    response.json()
                )
            except Exception:
                error_payload = (
                    response.text[:500]
                )

            raise RuntimeError(
                "YouTube API request failed.\n"
                f"Endpoint: {endpoint}\n"
                f"Status: {response.status_code}\n"
                f"Reasons: {reason_text}\n"
                f"Response: {error_payload}"
            )

    raise RuntimeError(
        "All configured YouTube API keys were unavailable "
        "or temporarily rate-limited for this request. "
        "No key was permanently disabled solely because of "
        "rateLimitExceeded. "
        f"Last error: {last_error}"
    )


# ============================================================
# OPTIONAL YOUTUBE KEY TEST
# ============================================================

def test_youtube_api_key():
    """
    Test the active key with a low-result videos.list request.
    This still consumes API quota.
    """
    if not YOUTUBE_API_KEYS:
        print(
            "YouTube test skipped: no keys configured."
        )

        return False

    try:
        payload = youtube_api_get(
            "videos",
            {
                "part": "snippet",
                "id": "dQw4w9WgXcQ",
                "maxResults": 1,
            },
        )

        success = bool(
            payload.get(
                "items",
                [],
            )
        )

        print(
            "YouTube active-key test:",
            (
                "passed"
                if success
                else "returned no items"
            ),
        )

        return success

    except Exception as error:
        print(
            "YouTube active-key test failed:",
            error,
        )

        return False


# ============================================================
# API STATUS
# ============================================================

print(
    "\nAPI key status:"
)


print(
    "YouTube API keys:",
    len(
        YOUTUBE_API_KEYS
    ),
    "configured",
)


for key_number, key in enumerate(
    YOUTUBE_API_KEYS,
    start=1,
):
    print(
        f"  Key {key_number}:",
        mask_api_key(
            key
        ),
    )


print(
    "Active YouTube key:",
    (
        active_youtube_key_number()
        if YOUTUBE_API_KEYS
        else "none"
    ),
)

print(
    "Remaining YouTube key slots:",
    max(
        0,
        MAX_YOUTUBE_KEYS
        - len(YOUTUBE_API_KEYS)
    ),
)


print(
    "Last.fm API:",
    (
        "set"
        if bool(
            LASTFM_API_KEY
        )
        else "skipped/missing"
    ),
)


print(
    "Discogs token:",
    (
        "set"
        if bool(
            DISCOGS_TOKEN
        )
        else "skipped/missing"
    ),
)


print(
    "Spotify Client ID:",
    (
        "set"
        if bool(
            SPOTIFY_CLIENT_ID
        )
        else "skipped/missing"
    ),
)


print(
    "Spotify Client Secret:",
    (
        "set"
        if bool(
            SPOTIFY_CLIENT_SECRET
        )
        else "skipped/missing"
    ),
)


print(
    "\nLibrary status:"
)


print(
    "yt_dlp:",
    (
        "available"
        if YTDLP_AVAILABLE
        else "missing"
    ),
)


print(
    "mutagen:",
    (
        "available"
        if MUTAGEN_AVAILABLE
        else "missing"
    ),
)


print(
    "\nCell 1 setup complete."
)

In [ ]:
# Local pre-flight check — no network requests.
import importlib.util
import shutil
import sys

required_packages = [
    "pandas",
    "numpy",
    "requests",
    "tqdm",
    "matplotlib",
    "mutagen",
    "yt_dlp",
]

print("Python:", sys.version.split()[0])

for package_name in required_packages:
    available = importlib.util.find_spec(package_name) is not None
    print(f"{package_name:12} -> {'OK' if available else 'MISSING'}")

ffmpeg_path = shutil.which("ffmpeg")
print(
    "ffmpeg      ->",
    ffmpeg_path
    if ffmpeg_path
    else "MISSING — install FFmpeg before audio download",
)


## 2. Shared helper functions

These functions standardise text, map genres, create song IDs and perform reusable count checks.


## Global Helper Functions

> **Note on Helper Functions:** All text normalization, ID generation, and genre mapping utilities are centralized here. Defining these globally prevents variable shadowing and keeps the logic strictly consistent across both the Human and AI data pipelines.

In [ ]:
# GLOBAL HELPER FUNCTIONS
def clean_text(text):
    return str(text).lower().replace("_", " ").replace("-", " ").strip()

def normalise_for_match(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\u0900-\u097f\u4e00-\u9fff]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def make_safe_genre_code(genre):
    return genre.replace("/", "_").replace("-", "_").replace(" ", "_").upper()

def make_song_id(language_code, label, genre, number):
    label_code = "HUMAN" if label.lower() == "human" else "AI"
    genre_code = make_safe_genre_code(genre)
    return f"{label_code}_{language_code}_{genre_code}_{number:03d}"

def map_genre_from_text(text):
    s = clean_text(text)
    if any(t in s for t in ["hip hop", "rap", "trap", "drill", "嘻哈", "说唱"]): return "Hip-Hop/Rap"
    if any(t in s for t in ["r&b", "rnb", "soul", "funk"]): return "R&B/Soul"
    if any(t in s for t in ["folk", "acoustic", "singer songwriter", "ghazal", "民谣"]): return "Folk/Acoustic"
    if any(t in s for t in ["rock", "punk", "metal", "摇滚"]): return "Rock"
    if any(t in s for t in ["pop", "bollywood", "filmi", "indipop", "流行"]): return "Pop"
    return None

def check_counts(df, title="Counts"):
    print(f"\n{title}\n{'=' * len(title)}")
    if len(df) == 0:
        print("No rows")
        return
    counts = df["final_genre"].value_counts().reindex(GENRES, fill_value=0)
    print(counts)
    print("Total:", len(df))
    return counts

def plot_genre_counts(df, title="Genre counts"):
    counts = df["final_genre"].value_counts().reindex(GENRES, fill_value=0)
    plt.figure(figsize=(9, 4))
    plt.bar(counts.index, counts.values)
    plt.axhline(TARGET_PER_GENRE, linestyle="--", color='red')
    plt.title(title)
    plt.xlabel("Genre")
    plt.ylabel("Count")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 3. Dataset schema and source plan

Defines the target genres, language codes, output columns and search plans. Keep genre labels consistent throughout the notebook.


## Source Registries & Settings
Configures the final schema and iTunes search plans for the human data discovery phase.

In [ ]:
# SETTINGS & SOURCE PLAN
# Same schema, same variables, only better Hindi/Chinese iTunes search plan

GENRES = ["Pop", "Rock", "Hip-Hop/Rap", "R&B/Soul", "Folk/Acoustic"]
TARGET_PER_GENRE = 60

LANGUAGE_CODES = {
    "English": "EN",
    "Hindi": "HI",
    "Chinese": "CN",   # keep same to avoid breaking old notebook
}

FINAL_SCHEMA = [
    "song_id", "language", "language_code", "label", "final_genre",
    "source_dataset", "model", "title", "artist", "audio_path",
    "preview_url", "track_view_url", "youtube_url", "youtube_video_id",
    "raw_genre", "search_query", "country", "verification_status", "notes"
]

def standardise_schema(df):
    out = df.copy()
    for col in FINAL_SCHEMA:
        if col not in out.columns:
            out[col] = ""
    return out[FINAL_SCHEMA].copy()


# iTunes Search Strategy
# Better commercial metadata search for Hindi + Chinese

SEARCH_PLAN = {
    "Hindi": {
        "country": "IN",
        "queries": {
            "Pop": [
                "Hindi pop",
                "Bollywood pop",
                "Hindi filmi pop",
                "Indipop",
                "Bollywood party songs",
                "Bollywood dance songs",
                "Bollywood romantic songs",
                "Arijit Singh",
                "Neha Kakkar",
                "Badshah",
                "Jubin Nautiyal",
                "Dhvani Bhanushali",
                "Darshan Raval",
                "Tulsi Kumar",
                "Asees Kaur",
                "Vishal Mishra",
                "Sachet Tandon Parampara Tandon",
                "T-Series Hindi songs",
                "Zee Music Company Hindi",
                "Bollywood chartbusters",
            ],
            "Rock": [
                "Hindi rock",
                "Indian rock Hindi",
                "Bollywood rock",
                "Hindi alternative rock",
                "Hindi indie rock band",
                "Indian Ocean band",
                "Agnee band",
                "Parikrama band",
                "Euphoria band Hindi",
                "The Local Train band",
                "Kailash Kher rock",
                "Bollywood high energy rock",
                "Hindi rock ballad",
            ],
            "Hip-Hop/Rap": [
                "Hindi rap",
                "Desi hip hop",
                "Indian hip hop Hindi",
                "Hindi trap",
                "Bollywood rap",
                "Divine rapper",
                "Emiway Bantai",
                "Raftaar rapper",
                "KR$NA rapper",
                "Seedhe Maut",
                "MC Stan",
                "Naezy",
                "Ikka rapper",
                "gully rap Hindi",
                "Bollywood rap songs 2023",
            ],
            "R&B/Soul": [
                "Hindi R&B",
                "Hindi soul",
                "Bollywood R&B",
                "Indian R&B",
                "Jonita Gandhi",
                "Armaan Malik R&B",
                "Benny Dayal soul",
                "Shashwat Singh songs",
                "Raghav R&B",
                "Bollywood slow romantic songs",
                "Bollywood soulful songs",
                "Hindi chill R&B",
            ],
            "Folk/Acoustic": [
                "Hindi folk",
                "Indian folk Hindi",
                "Hindi acoustic",
                "Hindi ghazal",
                "Hindi devotional",
                "Hindi unplugged songs",
                "Bollywood acoustic version",
                "Hindi sufi songs",
                "Prateek Kuhad",
                "Anuv Jain",
                "Papon singer",
                "Kabir Cafe",
                "Coke Studio Bharat Hindi",
                "Raghav Meattle",
            ],
        },
    },

    "Chinese": {
        "country": "CN",

        # Keep same key name as your old code
        # CN = China, HK = Hong Kong, TW = Taiwan, SG = Singapore, MO = Macau
        "extra_countries": ["CN", "HK", "TW", "SG", "MO", "US"],

        "queries": {
            "Pop": [
                "Mandopop",
                "Chinese pop",
                "C-Pop",
                "华语流行",
                "國語流行",
                "中文流行",
                "周杰伦",
                "林俊杰",
                "邓紫棋",
                "蔡依林",
                "张惠妹",
            ],

            "Rock": [
                "Chinese rock",
                "Mandarin rock",
                "华语摇滚",
                "中文摇滚",
                "中國搖滾",
                "Mayday 五月天",
                "Beyond",
                "伍佰",
                "汪峰",
                "痛仰乐队",
            ],

            "Hip-Hop/Rap": [
                "Chinese rap",
                "Mandarin rap",
                "Chinese hip hop",
                "中文说唱",
                "华语说唱",
                "中国说唱",
                "GAI",
                "Higher Brothers",
                "Vava",
                "MC HotDog",
            ],

            "R&B/Soul": [
                "Chinese R&B",
                "Mandarin R&B",
                "华语R&B",
                "华语 R&B",
                "中文 R&B",
                "陶喆",
                "方大同",
                "王力宏",
                "袁娅维",
            ],

            "Folk/Acoustic": [
                "Chinese folk",
                "Mandarin folk",
                "Chinese acoustic",
                "中文民谣",
                "华语民谣",
                "中國民謠",
                "赵雷",
                "陈粒",
                "毛不易",
                "李健",
            ],
        },
    },
}

## 4. Candidate discovery

This stage intentionally creates pools larger than the final dataset. API results and progress are cached in the manifests/cache folders.


## iTunes Candidate Discovery (Hindi & Chinese)
Leverages the Apple Search API to collect candidate metadata for Hindi and Chinese tracks.

In [ ]:
# QUERY-BASED MUSICBRAINZ/LASTFM DISCOVERY

# TARGET GENRES
#   Pop
#   Rock
#   Hip-Hop/Rap
#   R&B/Soul
#   Folk/Acoustic
# OUTPUT SAME NAME
#   hindi_itunes_candidates_updated.csv


from __future__ import annotations

import base64
import hashlib
import html
import json
import os
import re
import time
import unicodedata
from difflib import SequenceMatcher
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import pandas as pd
import requests

try:
    from tqdm.auto import tqdm
except ImportError as error:
    raise ImportError(
        "tqdm is required for progress bars. Run: pip install tqdm"
    ) from error


# PATHS

if "DATASET_ROOT" in globals():
    DATASET_ROOT = Path(DATASET_ROOT)
else:
    DATASET_ROOT = Path("dataset_collection_clean")

if "MANIFEST_DIR" in globals():
    MANIFEST_DIR = Path(MANIFEST_DIR)
else:
    MANIFEST_DIR = DATASET_ROOT / "manifests"

CACHE_DIR = DATASET_ROOT / "cache"

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MAIN_OUTPUT_CSV = (
    MANIFEST_DIR / "hindi_itunes_candidates_updated.csv"
)

RAW_OUTPUT_CSV = (
    MANIFEST_DIR / "hindi_itunes_candidates_raw.csv"
)

REJECTED_OUTPUT_CSV = (
    MANIFEST_DIR / "hindi_itunes_candidates_rejected.csv"
)

QUERY_LOG_CSV = (
    MANIFEST_DIR / "hindi_itunes_query_log.csv"
)

ARTIST_PROFILE_CSV = (
    MANIFEST_DIR / "hindi_itunes_artist_profiles.csv"
)

HINDI_APPLE_CHART_ARTISTS_CSV = (
    MANIFEST_DIR / "hindi_apple_india_chart_artist_seeds.csv"
)

HINDI_FAMOUS_ARTIST_LOOKUP_LOG_CSV = (
    MANIFEST_DIR / "hindi_famous_artist_lookup_log.csv"
)

HINDI_CROSS_SOURCE_MATCH_CSV = (
    MANIFEST_DIR / "hindi_itunes_ranked_artist_matches.csv"
)

HINDI_SHORTAGE_REPORT_CSV = (
    MANIFEST_DIR / "hindi_high_precision_shortage_report.csv"
)

HINDI_EXTERNAL_DISCOVERY_CSV = (
    MANIFEST_DIR / "hindi_external_query_discovery_audit.csv"
)

HINDI_EXTERNAL_ARTIST_CSV = (
    MANIFEST_DIR / "hindi_external_artist_discovery.csv"
)

MUSICBRAINZ_CACHE_FILE = (
    CACHE_DIR / "cell4_musicbrainz_cache_v1.json"
)

LASTFM_CACHE_FILE = (
    CACHE_DIR / "cell4_lastfm_cache_v1.json"
)

SOUNDCLOUD_CACHE_FILE = (
    CACHE_DIR / "cell4_soundcloud_cache_v1.json"
)

CACHE_FILE = (
    CACHE_DIR / "cell4h_itunes_only_high_precision_cache_v6.json"
)

CHINESE_MAIN_OUTPUT_CSV = (
    MANIFEST_DIR / "chinese_itunes_candidates_updated.csv"
)

CHINESE_RAW_OUTPUT_CSV = (
    MANIFEST_DIR / "chinese_itunes_candidates_raw.csv"
)

CHINESE_REJECTED_OUTPUT_CSV = (
    MANIFEST_DIR / "chinese_itunes_candidates_rejected.csv"
)

CHINESE_QUERY_LOG_CSV = (
    MANIFEST_DIR / "chinese_itunes_query_log.csv"
)

CHINESE_APPLE_CHART_CSV = (
    MANIFEST_DIR / "chinese_apple_chart_candidates.csv"
)

CHINESE_CACHE_FILE = (
    CACHE_DIR / "cell4_chinese_itunes_cache_v3.json"
)


# CONFIGURATION

ITUNES_SEARCH_URL = "https://itunes.apple.com/search"
ITUNES_LOOKUP_URL = "https://itunes.apple.com/lookup"

TARGET_GENRES = [
    "Pop",
    "Rock",
    "Hip-Hop/Rap",
    "R&B/Soul",
    "Folk/Acoustic",
]

MIN_RELEASE_YEAR = 2019
MAX_RELEASE_YEAR = 2026

# iTunes documents a maximum search limit of 200.
ITUNES_LIMIT = 200

# Approximately 17–18 calls/minute, below the documented approximate
# 20-calls/minute Search API guidance.
REQUEST_INTERVAL_SECONDS = 1.5
REQUEST_TIMEOUT_SECONDS = 45
MAX_RETRIES = 5

# No fixed final target is applied here.
# Cell 4H collects every unique usable result returned by the query plan.
KEEP_EXISTING_ITUNES_ROWS = False

PRINT_EVERY_ACCEPTED_TRACK = False
SAVE_CACHE_AFTER_EVERY_QUERY = True

# Second pass: automatically discover Hindi-dominant artist candidates from
# repeated Hindi-query appearances, then search their exact iTunes catalogues.
EXPAND_HINDI_DOMINANT_ARTISTS = True

# Famous-artist methodology.
KEEP_ONLY_FAMOUS_HINDI_ARTISTS = True
USE_CURATED_ESTABLISHED_HINDI_ARTISTS = True
USE_APPLE_INDIA_CHART_FOR_FAME = True
USE_AUTOMATIC_HINDI_QUERY_PROFILES = True

MIN_SEED_TRACKS_PER_ARTIST = 2
MIN_DISTINCT_HINDI_QUERIES_PER_ARTIST = 2
MIN_AUTOMATIC_FAME_SCORE = 24

# Artist catalogue retrieval uses an ID lookup after resolving the exact artist.
MAX_FAMOUS_ARTISTS_TO_EXPAND = 220
ARTIST_SEARCH_LIMIT = 25
ARTIST_CATALOGUE_LIMIT = 200

# Apple Marketing Tools RSS feed supports the official India most-played chart.
APPLE_INDIA_CHART_LIMIT = 50
APPLE_INDIA_CHART_URL = (
    "https://rss.marketingtools.apple.com/"
    f"api/v2/in/music/most-played/{APPLE_INDIA_CHART_LIMIT}/songs.json"
)

# Fame weights. A chart appearance boosts an artist, but chart-only artists
# do not qualify unless they are also curated or repeatedly found in Hindi searches.
CURATED_ARTIST_BASE_SCORE = 100
APPLE_CHART_MAX_SCORE = 75

# iTUNES-ONLY HIGH-PRECISION HINDI SONG SETTINGS

STRICT_HINDI_PRECISION_MODE = True

# No ranked-search cutoff. Keep every valid exact-artist catalogue track.
MAX_ITUNES_SEARCH_RANK_PER_ARTIST = None
REQUIRE_RANKED_ITUNES_OR_APPLE_CHART = False

# No fixed total per genre and no per-artist song cap.
MAX_TRACKS_PER_ARTIST_PER_GENRE = None
HINDI_FIXED_GENRE_TOTAL_LIMIT = None

# Both Hindi and Chinese: 2 to 7 minutes.
HINDI_MIN_DURATION_SECONDS = 120
HINDI_MAX_DURATION_SECONDS = 420
MIN_DURATION_SECONDS = 120
MAX_DURATION_SECONDS = 420

# Popularity scoring.
APPLE_CHART_TRACK_BONUS = 120
ITUNES_RANK_MAX_POINTS = 70
RECENT_RELEASE_MAX_POINTS = 8

# OPTIONAL MULTI-SOURCE DISCOVERY
# These sources discover titles/artists only. A row enters the final
# Hindi CSV only after exact iTunes confirmation.

RUN_MUSICBRAINZ_DISCOVERY = True
RUN_LASTFM_DISCOVERY = True
RUN_LASTFM_SIMILAR_ARTIST_DISCOVERY = False
RUN_SOUNDCLOUD_DISCOVERY = True

MAX_EXTERNAL_ARTISTS_TOTAL = 30
MAX_EXTERNAL_TRACKS_PER_ARTIST_PER_SOURCE = 5
MAX_LASTFM_SIMILAR_ARTISTS_PER_GENRE = 8
LASTFM_SIMILAR_ANCHORS_PER_GENRE = 2
LASTFM_SIMILAR_PER_ANCHOR = 5

MUSICBRAINZ_REQUEST_INTERVAL_SECONDS = 1.1
LASTFM_REQUEST_INTERVAL_SECONDS = 0.35
SOUNDCLOUD_REQUEST_INTERVAL_SECONDS = 0.45

MUSICBRAINZ_RECORDING_LIMIT = 100
LASTFM_TOP_TRACK_LIMIT = 50
SOUNDCLOUD_TRACK_LIMIT = 50

EXTERNAL_ITUNES_MATCH_LIMIT = 25
MIN_EXTERNAL_TITLE_SIMILARITY = 0.88
MAX_EXTERNAL_DURATION_DIFFERENCE_SECONDS = 20

MUSICBRAINZ_CONTACT = (
    os.getenv("MUSICBRAINZ_CONTACT", "").strip()
    or "MScMusicDataset/1.0 academic-research"
)

LASTFM_API_KEY = str(
    os.getenv("LASTFM_API_KEY", "")
    or globals().get("LASTFM_API_KEY", "")
).strip()

SOUNDCLOUD_ACCESS_TOKEN = (
    os.getenv("SOUNDCLOUD_ACCESS_TOKEN", "").strip()
)

SOUNDCLOUD_CLIENT_ID = (
    os.getenv("SOUNDCLOUD_CLIENT_ID", "").strip()
)

SOUNDCLOUD_CLIENT_SECRET = (
    os.getenv("SOUNDCLOUD_CLIENT_SECRET", "").strip()
)


# QUERY-BASED EXTERNAL DISCOVERY
#
# MusicBrainz and Last.fm are searched by genre/tag query rather
# than separately for every artist. Each query:


RUN_QUERY_BASED_EXTERNAL_DISCOVERY = True
RUN_ARTIST_BASED_EXTERNAL_DISCOVERY = False

MAX_QUERY_RESULTS_PER_SOURCE = 20
MAX_EXTERNAL_ITUNES_ATTEMPTS_PER_QUERY = 8
MAX_EXTERNAL_ACCEPTED_PER_QUERY = 3

EXTERNAL_HINDI_QUERY_PLAN: Dict[str, List[str]] = {
    "Pop": [
        "bollywood",
        "indian pop",
        "indipop",
    ],
    "Rock": [
        "indian rock",
        "hindi rock",
        "indian indie rock",
    ],
    "Hip-Hop/Rap": [
        "desi hip hop",
        "indian hip hop",
        "hindi rap",
    ],
    "R&B/Soul": [
        "indian r&b",
        "indian soul",
        "hindi r&b",
    ],
    "Folk/Acoustic": [
        "indian folk",
        "sufi",
        "indian acoustic",
    ],
}

# Chinese methodology from the earlier combined Cell 4.
RUN_CHINESE_ITUNES = True
RUN_CHINESE_YEARLY_QUERIES = False
RUN_CHINESE_APPLE_CHARTS = True

CHINESE_ITUNES_COUNTRIES = ["CN", "TW", "HK"]
CHINESE_APPLE_STOREFRONTS = ["cn", "tw", "hk"]

CHINESE_APPLE_CHART_LIMIT = 100
CHINESE_FILTER_DURATION = True


# HINDI GENRE QUERY PLAN
#
# There is no manually supplied artist list.
# Artists are discovered automatically from repeated Hindi-query evidence.

HINDI_ITUNES_QUERY_PLAN: Dict[str, List[str]] = {
    "Pop": [
        "Hindi Pop",
        "Latest Hindi Pop",
        "Bollywood Pop",
        "Indian Pop Hindi",
        "Hindi Indie Pop",
        "Latest Hindi Songs",
        "New Hindi Songs",
        "New Bollywood Songs",
        "Hindi Chart Hits",
        "Hindi Hit Songs",
        "Hindi Singles",
        "Hindi Independent Music",
        "Hindi Pop 2023",
        "Hindi Pop 2024",
        "Hindi Pop 2025",
        "Hindi Pop 2026",
        "नए हिंदी गाने",
        "हिंदी पॉप",
    ],
    "Rock": [
        "Hindi Rock",
        "Bollywood Rock",
        "Indian Rock Hindi",
        "Hindi Indie Rock",
        "Hindi Alternative Rock",
        "Hindi Progressive Rock",
        "Hindi Soft Rock",
        "Rock Songs Hindi",
        "Hindi Rock Band",
        "Indian Indie Rock Hindi",
        "Hindi Rock 2023",
        "Hindi Rock 2024",
        "Hindi Rock 2025",
        "Hindi Rock 2026",
        "हिंदी रॉक",
    ],
    "Hip-Hop/Rap": [
        "Hindi Rap",
        "Hindi Hip Hop",
        "Desi Hip Hop Hindi",
        "Indian Hip Hop Hindi",
        "Bollywood Rap",
        "Hindi Trap",
        "Hindi Drill",
        "Hindi Rap Songs",
        "Hindi Underground Rap",
        "Hindi Independent Rap",
        "Hindi Rap 2023",
        "Hindi Rap 2024",
        "Hindi Rap 2025",
        "Hindi Rap 2026",
        "हिंदी रैप",
        "हिंदी हिप हॉप",
    ],
    "R&B/Soul": [
        "Hindi R&B",
        "Hindi RnB",
        "Hindi Soul",
        "Indian R&B Hindi",
        "Bollywood R&B",
        "Hindi Indie R&B",
        "Hindi Neo Soul",
        "Indian Soul Hindi",
        "Hindi Contemporary R&B",
        "Hindi Alternative R&B",
        "Hindi R&B 2023",
        "Hindi R&B 2024",
        "Hindi R&B 2025",
        "Hindi R&B 2026",
        "हिंदी सोल",
    ],
    "Folk/Acoustic": [
        "Hindi Folk",
        "Hindi Acoustic",
        "Hindi Sufi",
        "Indian Folk Hindi",
        "Hindi Unplugged",
        "Hindi Singer Songwriter",
        "Hindi Folk Songs",
        "Hindi Indie Folk",
        "Hindi Acoustic Songs",
        "Hindi Ghazal",
        "Hindi Contemporary Folk",
        "Hindi Folk 2023",
        "Hindi Acoustic 2024",
        "Hindi Folk 2025",
        "Hindi Acoustic 2026",
        "हिंदी लोक गीत",
        "हिंदी सूफी",
        "हिंदी अकूस्टिक",
    ],
}

# CURATED ESTABLISHED / FAMOUS HINDI-DOMINANT ARTISTS
#
# This is a discovery seed list, not a claim that every catalogue track
# by the artist is Hindi or belongs to the assigned genre.
# Every returned track still goes through year, version, script and Cell 5B checks.

FAMOUS_HINDI_ARTIST_SEEDS = {
    "Pop": [
        # Clean/Sanitized Originals & Playback Legends 
        "Arijit Singh", "Shreya Ghoshal", "Sonu Nigam", "Sunidhi Chauhan", 
        "Shaan", "KK", "Mohit Chauhan", "Jubin Nautiyal", "Vishal Mishra", 
        "Monali Thakur", "Palak Muchhal", "Tulsi Kumar", "Shilpa Rao", 
        "Javed Ali", "Darshan Raval", "Sachet Tandon", "Parampara Tandon", 
        "Yasser Desai", "Stebin Ben", "Akhil Sachdeva", "Antara Mitra",
        "Kumar Sanu", "Alka Yagnik", "Udit Narayan", "Kavita Krishnamurthy", 
        "Abhijeet Bhattacharya", "Sadhana Sargam", "Anuradha Paudwal", 
        "Hariharan", "Roop Kumar Rathod", "Kunal Ganjawala", "Babul Supriyo", 
        "Vinod Rathod", "Sudesh Bhosle", "Nitin Mukesh", "Lucky Ali",

        # Safe Modern Hindi Pop Vocalists 
        "Armaan Malik", "Amaal Mallik", "Himesh Reshammiya", "Ankit Tiwari", 
        "Sanam Puri", "Adnan Sami", "Atif Aslam", "Neeti Mohan", 
        "Shalmali Kholgade", "Dhvani Bhanushali", "Shirley Setia", 
        "Arjun Kanungo", "Akasa Singh", "Prakriti Kakar", "Sukriti Kakar", 
        "Alisha Chinai", "Neeraj Shridhar", "Ash King", "Amit Mishra", 
        "Dev Negi", "Mohammed Irfan", "Vishal Dadlani", "Shekhar Ravjiani", 
        "Shankar Mahadevan", "Siddharth Mahadevan", "Clinton Cerejo", 
        "Sachin Sanghvi", "Jigar Saraiya", "Raghav Chaitanya", "Rochak Kohli"
    ],
    
    "Rock": [
        # Retained Pure Hindi/Urdu Rock 
        "The Local Train", "Indian Ocean", "Euphoria", "Agnee", 
        "Anand Bhaskar Collective", "Coshish", "Faridkot", "Kabir Cafe", 
        "The Yellow Diary", "Naalayak", "Raman Negi", "Strings", 
        "Silk Route", "Advaita", "Sifar", "Antariksh", "Daira", 
        "Ankur & The Ghalat Family", "Kailasa", "Jal The Band", 
        "Raeth", "Call", "Roxen", "Astitva", "Eka", "Bandish", 

        # New Pure Hindi/Urdu Rock & Alt-Rock additions 
        "Junoon", "Vital Signs", "Noori", "Entity Paradigm", 
        "Aaroh", "Karavan", "Mizraab", "Ali Azmat", "Mustafa Zahid", 
        "Kaavish", "Sparsh", "Paradigm Shift", "Bayaan", "Kashmir", 
        "Aavishkaar", "Parashara", "Ektara", "Fuzon", "Soch The Band", 
        "Swastik The Band", "Bombay Bandook", "Nasha", "Spunk!", 
        "Antaragni", "Udaan The Band", "Vayu", "Highway 61", "Fitoor",
        "Aurko", "Aarohan", "Farhan Akhtar", "Shadow and Light"
    ],
    
    "Hip-Hop/Rap": [
        # Retained Pure Hindi/Urdu Rap (Punjabi Slang Purged) 
        "DIVINE", "Emiway Bantai", "KR$NA", "Seedhe Maut", "King", 
        "Dino James", "Ikka", "Fotty Seven", "Karma", "MC Stan", 
        "Yashraj", "Frappe Ash", "Chaar Diwaari", "Naezy", "Rawal", 
        "Bharg", "Dakait", "EPR Iyer", "Panther", "Raga", "Muhfaad", 
        "Bella", "MC Altaf", "Void", "Rebel 7", "Talha Anjum", 
        "Talhah Yunus", "Jokhay", "Gravity", "Dhanji", "Qaab", 
        "Revo Lekhak", "Spectra", "Siege", "Karan Kanchan",

        #  New Pure Hindi/Urdu Rappers & Collaborators 
        "Yungsta", "Calm", "Encore ABJ", "Kidshot", "Shah Rule", 
        "J Trix", "Gaush", "Spitfire", "D'Evil", "Aavrutti", 
        "Umair", "Savage", "JJ 47", "Shareh", "Sunny Khan Durrani", 
        "Jani", "Rap Demon", "Hashim Nawaz", "Umer Anjum", 
        "Munawar Faruqui", "Faris Shafi", "Loka", "7Bantaiz", 
        "Dopeadelicz", "Shloka", "MC Heam", "Khullar G"
    ],
    
    "R&B/Soul": [
        #  Retained Pure Hindi/Urdu R&B & Soulful Pop 
        "Prateek Kuhad", "Anuv Jain", "Lisa Mishra", "Akanksha Bhandari", 
        "Anumita Nadesan", "Savera", "Osho Jain", "Akshath", 
        "Aditya Rikhari", "Mitraz", "Twin Strings", "Zaeden", 
        "Raghav Chaitanya", "Iqlipse Nova", "Kanishk Seth", 
        "Bharat Chauhan", "Nanku", "Lifafa", "Ankur Tewari", 

        #  New Pure Hindi/Urdu Lo-Fi, R&B, and Soul Vocalists 
        "Hasan Raheem", "Abdul Hannan", "Kaifi Khalil", "Asim Azhar", 
        "Maanu", "Taha G", "Annural Khalid", "Shamoon Ismail", 
        "Rito Riba", "Salman Elahi", "Vilen", "Ritrisha", 
        "Nikhil D'Souza", "Bhuvan Bam", "Shuja Haider", 
        "Arooj Aftab", "Zeeshan Ali", "Khudgharz", "Lothika", 
        "OAFF", "Gajendra Verma", "Jasleen Royal", "Vishal Mishra", 
        "Akhil Sachdeva", "Goldie Sohel", "Shashwat Singh", 
        "Kavita Seth", "Armaan Malik", "Ali Sethi"
    ],
    
    "Folk/Acoustic": [
        # Retained Strictly Hindi/Urdu Folk & Acoustic 
        "Papon", "Kailash Kher", "Rekha Bhardwaj", "Kavita Seth", 
        "Lucky Ali", "Suzonn", "Piyush Mishra", "Nilotpal Bora", 
        "Bawari Basanti", "Shafqat Amanat Ali", "Rahat Fateh Ali Khan", 
        "Shubha Mudgal", "Swanand Kirkire", "Varun Grover", 
        "Arko", "Rituraj Mohanty",

        #  New Classic Ghazal, Qawwali & Acoustic Hindi Indie 
        "Jagjit Singh", "Pankaj Udhas", "Tochi Raina", "Richa Sharma", 
        "Sona Mohapatra", "Madhushree", "Nizami Bandhu", 
        "Bhupinder Singh", "Talat Aziz", "Ghulam Ali", "Javed Bashir", 
        "Chitra Singh", "Begum Akhtar", "Mehdi Hassan", "Farida Khanum", 
        "Iqbal Bano", "Nusrat Fateh Ali Khan", "Sabri Brothers", 
        "Amjad Sabri", "Aziz Mian", "Abhay Jodhpurkar", "Sameer Rahat", 
        "Hanita Bhambri", "Kamakshi Khanna", "Sanjeeta Bhattacharya", 
        "Gaurav Dagaonkar", "Neeraj Arya", "Bismil", "Farhan Saeed"
    ]
}

EXTRA_HINDI_ARTIST_SEEDS: Dict[str, List[str]] = {}

for _genre, _names in EXTRA_HINDI_ARTIST_SEEDS.items():
    FAMOUS_HINDI_ARTIST_SEEDS[_genre] = list(
        dict.fromkeys(
            FAMOUS_HINDI_ARTIST_SEEDS.get(_genre, [])
            + _names
        )
    )

# Chinese discovery stays aligned with the previous combined methodology:
# iTunes direct search in CN/TW/HK plus official Apple Music charts.
CHINESE_ITUNES_QUERY_PLAN: Dict[str, List[str]] = {
    # Artist-name queries removed artists are handled by
    # collect_chinese_famous_artist_candidates() via ID-based catalogue
    # lookup (same method as Hindi).  Only genre-signal queries are kept.
    "Pop": [
        "Mandopop",
        "華語流行",
        "国语流行",
        "Chinese pop",
        "C-pop songs",
        "Cantopop",
        "广东话流行",
    ],
    "Rock": [
        "Chinese rock",
        "華語搖溻",
        "国语摇滚",
        "Chinese indie rock",
        "中国摇滚",
        "台湾摇滚",
    ],
    "Hip-Hop/Rap": [
        "Chinese hip hop",
        "中文說唱",
        "国语说唱",
        "Mandarin rap",
        "中国说唱",
        "华语嘻哈",
    ],
    "R&B/Soul": [
        "Chinese R&B",
        "華語 R&B",
        "国语 R&B",
        "Mandarin soul",
        "华语 soul",
    ],
    "Folk/Acoustic": [
        "Chinese folk",
        "華語民謠",
        "国语民谣",
        "Chinese acoustic",
        "中国民谣",
        "台湾民谣",
        "民谣 singer songwriter Chinese",
    ],
}


# FAMOUS CHINESE ARTIST SEEDS
# Mirrors the Hindi approach: each artist is resolved to an exact
# iTunes artist identity, then their full catalogue is pulled and
# filtered. Every track still requires Cell 5B Chinese-language
# and genre verification.


FAMOUS_CHINESE_ARTIST_SEEDS = {
    "Pop": [
        "Jay Chou",
        "JJ Lin",
        "G.E.M.",
        "Eason Chan",
        "Jolin Tsai",
        "A-Mei",
        "Leehom Wang",
        "Fish Leong",
        "Stefanie Sun",
        "Li Ronghao",
        "Xue Zhiqian",
        "Hua Chenyu",
        "A-Lin",
        "Crowd Lu",
        "Hebe Tien",
        "Angela Chang",
        "Show Luo",
        "Rainie Yang",
        "William Wei",
        "Jason Zhang",
        "Silence Wang",
        "Xu Song",
        "Yisa Yu",
        "Jam Hsiao",
        "Tanya Chua",
        "Elva Hsiao",
        "Richie Jen",
        "Cyndi Wang",
        "Della Ding",
        "Michael Wong",
        "Bii",
        "WeiBird",
        "Power Station",
        "Faye Wong",
        "Karen Mok",
        "Rene Liu",
        "Zhou Shen",
        "Mavis Fan",
        "Gen Neo",
        "Nine Chen"
    ],

    "Rock": [
        "Mayday",
        "Beyond",
        "Wu Bai",
        "Sodagreen",
        "Wang Feng",
        "Zhang Zhenyue",
        "New Pants",
        "No Party for Cao Dong",
        "Re-TROS",
        "Second Hand Rose",
        "Reflector",
        "PK14",
        "Carsick Cars",
        "Omnipotent Youth Society",
        "Escape Plan",
        "Miserable Faith",
        "Queen Sea Big Shark",
        "Brain Failure",
        "Tang Dynasty",
        "Black Panther",
        "Nova Heart",
        "Snapline",
        "Yaksa",
        "Muma",
        "Joyside",
        "Hang on the Box",
        "Thin Man",
        "The Face",
        "Life Journey",
        "Hua Lun"
    ],

    "Hip-Hop/Rap": [
        "GAI",
        "Higher Brothers",
        "MC HotDog",
        "VaVa",
        "GALI",
        "TT",
        "After Journey",
        "PG One",
        "Jony J",
        "Bridge",
        "Tizzy T",
        "KnowKnow",
        "Psy.P",
        "Melo",
        "ICE",
        "Capper",
        "KeyNG",
        "Vinida Weng",
        "Liu Cong",
        "Lil Ghost",
        "Kc",
        "Yamy",
        "PACT",
        "KKECHO",
        "AIR",
        "Step.jad",
        "Maxone",
        "OB03",
        "YoungJack",
        "Mandarin MC"
    ],

    "R&B/Soul": [
        "David Tao",
        "Khalil Fong",
        "Tia Ray",
        "Jane Zhang",
        "Wanting",
        "Waa Wei",
        "Wu Qingfeng",
        "Chris Lee",
        "Yoga Lin",
        "Eric Chou",
        "Lala Hsu",
        "Sandy Lam",
        "Cass Phang",
        "J.Sheon",
        "Shi Shi",
        "YELLOW",
        "Karencici",
        "9m88",
        "Jocelyn 9.4.0",
        "Eve Ai",
        "Penny Tai",
        "Karen Mok",
        "Faye Wong",
        "Della Ding",
        "Michael Wong",
        "Jam Hsiao",
        "Bii",
        "WeiBird",
        "Yisa Yu",
        "Tanya Chua"
    ],

    "Folk/Acoustic": [
        "Zhao Lei",
        "Chen Li",
        "Mao Buyi",
        "Li Jian",
        "Song Dongye",
        "Ma Di",
        "Cheng Bi",
        "Chen Hongyu",
        "Zhou Yunpeng",
        "Wan Xiaoli",
        "Li Zhi",
        "Lao Lang",
        "Liu Shuang",
        "Zhang Weiwei",
        "Zhong Lifeng",
        "Xiao He",
        "Buyi Band",
        "Liu Sen",
        "Xu Fei",
        "Liu Haolin",
        "Li Xiaodong",
        "Ye Bei",
        "Yao Si Ting",
        "Zhao Zhao",
        "Xu Jun",
        "Su Yunying",
        "An Ran",
        "Chen Yonglong",
        "Chen Liang",
        "Pu Shu"
    ]
}


# ============================================================
# EXCLUSION RULES
# ============================================================

# ============================================================
# HINDI LANGUAGE REJECT MARKERS
#
# If any of these words appear in the title, album name, or
# iTunes primaryGenreName the track is rejected in Cell 4 before
# any iTunes quota is spent on verification.
#
# Rule: applied to the combined "title + collection" blob only
# (same as LOW_QUALITY_MARKERS) so artist names are not affected.
# ============================================================

HINDI_LANGUAGE_REJECT_MARKERS = {
    # Non-Hindi Indian languages
    "punjabi",
    "bhojpuri",
    "haryanvi",
    "rajasthani",
    "maithili",
    "chhattisgarhi",
    "awadhi",
    "braj",
    "kannada",
    "tamil",
    "telugu",
    "malayalam",
    "bengali",
    "marathi",
    "gujarati",
    "odia",
    "assamese",
    "sindhi",
    "nepali",
    "urdu",
    # Punjabi-associated genre/style words
    "bhangra",
    "giddha",
    "jatti",
    "pind",
    "gabru",
    "punjab",
    # Bhojpuri-associated words
    "bhojpuri gana",
    "bhojpuri song",
}

HINDI_GENRE_REJECT_MARKERS = {
    # iTunes primaryGenreName values that signal a non-Hindi language
    "punjabi",
    "bhojpuri",
    "haryanvi",
    "regional indian",
    "kannada",
    "tamil",
    "telugu",
    "malayalam",
    "bengali",
    "marathi",
    "gujarati",
}


LOW_QUALITY_MARKERS = {
    "karaoke",
    "instrumental",
    "vocals only",
    "vocal only",
    "acapella",
    "a cappella",
    "tribute",
    "cover version",
    "cover song",
    "lofi",
    "lo-fi",
    "slowed",
    "reverb",
    "sped up",
    "nightcore",
    "8d audio",
    "dj mix",
    "dj remix",
    "remix",
    "remake",
    "recreated",
    "recreation",
    "medley",
    "jukebox",
    "non stop",
    "non-stop",
    "male version",
    "female version",
    "worship",
    "bhajan",
    "aarti",
    "arti",
    "mantra",
    "chalisa",
    "devotional",
    "prayer",
    "meditation",
    "chant",
    "nursery rhyme",
    "kids song",
    "kids poem",
    "poem for kids",
}

NON_SONG_MARKERS = {
    "podcast",
    "audiobook",
    "interview",
    "speech",
    "dialogue",
    "dialogues",
    "trailer",
    "teaser",
    "commentary",
    "episode",
}


# BROAD LABELS NOT FINAL GENRE EVIDENCE

BROAD_ITUNES_GENRES = {
    "bollywood",
    "filmi",
    "indian pop",
    "indipop",
    "regional indian",
    "indian",
    "indian music",
    "hindi music",
    "soundtrack",
    "original soundtrack",
    "motion picture soundtrack",
    "stage & screen",
    "world",
    "worldwide",
    "music",
}


# SESSION AND CACHE


SESSION = requests.Session()
SESSION.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 MSc-AI-Music-Research/1.0 "
            "(iTunes metadata-only)"
        ),
        "Accept": "application/json",
    }
)

LAST_REQUEST_TIME = 0.0


def load_json_cache(path: Path) -> Dict[str, Any]:
    if not path.exists():
        return {}

    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(payload, dict):
            return payload
    except Exception as error:
        print(
            "WARNING | Could not load cache:",
            type(error).__name__,
        )

    return {}


def save_json_cache(cache: Dict[str, Any], path: Path) -> None:
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(
            cache,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    temporary_path.replace(path)


ITUNES_CACHE = load_json_cache(CACHE_FILE)
CHINESE_ITUNES_CACHE = load_json_cache(CHINESE_CACHE_FILE)
MUSICBRAINZ_CACHE = load_json_cache(MUSICBRAINZ_CACHE_FILE)
LASTFM_CACHE = load_json_cache(LASTFM_CACHE_FILE)
SOUNDCLOUD_CACHE = load_json_cache(SOUNDCLOUD_CACHE_FILE)


# GENERAL HELPERS

def safe_text(value: Any) -> str:
    if value is None:
        return ""

    if isinstance(value, float) and pd.isna(value):
        return ""

    return str(value).strip()


def clean_html(value: Any) -> str:
    text = html.unescape(safe_text(value))
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalise_text(value: Any) -> str:
    text = clean_html(value).casefold()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )
    text = re.sub(r"[^a-z0-9\u0900-\u097f\u3400-\u9fff]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def unique_nonempty(values: Iterable[Any]) -> List[str]:
    output: List[str] = []
    seen = set()

    for value in values:
        text = safe_text(value)
        key = normalise_text(text)

        if not text or not key or key in seen:
            continue

        seen.add(key)
        output.append(text)

    return output


def parse_release_year(value: Any) -> Optional[int]:
    match = re.search(
        r"\b(?:19|20)\d{2}\b",
        safe_text(value),
    )
    return int(match.group(0)) if match else None


def parse_duration_seconds(value: Any) -> Optional[float]:
    try:
        milliseconds = float(value)
    except (TypeError, ValueError):
        return None

    if milliseconds <= 0:
        return None

    return round(milliseconds / 1000.0, 2)


def contains_devanagari(value: Any) -> bool:
    return bool(re.search(r"[\u0900-\u097F]", safe_text(value)))


def contains_han(value: Any) -> bool:
    return bool(
        re.search(
            r"[\u3400-\u4DBF\u4E00-\u9FFF]",
            safe_text(value),
        )
    )


def contains_kana_or_hangul(value: Any) -> bool:
    return bool(
        re.search(
            r"[\u3040-\u30FF\uAC00-\uD7AF]",
            safe_text(value),
        )
    )


def contains_gurmukhi(value: Any) -> bool:
    return bool(re.search(r"[\u0A00-\u0A7F]", safe_text(value)))


def contains_other_indic_script(value: Any) -> bool:
    text = safe_text(value)
    ranges = [
        r"[\u0980-\u09FF]",  # Bengali
        r"[\u0A80-\u0AFF]",  # Gujarati
        r"[\u0B00-\u0B7F]",  # Odia
        r"[\u0B80-\u0BFF]",  # Tamil
        r"[\u0C00-\u0C7F]",  # Telugu
        r"[\u0C80-\u0CFF]",  # Kannada
        r"[\u0D00-\u0D7F]",  # Malayalam
    ]
    return any(re.search(pattern, text) for pattern in ranges)


def split_artist_names(value: Any) -> List[str]:
    return unique_nonempty(
        part.strip()
        for part in re.split(
            r"\s*(?:,|&| feat\.?| ft\.?| featuring | with )\s*",
            safe_text(value),
            flags=re.IGNORECASE,
        )
        if part.strip()
    )


def exact_artist_match(seed_artist: str, returned_artist: str) -> bool:
    seed = normalise_text(seed_artist)
    returned = {
        normalise_text(name)
        for name in split_artist_names(returned_artist)
        if normalise_text(name)
    }
    return bool(seed and seed in returned)


def obvious_non_hindi_script_reason(
    title: str,
    collection: str,
    primary_genre: str = "",
) -> str:
    text = f"{title} {collection}"
    if contains_gurmukhi(text):
        return "obvious_gurmukhi_script"
    if contains_other_indic_script(text):
        return "obvious_other_indic_script"
    # Reject tracks whose title/album or iTunes genre name explicitly
    # signals a non-Hindi Indian language (Punjabi, Bhojpuri, etc.).
    text_norm = normalise_text(text)
    for marker in HINDI_LANGUAGE_REJECT_MARKERS:
        marker_norm = normalise_text(marker)
        if marker_norm and re.search(
            rf"(?<![a-z0-9]){re.escape(marker_norm)}(?![a-z0-9])",
            text_norm,
        ):
            return f"non_hindi_language_marker_{marker_norm.replace(' ', '_')}"
    if primary_genre:
        genre_norm = normalise_text(primary_genre)
        for marker in HINDI_GENRE_REJECT_MARKERS:
            if normalise_text(marker) in genre_norm:
                return f"non_hindi_itunes_genre_{normalise_text(marker).replace(' ', '_')}"
    return ""


def canonical_title(value: Any) -> str:
    text = safe_text(value)

    # Merge album/single duplicates while preserving the underlying song title.
    text = re.sub(
        r"\s*[\(\[]\s*from\s+[^\)\]]+[\)\]]\s*",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\s*[-–—]\s*from\s+[\"']?.+?[\"']?\s*$",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    return normalise_text(text)


def canonical_artist(value: Any) -> str:
    text = safe_text(value)

    # Use the first listed artist as the primary dedupe identity.
    first_artist = re.split(
        r"\s*(?:,|&| feat\.?| ft\.?| featuring )\s*",
        text,
        maxsplit=1,
        flags=re.IGNORECASE,
    )[0]

    return normalise_text(first_artist)


def track_dedupe_key(title: Any, artist: Any) -> str:
    return (
        f"{canonical_title(title)}"
        f"||{canonical_artist(artist)}"
    )


def list_to_pipe(values: Iterable[Any]) -> str:
    return " | ".join(unique_nonempty(values))


def log_event(event: str, **fields: Any) -> None:
    parts = [event]
    for key, value in fields.items():
        parts.append(f"{key}={safe_text(value)}")
    tqdm.write(" | ".join(parts))


# GENRE MAPPING

def map_specific_itunes_genre(
    primary_genre: Any,
) -> Tuple[str, str]:
    raw = safe_text(primary_genre)
    value = normalise_text(raw)

    if not value:
        return "", "itunes_primary_genre_missing"

    if value in BROAD_ITUNES_GENRES:
        return "", f"itunes_broad_genre_{value.replace(' ', '_')}"

    if any(
        marker in value
        for marker in [
            "hip hop",
            "hip-hop",
            "rap",
            "trap",
            "嘻哈",
            "說唱",
            "说唱",
        ]
    ):
        return (
            "Hip-Hop/Rap",
            f"itunes_specific_primary_genre_{raw}",
        )

    if any(
        marker in value
        for marker in [
            "r b",
            "r&b",
            "rhythm and blues",
            "soul",
            "neo soul",
        ]
    ):
        return (
            "R&B/Soul",
            f"itunes_specific_primary_genre_{raw}",
        )

    if any(
        marker in value
        for marker in [
            "rock",
            "alternative",
            "metal",
            "punk",
            "搖滾",
            "摇滚",
        ]
    ):
        return (
            "Rock",
            f"itunes_specific_primary_genre_{raw}",
        )

    if any(
        marker in value
        for marker in [
            "folk",
            "acoustic",
            "singer songwriter",
            "singer-songwriter",
            "sufi",
            "民謠",
            "民谣",
        ]
    ):
        return (
            "Folk/Acoustic",
            f"itunes_specific_primary_genre_{raw}",
        )

    if (
        value == "pop"
        or "mandopop" in value
        or "cantopop" in value
        or "流行" in value
    ):
        return (
            "Pop",
            f"itunes_specific_primary_genre_{raw}",
        )

    return "", f"itunes_unmapped_primary_genre_{raw}"


def choose_provisional_genre(
    raw_itunes_genres: Sequence[str],
    query_genres: Sequence[str],
) -> Tuple[str, str]:
    specific_votes: List[str] = []
    specific_bases: List[str] = []

    for raw_genre in unique_nonempty(raw_itunes_genres):
        mapped, basis = map_specific_itunes_genre(raw_genre)
        if mapped:
            specific_votes.append(mapped)
            specific_bases.append(basis)

    if specific_votes:
        counts = Counter(specific_votes)
        winner = counts.most_common(1)[0][0]
        return (
            winner,
            " | ".join(unique_nonempty(specific_bases)),
        )

    valid_query_genres = [
        genre
        for genre in unique_nonempty(query_genres)
        if genre in TARGET_GENRES
    ]

    if valid_query_genres:
        counts = Counter(valid_query_genres)
        winner = counts.most_common(1)[0][0]
        return (
            winner,
            "query_genre_only_pending_exact_track_verification",
        )

    return (
        "Unclassified",
        "no_specific_genre_evidence",
    )


# QUALITY FILTERING

def contains_normalised_phrase(
    text: Any,
    phrase: Any,
) -> bool:
    text_norm = normalise_text(text)
    phrase_norm = normalise_text(phrase)

    if not text_norm or not phrase_norm:
        return False

    return bool(
        re.search(
            rf"(?<![a-z0-9]){re.escape(phrase_norm)}(?![a-z0-9])",
            text_norm,
        )
    )


def exclusion_reason(
    title: str,
    artist: str,
    collection: str,
    primary_genre: str,
) -> str:
    # Deliberately exclude artist, source and notes from version matching.
    # For example, "arti" must not reject an artist named Kartik.
    combined = f"{title} {collection}"

    for marker in NON_SONG_MARKERS:
        if contains_normalised_phrase(combined, marker):
            return (
                "excluded_non_song_"
                + normalise_text(marker).replace(" ", "_")
            )

    for marker in LOW_QUALITY_MARKERS:
        if contains_normalised_phrase(combined, marker):
            return (
                "excluded_low_quality_or_out_of_scope_"
                + normalise_text(marker).replace(" ", "_")
            )

    return ""


def commercial_score(item: Dict[str, Any]) -> int:
    score = 0

    if safe_text(item.get("trackViewUrl")):
        score += 4

    if safe_text(item.get("artistViewUrl")):
        score += 2

    if safe_text(item.get("collectionViewUrl")):
        score += 2

    if safe_text(item.get("collectionName")):
        score += 1

    if safe_text(item.get("primaryGenreName")):
        score += 1

    if item.get("trackPrice") not in (None, ""):
        score += 1

    if item.get("collectionPrice") not in (None, ""):
        score += 1

    if safe_text(item.get("previewUrl")):
        # Used only as catalogue-confidence evidence.
        # The preview URL itself is not saved in the final CSV.
        score += 1

    return score


# iTUNES API

def cache_key(query: str, mode: str = "genre_query") -> str:
    payload = {
        "version": "cell4h_famous_artist_v4",
        "mode": mode,
        "query": query,
        "country": "IN",
        "media": "music",
        "entity": "song",
        "limit": ITUNES_LIMIT,
    }

    digest = hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True,
            ensure_ascii=False,
        ).encode("utf-8")
    ).hexdigest()

    return digest


def wait_for_itunes() -> None:
    global LAST_REQUEST_TIME

    elapsed = time.monotonic() - LAST_REQUEST_TIME
    remaining = REQUEST_INTERVAL_SECONDS - elapsed

    if remaining > 0:
        time.sleep(remaining)


def request_itunes(
    query: str,
) -> Tuple[List[Dict[str, Any]], str]:
    global LAST_REQUEST_TIME

    key = cache_key(query, mode="genre_query")

    cached = ITUNES_CACHE.get(key)
    if isinstance(cached, dict):
        results = cached.get("results")
        if isinstance(results, list):
            return (
                [
                    item
                    for item in results
                    if isinstance(item, dict)
                ],
                "cache",
            )

    params = {
        "term": query,
        "country": "IN",
        "media": "music",
        "entity": "song",
        "limit": ITUNES_LIMIT,
        "lang": "en_us",
        "version": "2",
    }

    last_status = "not_requested"

    for attempt in range(1, MAX_RETRIES + 1):
        wait_for_itunes()

        try:
            response = SESSION.get(
                ITUNES_SEARCH_URL,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            LAST_REQUEST_TIME = time.monotonic()
        except requests.RequestException as error:
            last_status = (
                f"network_{type(error).__name__}"
            )
            if attempt < MAX_RETRIES:
                time.sleep(attempt * 3)
                continue
            return [], last_status

        if response.status_code == 200:
            try:
                payload = response.json()
            except ValueError:
                return [], "invalid_json"

            results = payload.get("results", [])
            if not isinstance(results, list):
                results = []

            clean_results = [
                item
                for item in results
                if isinstance(item, dict)
            ]

            ITUNES_CACHE[key] = {
                "query": query,
                "retrieved_at_unix": int(time.time()),
                "results": clean_results,
            }

            if SAVE_CACHE_AFTER_EVERY_QUERY:
                save_json_cache(
                    ITUNES_CACHE,
                    CACHE_FILE,
                )

            return clean_results, "http_200"

        if response.status_code in {403, 429}:
            last_status = f"http_{response.status_code}"
            retry_after = response.headers.get(
                "Retry-After",
                "",
            )

            sleep_seconds = (
                float(retry_after)
                if safe_text(retry_after).replace(".", "", 1).isdigit()
                else max(15, attempt * 15)
            )

            if attempt < MAX_RETRIES:
                print(
                    f"RATE_LIMIT | query={query} | "
                    f"status={response.status_code} | "
                    f"sleep={sleep_seconds}s"
                )
                time.sleep(sleep_seconds)
                continue

            return [], last_status

        if response.status_code >= 500:
            last_status = f"http_{response.status_code}"
            if attempt < MAX_RETRIES:
                time.sleep(attempt * 5)
                continue
            return [], last_status

        return [], f"http_{response.status_code}"

    return [], last_status


def request_itunes_artist_catalogue(
    artist_name: str,
) -> Tuple[List[Dict[str, Any]], str]:
    global LAST_REQUEST_TIME

    key = cache_key(
        artist_name,
        mode="exact_artist_catalogue",
    )

    cached = ITUNES_CACHE.get(key)
    if isinstance(cached, dict):
        results = cached.get("results")
        if isinstance(results, list):
            return (
                [
                    item
                    for item in results
                    if isinstance(item, dict)
                ],
                "cache",
            )

    params = {
        "term": artist_name,
        "country": "IN",
        "media": "music",
        "entity": "song",
        "attribute": "artistTerm",
        "limit": ARTIST_CATALOGUE_LIMIT,
        "lang": "en_us",
        "version": "2",
    }

    last_status = "not_requested"

    for attempt in range(1, MAX_RETRIES + 1):
        wait_for_itunes()

        try:
            response = SESSION.get(
                ITUNES_SEARCH_URL,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            LAST_REQUEST_TIME = time.monotonic()
        except requests.RequestException as error:
            last_status = f"network_{type(error).__name__}"
            if attempt < MAX_RETRIES:
                time.sleep(attempt * 3)
                continue
            return [], last_status

        if response.status_code == 200:
            try:
                payload = response.json()
            except ValueError:
                return [], "invalid_json"

            results = payload.get("results", [])
            if not isinstance(results, list):
                results = []

            clean_results = [
                item
                for item in results
                if isinstance(item, dict)
            ]

            ITUNES_CACHE[key] = {
                "artist_name": artist_name,
                "mode": "exact_artist_catalogue",
                "retrieved_at_unix": int(time.time()),
                "results": clean_results,
            }
            save_json_cache(ITUNES_CACHE, CACHE_FILE)
            return clean_results, "http_200"

        if response.status_code in {403, 429}:
            last_status = f"http_{response.status_code}"
            retry_after = response.headers.get("Retry-After", "")
            sleep_seconds = (
                float(retry_after)
                if safe_text(retry_after).replace(".", "", 1).isdigit()
                else max(15, attempt * 15)
            )
            if attempt < MAX_RETRIES:
                time.sleep(sleep_seconds)
                continue
            return [], last_status

        if response.status_code >= 500:
            last_status = f"http_{response.status_code}"
            if attempt < MAX_RETRIES:
                time.sleep(attempt * 5)
                continue

        return [], f"http_{response.status_code}"

    return [], last_status



# FAMOUS-ARTIST SEEDING AND ID-BASED CATALOGUE LOOKUP

def collect_apple_india_chart_artist_seeds() -> Tuple[
    Dict[str, Dict[str, Any]],
    List[Dict[str, Any]],
]:
    """
    Return artist fame signals from the official Apple India most-played feed.

    Chart presence is a popularity signal only. It does not prove that an
    artist or track is Hindi.
    """
    if not USE_APPLE_INDIA_CHART_FOR_FAME:
        return {}, []

    try:
        response = SESSION.get(
            APPLE_INDIA_CHART_URL,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
    except requests.RequestException as error:
        log_event(
            "HINDI_APPLE_CHART_FAILED",
            error_type=type(error).__name__,
        )
        return {}, []

    if response.status_code != 200:
        log_event(
            "HINDI_APPLE_CHART_FAILED",
            status=response.status_code,
        )
        return {}, []

    try:
        payload = response.json()
    except ValueError:
        log_event(
            "HINDI_APPLE_CHART_FAILED",
            status="invalid_json",
        )
        return {}, []

    results = payload.get("feed", {}).get("results", [])
    if not isinstance(results, list):
        results = []

    artist_signals: Dict[str, Dict[str, Any]] = {}
    audit_rows: List[Dict[str, Any]] = []

    for rank, item in enumerate(results, start=1):
        if not isinstance(item, dict):
            continue

        title = clean_html(item.get("name"))
        artist_text = clean_html(item.get("artistName"))
        collection = clean_html(item.get("collectionName"))

        for artist_name in split_artist_names(artist_text):
            key = normalise_text(artist_name)
            if not key:
                continue

            record = artist_signals.setdefault(
                key,
                {
                    "artist": artist_name,
                    "best_chart_rank": rank,
                    "chart_track_titles": [],
                    "chart_collections": [],
                },
            )
            record["best_chart_rank"] = min(
                int(record["best_chart_rank"]),
                rank,
            )
            record["chart_track_titles"].append(title)
            record["chart_collections"].append(collection)

            audit_rows.append(
                {
                    "chart_rank": rank,
                    "artist": artist_name,
                    "track_title": title,
                    "collection": collection,
                    "apple_url": safe_text(item.get("url")),
                }
            )

    pd.DataFrame(audit_rows).to_csv(
        HINDI_APPLE_CHART_ARTISTS_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    log_event(
        "HINDI_APPLE_CHART_COMPLETE",
        returned_tracks=len(results),
        unique_artists=len(artist_signals),
    )

    return artist_signals, audit_rows


def request_itunes_artist_identity(
    artist_name: str,
) -> Tuple[Optional[Dict[str, Any]], str]:
    """
    Resolve an artist name to the best exact iTunes musicArtist result.

    The artist ID is then used with /lookup to reduce false positives.
    """
    global LAST_REQUEST_TIME

    key = cache_key(
        artist_name,
        mode="famous_artist_identity_search",
    )

    cached = ITUNES_CACHE.get(key)
    if isinstance(cached, dict):
        result = cached.get("best_artist")
        if isinstance(result, dict):
            return result, "cache"
        if cached.get("no_exact_match"):
            return None, "cache_no_exact_match"

    params = {
        "term": artist_name,
        "country": "IN",
        "media": "music",
        "entity": "musicArtist",
        "attribute": "artistTerm",
        "limit": ARTIST_SEARCH_LIMIT,
        "lang": "en_us",
        "version": "2",
    }

    for attempt in range(1, MAX_RETRIES + 1):
        wait_for_itunes()

        try:
            response = SESSION.get(
                ITUNES_SEARCH_URL,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            LAST_REQUEST_TIME = time.monotonic()
        except requests.RequestException as error:
            if attempt < MAX_RETRIES:
                time.sleep(attempt * 3)
                continue
            return None, f"network_{type(error).__name__}"

        if response.status_code == 200:
            try:
                payload = response.json()
            except ValueError:
                return None, "invalid_json"

            results = [
                item
                for item in payload.get("results", [])
                if isinstance(item, dict)
            ]

            requested = normalise_text(artist_name)
            exact = [
                item
                for item in results
                if normalise_text(item.get("artistName")) == requested
                and safe_text(item.get("artistId"))
            ]

            # Exact normalised-name match first. If unavailable, accept only a
            # single strong containment result, never a broad fuzzy match.
            best = exact[0] if exact else None

            if best is None:
                contained = [
                    item
                    for item in results
                    if safe_text(item.get("artistId"))
                    and (
                        requested
                        in normalise_text(item.get("artistName"))
                        or normalise_text(item.get("artistName"))
                        in requested
                    )
                ]
                if len(contained) == 1:
                    best = contained[0]

            ITUNES_CACHE[key] = {
                "artist_name": artist_name,
                "best_artist": best,
                "no_exact_match": best is None,
                "retrieved_at_unix": int(time.time()),
            }
            save_json_cache(ITUNES_CACHE, CACHE_FILE)

            return (
                best,
                "http_200" if best else "http_200_no_exact_match",
            )

        if response.status_code in {403, 429}:
            retry_after = response.headers.get("Retry-After", "")
            sleep_seconds = (
                float(retry_after)
                if safe_text(retry_after).replace(".", "", 1).isdigit()
                else max(15, attempt * 15)
            )
            if attempt < MAX_RETRIES:
                time.sleep(sleep_seconds)
                continue
            return None, f"http_{response.status_code}"

        if response.status_code >= 500 and attempt < MAX_RETRIES:
            time.sleep(attempt * 5)
            continue

        return None, f"http_{response.status_code}"

    return None, "max_retries_exceeded"


def request_itunes_artist_catalogue_by_id(
    artist_id: str,
) -> Tuple[List[Dict[str, Any]], str]:
    """
    ID-based lookup is used instead of a broad artist-name song search.
    """
    global LAST_REQUEST_TIME

    key = cache_key(
        artist_id,
        mode="famous_artist_id_catalogue_lookup",
    )

    cached = ITUNES_CACHE.get(key)
    if isinstance(cached, dict):
        results = cached.get("results")
        if isinstance(results, list):
            return (
                [
                    item
                    for item in results
                    if isinstance(item, dict)
                ],
                "cache",
            )

    params = {
        "id": artist_id,
        "country": "IN",
        "entity": "song",
        "limit": ARTIST_CATALOGUE_LIMIT,
        "lang": "en_us",
    }

    for attempt in range(1, MAX_RETRIES + 1):
        wait_for_itunes()

        try:
            response = SESSION.get(
                ITUNES_LOOKUP_URL,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            LAST_REQUEST_TIME = time.monotonic()
        except requests.RequestException as error:
            if attempt < MAX_RETRIES:
                time.sleep(attempt * 3)
                continue
            return [], f"network_{type(error).__name__}"

        if response.status_code == 200:
            try:
                payload = response.json()
            except ValueError:
                return [], "invalid_json"

            # The first lookup object can be the artist itself; retain tracks only.
            results = [
                item
                for item in payload.get("results", [])
                if isinstance(item, dict)
                and normalise_text(item.get("wrapperType")) == "track"
                and normalise_text(item.get("kind")) in {"song", ""}
            ]

            ITUNES_CACHE[key] = {
                "artist_id": artist_id,
                "retrieved_at_unix": int(time.time()),
                "results": results,
            }
            save_json_cache(ITUNES_CACHE, CACHE_FILE)

            return results, "http_200"

        if response.status_code in {403, 429}:
            retry_after = response.headers.get("Retry-After", "")
            sleep_seconds = (
                float(retry_after)
                if safe_text(retry_after).replace(".", "", 1).isdigit()
                else max(15, attempt * 15)
            )
            if attempt < MAX_RETRIES:
                time.sleep(sleep_seconds)
                continue
            return [], f"http_{response.status_code}"

        if response.status_code >= 500 and attempt < MAX_RETRIES:
            time.sleep(attempt * 5)
            continue

        return [], f"http_{response.status_code}"

    return [], "max_retries_exceeded"


def record_has_famous_artist(
    artist_text: str,
    famous_artist_keys: set[str],
) -> bool:
    return any(
        normalise_text(name) in famous_artist_keys
        for name in split_artist_names(artist_text)
    )


# NORMALISE RESULT

def normalise_itunes_result(
    item: Dict[str, Any],
    *,
    query: str,
    query_genre: str,
    result_rank: int,
    source_dataset: str = "iTunes Hindi Search Metadata",
    artist_source: str = "itunes_search_result",
) -> Dict[str, Any]:
    title = clean_html(item.get("trackName"))
    artist = clean_html(item.get("artistName"))
    collection = clean_html(item.get("collectionName"))
    primary_genre = clean_html(item.get("primaryGenreName"))

    release_year = parse_release_year(
        item.get("releaseDate")
    )

    duration_seconds = parse_duration_seconds(
        item.get("trackTimeMillis")
    )

    return {
        "language": "Hindi",
        "language_code": "hi",
        "label": "Human",
        "model": "Human",
        "source_dataset": source_dataset,
        "title": title,
        "artist": artist,
        "collection": collection,
        "album": collection,
        "raw_genre": primary_genre,
        "primary_genre_name": primary_genre,
        "search_query": query,
        "query_genre_group": query_genre,
        "country": safe_text(item.get("country")) or "IND",
        "release_year": release_year,
        "release_date": safe_text(item.get("releaseDate")),
        "duration_seconds": duration_seconds,
        "source_track_id": safe_text(item.get("trackId")),
        "source_artist_id": safe_text(item.get("artistId")),
        "source_collection_id": safe_text(
            item.get("collectionId")
        ),
        "track_view_url": safe_text(item.get("trackViewUrl")),
        "artist_view_url": safe_text(item.get("artistViewUrl")),
        "collection_view_url": safe_text(
            item.get("collectionViewUrl")
        ),
        "currency": safe_text(item.get("currency")),
        "track_price": item.get("trackPrice"),
        "collection_price": item.get("collectionPrice"),
        "content_advisory_rating": safe_text(
            item.get("contentAdvisoryRating")
        ),
        "itunes_kind": safe_text(item.get("kind")),
        "itunes_wrapper_type": safe_text(
            item.get("wrapperType")
        ),
        "itunes_result_rank": result_rank,
        "itunes_preview_available": bool(
            safe_text(item.get("previewUrl"))
        ),
        "commercial_catalogue_score": commercial_score(item),
        "commercial_catalogue_match": bool(
            safe_text(item.get("trackViewUrl"))
            and title
            and artist
        ),
        "candidate_language_verified": False,
        "requires_cell5b_language_verification": True,
        "requires_cell5b_genre_verification": True,
        "candidate_language_basis": (
            "devanagari_title_plus_hindi_itunes_query_pending_cell5b"
            if contains_devanagari(title)
            else "hindi_itunes_query_pending_cell5b_language_verification"
        ),
        "verification_status": (
            "itunes_commercial_candidate_pending_cell5b_language_and_genre"
        ),
        "audio_path": "",
        "preview_url": "",
        "youtube_url": "",
        "youtube_video_id": "",
        "youtube_channel": "",
        "youtube_views": 0,
        "youtube_likes": 0,
        "chart_rank": "",
        "artist_source": artist_source,
    }


# COLLECT ALL HINDI iTUNES CANDIDATES


# HIGH-PRECISION HINDI: iTUNES-ONLY RANKED ARTIST WORKFLOW

def build_apple_chart_track_keys(
    chart_rows: Sequence[Dict[str, Any]],
) -> set[str]:
    keys: set[str] = set()

    for row in chart_rows:
        title = safe_text(row.get("track_title"))
        artist_text = safe_text(row.get("artist"))

        if not title or not artist_text:
            continue

        for artist_name in split_artist_names(artist_text):
            keys.add(
                track_dedupe_key(
                    title,
                    artist_name,
                )
            )

    return keys


def build_ranked_itunes_track_map(
    results: Sequence[Dict[str, Any]],
    *,
    resolved_artist_id: str,
    resolved_artist_name: str,
) -> Dict[str, int]:
    """
    Convert relevance-ranked iTunes artistTerm search results into a
    track-ID/dedupe-key rank map, while requiring the exact artist.
    """
    rank_map: Dict[str, int] = {}

    for rank, item in enumerate(results, start=1):
        if not isinstance(item, dict):
            continue

        returned_id = safe_text(item.get("artistId"))
        returned_name = clean_html(item.get("artistName"))

        if (
            returned_id
            and resolved_artist_id
            and returned_id != resolved_artist_id
        ):
            continue

        if not exact_artist_match(
            resolved_artist_name,
            returned_name,
        ):
            continue

        track_id = safe_text(item.get("trackId"))
        title = clean_html(item.get("trackName"))

        if track_id:
            rank_map[f"id::{track_id}"] = min(
                rank,
                rank_map.get(f"id::{track_id}", rank),
            )

        if title and returned_name:
            dedupe_key = track_dedupe_key(
                title,
                returned_name,
            )
            rank_map[f"key::{dedupe_key}"] = min(
                rank,
                rank_map.get(f"key::{dedupe_key}", rank),
            )

    return rank_map


def ranked_itunes_position(
    item: Dict[str, Any],
    rank_map: Dict[str, int],
) -> Optional[int]:
    track_id = safe_text(item.get("trackId"))

    if track_id:
        value = rank_map.get(f"id::{track_id}")
        if value is not None:
            return int(value)

    title = clean_html(item.get("trackName"))
    artist = clean_html(item.get("artistName"))
    key = track_dedupe_key(title, artist)

    value = rank_map.get(f"key::{key}")
    return int(value) if value is not None else None


def validate_high_precision_filter() -> None:
    ordinary_examples = [
        ("Tum Hi Ho", "Aashiqui 2"),
        ("Kesariya", "Brahmastra"),
        ("Apna Bana Le", "Bhediya"),
    ]

    for title, album in ordinary_examples:
        reason = exclusion_reason(
            title,
            "Test Artist",
            album,
            "Bollywood",
        )
        if reason:
            raise RuntimeError(
                "Quality-filter self-test failed: "
                f"{title!r} was rejected as {reason!r}."
            )

    bad_examples = [
        ("Song Name (Karaoke)", "Single"),
        ("Song Name - Remix", "Single"),
        ("Song Name (Vocals Only)", "Single"),
    ]

    for title, album in bad_examples:
        reason = exclusion_reason(
            title,
            "Test Artist",
            album,
            "Bollywood",
        )
        if not reason:
            raise RuntimeError(
                "Quality-filter self-test failed: "
                f"{title!r} was not rejected."
            )

    print("iTunes-only local filter self-test: passed")



# MUSICBRAINZ / LAST.FM / SOUNDCLOUD DISCOVERY
# No external-source row is accepted directly. Every candidate must
# match an exact iTunes song first.

MUSICBRAINZ_BASE_URL = "https://musicbrainz.org/ws/2"
LASTFM_API_URL = "https://ws.audioscrobbler.com/2.0/"
SOUNDCLOUD_API_URL = "https://api.soundcloud.com"
SOUNDCLOUD_TOKEN_URL = "https://secure.soundcloud.com/oauth/token"

_LAST_MUSICBRAINZ_REQUEST = 0.0
_LAST_LASTFM_REQUEST = 0.0
_LAST_SOUNDCLOUD_REQUEST = 0.0
_SOUNDCLOUD_TOKEN = SOUNDCLOUD_ACCESS_TOKEN
_SOUNDCLOUD_TOKEN_EXPIRES_AT = 0.0


def _paced_wait(
    last_request_time: float,
    interval_seconds: float,
) -> None:
    elapsed = time.monotonic() - last_request_time
    remaining = interval_seconds - elapsed
    if remaining > 0:
        time.sleep(remaining)


def external_cache_key(
    source: str,
    mode: str,
    payload: Dict[str, Any],
) -> str:
    serialised = json.dumps(
        {
            "source": source,
            "mode": mode,
            "version": "cell4_external_v1",
            **payload,
        },
        sort_keys=True,
        ensure_ascii=False,
        default=str,
    )
    return hashlib.sha256(
        serialised.encode("utf-8")
    ).hexdigest()


def musicbrainz_get(
    endpoint: str,
    params: Dict[str, Any],
) -> Tuple[Optional[Dict[str, Any]], str]:
    global _LAST_MUSICBRAINZ_REQUEST

    key = external_cache_key(
        "musicbrainz",
        endpoint,
        params,
    )

    cached = MUSICBRAINZ_CACHE.get(key)
    if isinstance(cached, dict):
        return cached.get("payload"), "cache"

    _paced_wait(
        _LAST_MUSICBRAINZ_REQUEST,
        MUSICBRAINZ_REQUEST_INTERVAL_SECONDS,
    )

    headers = {
        "User-Agent": MUSICBRAINZ_CONTACT,
        "Accept": "application/json",
    }

    try:
        response = requests.get(
            f"{MUSICBRAINZ_BASE_URL}/{endpoint}",
            params={**params, "fmt": "json"},
            headers=headers,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        _LAST_MUSICBRAINZ_REQUEST = time.monotonic()
    except requests.RequestException as error:
        return None, f"network_{type(error).__name__}"

    if response.status_code == 503:
        time.sleep(5)
        return None, "http_503_rate_limited"

    if response.status_code != 200:
        return None, f"http_{response.status_code}"

    try:
        payload = response.json()
    except ValueError:
        return None, "invalid_json"

    MUSICBRAINZ_CACHE[key] = {
        "payload": payload,
        "retrieved_at_unix": int(time.time()),
    }
    save_json_cache(
        MUSICBRAINZ_CACHE,
        MUSICBRAINZ_CACHE_FILE,
    )
    return payload, "http_200"


def resolve_musicbrainz_artist(
    artist_name: str,
) -> Tuple[Optional[Dict[str, Any]], str]:
    payload, status = musicbrainz_get(
        "artist/",
        {
            "query": f'artist:"{artist_name}"',
            "limit": 10,
        },
    )

    if not payload:
        return None, status

    artists = [
        item
        for item in payload.get("artists", [])
        if isinstance(item, dict)
    ]

    requested = normalise_text(artist_name)

    exact = [
        item
        for item in artists
        if normalise_text(item.get("name")) == requested
        and safe_text(item.get("id"))
    ]

    if exact:
        exact.sort(
            key=lambda item: int(
                item.get("score", 0) or 0
            ),
            reverse=True,
        )
        return exact[0], status

    return None, status + "_no_exact_artist"


def discover_musicbrainz_tracks(
    artist_name: str,
) -> Tuple[List[Dict[str, Any]], str]:
    artist, artist_status = (
        resolve_musicbrainz_artist(artist_name)
    )

    if not artist:
        return [], artist_status

    mbid = safe_text(artist.get("id"))

    query = (
        f"arid:{mbid} AND "
        f"firstreleasedate:[{MIN_RELEASE_YEAR}-01-01 TO "
        f"{MAX_RELEASE_YEAR}-12-31]"
    )

    payload, status = musicbrainz_get(
        "recording/",
        {
            "query": query,
            "limit": MUSICBRAINZ_RECORDING_LIMIT,
            "offset": 0,
        },
    )

    if not payload:
        return [], status

    rows: List[Dict[str, Any]] = []

    for rank, recording in enumerate(
        payload.get("recordings", []),
        start=1,
    ):
        if not isinstance(recording, dict):
            continue

        title = clean_html(recording.get("title"))
        duration_seconds = parse_duration_seconds(
            recording.get("length")
        )
        release_year = parse_release_year(
            recording.get("first-release-date")
        )

        if not title:
            continue

        if (
            duration_seconds is not None
            and not (
                HINDI_MIN_DURATION_SECONDS
                <= duration_seconds
                <= HINDI_MAX_DURATION_SECONDS
            )
        ):
            continue

        rows.append(
            {
                "external_source": "MusicBrainz",
                "external_rank": rank,
                "external_title": title,
                "external_artist": artist_name,
                "external_duration_seconds": duration_seconds,
                "external_release_year": release_year,
                "external_track_id": safe_text(
                    recording.get("id")
                ),
                "external_url": "",
                "external_playcount": "",
                "external_listeners": "",
                "external_tags": list_to_pipe(
                    tag.get("name", "")
                    for tag in recording.get("tags", [])
                    if isinstance(tag, dict)
                ),
                "musicbrainz_artist_id": mbid,
            }
        )

        if len(rows) >= MAX_EXTERNAL_TRACKS_PER_ARTIST_PER_SOURCE:
            break

    return rows, status


def lastfm_get(
    method: str,
    params: Dict[str, Any],
) -> Tuple[Optional[Dict[str, Any]], str]:
    global _LAST_LASTFM_REQUEST

    if not LASTFM_API_KEY:
        return None, "skipped_missing_lastfm_api_key"

    full_params = {
        "method": method,
        "api_key": LASTFM_API_KEY,
        "format": "json",
        **params,
    }

    key = external_cache_key(
        "lastfm",
        method,
        full_params,
    )

    cached = LASTFM_CACHE.get(key)
    if isinstance(cached, dict):
        return cached.get("payload"), "cache"

    _paced_wait(
        _LAST_LASTFM_REQUEST,
        LASTFM_REQUEST_INTERVAL_SECONDS,
    )

    try:
        response = requests.get(
            LASTFM_API_URL,
            params=full_params,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        _LAST_LASTFM_REQUEST = time.monotonic()
    except requests.RequestException as error:
        return None, f"network_{type(error).__name__}"

    if response.status_code != 200:
        return None, f"http_{response.status_code}"

    try:
        payload = response.json()
    except ValueError:
        return None, "invalid_json"

    if "error" in payload:
        return (
            None,
            f"lastfm_error_{payload.get('error')}_"
            f"{safe_text(payload.get('message'))}",
        )

    LASTFM_CACHE[key] = {
        "payload": payload,
        "retrieved_at_unix": int(time.time()),
    }
    save_json_cache(
        LASTFM_CACHE,
        LASTFM_CACHE_FILE,
    )
    return payload, "http_200"


def discover_lastfm_tracks(
    artist_name: str,
) -> Tuple[List[Dict[str, Any]], str]:
    payload, status = lastfm_get(
        "artist.getTopTracks",
        {
            "artist": artist_name,
            "autocorrect": 1,
            "limit": LASTFM_TOP_TRACK_LIMIT,
            "page": 1,
        },
    )

    if not payload:
        return [], status

    toptracks = payload.get("toptracks", {})
    tracks = (
        toptracks.get("track", [])
        if isinstance(toptracks, dict)
        else []
    )

    if isinstance(tracks, dict):
        tracks = [tracks]

    rows: List[Dict[str, Any]] = []

    for rank, track in enumerate(tracks, start=1):
        if not isinstance(track, dict):
            continue

        title = clean_html(track.get("name"))
        if not title:
            continue

        rows.append(
            {
                "external_source": "Last.fm",
                "external_rank": rank,
                "external_title": title,
                "external_artist": artist_name,
                "external_duration_seconds": "",
                "external_release_year": "",
                "external_track_id": safe_text(
                    track.get("mbid")
                ),
                "external_url": safe_text(
                    track.get("url")
                ),
                "external_playcount": safe_text(
                    track.get("playcount")
                ),
                "external_listeners": safe_text(
                    track.get("listeners")
                ),
                "external_tags": "",
                "musicbrainz_artist_id": "",
            }
        )

        if len(rows) >= MAX_EXTERNAL_TRACKS_PER_ARTIST_PER_SOURCE:
            break

    return rows, status


def lastfm_artist_tags(
    artist_name: str,
) -> Tuple[List[str], str]:
    payload, status = lastfm_get(
        "artist.getTopTags",
        {
            "artist": artist_name,
            "autocorrect": 1,
        },
    )

    if not payload:
        return [], status

    top = payload.get("toptags", {})
    tags = (
        top.get("tag", [])
        if isinstance(top, dict)
        else []
    )

    if isinstance(tags, dict):
        tags = [tags]

    return [
        normalise_text(tag.get("name"))
        for tag in tags
        if isinstance(tag, dict)
        and normalise_text(tag.get("name"))
    ], status


def discover_lastfm_similar_artists(
    seed_plan: Dict[str, List[str]],
) -> Tuple[Dict[str, List[str]], List[Dict[str, Any]]]:
    discovered: Dict[str, List[str]] = {
        genre: []
        for genre in TARGET_GENRES
    }
    audit: List[Dict[str, Any]] = []

    if (
        not RUN_LASTFM_SIMILAR_ARTIST_DISCOVERY
        or not LASTFM_API_KEY
    ):
        return discovered, audit

    positive_tags = {
        "hindi",
        "bollywood",
        "indian",
        "indian indie",
        "desi",
        "hindustani",
        "indipop",
        "indie india",
    }

    negative_tags = {
        "punjabi",
        "tamil",
        "telugu",
        "bengali",
        "malayalam",
        "kannada",
        "marathi",
        "gujarati",
    }

    known = {
        normalise_text(name)
        for names in seed_plan.values()
        for name in names
    }

    for genre, anchors in seed_plan.items():
        for anchor in anchors[
            :LASTFM_SIMILAR_ANCHORS_PER_GENRE
        ]:
            payload, status = lastfm_get(
                "artist.getSimilar",
                {
                    "artist": anchor,
                    "autocorrect": 1,
                    "limit": LASTFM_SIMILAR_PER_ANCHOR,
                },
            )

            if not payload:
                audit.append(
                    {
                        "genre": genre,
                        "anchor_artist": anchor,
                        "candidate_artist": "",
                        "status": status,
                        "accepted": False,
                        "tag_basis": "",
                    }
                )
                continue

            similar = payload.get(
                "similarartists",
                {},
            )
            artists = (
                similar.get("artist", [])
                if isinstance(similar, dict)
                else []
            )

            if isinstance(artists, dict):
                artists = [artists]

            for item in artists:
                if not isinstance(item, dict):
                    continue

                candidate = clean_html(
                    item.get("name")
                )
                candidate_key = normalise_text(
                    candidate
                )

                if (
                    not candidate_key
                    or candidate_key in known
                    or candidate_key
                    in {
                        normalise_text(name)
                        for name in discovered[genre]
                    }
                ):
                    continue

                tags, tag_status = (
                    lastfm_artist_tags(candidate)
                )
                tag_set = set(tags)

                accepted = bool(
                    tag_set & positive_tags
                    and not tag_set & negative_tags
                )

                audit.append(
                    {
                        "genre": genre,
                        "anchor_artist": anchor,
                        "candidate_artist": candidate,
                        "status": (
                            status + "|" + tag_status
                        ),
                        "accepted": accepted,
                        "tag_basis": " | ".join(tags[:20]),
                    }
                )

                if accepted:
                    discovered[genre].append(
                        candidate
                    )
                    known.add(candidate_key)

                if (
                    len(discovered[genre])
                    >= MAX_LASTFM_SIMILAR_ARTISTS_PER_GENRE
                ):
                    break

            if (
                len(discovered[genre])
                >= MAX_LASTFM_SIMILAR_ARTISTS_PER_GENRE
            ):
                break

    return discovered, audit


def get_soundcloud_token() -> Tuple[str, str]:
    global _SOUNDCLOUD_TOKEN
    global _SOUNDCLOUD_TOKEN_EXPIRES_AT

    if (
        _SOUNDCLOUD_TOKEN
        and (
            _SOUNDCLOUD_TOKEN_EXPIRES_AT == 0
            or time.time() < _SOUNDCLOUD_TOKEN_EXPIRES_AT
        )
    ):
        return _SOUNDCLOUD_TOKEN, "existing_token"

    if (
        not SOUNDCLOUD_CLIENT_ID
        or not SOUNDCLOUD_CLIENT_SECRET
    ):
        return "", "skipped_missing_soundcloud_credentials"

    credentials = (
        f"{SOUNDCLOUD_CLIENT_ID}:"
        f"{SOUNDCLOUD_CLIENT_SECRET}"
    )
    encoded = base64.b64encode(
        credentials.encode("utf-8")
    ).decode("ascii")

    try:
        response = requests.post(
            SOUNDCLOUD_TOKEN_URL,
            headers={
                "Authorization": f"Basic {encoded}",
                "Accept": "application/json; charset=utf-8",
                "Content-Type": (
                    "application/x-www-form-urlencoded"
                ),
            },
            data={
                "grant_type": "client_credentials",
            },
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
    except requests.RequestException as error:
        return "", f"network_{type(error).__name__}"

    if response.status_code != 200:
        return "", f"http_{response.status_code}"

    try:
        payload = response.json()
    except ValueError:
        return "", "invalid_json"

    token = safe_text(
        payload.get("access_token")
    )
    if not token:
        return "", "missing_access_token"

    _SOUNDCLOUD_TOKEN = token

    try:
        expires_in = int(
            payload.get("expires_in", 3600)
        )
    except (TypeError, ValueError):
        expires_in = 3600

    _SOUNDCLOUD_TOKEN_EXPIRES_AT = (
        time.time() + expires_in - 60
    )

    return token, "client_credentials"


def discover_soundcloud_tracks(
    artist_name: str,
) -> Tuple[List[Dict[str, Any]], str]:
    global _LAST_SOUNDCLOUD_REQUEST

    token, token_status = get_soundcloud_token()
    if not token:
        return [], token_status

    params = {
        "q": artist_name,
        "access": "playable",
        "duration[from]": (
            HINDI_MIN_DURATION_SECONDS * 1000
        ),
        "duration[to]": (
            HINDI_MAX_DURATION_SECONDS * 1000
        ),
        "limit": SOUNDCLOUD_TRACK_LIMIT,
        "linked_partitioning": "true",
    }

    key = external_cache_key(
        "soundcloud",
        "tracks_search",
        params,
    )

    cached = SOUNDCLOUD_CACHE.get(key)
    if isinstance(cached, dict):
        payload = cached.get("payload")
        status = "cache"
    else:
        _paced_wait(
            _LAST_SOUNDCLOUD_REQUEST,
            SOUNDCLOUD_REQUEST_INTERVAL_SECONDS,
        )

        try:
            response = requests.get(
                f"{SOUNDCLOUD_API_URL}/tracks",
                params=params,
                headers={
                    "Authorization": (
                        f"OAuth {token}"
                    ),
                    "Accept": (
                        "application/json; charset=utf-8"
                    ),
                },
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            _LAST_SOUNDCLOUD_REQUEST = (
                time.monotonic()
            )
        except requests.RequestException as error:
            return [], (
                f"network_{type(error).__name__}"
            )

        if response.status_code != 200:
            return [], f"http_{response.status_code}"

        try:
            payload = response.json()
        except ValueError:
            return [], "invalid_json"

        SOUNDCLOUD_CACHE[key] = {
            "payload": payload,
            "retrieved_at_unix": int(
                time.time()
            ),
        }
        save_json_cache(
            SOUNDCLOUD_CACHE,
            SOUNDCLOUD_CACHE_FILE,
        )
        status = "http_200"

    collection = (
        payload.get("collection", [])
        if isinstance(payload, dict)
        else payload
    )

    if not isinstance(collection, list):
        collection = []

    seed_key = normalise_text(artist_name)
    rows: List[Dict[str, Any]] = []

    for rank, track in enumerate(
        collection,
        start=1,
    ):
        if not isinstance(track, dict):
            continue

        publisher = track.get(
            "publisher_metadata",
            {},
        )
        publisher = (
            publisher
            if isinstance(publisher, dict)
            else {}
        )
        user = track.get("user", {})
        user = (
            user
            if isinstance(user, dict)
            else {}
        )

        credited_artist = clean_html(
            publisher.get("artist")
            or track.get("display_artist")
            or user.get("username")
        )

        credited_key = normalise_text(
            credited_artist
        )

        if (
            seed_key not in credited_key
            and credited_key not in seed_key
        ):
            continue

        title = clean_html(track.get("title"))
        if not title:
            continue

        try:
            duration_seconds = round(
                float(track.get("duration"))
                / 1000.0,
                2,
            )
        except (TypeError, ValueError):
            duration_seconds = None

        rows.append(
            {
                "external_source": "SoundCloud",
                "external_rank": rank,
                "external_title": title,
                "external_artist": (
                    credited_artist
                    or artist_name
                ),
                "external_duration_seconds": (
                    duration_seconds
                ),
                "external_release_year": parse_release_year(
                    track.get("release_date")
                ),
                "external_track_id": safe_text(
                    track.get("urn")
                    or track.get("id")
                ),
                "external_url": safe_text(
                    track.get("permalink_url")
                ),
                "external_playcount": safe_text(
                    track.get("playback_count")
                ),
                "external_listeners": "",
                "external_tags": safe_text(
                    track.get("genre")
                ),
                "musicbrainz_artist_id": "",
            }
        )

        if len(rows) >= MAX_EXTERNAL_TRACKS_PER_ARTIST_PER_SOURCE:
            break

    return rows, status


def base_external_title(
    value: Any,
) -> str:
    text = clean_html(value)
    text = re.sub(
        r"\s*[\(\[]\s*(?:from|feat\.?|ft\.?|"
        r"version|remix|mix|soundtrack)[^\)\]]*"
        r"[\)\]]\s*",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    return normalise_text(text)


def external_title_similarity(
    left: Any,
    right: Any,
) -> float:
    left_key = base_external_title(left)
    right_key = base_external_title(right)

    if not left_key or not right_key:
        return 0.0

    if left_key == right_key:
        return 1.0

    return SequenceMatcher(
        None,
        left_key,
        right_key,
    ).ratio()


def request_itunes_external_track_match(
    title: str,
    artist: str,
    external_duration_seconds: Any = None,
) -> Tuple[
    Optional[Dict[str, Any]],
    Dict[str, Any],
    str,
]:
    global LAST_REQUEST_TIME

    key = cache_key(
        f"{title}||{artist}",
        mode="external_exact_track_match_v1",
    )

    cached = ITUNES_CACHE.get(key)
    if isinstance(cached, dict):
        results = cached.get("results")
        status = "cache"
    else:
        params = {
            "term": f"{title} {artist}",
            "country": "IN",
            "media": "music",
            "entity": "song",
            "limit": EXTERNAL_ITUNES_MATCH_LIMIT,
            "lang": "en_us",
            "version": "2",
        }

        wait_for_itunes()

        try:
            response = SESSION.get(
                ITUNES_SEARCH_URL,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            LAST_REQUEST_TIME = (
                time.monotonic()
            )
        except requests.RequestException as error:
            return (
                None,
                {},
                f"network_{type(error).__name__}",
            )

        if response.status_code != 200:
            return (
                None,
                {},
                f"http_{response.status_code}",
            )

        try:
            payload = response.json()
        except ValueError:
            return None, {}, "invalid_json"

        results = [
            item
            for item in payload.get("results", [])
            if isinstance(item, dict)
        ]

        ITUNES_CACHE[key] = {
            "query_title": title,
            "query_artist": artist,
            "results": results,
            "retrieved_at_unix": int(time.time()),
        }
        save_json_cache(
            ITUNES_CACHE,
            CACHE_FILE,
        )
        status = "http_200"

    best: Optional[Dict[str, Any]] = None
    best_meta: Dict[str, Any] = {}
    best_score = -1.0

    try:
        external_duration = float(
            external_duration_seconds
        )
    except (TypeError, ValueError):
        external_duration = None

    for item in results or []:
        returned_title = clean_html(
            item.get("trackName")
        )
        returned_artist = clean_html(
            item.get("artistName")
        )
        collection = clean_html(
            item.get("collectionName")
        )

        if not exact_artist_match(
            artist,
            returned_artist,
        ):
            continue

        similarity = external_title_similarity(
            title,
            returned_title,
        )

        if similarity < MIN_EXTERNAL_TITLE_SIMILARITY:
            continue

        duration = parse_duration_seconds(
            item.get("trackTimeMillis")
        )

        duration_difference = None

        if (
            external_duration is not None
            and duration is not None
        ):
            duration_difference = abs(
                external_duration - duration
            )

            if (
                duration_difference
                > MAX_EXTERNAL_DURATION_DIFFERENCE_SECONDS
            ):
                continue

        release_year = parse_release_year(
            item.get("releaseDate")
        )

        reason = ""

        if release_year is None or not (
            MIN_RELEASE_YEAR
            <= release_year
            <= MAX_RELEASE_YEAR
        ):
            reason = "release_year_outside_range"
        elif duration is None or not (
            HINDI_MIN_DURATION_SECONDS
            <= duration
            <= HINDI_MAX_DURATION_SECONDS
        ):
            reason = "duration_outside_range"
        elif obvious_non_hindi_script_reason(
            returned_title,
            collection,
            item.get("primaryGenreName", ""),
        ):
            reason = obvious_non_hindi_script_reason(
                returned_title,
                collection,
                item.get("primaryGenreName", ""),
            )
        else:
            reason = exclusion_reason(
                returned_title,
                returned_artist,
                collection,
                item.get(
                    "primaryGenreName",
                    "",
                ),
            )

        if reason:
            continue

        score = similarity * 100.0
        score += commercial_score(item) * 2

        if duration_difference is not None:
            score += max(
                0.0,
                20.0 - duration_difference,
            )

        if score > best_score:
            best_score = score
            best = item
            best_meta = {
                "external_title_similarity": round(
                    similarity,
                    4,
                ),
                "external_duration_difference_seconds": (
                    round(duration_difference, 2)
                    if duration_difference is not None
                    else ""
                ),
                "external_itunes_match_score": round(
                    score,
                    2,
                ),
            }

    return best, best_meta, status


def external_tags_to_genre(
    tags: Any,
    seed_genre: str,
) -> Tuple[str, str]:
    text = normalise_text(tags)

    mappings = [
        (
            "Hip-Hop/Rap",
            [
                "hip hop",
                "hiphop",
                "rap",
                "trap",
                "drill",
                "desi hip hop",
            ],
        ),
        (
            "R&B/Soul",
            [
                "r b",
                "rnb",
                "soul",
                "neo soul",
                "rhythm and blues",
            ],
        ),
        (
            "Rock",
            [
                "rock",
                "alternative rock",
                "indie rock",
                "metal",
                "punk",
            ],
        ),
        (
            "Folk/Acoustic",
            [
                "folk",
                "acoustic",
                "sufi",
                "ghazal",
                "singer songwriter",
            ],
        ),
        (
            "Pop",
            [
                "pop",
                "indie pop",
                "bollywood",
                "indian pop",
            ],
        ),
    ]

    for genre, markers in mappings:
        if any(
            contains_normalised_phrase(
                text,
                marker,
            )
            for marker in markers
        ):
            return (
                genre,
                f"external_tags_{genre}",
            )

    return (
        seed_genre,
        "external_source_artist_genre_seed_"
        "pending_cell5b",
    )



def musicbrainz_artist_credit_name(
    recording: Dict[str, Any],
) -> str:
    credits = recording.get("artist-credit", [])

    if isinstance(credits, dict):
        credits = [credits]

    names: List[str] = []

    for credit in credits or []:
        if isinstance(credit, str):
            continue

        if not isinstance(credit, dict):
            continue

        artist = credit.get("artist", {})
        artist = artist if isinstance(artist, dict) else {}

        name = clean_html(
            credit.get("name")
            or artist.get("name")
        )

        if name:
            names.append(name)

    return names[0] if names else ""


def discover_musicbrainz_by_query(
    query_text: str,
    seed_genre: str,
) -> Tuple[List[Dict[str, Any]], str]:
    query = (
        f'tag:"{query_text}" AND '
        f"country:IN AND "
        f"firstreleasedate:[{MIN_RELEASE_YEAR}-01-01 TO "
        f"{MAX_RELEASE_YEAR}-12-31] AND "
        f"dur:[{HINDI_MIN_DURATION_SECONDS * 1000} TO "
        f"{HINDI_MAX_DURATION_SECONDS * 1000}] AND "
        "video:false"
    )

    payload, status = musicbrainz_get(
        "recording/",
        {
            "query": query,
            "limit": MAX_QUERY_RESULTS_PER_SOURCE,
            "offset": 0,
        },
    )

    if not payload:
        return [], status

    rows: List[Dict[str, Any]] = []

    for rank, recording in enumerate(
        payload.get("recordings", []),
        start=1,
    ):
        if not isinstance(recording, dict):
            continue

        title = clean_html(recording.get("title"))
        artist = musicbrainz_artist_credit_name(recording)
        duration_seconds = parse_duration_seconds(
            recording.get("length")
        )
        release_year = parse_release_year(
            recording.get("first-release-date")
        )

        if not title or not artist:
            continue

        if (
            release_year is not None
            and not (
                MIN_RELEASE_YEAR
                <= release_year
                <= MAX_RELEASE_YEAR
            )
        ):
            continue

        if (
            duration_seconds is not None
            and not (
                HINDI_MIN_DURATION_SECONDS
                <= duration_seconds
                <= HINDI_MAX_DURATION_SECONDS
            )
        ):
            continue

        tags = list_to_pipe(
            tag.get("name", "")
            for tag in recording.get("tags", [])
            if isinstance(tag, dict)
        )

        rows.append(
            {
                "external_source": "MusicBrainz",
                "external_query": query_text,
                "query_genre": seed_genre,
                "external_rank": rank,
                "external_title": title,
                "external_artist": artist,
                "external_duration_seconds": duration_seconds,
                "external_release_year": release_year,
                "external_track_id": safe_text(
                    recording.get("id")
                ),
                "external_url": "",
                "external_playcount": "",
                "external_listeners": "",
                "external_tags": tags or query_text,
                "musicbrainz_artist_id": "",
                "external_source_score": int(
                    recording.get("score", 0) or 0
                ),
            }
        )

    return rows, status


def discover_lastfm_by_query(
    query_text: str,
    seed_genre: str,
) -> Tuple[List[Dict[str, Any]], str]:
    payload, status = lastfm_get(
        "tag.getTopTracks",
        {
            "tag": query_text,
            "limit": MAX_QUERY_RESULTS_PER_SOURCE,
            "page": 1,
        },
    )

    if not payload:
        return [], status

    toptracks = payload.get("toptracks", {})
    tracks = (
        toptracks.get("track", [])
        if isinstance(toptracks, dict)
        else []
    )

    if isinstance(tracks, dict):
        tracks = [tracks]

    rows: List[Dict[str, Any]] = []

    for rank, track in enumerate(tracks, start=1):
        if not isinstance(track, dict):
            continue

        title = clean_html(track.get("name"))

        artist_payload = track.get("artist", {})
        if isinstance(artist_payload, dict):
            artist = clean_html(
                artist_payload.get("name")
            )
            artist_mbid = safe_text(
                artist_payload.get("mbid")
            )
        else:
            artist = clean_html(artist_payload)
            artist_mbid = ""

        if not title or not artist:
            continue

        rows.append(
            {
                "external_source": "Last.fm",
                "external_query": query_text,
                "query_genre": seed_genre,
                "external_rank": rank,
                "external_title": title,
                "external_artist": artist,
                "external_duration_seconds": "",
                "external_release_year": "",
                "external_track_id": safe_text(
                    track.get("mbid")
                ),
                "external_url": safe_text(
                    track.get("url")
                ),
                "external_playcount": safe_text(
                    track.get("playcount")
                ),
                "external_listeners": safe_text(
                    track.get("listeners")
                ),
                "external_tags": query_text,
                "musicbrainz_artist_id": artist_mbid,
                "external_source_score": max(
                    0,
                    100 - rank,
                ),
            }
        )

    return rows, status


def add_external_query_discovery_candidates(
    records: Dict[str, Dict[str, Any]],
    apple_chart_keys: set[str],
    rejected_rows: List[Dict[str, Any]],
    match_rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    audit_rows: List[Dict[str, Any]] = []

    query_tasks = [
        (genre, query_text)
        for genre, queries
        in EXTERNAL_HINDI_QUERY_PLAN.items()
        for query_text in queries
    ]

    print(
        "\nEXTERNAL_QUERY_DISCOVERY_START",
        flush=True,
    )
    print(
        "External genre queries:",
        len(query_tasks),
        flush=True,
    )
    print(
        "Maximum iTunes attempts per query:",
        MAX_EXTERNAL_ITUNES_ATTEMPTS_PER_QUERY,
        flush=True,
    )
    print(
        "Maximum accepted tracks per query:",
        MAX_EXTERNAL_ACCEPTED_PER_QUERY,
        flush=True,
    )
    print(
        "MusicBrainz query search:",
        RUN_MUSICBRAINZ_DISCOVERY,
        "| Last.fm tag search:",
        bool(
            RUN_LASTFM_DISCOVERY
            and LASTFM_API_KEY
        ),
        flush=True,
    )

    progress = tqdm(
        query_tasks,
        desc="External Hindi genre queries",
        unit="query",
        dynamic_ncols=True,
        leave=True,
    )

    for seed_genre, query_text in progress:
        progress.set_postfix(
            genre=seed_genre,
            query=query_text,
            unique=len(records),
            refresh=False,
        )

        discovered: List[Dict[str, Any]] = []
        statuses: List[str] = []

        tqdm.write(
            f"EXTERNAL_QUERY_START | {seed_genre} | {query_text}"
        )

        if RUN_MUSICBRAINZ_DISCOVERY:
            rows, status = discover_musicbrainz_by_query(
                query_text,
                seed_genre,
            )
            discovered.extend(rows)
            statuses.append(
                f"MusicBrainz={status}:{len(rows)}"
            )

        if RUN_LASTFM_DISCOVERY and LASTFM_API_KEY:
            rows, status = discover_lastfm_by_query(
                query_text,
                seed_genre,
            )
            discovered.extend(rows)
            statuses.append(
                f"Last.fm={status}:{len(rows)}"
            )

        # Deduplicate title + artist across both sources.
        deduped: Dict[str, Dict[str, Any]] = {}

        for candidate in discovered:
            key = track_dedupe_key(
                candidate.get("external_title"),
                candidate.get("external_artist"),
            )

            if not key or key == "||":
                continue

            existing = deduped.get(key)
            candidate_score = float(
                candidate.get(
                    "external_source_score",
                    0,
                )
                or 0
            )

            existing_score = float(
                existing.get(
                    "external_source_score",
                    0,
                )
                or 0
            ) if existing else -1

            if existing is None or candidate_score > existing_score:
                deduped[key] = candidate

        ranked_candidates = sorted(
            deduped.values(),
            key=lambda row: (
                -float(
                    row.get(
                        "external_source_score",
                        0,
                    )
                    or 0
                ),
                int(
                    row.get(
                        "external_rank",
                        9999,
                    )
                    or 9999
                ),
            ),
        )

        attempts = 0
        accepted = 0

        for candidate in ranked_candidates:
            if (
                attempts
                >= MAX_EXTERNAL_ITUNES_ATTEMPTS_PER_QUERY
                or accepted
                >= MAX_EXTERNAL_ACCEPTED_PER_QUERY
            ):
                break

            candidate_artist = clean_html(
                candidate.get("external_artist")
            )
            candidate_title = clean_html(
                candidate.get("external_title")
            )

            if not candidate_artist or not candidate_title:
                continue

            attempts += 1

            matched, match_meta, match_status = (
                request_itunes_external_track_match(
                    candidate_title,
                    candidate_artist,
                    candidate.get(
                        "external_duration_seconds",
                    ),
                )
            )

            audit = {
                "seed_genre": seed_genre,
                "external_query": query_text,
                **candidate,
                "itunes_attempt_number": attempts,
                "itunes_match_status": match_status,
                **match_meta,
                "accepted_into_main_pool": bool(matched),
                "itunes_title": (
                    clean_html(matched.get("trackName"))
                    if matched
                    else ""
                ),
                "itunes_artist": (
                    clean_html(matched.get("artistName"))
                    if matched
                    else ""
                ),
                "itunes_track_id": (
                    safe_text(matched.get("trackId"))
                    if matched
                    else ""
                ),
            }
            audit_rows.append(audit)

            if not matched:
                continue

            returned_title = clean_html(
                matched.get("trackName")
            )
            returned_artist = clean_html(
                matched.get("artistName")
            )
            raw_genre = clean_html(
                matched.get("primaryGenreName")
            )

            mapped_genre, mapped_basis = (
                map_specific_itunes_genre(raw_genre)
            )

            provisional_genre = (
                mapped_genre or seed_genre
            )
            genre_basis = (
                mapped_basis
                if mapped_genre
                else (
                    "external_genre_query_"
                    f"{normalise_text(query_text).replace(' ', '_')}"
                    "_pending_cell5b"
                )
            )

            source_name = safe_text(
                candidate.get(
                    "external_source",
                    "External",
                )
            )
            source_rank = candidate.get(
                "external_rank",
                "",
            )

            try:
                source_rank_points = max(
                    0,
                    50 - int(source_rank),
                )
            except (TypeError, ValueError):
                source_rank_points = 0

            chart_key = track_dedupe_key(
                returned_title,
                returned_artist,
            )
            is_chart = chart_key in apple_chart_keys

            score = (
                source_rank_points
                + commercial_score(matched) * 2
                + (
                    APPLE_CHART_TRACK_BONUS
                    if is_chart
                    else 0
                )
                + float(
                    match_meta.get(
                        "external_itunes_match_score",
                        0,
                    )
                )
            )

            row = normalise_itunes_result(
                matched,
                query=(
                    f"{source_name.upper()}_QUERY::"
                    f"{query_text}"
                ),
                query_genre=seed_genre,
                result_rank=(
                    int(source_rank)
                    if safe_text(source_rank).isdigit()
                    else 9999
                ),
                source_dataset=(
                    f"{source_name} Genre Query "
                    "+ Exact iTunes Track Match"
                ),
                artist_source=(
                    f"{source_name.lower()}_query_"
                    "exact_itunes_confirmation"
                ),
            )

            row["final_genre"] = provisional_genre
            row["candidate_genre_basis"] = genre_basis
            row["seed_genre"] = seed_genre
            row["seed_artist"] = ""
            row["external_query"] = query_text
            row["external_source"] = source_name
            row["external_rank"] = source_rank
            row["external_track_id"] = safe_text(
                candidate.get("external_track_id")
            )
            row["external_url"] = safe_text(
                candidate.get("external_url")
            )
            row["external_playcount"] = safe_text(
                candidate.get("external_playcount")
            )
            row["external_listeners"] = safe_text(
                candidate.get("external_listeners")
            )
            row["external_tags"] = safe_text(
                candidate.get("external_tags")
            )
            row["official_apple_india_chart_match"] = is_chart
            row["high_precision_score"] = round(score, 2)
            row.update(match_meta)
            row["candidate_language_basis"] = (
                "external_genre_query_plus_exact_"
                "itunes_confirmation_pending_cell5b"
            )
            row["verification_status"] = (
                "external_query_exact_itunes_"
                "candidate_pending_cell5b"
            )
            row["notes"] = (
                f"discovered_by={source_name};"
                f"external_query={query_text};"
                "exact_iTunes_title_artist_confirmation;"
                f"genre_basis={genre_basis};"
                "no_audio_download;"
                "strict_Hindi_language_and_final_genre_"
                "verification_required_in_Cell5B"
            )

            dedupe_key = (
                safe_text(matched.get("trackId"))
                or track_dedupe_key(
                    returned_title,
                    returned_artist,
                )
            )

            existing = records.get(dedupe_key)

            if (
                existing is None
                or float(
                    row.get("high_precision_score", 0)
                )
                > float(
                    existing.get("high_precision_score", 0)
                )
            ):
                records[dedupe_key] = row

            match_rows.append(
                {
                    "seed_genre": seed_genre,
                    "seed_artist": "",
                    "external_query": query_text,
                    "title": returned_title,
                    "artist": returned_artist,
                    "collection": clean_html(
                        matched.get("collectionName")
                    ),
                    "release_year": parse_release_year(
                        matched.get("releaseDate")
                    ),
                    "duration_seconds": parse_duration_seconds(
                        matched.get("trackTimeMillis")
                    ),
                    "itunes_track_id": safe_text(
                        matched.get("trackId")
                    ),
                    "external_source": source_name,
                    "external_rank": source_rank,
                    "external_title": candidate_title,
                    "external_tags": candidate.get(
                        "external_tags",
                        "",
                    ),
                    "provisional_genre": provisional_genre,
                    "genre_basis": genre_basis,
                    "high_precision_score": round(score, 2),
                    **match_meta,
                }
            )

            accepted += 1

        log_event(
            "EXTERNAL_QUERY_COMPLETE",
            genre=seed_genre,
            query=query_text,
            statuses=" | ".join(statuses),
            discovered=len(discovered),
            deduped=len(ranked_candidates),
            itunes_attempts=attempts,
            accepted=accepted,
            cumulative_unique=len(records),
        )

        tqdm.write(
            "EXTERNAL_QUERY_COMPLETE | "
            f"{seed_genre} | {query_text} | "
            f"source_candidates={len(discovered)} | "
            f"deduped={len(ranked_candidates)} | "
            f"itunes_attempts={attempts} | "
            f"accepted={accepted}"
        )

    return audit_rows


def add_external_discovery_candidates(
    records: Dict[str, Dict[str, Any]],
    unique_tasks: Sequence[Tuple[str, str]],
    apple_chart_keys: set[str],
    rejected_rows: List[Dict[str, Any]],
    match_rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    audit_rows: List[Dict[str, Any]] = []

    external_tasks = list(unique_tasks)[
        :MAX_EXTERNAL_ARTISTS_TOTAL
    ]

    print(
        "\nEXTERNAL_DISCOVERY_START",
        flush=True,
    )
    print(
        "External discovery artists:",
        len(external_tasks),
        flush=True,
    )
    print(
        "Maximum discovered tracks per artist/source:",
        MAX_EXTERNAL_TRACKS_PER_ARTIST_PER_SOURCE,
        flush=True,
    )
    print(
        "Every external candidate still requires exact iTunes confirmation.",
        flush=True,
    )
    print(
        "MusicBrainz:",
        RUN_MUSICBRAINZ_DISCOVERY,
        "| Last.fm:",
        bool(
            RUN_LASTFM_DISCOVERY
            and LASTFM_API_KEY
        ),
        "| SoundCloud:",
        bool(
            RUN_SOUNDCLOUD_DISCOVERY
            and (
                SOUNDCLOUD_ACCESS_TOKEN
                or (
                    SOUNDCLOUD_CLIENT_ID
                    and SOUNDCLOUD_CLIENT_SECRET
                )
            )
        ),
        flush=True,
    )

    for seed_genre, seed_artist in tqdm(
        external_tasks,
        desc="External Hindi discovery",
        unit="artist",
        dynamic_ncols=True,
        leave=True,
    ):
        discovered: List[Dict[str, Any]] = []
        statuses: List[str] = []

        tqdm.write(
            f"EXTERNAL_ARTIST_START | {seed_genre} | {seed_artist}"
        )

        if RUN_MUSICBRAINZ_DISCOVERY:
            rows, status = (
                discover_musicbrainz_tracks(
                    seed_artist
                )
            )
            discovered.extend(rows)
            statuses.append(
                f"MusicBrainz={status}:{len(rows)}"
            )

        if (
            RUN_LASTFM_DISCOVERY
            and LASTFM_API_KEY
        ):
            rows, status = (
                discover_lastfm_tracks(
                    seed_artist
                )
            )
            discovered.extend(rows)
            statuses.append(
                f"Last.fm={status}:{len(rows)}"
            )

        if RUN_SOUNDCLOUD_DISCOVERY:
            rows, status = (
                discover_soundcloud_tracks(
                    seed_artist
                )
            )
            discovered.extend(rows)
            statuses.append(
                f"SoundCloud={status}:{len(rows)}"
            )

        tqdm.write(
            "EXTERNAL_SOURCE_RESULTS | "
            f"{seed_artist} | "
            + " | ".join(statuses)
        )

        local_seen = set()

        for candidate in discovered:
            candidate_key = (
                f"{normalise_text(candidate.get('external_title'))}"
                f"||{normalise_text(seed_artist)}"
            )

            if candidate_key in local_seen:
                continue
            local_seen.add(candidate_key)

            matched, match_meta, match_status = (
                request_itunes_external_track_match(
                    candidate.get(
                        "external_title",
                        "",
                    ),
                    seed_artist,
                    candidate.get(
                        "external_duration_seconds",
                    ),
                )
            )

            audit = {
                "seed_genre": seed_genre,
                "seed_artist": seed_artist,
                **candidate,
                "itunes_match_status": match_status,
                **match_meta,
                "accepted_into_main_pool": bool(
                    matched
                ),
                "itunes_title": (
                    clean_html(
                        matched.get("trackName")
                    )
                    if matched
                    else ""
                ),
                "itunes_artist": (
                    clean_html(
                        matched.get("artistName")
                    )
                    if matched
                    else ""
                ),
                "itunes_track_id": (
                    safe_text(
                        matched.get("trackId")
                    )
                    if matched
                    else ""
                ),
            }
            audit_rows.append(audit)

            if not matched:
                continue

            returned_title = clean_html(
                matched.get("trackName")
            )
            returned_artist = clean_html(
                matched.get("artistName")
            )
            raw_genre = clean_html(
                matched.get("primaryGenreName")
            )

            mapped_genre, mapped_basis = (
                map_specific_itunes_genre(
                    raw_genre
                )
            )

            external_genre, external_basis = (
                external_tags_to_genre(
                    candidate.get(
                        "external_tags",
                        "",
                    ),
                    seed_genre,
                )
            )

            if mapped_genre:
                provisional_genre = mapped_genre
                genre_basis = mapped_basis
            else:
                provisional_genre = external_genre
                genre_basis = external_basis

            source_name = safe_text(
                candidate.get(
                    "external_source",
                    "External",
                )
            )

            source_rank = candidate.get(
                "external_rank",
                "",
            )

            try:
                source_rank_points = max(
                    0,
                    50 - int(source_rank),
                )
            except (TypeError, ValueError):
                source_rank_points = 0

            chart_key = track_dedupe_key(
                returned_title,
                seed_artist,
            )
            is_chart = (
                chart_key in apple_chart_keys
            )

            score = (
                source_rank_points
                + commercial_score(matched) * 2
                + (
                    APPLE_CHART_TRACK_BONUS
                    if is_chart
                    else 0
                )
                + float(
                    match_meta.get(
                        "external_itunes_match_score",
                        0,
                    )
                )
            )

            row = normalise_itunes_result(
                matched,
                query=(
                    f"{source_name.upper()}::"
                    f"{seed_artist}"
                ),
                query_genre=seed_genre,
                result_rank=(
                    int(source_rank)
                    if safe_text(source_rank).isdigit()
                    else 9999
                ),
                source_dataset=(
                    f"{source_name} Discovery "
                    "+ Exact iTunes Track Match"
                ),
                artist_source=(
                    f"{source_name.lower()}_"
                    "discovery_exact_itunes_confirmation"
                ),
            )

            row["final_genre"] = (
                provisional_genre
            )
            row["candidate_genre_basis"] = (
                genre_basis
            )
            row["seed_genre"] = seed_genre
            row["seed_artist"] = seed_artist
            row["external_source"] = source_name
            row["external_rank"] = source_rank
            row["external_track_id"] = safe_text(
                candidate.get(
                    "external_track_id"
                )
            )
            row["external_url"] = safe_text(
                candidate.get("external_url")
            )
            row["external_playcount"] = safe_text(
                candidate.get(
                    "external_playcount"
                )
            )
            row["external_listeners"] = safe_text(
                candidate.get(
                    "external_listeners"
                )
            )
            row["external_tags"] = safe_text(
                candidate.get(
                    "external_tags"
                )
            )
            row["official_apple_india_chart_match"] = (
                is_chart
            )
            row["high_precision_score"] = (
                round(score, 2)
            )
            row.update(match_meta)
            row["candidate_language_basis"] = (
                f"{source_name}_discovery_plus_"
                "exact_itunes_confirmation_"
                "pending_cell5b"
            )
            row["verification_status"] = (
                "external_discovery_exact_itunes_"
                "candidate_pending_cell5b"
            )
            row["notes"] = (
                f"discovered_by={source_name};"
                "exact_iTunes_title_artist_confirmation;"
                f"genre_basis={genre_basis};"
                "no_audio_download;"
                "strict_Hindi_language_and_final_genre_"
                "verification_required_in_Cell5B"
            )

            dedupe_key = (
                safe_text(matched.get("trackId"))
                or track_dedupe_key(
                    returned_title,
                    returned_artist,
                )
            )

            existing = records.get(dedupe_key)

            if (
                existing is None
                or float(
                    row.get(
                        "high_precision_score",
                        0,
                    )
                )
                > float(
                    existing.get(
                        "high_precision_score",
                        0,
                    )
                )
            ):
                records[dedupe_key] = row

            match_rows.append(
                {
                    "seed_genre": seed_genre,
                    "seed_artist": seed_artist,
                    "title": returned_title,
                    "artist": returned_artist,
                    "collection": clean_html(
                        matched.get(
                            "collectionName"
                        )
                    ),
                    "release_year": parse_release_year(
                        matched.get(
                            "releaseDate"
                        )
                    ),
                    "duration_seconds": (
                        parse_duration_seconds(
                            matched.get(
                                "trackTimeMillis"
                            )
                        )
                    ),
                    "itunes_track_id": safe_text(
                        matched.get("trackId")
                    ),
                    "external_source": source_name,
                    "external_rank": source_rank,
                    "external_title": candidate.get(
                        "external_title",
                        "",
                    ),
                    "external_tags": candidate.get(
                        "external_tags",
                        "",
                    ),
                    "provisional_genre": (
                        provisional_genre
                    ),
                    "genre_basis": genre_basis,
                    "high_precision_score": (
                        round(score, 2)
                    ),
                    **match_meta,
                }
            )

        log_event(
            "EXTERNAL_DISCOVERY_ARTIST_COMPLETE",
            genre=seed_genre,
            artist=seed_artist,
            statuses=" | ".join(statuses),
            candidates=len(discovered),
            cumulative_unique=len(records),
        )

    return audit_rows


def collect_hindi_itunes_candidates() -> pd.DataFrame:
    """
    High-precision iTunes-only Hindi candidate discovery.

    A track is retained only when:
      1. The curated artist resolves to an exact iTunes artist identity.
      2. The track belongs to that artist ID catalogue.
      3. The same track appears in relevance-ranked iTunes artist search,
         or is present on the official Apple India chart.
      4. It passes year, duration, script and version filters.

    Old broad-query Hindi CSV rows are never merged.
    """
    validate_high_precision_filter()

    records: Dict[str, Dict[str, Any]] = {}
    rejected_rows: List[Dict[str, Any]] = []
    artist_rows: List[Dict[str, Any]] = []
    match_rows: List[Dict[str, Any]] = []

    _, apple_chart_rows = (
        collect_apple_india_chart_artist_seeds()
    )
    apple_chart_keys = build_apple_chart_track_keys(
        apple_chart_rows
    )

    print(
        "NEXT_STAGE | preparing Hindi artist tasks",
        flush=True,
    )
    print(
        "LASTFM_SIMILAR_ARTIST_PRE_SCAN |",
        (
            "enabled"
            if RUN_LASTFM_SIMILAR_ARTIST_DISCOVERY
            else "disabled_fast_mode"
        ),
        flush=True,
    )

    lastfm_similar_plan, similar_artist_audit = (
        discover_lastfm_similar_artists(
            FAMOUS_HINDI_ARTIST_SEEDS
        )
    )

    print(
        "LASTFM_SIMILAR_ARTIST_PRE_SCAN_COMPLETE |",
        "new_artists=",
        sum(
            len(names)
            for names in lastfm_similar_plan.values()
        ),
        flush=True,
    )

    for genre, names in lastfm_similar_plan.items():
        FAMOUS_HINDI_ARTIST_SEEDS[genre] = list(
            dict.fromkeys(
                FAMOUS_HINDI_ARTIST_SEEDS.get(
                    genre,
                    [],
                )
                + names
            )
        )

    pd.DataFrame(
        similar_artist_audit
    ).to_csv(
        HINDI_EXTERNAL_ARTIST_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    artist_tasks: List[Tuple[str, str]] = [
        (genre, artist_name)
        for genre, artist_names
        in FAMOUS_HINDI_ARTIST_SEEDS.items()
        for artist_name in artist_names
    ]

    seen_artist_names = set()
    unique_tasks: List[Tuple[str, str]] = []

    for genre, artist_name in artist_tasks:
        artist_key = normalise_text(artist_name)
        if not artist_key or artist_key in seen_artist_names:
            continue

        seen_artist_names.add(artist_key)
        unique_tasks.append((genre, artist_name))

    print("=" * 78, flush=True)
    print("iTUNES-ONLY HIGH-PRECISION FAMOUS HINDI SONG DISCOVERY", flush=True)
    print("=" * 78, flush=True)
    print(
        "Method: exact artist identity + artist-ID catalogue + "
        "ranked artistTerm song search"
    )
    print("Curated artist tasks:", len(unique_tasks))
    print("Release years:", MIN_RELEASE_YEAR, "-", MAX_RELEASE_YEAR)
    print("Spotify used: False")
    print("Old Hindi CSV rows retained: False")
    print(
        "Maximum accepted ranked-search position:",
        "None — all valid exact-artist tracks",
    )
    print(
        "Fixed Hindi total per genre:",
        HINDI_FIXED_GENRE_TOTAL_LIMIT,
    )
    print(
        "Per-artist song cap:",
        "None",
    )
    print(
        "Hindi duration:",
        f"{HINDI_MIN_DURATION_SECONDS}–{HINDI_MAX_DURATION_SECONDS} seconds",
    )

    progress = tqdm(
        unique_tasks,
        desc="iTunes famous Hindi artists",
        unit="artist",
        dynamic_ncols=True,
        leave=True,
    )

    for seed_genre, seed_artist in progress:
        progress.set_postfix(
            genre=seed_genre,
            kept=len(records),
            refresh=False,
        )

        identity, identity_status = (
            request_itunes_artist_identity(
                seed_artist
            )
        )

        if not identity:
            artist_rows.append(
                {
                    "seed_genre": seed_genre,
                    "seed_artist": seed_artist,
                    "identity_status": identity_status,
                    "itunes_artist_id": "",
                    "itunes_resolved_artist": "",
                    "catalogue_status": "not_requested",
                    "ranked_search_status": "not_requested",
                    "catalogue_rows": 0,
                    "ranked_exact_artist_rows": 0,
                    "accepted_rows": 0,
                }
            )
            log_event(
                "ITUNES_ARTIST_NOT_RESOLVED",
                genre=seed_genre,
                artist=seed_artist,
                status=identity_status,
            )
            continue

        artist_id = safe_text(
            identity.get("artistId")
        )
        resolved_name = clean_html(
            identity.get("artistName")
        )

        catalogue, catalogue_status = (
            request_itunes_artist_catalogue_by_id(
                artist_id
            )
        )
        ranked_results, ranked_status = (
            request_itunes_artist_catalogue(
                resolved_name or seed_artist
            )
        )

        rank_map = build_ranked_itunes_track_map(
            ranked_results,
            resolved_artist_id=artist_id,
            resolved_artist_name=(
                resolved_name or seed_artist
            ),
        )

        accepted_for_artist = 0
        ranked_exact_count = len(
            {
                key
                for key in rank_map
                if key.startswith("id::")
            }
        )

        for catalogue_rank, item in enumerate(
            catalogue,
            start=1,
        ):
            returned_id = safe_text(
                item.get("artistId")
            )
            returned_artist = clean_html(
                item.get("artistName")
            )

            if (
                returned_id
                and artist_id
                and returned_id != artist_id
            ):
                continue

            if not exact_artist_match(
                resolved_name or seed_artist,
                returned_artist,
            ):
                continue

            title = clean_html(item.get("trackName"))
            collection = clean_html(
                item.get("collectionName")
            )
            raw_genre = clean_html(
                item.get("primaryGenreName")
            )
            release_year = parse_release_year(
                item.get("releaseDate")
            )
            duration_seconds = parse_duration_seconds(
                item.get("trackTimeMillis")
            )

            search_rank = ranked_itunes_position(
                item,
                rank_map,
            )

            chart_key = track_dedupe_key(
                title,
                seed_artist,
            )
            is_chart_track = (
                chart_key in apple_chart_keys
            )

            reason = ""

            if not title or not returned_artist:
                reason = "missing_title_or_artist"
            elif normalise_text(
                item.get("wrapperType")
            ) != "track":
                reason = "itunes_wrapper_type_not_track"
            elif normalise_text(
                item.get("kind")
            ) not in {"song", ""}:
                reason = "itunes_kind_not_song"
            elif not safe_text(
                item.get("trackViewUrl")
            ):
                reason = "missing_itunes_track_view_url"
            elif release_year is None:
                reason = "missing_release_year"
            elif not (
                MIN_RELEASE_YEAR
                <= int(release_year)
                <= MAX_RELEASE_YEAR
            ):
                reason = "release_year_outside_configured_range"
            elif (
                duration_seconds is None
                or not (
                    HINDI_MIN_DURATION_SECONDS
                    <= float(duration_seconds)
                    <= HINDI_MAX_DURATION_SECONDS
                )
            ):
                reason = "duration_outside_hindi_range"
            elif obvious_non_hindi_script_reason(
                title,
                collection,
                raw_genre,
            ):
                reason = obvious_non_hindi_script_reason(
                    title,
                    collection,
                    raw_genre,
                )
            else:
                reason = exclusion_reason(
                    title,
                    returned_artist,
                    collection,
                    raw_genre,
                )

            # No ranked-search rejection in v8.
            # Exact artist identity, year, duration, script and version
            # filters still apply.

            if reason:
                rejected_rows.append(
                    {
                        "seed_genre": seed_genre,
                        "seed_artist": seed_artist,
                        "resolved_artist": resolved_name,
                        "itunes_artist_id": artist_id,
                        "title": title,
                        "artist": returned_artist,
                        "collection": collection,
                        "release_year": release_year,
                        "duration_seconds": duration_seconds,
                        "itunes_artist_search_rank": (
                            search_rank
                            if search_rank is not None
                            else ""
                        ),
                        "official_apple_india_chart_match": (
                            is_chart_track
                        ),
                        "rejection_reason": reason,
                    }
                )
                continue

            mapped_genre, mapped_basis = (
                map_specific_itunes_genre(
                    raw_genre
                )
            )

            if not mapped_genre:
                provisional_genre = seed_genre
                genre_basis = (
                    "curated_primary_artist_genre_"
                    "pending_lastfm_track_verification"
                )
                genre_conflict = False
            elif mapped_genre == seed_genre:
                provisional_genre = seed_genre
                genre_basis = mapped_basis
                genre_conflict = False
            else:
                provisional_genre = "Unclassified"
                genre_basis = (
                    f"genre_conflict_seed_{seed_genre}_"
                    f"itunes_{mapped_genre}"
                )
                genre_conflict = True

            rank_points = (
                max(
                    0,
                    ITUNES_RANK_MAX_POINTS
                    - int(search_rank)
                    + 1,
                )
                if search_rank is not None
                else 0
            )
            chart_points = (
                APPLE_CHART_TRACK_BONUS
                if is_chart_track
                else 0
            )
            recent_points = max(
                0,
                min(
                    RECENT_RELEASE_MAX_POINTS,
                    int(release_year)
                    - MIN_RELEASE_YEAR
                    + 1,
                ),
            )
            catalogue_points = commercial_score(
                item
            ) * 2

            high_precision_score = (
                chart_points
                + rank_points
                + recent_points
                + catalogue_points
            )

            row = normalise_itunes_result(
                item,
                query=(
                    f'ITUNES_ARTIST_ID::{seed_artist}'
                ),
                query_genre=seed_genre,
                result_rank=(
                    search_rank
                    if search_rank is not None
                    else catalogue_rank
                ),
                source_dataset=(
                    "iTunes Famous Hindi Artist ID Catalogue "
                    "+ Ranked Artist Search"
                ),
                artist_source=(
                    "itunes_only_curated_famous_artist"
                ),
            )

            row["final_genre"] = provisional_genre
            row["candidate_genre_basis"] = (
                genre_basis
            )
            row["seed_genre"] = seed_genre
            row["seed_artist"] = seed_artist
            row["famous_artist_resolved_name"] = (
                resolved_name
            )
            row["famous_artist_itunes_id"] = (
                artist_id
            )
            row["artist_identity_status"] = (
                identity_status
            )
            row["artist_catalogue_status"] = (
                catalogue_status
            )
            row["ranked_artist_search_status"] = (
                ranked_status
            )
            row["itunes_artist_search_rank"] = (
                search_rank
                if search_rank is not None
                else ""
            )
            row["itunes_artist_catalogue_rank"] = (
                catalogue_rank
            )
            row[
                "official_apple_india_chart_match"
            ] = is_chart_track
            row["genre_conflict"] = genre_conflict
            row["high_precision_score"] = (
                high_precision_score
            )
            row["fame_basis"] = (
                "official_apple_india_chart"
                if is_chart_track
                else (
                    "top_ranked_exact_itunes_artist_search_"
                    f"rank_{search_rank}"
                )
            )
            row["candidate_language_basis"] = (
                "famous_curated_artist_plus_exact_itunes_"
                "artist_ID_and_ranked_track_pending_cell5b"
            )
            row["candidate_language_verified"] = False
            row[
                "requires_cell5b_language_verification"
            ] = True
            row[
                "requires_cell5b_genre_verification"
            ] = True
            row["verification_status"] = (
                "itunes_only_high_precision_candidate_"
                "pending_cell5b"
            )
            row["notes"] = (
                "iTunes_only_metadata;"
                "exact_artist_identity;"
                "artist_ID_catalogue;"
                "ranked_artistTerm_search_confirmation;"
                f"fame_basis={row['fame_basis']};"
                f"genre_basis={genre_basis};"
                "no_Spotify;"
                "no_audio_download;"
                "strict_Hindi_language_and_final_genre_"
                "verification_required_in_Cell5B"
            )

            dedupe_key = (
                safe_text(item.get("trackId"))
                or track_dedupe_key(
                    title,
                    returned_artist,
                )
            )

            existing = records.get(dedupe_key)
            if (
                existing is None
                or float(high_precision_score)
                > float(
                    existing.get(
                        "high_precision_score",
                        0,
                    )
                )
            ):
                records[dedupe_key] = row

            accepted_for_artist += 1

            match_rows.append(
                {
                    "seed_genre": seed_genre,
                    "seed_artist": seed_artist,
                    "resolved_artist": resolved_name,
                    "itunes_artist_id": artist_id,
                    "title": title,
                    "artist": returned_artist,
                    "collection": collection,
                    "release_year": release_year,
                    "duration_seconds": duration_seconds,
                    "itunes_track_id": safe_text(
                        item.get("trackId")
                    ),
                    "itunes_artist_search_rank": (
                        search_rank
                        if search_rank is not None
                        else ""
                    ),
                    "itunes_artist_catalogue_rank": (
                        catalogue_rank
                    ),
                    "official_apple_india_chart_match": (
                        is_chart_track
                    ),
                    "raw_genre": raw_genre,
                    "provisional_genre": (
                        provisional_genre
                    ),
                    "genre_basis": genre_basis,
                    "high_precision_score": (
                        high_precision_score
                    ),
                }
            )

        artist_rows.append(
            {
                "seed_genre": seed_genre,
                "seed_artist": seed_artist,
                "identity_status": identity_status,
                "itunes_artist_id": artist_id,
                "itunes_resolved_artist": resolved_name,
                "catalogue_status": catalogue_status,
                "ranked_search_status": ranked_status,
                "catalogue_rows": len(catalogue),
                "ranked_exact_artist_rows": (
                    ranked_exact_count
                ),
                "accepted_rows": accepted_for_artist,
            }
        )

        log_event(
            "ITUNES_HIGH_PRECISION_ARTIST_COMPLETE",
            genre=seed_genre,
            artist=seed_artist,
            resolved_artist=resolved_name,
            artist_id=artist_id,
            catalogue_rows=len(catalogue),
            ranked_exact_rows=ranked_exact_count,
            accepted_rows=accepted_for_artist,
            cumulative_unique=len(records),
        )

    if RUN_QUERY_BASED_EXTERNAL_DISCOVERY:
        external_audit_rows = (
            add_external_query_discovery_candidates(
                records=records,
                apple_chart_keys=apple_chart_keys,
                rejected_rows=rejected_rows,
                match_rows=match_rows,
            )
        )
    elif RUN_ARTIST_BASED_EXTERNAL_DISCOVERY:
        external_audit_rows = (
            add_external_discovery_candidates(
                records=records,
                unique_tasks=unique_tasks,
                apple_chart_keys=apple_chart_keys,
                rejected_rows=rejected_rows,
                match_rows=match_rows,
            )
        )
    else:
        external_audit_rows = []

    pd.DataFrame(
        external_audit_rows
    ).to_csv(
        HINDI_EXTERNAL_DISCOVERY_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    output = pd.DataFrame(
        list(records.values())
    )

    if output.empty:
        raise RuntimeError(
            "No iTunes-only high-precision Hindi candidates survived. "
            "Inspect the artist and rejected audit CSV files."
        )

    # NO FIXED GENRE TOTAL.

    # Keep every valid result produced by the full famous-artist
    # list. The only remaining selection is a five-track cap per
    # artist within the assigned genre, which prevents one artist
    # from dominating the dataset.

    output["_artist_key"] = output.apply(
        lambda row: canonical_artist(
            row.get("seed_artist")
            or row.get("artist")
        ),
        axis=1,
    )

    output["_genre_key"] = output[
        "final_genre"
    ].fillna("Unclassified")

    output["_score"] = pd.to_numeric(
        output["high_precision_score"],
        errors="coerce",
    ).fillna(0)

    # Keep all valid rows. No per-artist cap is applied.
    output = (
        output.sort_values(
            by=[
                "_genre_key",
                "_artist_key",
                "_score",
                "official_apple_india_chart_match",
                "itunes_artist_search_rank",
                "release_year",
            ],
            ascending=[
                True,
                True,
                False,
                False,
                True,
                False,
            ],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    output = (
        output.sort_values(
            by=[
                "_genre_key",
                "_score",
                "official_apple_india_chart_match",
                "itunes_artist_search_rank",
                "release_year",
                "artist",
                "title",
            ],
            ascending=[
                True,
                False,
                False,
                True,
                False,
                True,
                True,
            ],
            na_position="last",
        )
        .drop(
            columns=[
                "_artist_key",
                "_genre_key",
                "_score",
            ],
            errors="ignore",
        )
        .reset_index(drop=True)
    )

    artist_df = pd.DataFrame(artist_rows)
    matches_df = pd.DataFrame(match_rows)
    rejected_df = pd.DataFrame(rejected_rows)

    artist_df.to_csv(
        ARTIST_PROFILE_CSV,
        index=False,
        encoding="utf-8-sig",
    )
    artist_df.to_csv(
        HINDI_FAMOUS_ARTIST_LOOKUP_LOG_CSV,
        index=False,
        encoding="utf-8-sig",
    )
    matches_df.to_csv(
        HINDI_CROSS_SOURCE_MATCH_CSV,
        index=False,
        encoding="utf-8-sig",
    )
    matches_df.to_csv(
        RAW_OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig",
    )
    artist_df.to_csv(
        QUERY_LOG_CSV,
        index=False,
        encoding="utf-8-sig",
    )
    rejected_df.to_csv(
        REJECTED_OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig",
    )
    output.to_csv(
        MAIN_OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    save_json_cache(
        ITUNES_CACHE,
        CACHE_FILE,
    )

    counts = (
        output["final_genre"]
        .value_counts()
        .reindex(
            TARGET_GENRES + ["Unclassified"],
            fill_value=0,
        )
    )

    count_summary_rows = []

    for genre in TARGET_GENRES:
        count_summary_rows.append(
            {
                "genre": genre,
                "collected_candidates": int(
                    counts.get(genre, 0)
                ),
                "fixed_genre_total_limit": "",
                "fixed_limit_applied": False,
                "per_artist_diversity_cap": "",
                "policy": (
                    "all_valid_exact_artist_results_kept_"
                    "without_genre_rank_or_artist_caps"
                ),
            }
        )

    pd.DataFrame(
        count_summary_rows
    ).to_csv(
        HINDI_SHORTAGE_REPORT_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    print("\nHindi iTunes-only provisional counts")
    print(counts)
    print(
        "\nIMPORTANT | Spotify was not used. "
        "No fixed 70/75 Hindi genre total was applied."
    )
    print("Main Hindi CSV:", MAIN_OUTPUT_CSV)
    print(
        "Ranked iTunes match audit:",
        HINDI_CROSS_SOURCE_MATCH_CSV,
    )
    print(
        "No-limit count summary:",
        HINDI_SHORTAGE_REPORT_CSV,
    )

    return output


# CHINESE iTUNES + APPLE MUSIC CHART DISCOVERY

def chinese_cache_key(
    query: str,
    country: str,
) -> str:
    payload = {
        "version": "cell4_chinese_v3",
        "query": query,
        "country": country,
        "media": "music",
        "entity": "song",
        "limit": ITUNES_LIMIT,
    }
    return hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True,
            ensure_ascii=False,
        ).encode("utf-8")
    ).hexdigest()


def request_chinese_itunes(
    query: str,
    country: str,
) -> Tuple[List[Dict[str, Any]], str]:
    global LAST_REQUEST_TIME

    key = chinese_cache_key(query, country)

    cached = CHINESE_ITUNES_CACHE.get(key)
    if isinstance(cached, dict):
        results = cached.get("results")
        if isinstance(results, list):
            return (
                [
                    item
                    for item in results
                    if isinstance(item, dict)
                ],
                "cache",
            )

    params = {
        "term": query,
        "country": country,
        "media": "music",
        "entity": "song",
        "limit": ITUNES_LIMIT,
        "lang": "en_us",
        "version": "2",
    }

    last_status = "not_requested"

    for attempt in range(1, MAX_RETRIES + 1):
        wait_for_itunes()

        try:
            response = SESSION.get(
                ITUNES_SEARCH_URL,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            LAST_REQUEST_TIME = time.monotonic()
        except requests.RequestException as error:
            last_status = (
                f"network_{type(error).__name__}"
            )
            if attempt < MAX_RETRIES:
                time.sleep(attempt * 3)
                continue
            return [], last_status

        if response.status_code == 200:
            try:
                payload = response.json()
            except ValueError:
                return [], "invalid_json"

            results = payload.get("results", [])
            if not isinstance(results, list):
                results = []

            clean_results = [
                item
                for item in results
                if isinstance(item, dict)
            ]

            CHINESE_ITUNES_CACHE[key] = {
                "query": query,
                "country": country,
                "retrieved_at_unix": int(time.time()),
                "results": clean_results,
            }
            save_json_cache(
                CHINESE_ITUNES_CACHE,
                CHINESE_CACHE_FILE,
            )
            return clean_results, "http_200"

        if response.status_code in {403, 429}:
            last_status = f"http_{response.status_code}"
            retry_after = response.headers.get(
                "Retry-After",
                "",
            )
            sleep_seconds = (
                float(retry_after)
                if safe_text(retry_after)
                .replace(".", "", 1)
                .isdigit()
                else max(15, attempt * 15)
            )

            if attempt < MAX_RETRIES:
                tqdm.write(
                    f"CHINESE_RATE_LIMIT | query={query} | "
                    f"country={country} | "
                    f"sleep={sleep_seconds}s"
                )
                time.sleep(sleep_seconds)
                continue

            return [], last_status

        if response.status_code >= 500:
            last_status = f"http_{response.status_code}"
            if attempt < MAX_RETRIES:
                time.sleep(attempt * 5)
                continue
            return [], last_status

        return [], f"http_{response.status_code}"

    return [], last_status


# Chinese-language storefronts that can serve non-Chinese content.
# We require Han characters as positive evidence when the only signal
# is regional catalogue presence — this stops K-pop / J-pop from
# slipping through on the CN/TW/HK country tag alone.
_CHINESE_STOREFRONTS = {"CN", "TW", "HK", "MO", "SG"}


def chinese_language_basis(
    title: str,
    artist: str,
    collection: str,
    country: str,
) -> Tuple[bool, str]:
    blob = f"{title} {artist} {collection}"

    # Hard reject: Japanese kana or Korean Hangul anywhere in metadata.
    if contains_kana_or_hangul(blob):
        return (
            False,
            "reject_japanese_or_korean_script",
        )

    # Positive pass: Han (Chinese) characters present in title/artist/album.
    if contains_han(blob):
        return (
            True,
            "provisional_chinese_han_metadata",
        )

    # Country-only fallback: require non-ASCII characters in the metadata.
    # Purely Latin-script results in a Chinese storefront are almost always
    # international (English / K-pop) releases — reject them as ambiguous.
    if country.upper() in _CHINESE_STOREFRONTS:
        has_non_ascii = bool(
            re.search(r"[^\x00-\x7F]", blob)
        )
        if not has_non_ascii:
            return (
                False,
                "reject_latin_only_no_chinese_script_evidence",
            )
        return (
            True,
            "provisional_chinese_regional_catalogue_non_latin",
        )

    return False, "no_chinese_high_recall_basis"


def normalise_chinese_itunes_result(
    item: Dict[str, Any],
    *,
    query: str,
    query_genre: str,
    country: str,
    result_rank: int,
) -> Dict[str, Any]:
    title = clean_html(item.get("trackName"))
    artist = clean_html(item.get("artistName"))
    collection = clean_html(item.get("collectionName"))
    primary_genre = clean_html(
        item.get("primaryGenreName")
    )

    language_ok, language_basis = (
        chinese_language_basis(
            title,
            artist,
            collection,
            country,
        )
    )

    return {
        "language": "Chinese",
        "language_code": "zh",
        "label": "Human",
        "model": "Human",
        "source_dataset": (
            "iTunes Chinese Search Metadata"
        ),
        "title": title,
        "artist": artist,
        "collection": collection,
        "album": collection,
        "raw_genre": primary_genre,
        "primary_genre_name": primary_genre,
        "search_query": query,
        "query_genre_group": query_genre,
        "country": country,
        "release_year": parse_release_year(
            item.get("releaseDate")
        ),
        "release_date": safe_text(
            item.get("releaseDate")
        ),
        "duration_seconds": parse_duration_seconds(
            item.get("trackTimeMillis")
        ),
        "source_track_id": safe_text(
            item.get("trackId")
        ),
        "source_artist_id": safe_text(
            item.get("artistId")
        ),
        "source_collection_id": safe_text(
            item.get("collectionId")
        ),
        "track_view_url": safe_text(
            item.get("trackViewUrl")
        ),
        "artist_view_url": safe_text(
            item.get("artistViewUrl")
        ),
        "collection_view_url": safe_text(
            item.get("collectionViewUrl")
        ),
        "currency": safe_text(item.get("currency")),
        "track_price": item.get("trackPrice"),
        "collection_price": item.get(
            "collectionPrice"
        ),
        "content_advisory_rating": safe_text(
            item.get("contentAdvisoryRating")
        ),
        "itunes_kind": safe_text(item.get("kind")),
        "itunes_wrapper_type": safe_text(
            item.get("wrapperType")
        ),
        "itunes_result_rank": result_rank,
        "itunes_preview_available": bool(
            safe_text(item.get("previewUrl"))
        ),
        "commercial_catalogue_score": (
            commercial_score(item)
        ),
        "commercial_catalogue_match": bool(
            safe_text(item.get("trackViewUrl"))
            and title
            and artist
        ),
        "candidate_language_verified": False,
        "requires_cell5b_language_verification": True,
        "requires_cell5b_genre_verification": True,
        "candidate_language_basis": language_basis,
        "verification_status": (
            "itunes_chinese_candidate_pending_cell5b"
        ),
        "language_screen_passed": language_ok,
        "audio_path": "",
        "preview_url": "",
        "youtube_url": "",
        "youtube_video_id": "",
        "youtube_channel": "",
        "youtube_views": 0,
        "youtube_likes": 0,
        "chart_rank": "",
        "artist_source": "itunes_search_result",
    }


def build_chinese_itunes_tasks() -> List[
    Tuple[str, str, str, str]
]:
    tasks: List[Tuple[str, str, str, str]] = []

    for genre, queries in (
        CHINESE_ITUNES_QUERY_PLAN.items()
    ):
        for query in queries:
            for country in CHINESE_ITUNES_COUNTRIES:
                tasks.append(
                    (
                        "broad",
                        genre,
                        query,
                        country,
                    )
                )

                if RUN_CHINESE_YEARLY_QUERIES:
                    for year in range(
                        MIN_RELEASE_YEAR,
                        MAX_RELEASE_YEAR + 1,
                    ):
                        tasks.append(
                            (
                                "yearly",
                                genre,
                                f"{query} {year}",
                                country,
                            )
                        )

    return tasks


def collect_chinese_itunes_candidates() -> pd.DataFrame:
    if not RUN_CHINESE_ITUNES:
        return pd.DataFrame()

    records: Dict[str, Dict[str, Any]] = {}
    rejected_rows: List[Dict[str, Any]] = []
    raw_rows: List[Dict[str, Any]] = []
    query_logs: List[Dict[str, Any]] = []

    tasks = build_chinese_itunes_tasks()

    print("\n" + "=" * 72)
    print("CHINESE iTUNES COMMERCIAL DISCOVERY")
    print("=" * 72)
    print("iTunes query tasks:", len(tasks))
    print(
        "Storefronts:",
        ", ".join(CHINESE_ITUNES_COUNTRIES),
    )
    print(
        "Yearly queries:",
        RUN_CHINESE_YEARLY_QUERIES,
    )

    progress = tqdm(
        tasks,
        desc="Chinese iTunes queries",
        unit="query",
        dynamic_ncols=True,
        leave=True,
    )

    for task_type, query_genre, query, country in progress:
        progress.set_postfix(
            genre=query_genre,
            country=country,
            unique=len(records),
            refresh=False,
        )

        results, status = request_chinese_itunes(
            query,
            country,
        )

        accepted_this_query = 0
        rejected_this_query = 0
        new_unique_this_query = 0

        for result_rank, item in enumerate(
            results,
            start=1,
        ):
            row = normalise_chinese_itunes_result(
                item,
                query=query,
                query_genre=query_genre,
                country=country,
                result_rank=result_rank,
            )
            raw_rows.append(dict(row))

            reason = ""

            if not row["title"] or not row["artist"]:
                reason = "missing_title_or_artist"
            elif normalise_text(
                row["itunes_wrapper_type"]
            ) != "track":
                reason = (
                    "itunes_wrapper_type_not_track"
                )
            elif normalise_text(
                row["itunes_kind"]
            ) not in {"song", ""}:
                reason = "itunes_kind_not_song"
            elif not row["track_view_url"]:
                reason = (
                    "missing_itunes_track_view_url"
                )
            elif row["release_year"] is None:
                reason = "missing_release_year"
            elif not (
                MIN_RELEASE_YEAR
                <= int(row["release_year"])
                <= MAX_RELEASE_YEAR
            ):
                reason = (
                    "release_year_outside_configured_range"
                )
            elif not row["language_screen_passed"]:
                reason = row[
                    "candidate_language_basis"
                ]
            elif (
                CHINESE_FILTER_DURATION
                and (
                    row["duration_seconds"] is None
                    or not (
                        MIN_DURATION_SECONDS
                        <= float(
                            row["duration_seconds"]
                        )
                        <= MAX_DURATION_SECONDS
                    )
                )
            ):
                reason = "duration_missing_or_outside_120_420"
            else:
                reason = exclusion_reason(
                    row["title"],
                    row["artist"],
                    row["collection"],
                    row["raw_genre"],
                )

            if reason:
                rejected = dict(row)
                rejected["rejection_reason"] = reason
                rejected_rows.append(rejected)
                rejected_this_query += 1
                continue

            accepted_this_query += 1
            key = track_dedupe_key(
                row["title"],
                row["artist"],
            )

            if key not in records:
                records[key] = {
                    **row,
                    "_raw_genres": [],
                    "_queries": [],
                    "_query_genres": [],
                    "_countries": [],
                    "_track_ids": [],
                    "_result_ranks": [],
                    "_commercial_scores": [],
                    "_language_bases": [],
                }
                new_unique_this_query += 1

            record = records[key]

            if (
                int(row["commercial_catalogue_score"])
                > int(
                    record[
                        "commercial_catalogue_score"
                    ]
                )
            ):
                preserved = {
                    key_name: record[key_name]
                    for key_name in [
                        "_raw_genres",
                        "_queries",
                        "_query_genres",
                        "_countries",
                        "_track_ids",
                        "_result_ranks",
                        "_commercial_scores",
                        "_language_bases",
                    ]
                }
                record.clear()
                record.update(row)
                record.update(preserved)

            record["_raw_genres"].append(
                row["raw_genre"]
            )
            record["_queries"].append(query)
            record["_query_genres"].append(
                query_genre
            )
            record["_countries"].append(country)
            record["_track_ids"].append(
                row["source_track_id"]
            )
            record["_result_ranks"].append(
                row["itunes_result_rank"]
            )
            record["_commercial_scores"].append(
                row["commercial_catalogue_score"]
            )
            record["_language_bases"].append(
                row["candidate_language_basis"]
            )

        query_logs.append(
            {
                "task_type": task_type,
                "query_genre_group": query_genre,
                "query": query,
                "country": country,
                "request_status": status,
                "returned_rows": len(results),
                "accepted_rows": accepted_this_query,
                "rejected_rows": rejected_this_query,
                "new_unique_tracks": new_unique_this_query,
                "cumulative_unique_tracks": len(records),
            }
        )

        log_event(
            "CHINESE_QUERY_COMPLETE",
            task_type=task_type,
            query_genre=query_genre,
            query=query,
            country=country,
            request_status=status,
            returned_rows=len(results),
            accepted_rows=accepted_this_query,
            new_unique_tracks=new_unique_this_query,
            cumulative_unique_tracks=len(records),
        )

    final_rows: List[Dict[str, Any]] = []

    for record in records.values():
        provisional_genre, genre_basis = (
            choose_provisional_genre(
                record["_raw_genres"],
                record["_query_genres"],
            )
        )

        row = {
            key: value
            for key, value in record.items()
            if not key.startswith("_")
        }

        row["final_genre"] = provisional_genre
        row["raw_genre"] = list_to_pipe(
            record["_raw_genres"]
        )
        row["search_query"] = list_to_pipe(
            record["_queries"]
        )
        row["query_genre_group"] = list_to_pipe(
            record["_query_genres"]
        )
        row["country"] = list_to_pipe(
            record["_countries"]
        )
        row["source_track_id"] = list_to_pipe(
            record["_track_ids"]
        )
        row["candidate_genre_basis"] = (
            genre_basis
        )
        row["candidate_language_basis"] = (
            list_to_pipe(
                record["_language_bases"]
            )
        )
        row["itunes_best_result_rank"] = min(
            record["_result_ranks"]
        )
        row["itunes_query_count"] = len(
            set(record["_queries"])
        )
        row["itunes_query_genre_count"] = len(
            set(record["_query_genres"])
        )
        row["commercial_catalogue_score"] = max(
            record["_commercial_scores"]
        )
        row["notes"] = (
            "iTunes_Chinese_commercial_candidate;"
            f"query_count={row['itunes_query_count']};"
            f"provisional_genre_basis={genre_basis};"
            "regional_catalogue_or_Han_metadata_language_basis;"
            "strict_Chinese_language_and_final_genre_verification_required_in_Cell5B"
        )
        row.pop("language_screen_passed", None)
        final_rows.append(row)

    output = pd.DataFrame(final_rows)

    pd.DataFrame(raw_rows).to_csv(
        CHINESE_RAW_OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig",
    )
    pd.DataFrame(rejected_rows).to_csv(
        CHINESE_REJECTED_OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig",
    )
    pd.DataFrame(query_logs).to_csv(
        CHINESE_QUERY_LOG_CSV,
        index=False,
        encoding="utf-8-sig",
    )
    save_json_cache(
        CHINESE_ITUNES_CACHE,
        CHINESE_CACHE_FILE,
    )

    return output


def collect_chinese_apple_charts() -> pd.DataFrame:
    if not RUN_CHINESE_APPLE_CHARTS:
        return pd.DataFrame()

    rows: List[Dict[str, Any]] = []

    for storefront in tqdm(
        CHINESE_APPLE_STOREFRONTS,
        desc="Chinese Apple charts",
        unit="storefront",
        dynamic_ncols=True,
        leave=True,
    ):
        url = (
            "https://rss.marketingtools.apple.com/"
            "api/v2/"
            f"{storefront}/music/most-played/"
            f"{CHINESE_APPLE_CHART_LIMIT}/songs.json"
        )

        try:
            response = SESSION.get(
                url,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
        except requests.RequestException as error:
            log_event(
                "APPLE_CHART_FAILED",
                storefront=storefront,
                error_type=type(error).__name__,
            )
            continue

        if response.status_code != 200:
            log_event(
                "APPLE_CHART_FAILED",
                storefront=storefront,
                status=response.status_code,
            )
            continue

        try:
            payload = response.json()
        except ValueError:
            log_event(
                "APPLE_CHART_FAILED",
                storefront=storefront,
                status="invalid_json",
            )
            continue

        results = (
            payload.get("feed", {})
            .get("results", [])
        )

        accepted = 0

        for rank, item in enumerate(
            results,
            start=1,
        ):
            if not isinstance(item, dict):
                continue

            title = clean_html(item.get("name"))
            artist = clean_html(
                item.get("artistName")
            )
            collection = clean_html(
                item.get("collectionName")
            )
            release_year = parse_release_year(
                item.get("releaseDate")
            )

            if not title or not artist:
                continue

            if release_year is None or not (
                MIN_RELEASE_YEAR
                <= release_year
                <= MAX_RELEASE_YEAR
            ):
                continue

            language_ok, language_basis = (
                chinese_language_basis(
                    title,
                    artist,
                    collection,
                    storefront,
                )
            )
            if not language_ok:
                continue

            reason = exclusion_reason(
                title,
                artist,
                collection,
                "",
            )
            if reason:
                continue

            raw_genres: List[str] = []
            genres = item.get("genres", [])

            if isinstance(genres, list):
                raw_genres = [
                    clean_html(genre.get("name"))
                    for genre in genres
                    if isinstance(genre, dict)
                    and clean_html(
                        genre.get("name")
                    )
                ]

            provisional_genre, genre_basis = (
                choose_provisional_genre(
                    raw_genres,
                    [],
                )
            )

            rows.append(
                {
                    "language": "Chinese",
                    "language_code": "zh",
                    "label": "Human",
                    "model": "Human",
                    "source_dataset": (
                        "Apple Music Official Chart"
                    ),
                    "title": title,
                    "artist": artist,
                    "collection": collection,
                    "album": collection,
                    "final_genre": provisional_genre,
                    "raw_genre": list_to_pipe(
                        raw_genres
                    ),
                    "search_query": (
                        "official_most_played_chart"
                    ),
                    "query_genre_group": "",
                    "country": storefront.upper(),
                    "release_year": release_year,
                    "release_date": safe_text(
                        item.get("releaseDate")
                    ),
                    "duration_seconds": "",
                    "source_track_id": safe_text(
                        item.get("id")
                    ),
                    "track_view_url": safe_text(
                        item.get("url")
                    ),
                    "chart_rank": rank,
                    "commercial_catalogue_score": 8,
                    "commercial_catalogue_match": True,
                    "candidate_language_verified": False,
                    "requires_cell5b_language_verification": True,
                    "requires_cell5b_genre_verification": True,
                    "candidate_language_basis": language_basis,
                    "candidate_genre_basis": genre_basis,
                    "verification_status": (
                        "apple_chinese_chart_candidate_pending_cell5b"
                    ),
                    "audio_path": "",
                    "preview_url": "",
                    "youtube_url": "",
                    "youtube_video_id": "",
                    "youtube_channel": "",
                    "youtube_views": 0,
                    "youtube_likes": 0,
                    "artist_source": (
                        "apple_chart_artist_metadata"
                    ),
                    "notes": (
                        "Apple_Music_official_chart_candidate;"
                        "chart_inclusion_is_commercial_evidence;"
                        "strict_Chinese_language_and_final_genre_verification_required_in_Cell5B"
                    ),
                }
            )
            accepted += 1

        log_event(
            "APPLE_CHART_COMPLETE",
            storefront=storefront,
            returned_rows=len(results),
            accepted_rows=accepted,
            cumulative_rows=len(rows),
        )

    output = pd.DataFrame(rows)
    output.to_csv(
        CHINESE_APPLE_CHART_CSV,
        index=False,
        encoding="utf-8-sig",
    )
    return output


def load_existing_chinese_candidates() -> pd.DataFrame:
    if not KEEP_EXISTING_ITUNES_ROWS:
        return pd.DataFrame()

    if not CHINESE_MAIN_OUTPUT_CSV.exists():
        return pd.DataFrame()

    try:
        existing = pd.read_csv(
            CHINESE_MAIN_OUTPUT_CSV,
            low_memory=False,
        )
    except Exception as error:
        log_event(
            "CHINESE_EXISTING_LOAD_FAILED",
            error_type=type(error).__name__,
        )
        return pd.DataFrame()

    if existing.empty:
        return existing

    source_text = existing.get(
        "source_dataset",
        pd.Series("", index=existing.index),
    ).astype(str)

    keep = source_text.str.contains(
        r"iTunes Chinese Search Metadata|Apple Music Official Chart",
        case=False,
        regex=True,
        na=False,
    )

    return existing[keep].copy()


def merge_chinese_candidate_pool(
    frames: Sequence[pd.DataFrame],
) -> pd.DataFrame:
    parts = [
        frame.copy()
        for frame in frames
        if frame is not None and not frame.empty
    ]

    if not parts:
        return pd.DataFrame()

    combined = pd.concat(
        parts,
        ignore_index=True,
        sort=False,
    )

    combined["_dedupe_key"] = combined.apply(
        lambda row: track_dedupe_key(
            row.get("title", ""),
            row.get("artist", ""),
        ),
        axis=1,
    )

    combined = combined[
        combined["_dedupe_key"].astype(str).str.strip()
        != "||"
    ].copy()

    output_rows: List[Dict[str, Any]] = []

    for _, group in combined.groupby(
        "_dedupe_key",
        sort=False,
    ):
        group = group.copy()

        group["_priority"] = (
            group.get(
                "source_dataset",
                pd.Series("", index=group.index),
            )
            .astype(str)
            .apply(
                lambda value: (
                    0
                    if "iTunes Chinese Search Metadata"
                    in value
                    else 1
                )
            )
        )

        group["_score"] = pd.to_numeric(
            group.get(
                "commercial_catalogue_score",
                0,
            ),
            errors="coerce",
        ).fillna(0)

        group = group.sort_values(
            by=["_priority", "_score"],
            ascending=[True, False],
        )

        base = group.iloc[0].to_dict()

        sources = unique_nonempty(
            group.get(
                "source_dataset",
                pd.Series(dtype=object),
            ).tolist()
        )
        raw_genres = unique_nonempty(
            group.get(
                "raw_genre",
                pd.Series(dtype=object),
            ).tolist()
        )
        queries = unique_nonempty(
            group.get(
                "search_query",
                pd.Series(dtype=object),
            ).tolist()
        )
        countries = unique_nonempty(
            group.get(
                "country",
                pd.Series(dtype=object),
            ).tolist()
        )

        query_genres: List[str] = []

        for value in group.get(
            "query_genre_group",
            pd.Series(dtype=object),
        ).tolist():
            for token in re.split(
                r"[,|]",
                safe_text(value),
            ):
                token = token.strip()
                if token in TARGET_GENRES:
                    query_genres.append(token)

        provisional_genre, genre_basis = (
            choose_provisional_genre(
                raw_genres,
                query_genres,
            )
        )

        years = pd.to_numeric(
            group.get(
                "release_year",
                pd.Series(dtype=object),
            ),
            errors="coerce",
        ).dropna()

        durations = pd.to_numeric(
            group.get(
                "duration_seconds",
                pd.Series(dtype=object),
            ),
            errors="coerce",
        ).dropna()

        chart_ranks = pd.to_numeric(
            group.get(
                "chart_rank",
                pd.Series(dtype=object),
            ),
            errors="coerce",
        ).dropna()

        base["language"] = "Chinese"
        base["language_code"] = "zh"
        base["source_dataset"] = (
            " + ".join(sources)
        )
        base["candidate_sources"] = (
            ", ".join(sources)
        )
        base["source_count"] = len(sources)
        base["raw_genre"] = list_to_pipe(
            raw_genres
        )
        base["search_query"] = list_to_pipe(
            queries
        )
        base["query_genre_group"] = (
            list_to_pipe(query_genres)
        )
        base["country"] = list_to_pipe(
            countries
        )
        base["final_genre"] = provisional_genre
        base["candidate_genre_basis"] = (
            genre_basis
        )
        base["release_year"] = (
            int(years.min())
            if not years.empty
            else ""
        )
        base["duration_seconds"] = (
            round(float(durations.median()), 2)
            if not durations.empty
            else ""
        )
        base["chart_rank"] = (
            int(chart_ranks.min())
            if not chart_ranks.empty
            else ""
        )
        base["candidate_language_verified"] = False
        base["requires_cell5b_language_verification"] = True
        base["requires_cell5b_genre_verification"] = True
        base["verification_status"] = (
            "chinese_commercial_candidate_pending_cell5b"
        )
        base["notes"] = (
            f"candidate_sources={','.join(sources)};"
            f"source_count={len(sources)};"
            f"provisional_genre_basis={genre_basis};"
            "iTunes_and_Apple_commercial_catalogue_evidence;"
            "strict_Chinese_language_and_final_genre_verification_required_in_Cell5B"
        )

        for temporary in [
            "_dedupe_key",
            "_priority",
            "_score",
        ]:
            base.pop(temporary, None)

        output_rows.append(base)

    output = pd.DataFrame(output_rows)

    if output.empty:
        return output

    genre_order = {
        genre: index
        for index, genre in enumerate(
            TARGET_GENRES + ["Unclassified"]
        )
    }

    output["_genre_order"] = (
        output["final_genre"]
        .map(genre_order)
        .fillna(len(genre_order))
    )
    output["_source_count"] = pd.to_numeric(
        output.get("source_count", 1),
        errors="coerce",
    ).fillna(1)
    output["_chart_rank"] = pd.to_numeric(
        output.get("chart_rank", 999999),
        errors="coerce",
    ).fillna(999999)

    output = (
        output.sort_values(
            by=[
                "_genre_order",
                "_source_count",
                "_chart_rank",
                "artist",
                "title",
            ],
            ascending=[
                True,
                False,
                True,
                True,
                True,
            ],
        )
        .drop(
            columns=[
                "_genre_order",
                "_source_count",
                "_chart_rank",
            ],
            errors="ignore",
        )
        .reset_index(drop=True)
    )

    return output


def collect_chinese_famous_artist_candidates() -> pd.DataFrame:
    """
    ID-based catalogue lookup for FAMOUS_CHINESE_ARTIST_SEEDS.

    Mirrors collect_hindi_itunes_candidates() exactly:
      1. Resolve each seed name to an exact iTunes musicArtist identity.
      2. Pull the full song catalogue via /lookup?id=...
      3. Run the same year / duration / script / version / exclusion filters.
      4. Use the Apple China/TW/HK chart signal instead of the India chart.

    Every accepted track still requires Cell 5B Chinese-language and
    genre verification.
    """
    # Re-use the Apple China chart signal (CN storefront) as popularity evidence.
    cn_chart_url = (
        "https://rss.marketingtools.apple.com/"
        f"api/v2/cn/music/most-played/{CHINESE_APPLE_CHART_LIMIT}/songs.json"
    )
    apple_chart_keys: set[str] = set()
    try:
        resp = SESSION.get(cn_chart_url, timeout=REQUEST_TIMEOUT_SECONDS)
        if resp.status_code == 200:
            chart_results = resp.json().get("feed", {}).get("results", [])
            for item in chart_results:
                if isinstance(item, dict):
                    for name in split_artist_names(
                        clean_html(item.get("artistName"))
                    ):
                        apple_chart_keys.add(
                            track_dedupe_key(
                                clean_html(item.get("name")), name
                            )
                        )
    except Exception:
        pass

    records: Dict[str, Dict[str, Any]] = {}

    artist_tasks: List[Tuple[str, str]] = [
        (genre, artist_name)
        for genre, artist_names in FAMOUS_CHINESE_ARTIST_SEEDS.items()
        for artist_name in artist_names
    ]

    seen: set[str] = set()
    unique_tasks: List[Tuple[str, str]] = []
    for genre, name in artist_tasks:
        key = normalise_text(name)
        if key and key not in seen:
            seen.add(key)
            unique_tasks.append((genre, name))

    print("\n" + "=" * 72)
    print("CHINESE FAMOUS-ARTIST ID-CATALOGUE DISCOVERY")
    print("=" * 72)
    print(f"Curated Chinese artist tasks: {len(unique_tasks)}")

    for seed_genre, seed_artist in tqdm(
        unique_tasks,
        desc="Chinese famous artists",
        unit="artist",
        dynamic_ncols=True,
        leave=True,
    ):
        identity, identity_status = request_itunes_artist_identity(seed_artist)
        if not identity:
            log_event(
                "CHINESE_ARTIST_NOT_RESOLVED",
                genre=seed_genre,
                artist=seed_artist,
                status=identity_status,
            )
            continue

        artist_id = safe_text(identity.get("artistId"))
        resolved_name = clean_html(identity.get("artistName"))

        catalogue, _ = request_itunes_artist_catalogue_by_id(artist_id)
        ranked_results, _ = request_itunes_artist_catalogue(
            resolved_name or seed_artist
        )
        rank_map = build_ranked_itunes_track_map(
            ranked_results,
            resolved_artist_id=artist_id,
            resolved_artist_name=resolved_name or seed_artist,
        )

        accepted_for_artist = 0

        for catalogue_rank, item in enumerate(catalogue, start=1):
            returned_id = safe_text(item.get("artistId"))
            returned_artist = clean_html(item.get("artistName"))

            if returned_id and artist_id and returned_id != artist_id:
                continue
            if not exact_artist_match(resolved_name or seed_artist, returned_artist):
                continue

            title = clean_html(item.get("trackName"))
            collection = clean_html(item.get("collectionName"))
            raw_genre = clean_html(item.get("primaryGenreName"))
            release_year = parse_release_year(item.get("releaseDate"))
            duration_seconds = parse_duration_seconds(item.get("trackTimeMillis"))
            search_rank = ranked_itunes_position(item, rank_map)

            chart_key = track_dedupe_key(title, seed_artist)
            is_chart_track = chart_key in apple_chart_keys

            # ── Filters ──────────────────────────────────────────────────────
            reason = ""
            if not title or not returned_artist:
                reason = "missing_title_or_artist"
            elif normalise_text(item.get("wrapperType")) != "track":
                reason = "itunes_wrapper_type_not_track"
            elif normalise_text(item.get("kind")) not in {"song", ""}:
                reason = "itunes_kind_not_song"
            elif not safe_text(item.get("trackViewUrl")):
                reason = "missing_itunes_track_view_url"
            elif release_year is None:
                reason = "missing_release_year"
            elif not (MIN_RELEASE_YEAR <= int(release_year) <= MAX_RELEASE_YEAR):
                reason = "release_year_outside_configured_range"
            elif duration_seconds is None or not (
                MIN_DURATION_SECONDS <= float(duration_seconds) <= MAX_DURATION_SECONDS
            ):
                reason = "duration_outside_chinese_range"
            elif contains_kana_or_hangul(f"{title} {collection}"):
                reason = "japanese_or_korean_script_detected"
            elif not contains_han(f"{title} {returned_artist} {collection}"):
                # Require at least one Han character as positive language evidence.
                if not (
                    contains_han(title)
                    or contains_han(returned_artist)
                    or contains_han(collection)
                ):
                    reason = "no_chinese_script_evidence"
            else:
                reason = exclusion_reason(title, returned_artist, collection, raw_genre)

            if reason:
                continue

            #Genre mapping 
            mapped_genre, mapped_basis = map_specific_itunes_genre(raw_genre)
            if not mapped_genre:
                provisional_genre = seed_genre
                genre_basis = "curated_chinese_artist_genre_pending_cell5b"
                genre_conflict = False
            elif mapped_genre == seed_genre:
                provisional_genre = seed_genre
                genre_basis = mapped_basis
                genre_conflict = False
            else:
                provisional_genre = "Unclassified"
                genre_basis = (
                    f"genre_conflict_seed_{seed_genre}_itunes_{mapped_genre}"
                )
                genre_conflict = True

            # Scoring 
            rank_points = (
                max(0, ITUNES_RANK_MAX_POINTS - int(search_rank) + 1)
                if search_rank is not None
                else 0
            )
            chart_points = APPLE_CHART_TRACK_BONUS if is_chart_track else 0
            recent_points = max(
                0,
                min(RECENT_RELEASE_MAX_POINTS, int(release_year) - MIN_RELEASE_YEAR + 1),
            )
            score = chart_points + rank_points + recent_points + commercial_score(item) * 2

            row = normalise_chinese_itunes_result(
                item,
                query=f"CHINESE_ARTIST_ID::{seed_artist}",
                query_genre=seed_genre,
                country=safe_text(item.get("country")) or "CN",
                result_rank=search_rank if search_rank is not None else catalogue_rank,
            )
            row["final_genre"] = provisional_genre
            row["candidate_genre_basis"] = genre_basis
            row["seed_genre"] = seed_genre
            row["seed_artist"] = seed_artist
            row["famous_artist_resolved_name"] = resolved_name
            row["famous_artist_itunes_id"] = artist_id
            row["official_apple_china_chart_match"] = is_chart_track
            row["genre_conflict"] = genre_conflict
            row["high_precision_score"] = score
            row["artist_source"] = "itunes_only_curated_famous_chinese_artist"
            row["source_dataset"] = (
                "iTunes Famous Chinese Artist ID Catalogue + Ranked Artist Search"
            )
            row["notes"] = (
                "iTunes_only_metadata;"
                "exact_Chinese_artist_identity;"
                "artist_ID_catalogue;"
                f"genre_basis={genre_basis};"
                "strict_Chinese_language_and_final_genre_"
                "verification_required_in_Cell5B"
            )

            dedupe_key = (
                safe_text(item.get("trackId"))
                or track_dedupe_key(title, returned_artist)
            )
            existing = records.get(dedupe_key)
            if existing is None or score > float(
                existing.get("high_precision_score", 0)
            ):
                records[dedupe_key] = row

            accepted_for_artist += 1

        log_event(
            "CHINESE_FAMOUS_ARTIST_COMPLETE",
            genre=seed_genre,
            artist=seed_artist,
            resolved=resolved_name,
            accepted=accepted_for_artist,
            cumulative=len(records),
        )

    output = pd.DataFrame(list(records.values()))
    print(
        f"Chinese famous-artist catalogue: {len(output)} candidates "
        f"from {len(unique_tasks)} artists"
    )
    return output


def run_chinese_collection() -> pd.DataFrame:
    existing = load_existing_chinese_candidates()
    direct = collect_chinese_itunes_candidates()
    charts = collect_chinese_apple_charts()
    famous = collect_chinese_famous_artist_candidates()

    merged = merge_chinese_candidate_pool(
        [existing, direct, charts, famous]
    )

    if merged.empty:
        raise RuntimeError(
            "Chinese iTunes/Apple collection returned no usable candidates. "
            "The main Chinese CSV was not overwritten."
        )

    # Enforce the same 2–7 minute duration rule after merging.
    # This also removes chart-only rows with no duration unless an
    # iTunes match supplied a valid duration.
    merged["_duration_numeric"] = pd.to_numeric(
        merged.get("duration_seconds"),
        errors="coerce",
    )

    chinese_before_duration_filter = len(merged)

    merged = merged[
        merged["_duration_numeric"].between(
            MIN_DURATION_SECONDS,
            MAX_DURATION_SECONDS,
            inclusive="both",
        )
    ].copy()

    merged["duration_seconds"] = (
        merged["_duration_numeric"].round(2)
    )

    merged = merged.drop(
        columns=["_duration_numeric"],
        errors="ignore",
    ).reset_index(drop=True)

    print(
        "Chinese duration filter 120–420 seconds:",
        f"{chinese_before_duration_filter} -> {len(merged)}",
        flush=True,
    )

    if merged.empty:
        raise RuntimeError(
            "All Chinese candidates were removed by the "
            "120–420 second duration rule."
        )

    merged.to_csv(
        CHINESE_MAIN_OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    return merged



# EXECUTE  ORIGINAL VARIABLE AND CSV NAMES

print("\n" + "=" * 80)
print("COMBINED CELL 4 START", flush=True)
print("=" * 80)
print("Stage 1/2: Hindi iTunes + query-based MusicBrainz/Last.fm", flush=True)
print(
    "QUERY_EXTERNAL_MODE | "
    "15 genre queries; max 8 iTunes attempts and "
    "3 accepted tracks per query",
    flush=True,
)

hindi_itunes_candidates = collect_hindi_itunes_candidates()

print("\nStage 2/2: Chinese iTunes + Apple charts — duration 2–7 min", flush=True)

chinese_itunes_candidates = run_chinese_collection()

save_json_cache(
    ITUNES_CACHE,
    CACHE_FILE,
)
save_json_cache(
    CHINESE_ITUNES_CACHE,
    CHINESE_CACHE_FILE,
)
save_json_cache(
    MUSICBRAINZ_CACHE,
    MUSICBRAINZ_CACHE_FILE,
)
save_json_cache(
    LASTFM_CACHE,
    LASTFM_CACHE_FILE,
)
save_json_cache(
    SOUNDCLOUD_CACHE,
    SOUNDCLOUD_CACHE_FILE,
)


# REPORT

print("\n" + "=" * 80)
print("COMBINED CELL 4 COMPLETE", flush=True)
print("=" * 80)

print("\nHindi provisional genre counts")
print("-" * 40)
print(
    hindi_itunes_candidates["final_genre"]
    .value_counts()
    .reindex(
        TARGET_GENRES + ["Unclassified"],
        fill_value=0,
    )
)

print("\nChinese provisional genre counts")
print("-" * 40)
print(
    chinese_itunes_candidates["final_genre"]
    .value_counts()
    .reindex(
        TARGET_GENRES + ["Unclassified"],
        fill_value=0,
    )
)

print(
    "\nHindi total:",
    len(hindi_itunes_candidates),
)
print(
    "Chinese total:",
    len(chinese_itunes_candidates),
)

print("\nHindi CSV:", MAIN_OUTPUT_CSV)
print("Chinese CSV:", CHINESE_MAIN_OUTPUT_CSV)

print("\nHindi audit files:")
print(" ", RAW_OUTPUT_CSV)
print(" ", REJECTED_OUTPUT_CSV)
print(" ", QUERY_LOG_CSV)
print(" ", ARTIST_PROFILE_CSV)
print(" ", HINDI_APPLE_CHART_ARTISTS_CSV)
print(" ", HINDI_FAMOUS_ARTIST_LOOKUP_LOG_CSV)
print(" ", HINDI_CROSS_SOURCE_MATCH_CSV)
print(" ", HINDI_SHORTAGE_REPORT_CSV)
print(" ", HINDI_EXTERNAL_ARTIST_CSV)
print(" ", HINDI_EXTERNAL_DISCOVERY_CSV)

print("\nChinese audit files:")
print(" ", CHINESE_RAW_OUTPUT_CSV)
print(" ", CHINESE_REJECTED_OUTPUT_CSV)
print(" ", CHINESE_QUERY_LOG_CSV)
print(" ", CHINESE_APPLE_CHART_CSV)

print(
    "\nNEXT STEP | Run Cell 5B on the high-precision iTunes-only Hindi pool for strict Hindi/Punjabi filtering, "
    "strict Chinese-language verification, and final exact-track genres."
)

display(
    hindi_itunes_candidates[
        [
            "title",
            "artist",
            "collection",
            "final_genre",
            "raw_genre",
            "release_year",
            "candidate_genre_basis",
        ]
    ].head(20)
)

display(
    chinese_itunes_candidates[
        [
            "title",
            "artist",
            "collection",
            "final_genre",
            "raw_genre",
            "release_year",
            "country",
            "candidate_genre_basis",
        ]
    ].head(20)
)

In [ ]:
#
#  COLLECTOR V2 (KAGGLE + GENIUS API INTEGRATED)

import pandas as pd
import kagglehub
import requests
import getpass
from pathlib import Path
from tqdm.auto import tqdm

# Setup
BASE_DIR = Path("dataset_collection")
BASE_DIR.mkdir(exist_ok=True)

# 1. REGISTRIES
KAGGLE_SOURCES = [
    ("Hindi", "asacker/all-bollywood-songs-and-albums-dataset-19312025", ["song_title"], ["singers"]),
    ("Hindi", "ranirathore/bollywood-hindi-songs", ["title"], ["artist"]),
    ("Hindi", "makvel/indian-hindi-songs-lyrics-dataset", ["title"], ["artist"]),
    ("Hindi", "pranavinani/desi-hip-hop-lyrics-verses-reverse-prompt", ["title"], ["artist"]),
    ("Hindi", "unnatgupta/songs-dataset", ["name"], ["artist"]),
    ("Chinese", "jyesawtellrickson/chinese-song-lyrics", ["title"], ["author"]),
]

# Seed artists to search on Genius (Expand this list as needed)
GENIUS_SEED_ARTISTS = {
    "Hindi": ["Arijit Singh", "Rahat Fateh Ali Khan", "Shreya Ghoshal", "DIVINE", "Seedhe Maut", "King"],
    "Chinese": ["Jay Chou", "JJ Lin", "G.E.M.", "Eason Chan", "Higher Brothers"]
}

# 2. INGESTION FUNCTIONS
def ingest_kaggle():
    for lang, slug, t_hints, a_hints in KAGGLE_SOURCES:
        print(f"\n>>> Collecting: {slug}")
        try:
            path = kagglehub.dataset_download(slug)
            csv_path = max(Path(path).rglob("*.csv"), key=lambda p: p.stat().st_size)
            df = pd.read_csv(csv_path, low_memory=False)
            
            t_col = next((c for c in df.columns if any(h in c.lower() for h in t_hints)), None)
            a_col = next((c for c in df.columns if any(h in c.lower() for h in a_hints)), None)
            
            if t_col and a_col:
                res = df[[t_col, a_col]].rename(columns={t_col: "title", a_col: "artist"})
                res["language"] = lang
                res["source"] = slug
                res.to_csv(BASE_DIR / f"{lang.lower()}_raw.csv", mode='a', index=False, header=not (BASE_DIR / f"{lang.lower()}_raw.csv").exists())
                print(f"Added {len(res)} tracks to {lang} pool.")
        except Exception as e:
            print(f"Skipping {slug}: {e}")

def ingest_genius(api_key):
    if not api_key: return
    for lang, artists in GENIUS_SEED_ARTISTS.items():
        print(f"\n>>> Searching Genius for {lang} artists...")
        all_hits = []
        for artist in tqdm(artists):
            url = "https://api.genius.com/search"
            headers = {"Authorization": f"Bearer {api_key}"}
            try:
                r = requests.get(url, params={"q": artist}, headers=headers).json()
                hits = r.get("response", {}).get("hits", [])
                for h in hits:
                    all_hits.append({"title": h["result"]["title"], "artist": h["result"]["primary_artist"]["name"], "language": lang, "source": "genius_api"})
            except: continue
        
        if all_hits:
            pd.DataFrame(all_hits).to_csv(BASE_DIR / f"{lang.lower()}_raw.csv", mode='a', index=False, header=not (BASE_DIR / f"{lang.lower()}_raw.csv").exists())
            print(f"Added {len(all_hits)} Genius tracks to {lang} pool.")

# 3. MAIN EXECUTION
ingest_kaggle()

# Ask for key securely
genius_key = getpass.getpass("Enter Genius API Key (or leave blank to skip): ")
ingest_genius(genius_key)

print("\nCollection complete. Now run the Purge and Balancing cells.")

In [ ]:
print("\nPreview of Hindi Candidates:")
display(hindi_itunes_candidates.head())

print("\nPreview of Chinese Candidates:")
display(chinese_itunes_candidates.head())

## 5. English candidate verification

English SONICS metadata is externally checked to build a verified buffer larger than 60 songs per genre.


## SONICS External Verification (English)

> **Note on Deprecated Code:** The original bulk iTunes search function for English tracks has been removed because it frequently triggered HTTP 429 Rate Limit blocks. We now strictly use this `build_english_candidates_from_sonics_safe` function, which relies on Last.fm and Discogs metadata to verify SONICS genres reliably.

In [ ]:
# ENGLISH REAL CSV + LAST.FM + iTUNES STRICT VERIFICATION
# UPDATED STRICT VERSION v2
# Source = real_songs.csv only
# No artist fallback
# Buffer = 75 per genre
# Metadata only
# Strict rule: Last.fm genre MUST equal iTunes genre
#   (with a relaxed "Alternative -> Folk/Acoustic" allowance,
#    ONLY when Last.fm already strongly says Folk/Acoustic)
# NEW: targeted Folk/Acoustic pass using Last.fm tag.getTopTracks
#      so Folk doesn't rely on slow random scanning
# NOTHING is deleted. Existing strict CSV is loaded and appended to.
# ============================================================

import pandas as pd
import requests
import time
import json
import re
import shutil
from pathlib import Path
from tqdm.auto import tqdm
from getpass import getpass
from datetime import datetime

# CONFIG

LASTFM_URL = "https://ws.audioscrobbler.com/2.0/"
ITUNES_SEARCH_URL = "https://itunes.apple.com/search"

SONICS_REAL_CSV = Path("real_songs.csv")

ENGLISH_TARGET_BUFFER = 65   # This now represents the ARTIST-CAP-SAFE count needed
                             # (i.e. how many tracks would survive Cell 6's max-2-per-artist
                             # rule). Small margin above Cell 6's TARGET_PER_GENRE=60.
MIN_LASTFM_LISTENERS = 1000
MAX_ROWS_TO_SCAN = 48090

# Must match Cell 6's "max 2 songs per artist per genre" rule.
# Enforcing this DURING collection means the buffer count we reach here
# already respects artist diversity, so Cell 6 won't shrink it later.
MAX_SONGS_PER_ARTIST_PER_GENRE = 2

LASTFM_CACHE_FILE = MANIFEST_DIR / "lastfm_english_track_cache.json"
ENGLISH_OUTPUT_CSV = MANIFEST_DIR / "english_real_lastfm_itunes_strict_candidates.csv"

LASTFM_SLEEP = 0.08
ITUNES_SLEEP = 0.08

# Last.fm tags to pull top tracks from, for the TARGETED Folk pass.
# Broad on purposewe still re-verify every track through the same
# strict Last.fm + iTunes pipeline below, this just gives us candidates
# that are already likely to be Folk instead of scanning randomly.
FOLK_TAG_QUERY_LIST = [
    "folk",
    "acoustic",
    "singer-songwriter",
    "americana",
    "country",
    "indie folk",
    "folk rock",
    "bluegrass",
    "country rock",
]

FOLK_TOPTRACKS_PAGES_PER_TAG = 4   # ~50 tracks per page from Last.fm



# BASIC FALLBACKS

try:
    GENRES
except NameError:
    GENRES = ["Pop", "Rock", "Hip-Hop/Rap", "R&B/Soul", "Folk/Acoustic"]


try:
    normalise_for_match
except NameError:
    def normalise_for_match(text):
        text = str(text).lower()
        text = re.sub(r"\(.*?\)|\[.*?\]", " ", text)
        text = re.sub(r"[^a-z0-9\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text


# API KEY

try:
    LASTFM_API_KEY
except NameError:
    LASTFM_API_KEY = ""

if not LASTFM_API_KEY:
    LASTFM_API_KEY = getpass("Enter your Last.fm API key: ")

if not LASTFM_API_KEY:
    raise ValueError("LASTFM_API_KEY missing.")


# NO DELETE, ONLY BACKUP

print("No files will be deleted. Existing strict CSV will be resumed safely.")

if ENGLISH_OUTPUT_CSV.exists():
    backup_csv = MANIFEST_DIR / f"backup_before_strict_tags_rerun_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    shutil.copy2(ENGLISH_OUTPUT_CSV, backup_csv)
    print("Backup created:", backup_csv)


# CACHE

def load_lastfm_cache():
    if LASTFM_CACHE_FILE.exists():
        try:
            with open(LASTFM_CACHE_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}


def save_lastfm_cache(cache):
    with open(LASTFM_CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)


LASTFM_TRACK_CACHE = load_lastfm_cache()

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 MScMusicDataset/1.0"})


# GENRE MAPPING - STRICT BUT MORE FOLK TAGS

def map_lastfm_tags_strict(tags):
    """
    Strict Last.fm track-tag mapping.
    More Folk/Country/Acoustic tags added.
    But final acceptance still needs Last.fm == iTunes (or the
    relaxed alternative-as-folk allowance below).
    """

    tags = [str(t).lower().strip() for t in tags if str(t).strip()]

    scores = {
        "Pop": 0,
        "Rock": 0,
        "Hip-Hop/Rap": 0,
        "R&B/Soul": 0,
        "Folk/Acoustic": 0,
    }

    for i, tag in enumerate(tags):
        weight = max(1, 20 - i)

        # Hip-Hop / Rap
        if any(x in tag for x in [
            "hip hop", "hip-hop", "rap", "trap", "drill", "grime"
        ]):
            scores["Hip-Hop/Rap"] += weight

        # R&B / Soul
        if any(x in tag for x in [
            "r&b", "rnb", "rhythm and blues", "soul",
            "neo soul", "contemporary r&b"
        ]):
            scores["R&B/Soul"] += weight

        # Folk / Acoustic / Country
        if any(x in tag for x in [
            "folk",
            "acoustic",
            "acoustic pop",
            "acoustic rock",
            "singer-songwriter",
            "singer songwriter",
            "singer/songwriter",
            "americana",
            "country",
            "country pop",
            "country rock",
            "contemporary country",
            "traditional country",
            "classic country",
            "alt-country",
            "alternative country",
            "bluegrass",
            "roots",
            "roots rock",
            "new americana",
            "indie folk",
            "folk rock",
            "folk pop",
            "contemporary folk",
            "traditional folk",
            "neo-folk",
            "chamber folk",
            "indie acoustic",
            "unplugged",
            "banjo",
            "mandolin",
            "old-time",
            "old time",
            "outlaw country",
            "nashville",
            "honky tonk",
            "honky-tonk",
            "southern rock",
            "country folk",
        ]):
            scores["Folk/Acoustic"] += int(weight * 1.4)

        # Rock
        if any(x in tag for x in [
            "rock", "punk", "metal", "grunge", "garage rock",
            "hard rock", "indie rock", "alternative rock"
        ]):
            scores["Rock"] += weight

        # Pop
        if any(x in tag for x in [
            "pop", "synthpop", "synth pop", "electropop",
            "dance pop", "dance-pop", "art pop"
        ]):
            scores["Pop"] += weight

        # Electronic-style songs can go to Pop bucket
        if any(x in tag for x in [
            "electronic", "electronica", "synth", "dance"
        ]):
            scores["Pop"] += int(weight * 0.7)

    best_genre = max(scores, key=scores.get)
    best_score = scores[best_genre]

    if best_score == 0:
        return None, scores

    sorted_scores = sorted(scores.values(), reverse=True)

    # strict tie rejection
    if len(sorted_scores) > 1 and sorted_scores[0] == sorted_scores[1]:
        return None, scores

    return best_genre, scores


def map_itunes_genre_strict(raw_genre, allow_alternative_as_folk=False):
    """
    Strict iTunes mapping.
    More Folk/Country labels included.

    allow_alternative_as_folk:
        iTunes often dumps genuine folk / singer-songwriter / country-folk
        songs under the generic "Alternative" label. Normally we return
        None for "alternative" because it's too messy to trust on its own.
        When this flag is True (only used when Last.fm has ALREADY scored
        the track strongly as Folk/Acoustic), we let "alternative" count
        as agreeing with Folk/Acoustic instead of rejecting the track.
    """

    g = str(raw_genre).lower().strip()

    if not g or g in ["nan", "none"]:
        return None

    if "hip-hop" in g or "hip hop" in g or "rap" in g:
        return "Hip-Hop/Rap"

    if "r&b" in g or "rnb" in g or "soul" in g:
        return "R&B/Soul"

    if any(x in g for x in [
        "singer/songwriter",
        "singer-songwriter",
        "folk",
        "alternative folk",
        "country",
        "americana",
        "bluegrass",
        "traditional country",
        "contemporary country",
        "classic country",
        "country pop",
        "country rock",
        "alt-country",
        "alternative country",
        "roots",
        "outlaw country",
        "nashville",
        "honky tonk",
        "honky-tonk",
    ]):
        return "Folk/Acoustic"

    if "rock" in g or "metal" in g or "punk" in g:
        return "Rock"

    # iTunes "Alternative" is messy - normally we refuse to guess.
    # But if Last.fm already strongly said Folk/Acoustic, let it agree.
    if g == "alternative" or ("alternative" in g and "rock" not in g):
        if allow_alternative_as_folk:
            return "Folk/Acoustic"
        return None

    if "pop" in g or "dance" in g or "electronic" in g:
        return "Pop"

    return None


# LAST.FM TRACK INFO

def get_lastfm_track_info(artist, title):
    """
    Track-level Last.fm verification only.
    No artist fallback.
    Returns tags, listeners, status.
    """

    cache_key = normalise_for_match(artist) + " | " + normalise_for_match(title)

    if cache_key in LASTFM_TRACK_CACHE:
        item = LASTFM_TRACK_CACHE[cache_key]
        return (
            item.get("tags", []),
            int(item.get("listeners", 0) or 0),
            item.get("status", "cached")
        )

    try:
        info_r = session.get(
            LASTFM_URL,
            params={
                "method": "track.getInfo",
                "artist": artist,
                "track": title,
                "api_key": LASTFM_API_KEY,
                "format": "json",
                "autocorrect": 1,
            },
            timeout=15
        )

        time.sleep(LASTFM_SLEEP)

        listeners = 0
        tags = []

        if info_r.status_code == 200:
            track_info = info_r.json().get("track", {})
            listeners = int(track_info.get("listeners", 0) or 0)

            tag_items = track_info.get("toptags", {}).get("tag", [])
            tags = [
                t.get("name", "").lower()
                for t in tag_items[:20]
                if t.get("name")
            ]

        if not tags:
            tag_r = session.get(
                LASTFM_URL,
                params={
                    "method": "track.getTopTags",
                    "artist": artist,
                    "track": title,
                    "api_key": LASTFM_API_KEY,
                    "format": "json",
                    "autocorrect": 1,
                },
                timeout=15
            )

            time.sleep(LASTFM_SLEEP)

            if tag_r.status_code == 200:
                tags = [
                    t.get("name", "").lower()
                    for t in tag_r.json().get("toptags", {}).get("tag", [])[:20]
                    if t.get("name")
                ]

        status = "lastfm_track_tags_found" if tags else "no_track_tags"

        LASTFM_TRACK_CACHE[cache_key] = {
            "tags": tags,
            "listeners": listeners,
            "status": status,
        }

        if len(LASTFM_TRACK_CACHE) % 25 == 0:
            save_lastfm_cache(LASTFM_TRACK_CACHE)

        return tags, listeners, status

    except Exception as e:
        status = f"lastfm_error_{type(e).__name__}"

        LASTFM_TRACK_CACHE[cache_key] = {
            "tags": [],
            "listeners": 0,
            "status": status,
        }

        return [], 0, status


# LAST.FM TAG - TOP TRACKS (for targeted Folk pass)

def get_lastfm_tag_toptracks(tag, pages=3):
    """
    Pulls top tracks for a given Last.fm tag (e.g. 'folk', 'americana').
    Used ONLY to generate a shortlist of likely-Folk candidates to check
    against our own SONICS csv. Every track still goes through the same
    strict verification pipeline afterwards - this is not a shortcut
    around verification, just a smarter place to start looking.
    """

    cache_key = f"tagtoptracks::{tag}::pages={pages}"

    if cache_key in LASTFM_TRACK_CACHE:
        return LASTFM_TRACK_CACHE[cache_key]

    results = []

    for page in range(1, pages + 1):
        try:
            r = session.get(
                LASTFM_URL,
                params={
                    "method": "tag.gettoptracks",
                    "tag": tag,
                    "api_key": LASTFM_API_KEY,
                    "format": "json",
                    "page": page,
                    "limit": 50,
                },
                timeout=15
            )

            time.sleep(LASTFM_SLEEP)

            if r.status_code != 200:
                continue

            items = r.json().get("tracks", {}).get("track", [])

            for item in items:
                t_name = str(item.get("name", "")).strip()
                a_name = str(item.get("artist", {}).get("name", "")).strip()

                if t_name and a_name:
                    results.append((t_name, a_name))

        except Exception:
            continue

    LASTFM_TRACK_CACHE[cache_key] = results
    return results


# iTUNES VERIFY SAME SONG


def search_itunes_track_verify(title, artist):
    """
    iTunes used only for metadata verification.
    It checks same song + artist and returns iTunes genre.
    """

    try:
        r = session.get(
            ITUNES_SEARCH_URL,
            params={
                "term": f"{title} {artist}",
                "country": "US",
                "media": "music",
                "entity": "song",
                "limit": 8,
            },
            timeout=15
        )

        time.sleep(ITUNES_SLEEP)

        if r.status_code != 200:
            return False, "", "", "", f"itunes_error_{r.status_code}"

        results = r.json().get("results", [])

        title_norm = normalise_for_match(title)
        artist_norm = normalise_for_match(artist)

        for item in results:
            it_title = str(item.get("trackName", "")).strip()
            it_artist = str(item.get("artistName", "")).strip()
            it_genre = str(item.get("primaryGenreName", "")).strip()

            it_title_norm = normalise_for_match(it_title)
            it_artist_norm = normalise_for_match(it_artist)

            title_match = (
                title_norm == it_title_norm
                or title_norm in it_title_norm
                or it_title_norm in title_norm
            )

            artist_match = (
                artist_norm == it_artist_norm
                or artist_norm in it_artist_norm
                or it_artist_norm in artist_norm
            )

            if title_match and artist_match:
                return True, it_title, it_artist, it_genre, "itunes_song_artist_match"

        return False, "", "", "", "itunes_no_good_match"

    except Exception as e:
        return False, "", "", "", f"itunes_error_{type(e).__name__}"


# SHARED VERIFY STEP (used by both targeted-Folk pass and random pass)

# EFFECTIVE (ARTIST-CAPPED) COUNT

def effective_capped_count(genre_artist_counts, genre):
    """
    Returns how many tracks would ACTUALLY survive Cell 6's
    'max 2 songs per artist per genre' rule, for a given genre.

    This matters because old data collected before the artist-cap fix
    can have artists appearing many times (e.g. one artist with 6 songs
    in a genre). Those extra songs beyond the cap don't help reach the
    Cell 6 target of 60 - Cell 6 will discard them anyway. So we must
    check THIS number against target_buffer, not the raw row count.
    """
    counts = genre_artist_counts.get(genre, {})
    return sum(min(c, MAX_SONGS_PER_ARTIST_PER_GENRE) for c in counts.values())


def verify_and_maybe_accept(
    title,
    artist,
    youtube_id,
    rows,
    seen,
    genre_counts,
    genre_artist_counts,
    target_buffer,
    min_listeners,
    source_dataset_label,
    stats,
    only_folk_missing_now=False,
):
    key = normalise_for_match(title) + " | " + normalise_for_match(artist)

    if key in seen:
        return False

    seen.add(key)
    stats["checked"] += 1

    artist_clean = normalise_for_match(artist)

    tags, listeners, lastfm_status = get_lastfm_track_info(artist, title)

    if not tags:
        stats["rejected_lastfm"] += 1
        return False

    if listeners < min_listeners:
        stats["rejected_lastfm"] += 1
        return False

    lastfm_genre, lastfm_scores = map_lastfm_tags_strict(tags)

    if lastfm_genre not in GENRES:
        stats["rejected_lastfm"] += 1
        return False

    if only_folk_missing_now and lastfm_genre != "Folk/Acoustic":
        stats["skipped_not_folk"] += 1
        return False

    # Artist-diversity cap, enforced at collection time so Cell 6's
    # "max 2 per artist per genre" rule never shrinks the buffer later.
    existing_for_artist = genre_artist_counts.get(lastfm_genre, {}).get(artist_clean, 0)
    if existing_for_artist >= MAX_SONGS_PER_ARTIST_PER_GENRE:
        stats["rejected_artist_cap"] += 1
        return False

    # Only relax "Alternative" -> Folk when Last.fm is already confident
    # about Folk/Acoustic for this track.
    allow_alt_folk = (lastfm_genre == "Folk/Acoustic")

    itunes_ok, it_title, it_artist, it_raw_genre, itunes_status = search_itunes_track_verify(title, artist)

    if not itunes_ok:
        stats["rejected_itunes"] += 1
        return False

    itunes_genre = map_itunes_genre_strict(it_raw_genre, allow_alternative_as_folk=allow_alt_folk)

    if itunes_genre not in GENRES:
        stats["rejected_itunes"] += 1
        return False

    if lastfm_genre != itunes_genre:
        stats["rejected_mismatch"] += 1
        if stats["rejected_mismatch"] <= 30:
            tqdm.write(
                f"REJECT genre mismatch | {title} - {artist} | "
                f"Last.fm={lastfm_genre} | iTunes={itunes_genre} | raw={it_raw_genre}"
            )
        return False

    final_genre = itunes_genre

    if only_folk_missing_now and final_genre != "Folk/Acoustic":
        return False

    if effective_capped_count(genre_artist_counts, final_genre) >= target_buffer:
        return False

    genre_counts[final_genre] += 1
    genre_artist_counts.setdefault(final_genre, {})
    genre_artist_counts[final_genre][artist_clean] = genre_artist_counts[final_genre].get(artist_clean, 0) + 1
    stats["accepted"] += 1

    youtube_url = ""
    if youtube_id and str(youtube_id).lower() not in ["nan", "none", ""]:
        youtube_url = f"https://www.youtube.com/watch?v={youtube_id}"

    used_alt_as_folk = (allow_alt_folk and "alternative" in str(it_raw_genre).lower())

    new_row = {
        "title": title,
        "artist": artist,
        "language": "English",
        "language_code": "EN",
        "label": "Human",
        "source_dataset": source_dataset_label,
        "model": "Human",
        "planned_genre": "",
        "raw_genre": it_raw_genre,
        "final_genre": final_genre,
        "preview_url": "",
        "track_view_url": "",
        "country": "US",
        "youtube_url": youtube_url,
        "youtube_video_id": youtube_id,
        "audio_path": "",
        "verification_status": (
            "lastfm_itunes_alternative_as_folk" if used_alt_as_folk
            else "lastfm_itunes_genre_agree_strict"
        ),
        "notes": f"Last.fm listeners={listeners}; " + (
            "Alternative accepted as Folk/Acoustic (Last.fm strongly folk)" if used_alt_as_folk
            else "Last.fm and iTunes agree strictly"
        ),
        "lastfm_track_tags": ", ".join(tags),
        "lastfm_listeners": listeners,
        "lastfm_genre": lastfm_genre,
        "itunes_title": it_title,
        "itunes_artist": it_artist,
        "itunes_raw_genre": it_raw_genre,
        "itunes_genre": itunes_genre,
        "lastfm_scores": str(lastfm_scores),
    }

    rows.append(new_row)

    tqdm.write(
        f"ACCEPT {stats['accepted']} | {final_genre} | "
        f"{title} - {artist} | listeners={listeners} | iTunes={it_raw_genre}"
    )

    if stats["accepted"] % 5 == 0:
        pd.DataFrame(rows).to_csv(ENGLISH_OUTPUT_CSV, index=False, encoding="utf-8-sig")
        save_lastfm_cache(LASTFM_TRACK_CACHE)

    if stats["accepted"] % 10 == 0:
        tqdm.write(f"Live counts -> {genre_counts}")

    return True


# TARGETED FOLK PASS

def run_targeted_folk_pass(df, rows, seen, genre_counts, genre_artist_counts, target_buffer, min_listeners, stats):
    """
    Instead of randomly scanning 48k rows hoping to hit a Folk track,
    pull Last.fm's own top tracks for folk/country/americana tags,
    then only keep the ones that also exist in our SONICS real_songs.csv
    (so we still have lyrics + vocal-filtered rows to work with).
    Every match still runs through the SAME strict verification as
    the random pass - this only changes where we look first.
    """

    if effective_capped_count(genre_artist_counts, "Folk/Acoustic") >= target_buffer:
        print(
            f"Folk/Acoustic already has {effective_capped_count(genre_artist_counts, 'Folk/Acoustic')} "
            f"artist-cap-safe tracks (target {target_buffer}), skipping targeted pass."
        )
        return

    print("\nRunning targeted Folk/Acoustic pass using Last.fm tag.getTopTracks...")

    # Build a lookup so we can check if a Last.fm top-track exists in our own CSV
    lookup = {}
    for _, row in df.iterrows():
        t = str(row.get("title", "")).strip()
        a = str(row.get("artist", "")).strip()
        if not t or not a:
            continue
        lookup_key = normalise_for_match(t) + " | " + normalise_for_match(a)
        lookup[lookup_key] = row

    candidates = []
    for tag in FOLK_TAG_QUERY_LIST:
        tqdm.write(f"Fetching Last.fm top tracks for tag: {tag}")
        tag_tracks = get_lastfm_tag_toptracks(tag, pages=FOLK_TOPTRACKS_PAGES_PER_TAG)
        candidates.extend(tag_tracks)

    print(f"Collected {len(candidates)} candidate (title, artist) pairs from Last.fm tags.")

    matched = 0
    for t_name, a_name in tqdm(candidates, desc="Matching Folk candidates against SONICS csv"):
        if effective_capped_count(genre_artist_counts, "Folk/Acoustic") >= target_buffer:
            tqdm.write("Folk/Acoustic artist-cap-safe target reached during targeted pass.")
            break

        lookup_key = normalise_for_match(t_name) + " | " + normalise_for_match(a_name)

        row = lookup.get(lookup_key)
        if row is None:
            continue

        matched += 1

        youtube_id = str(row.get("youtube_id", "")).strip()

        verify_and_maybe_accept(
            title=str(row.get("title", "")).strip(),
            artist=str(row.get("artist", "")).strip(),
            youtube_id=youtube_id,
            rows=rows,
            seen=seen,
            genre_counts=genre_counts,
            genre_artist_counts=genre_artist_counts,
            target_buffer=target_buffer,
            min_listeners=min_listeners,
            source_dataset_label="SONICS real_songs.csv (folk-targeted via Last.fm tag)",
            stats=stats,
            only_folk_missing_now=False,
        )

    print(f"Targeted Folk pass matched {matched} SONICS rows against Last.fm folk tags.")
    print(f"Folk/Acoustic raw count: {genre_counts.get('Folk/Acoustic', 0)}")
    print(f"Folk/Acoustic artist-cap-safe count: {effective_capped_count(genre_artist_counts, 'Folk/Acoustic')}")


# BUILD CANDIDATES (main runner)

def build_english_candidates_strict(
    csv_path,
    max_rows=MAX_ROWS_TO_SCAN,
    target_buffer=ENGLISH_TARGET_BUFFER,
    min_listeners=MIN_LASTFM_LISTENERS,
    resume=True
):
    if not Path(csv_path).exists():
        print("SONICS CSV not found:", csv_path)
        return pd.DataFrame()

    df_full = pd.read_csv(csv_path)

    print("Loaded SONICS rows:", len(df_full))
    print("Columns:", list(df_full.columns))

    # Keep vocal songs only
    if "no_vocal" in df_full.columns:
        before = len(df_full)
        no_vocal_text = df_full["no_vocal"].astype(str).str.lower().str.strip()
        keep_vocal = no_vocal_text.isin(["false", "0", "0.0"])
        df_full = df_full[keep_vocal].copy()
        print("After strict vocal filter:", len(df_full), "from", before)

    # Lyrics should exist because you need vocal songs
    if "lyrics" in df_full.columns:
        before = len(df_full)
        lyrics_text = df_full["lyrics"].astype(str).str.strip()
        df_full = df_full[
            (lyrics_text.str.lower() != "nan") &
            (lyrics_text.str.len() >= 50)
        ].copy()
        print("After lyrics-present filter:", len(df_full), "from", before)

    df = df_full.sample(frac=1, random_state=42).head(max_rows)

    rows = []
    seen = set()
    genre_counts = {g: 0 for g in GENRES}
    genre_artist_counts = {g: {} for g in GENRES}

    stats = {
        "checked": 0,
        "accepted": 0,
        "rejected_mismatch": 0,
        "rejected_itunes": 0,
        "rejected_lastfm": 0,
        "skipped_not_folk": 0,
        "rejected_artist_cap": 0,
    }

    # Resume from existing strict file - NOTHING is deleted here
    if resume and ENGLISH_OUTPUT_CSV.exists():
        old = pd.read_csv(ENGLISH_OUTPUT_CSV)

        if len(old) > 0 and "final_genre" in old.columns:
            print("Resume strict file found:", ENGLISH_OUTPUT_CSV)
            rows = old.to_dict("records")

            for _, r in old.iterrows():
                key = normalise_for_match(r.get("title", "")) + " | " + normalise_for_match(r.get("artist", ""))
                seen.add(key)

            old_counts = old["final_genre"].value_counts().to_dict()

            for g in GENRES:
                genre_counts[g] = int(old_counts.get(g, 0))

            for _, r in old.iterrows():
                g = r.get("final_genre", "")
                a_clean = normalise_for_match(r.get("artist", ""))
                if g in GENRES and a_clean:
                    genre_artist_counts.setdefault(g, {})
                    genre_artist_counts[g][a_clean] = genre_artist_counts[g].get(a_clean, 0) + 1

            stats["accepted"] = len(rows)

            print("Resume counts (raw):", genre_counts)
            print(
                "Resume counts (artist-cap-safe, i.e. what Cell 6 will actually see):",
                {g: effective_capped_count(genre_artist_counts, g) for g in GENRES}
            )
            print(
                "Note: old rows collected before the artist-cap fix may have artists "
                "repeated many times. Those extra duplicates count toward 'raw' but NOT "
                "toward 'artist-cap-safe' - the second number is what actually matters "
                "for Cell 6's max-2-per-artist rule."
            )

    # STEP 1: targeted Folk/Acoustic pass (fast, tag-based)
    run_targeted_folk_pass(
        df=df_full,   # search the FULL filtered SONICS pool, not just the random sample
        rows=rows,
        seen=seen,
        genre_counts=genre_counts,
        genre_artist_counts=genre_artist_counts,
        target_buffer=target_buffer,
        min_listeners=min_listeners,
        stats=stats,
    )

    # Save progress after targeted pass before starting the slower random pass
    pd.DataFrame(rows).to_csv(ENGLISH_OUTPUT_CSV, index=False, encoding="utf-8-sig")
    save_lastfm_cache(LASTFM_TRACK_CACHE)

    # STEP 2: random scan pass for remaining genres (original behaviour)
    if all(effective_capped_count(genre_artist_counts, g) >= target_buffer for g in GENRES):
        print("\nAll genres already at artist-cap-safe target after targeted Folk pass. Skipping random scan.")
    else:
        print("\nRunning random scan pass for remaining genres...")

        for _, row in tqdm(df.iterrows(), total=len(df), desc="English strict Last.fm + iTunes"):

            if all(effective_capped_count(genre_artist_counts, g) >= target_buffer for g in GENRES):
                tqdm.write(f"\nArtist-cap-safe target buffer {target_buffer} reached for all genres.")
                break

            title = str(row.get("title", "")).strip()
            artist = str(row.get("artist", "")).strip()
            youtube_id = str(row.get("youtube_id", "")).strip()

            if not title or not artist:
                continue

            if stats["checked"] % 20 == 0 and stats["checked"] > 0:
                tqdm.write(
                    f"Checked={stats['checked']} | Accepted={stats['accepted']} | "
                    f"Mismatch={stats['rejected_mismatch']} | Counts={genre_counts}"
                )

            only_folk_missing_now = (
                effective_capped_count(genre_artist_counts, "Folk/Acoustic") < target_buffer
                and all(
                    effective_capped_count(genre_artist_counts, g) >= target_buffer
                    for g in GENRES if g != "Folk/Acoustic"
                )
            )

            verify_and_maybe_accept(
                title=title,
                artist=artist,
                youtube_id=youtube_id,
                rows=rows,
                seen=seen,
                genre_counts=genre_counts,
                genre_artist_counts=genre_artist_counts,
                target_buffer=target_buffer,
                min_listeners=min_listeners,
                source_dataset_label="SONICS real_songs.csv",
                stats=stats,
                only_folk_missing_now=only_folk_missing_now,
            )

    save_lastfm_cache(LASTFM_TRACK_CACHE)

    out_df = pd.DataFrame(rows)

    if len(out_df) == 0:
        print("No strict English candidates collected.")
        return out_df

    out_df.to_csv(ENGLISH_OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print("\nSaved:", ENGLISH_OUTPUT_CSV)
    print("Final strict candidate counts:")
    print(out_df["final_genre"].value_counts())

    print("\nRejected mismatch:", stats["rejected_mismatch"])
    print("Rejected iTunes:", stats["rejected_itunes"])
    print("Rejected Last.fm:", stats["rejected_lastfm"])
    print("Rejected (artist cap reached):", stats["rejected_artist_cap"])
    print("Skipped non-Folk because only Folk missing:", stats["skipped_not_folk"])

    return out_df


# RUN

english_candidates = build_english_candidates_strict(
    SONICS_REAL_CSV,
    max_rows=48090,
    target_buffer=ENGLISH_TARGET_BUFFER,
    min_listeners=1000,
    resume=True
)

if english_candidates is None or len(english_candidates) == 0:
    print("No English candidates collected.")
else:
    display(english_candidates["final_genre"].value_counts())
    display(
        english_candidates[
            [
                "title",
                "artist",
                "final_genre",
                "lastfm_listeners",
                "lastfm_track_tags",
                "itunes_raw_genre",
                "verification_status",
            ]
        ].tail(50)
    )

## 6. Hindi and Chinese language/genre verification

Hindi: reject Punjabi/Gurmukhi and conflicting language evidence.
hinese: reject Japanese Kana and Korean Hangul.
 All languages: reject unsuitable covers, karaoke, remixes and unrelated versions.


## Strict Language & Genre Verification (Hindi & Chinese)

> **Methodology Note:** Candidate tracks pulled directly from regional iTunes charts frequently contain out-of-region tracks. To guarantee the linguistic and cultural accuracy of the dataset before balancing, this step applies strict filters:


In [ ]:
# FAST RESUME LANGUAGE + GENRE CHECK

import json
import os
import random
import re
import threading
import time
import unicodedata
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm


# SETTINGS


MANIFEST_DIR = Path(
    globals().get(
        "MANIFEST_DIR",
        Path("dataset_collection_clean") / "manifests",
    )
)

CACHE_DIR = Path(
    globals().get(
        "CACHE_DIR",
        Path("dataset_collection_clean") / "api_cache",
    )
)

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Lock used by save_cache()/save_all_caches() when snapshotting the
# in-memory cache dicts from multiple worker threads.
_CACHE_LOCK = threading.Lock()

GENRES = [
    "Pop",
    "Rock",
    "Hip-Hop/Rap",
    "R&B/Soul",
    "Folk/Acoustic",
]

USE_LYRICS_LANGUAGE_CHECK = True
USE_LASTFM_GENRE_CHECK = True

# Cell 4 already contains exact iTunes/Apple rows.
# Network iTunes rechecking is therefore unnecessary here.
RECHECK_ITUNES = False

# Resume already processed tracks from the live CSV files.
RESUME_FROM_CHECKPOINTS = False

# Fast mode:
#   - known Hindi catalogue rows are accepted before a lyrics call
#   - Last.fm is called only when iTunes/Cell 4 has no usable genre
FAST_MODE = True
LASTFM_ONLY_FOR_UNRESOLVED_GENRE = True

CHECKPOINT_EVERY = 250
PRINT_PROGRESS_EVERY = 25

# Do not wait repeatedly on missing lyrics or temporary API failures.
HTTP_TIMEOUT = 8
MAX_RETRIES = 1
MAX_429_WAIT_SECONDS = 5

LRCLIB_SEARCH_URL = "https://lrclib.net/api/search"
LASTFM_URL = "https://ws.audioscrobbler.com/2.0/"
ITUNES_SEARCH_URL = "https://itunes.apple.com/search"

LYRICS_CACHE_FILE = CACHE_DIR / "cell5b_lrclib_language_cache.json"
LASTFM_CACHE_FILE = CACHE_DIR / "cell5b_lastfm_track_cache.json"
ITUNES_CACHE_FILE = CACHE_DIR / "cell5b_itunes_verify_cache.json"

LASTFM_API_KEY = str(
    globals().get("LASTFM_API_KEY", "")
    or os.getenv("LASTFM_API_KEY", "")
).strip()

if USE_LASTFM_GENRE_CHECK and not LASTFM_API_KEY:
    LASTFM_API_KEY = getpass(
        "Enter Last.fm API key or press Enter to skip: "
    ).strip()

LASTFM_ENABLED = bool(
    USE_LASTFM_GENRE_CHECK and LASTFM_API_KEY
)

SESSION = requests.Session()
SESSION.headers.update(
    {
        "User-Agent": (
            "MScMusicDataset/1.0 "
            "student-research-language-check"
        ),
        "Accept": "application/json",
    }
)

API_CALLS = defaultdict(int)



# CACHE

def load_cache(path):
    path = Path(path)

    if not path.exists():
        return {}

    try:
        value = json.loads(
            path.read_text(encoding="utf-8")
        )
        return value if isinstance(value, dict) else {}
    except Exception:
        return {}


def save_cache(cache, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    # 1. Acquire lock and snapshot the dictionary
    with _CACHE_LOCK:
        # Create a shallow copy of the dict to iterate over safely
        snapshot = dict(cache)

    # 2. Serialize the snapshot outside the lock to keep threads moving
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(snapshot, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    temporary.replace(path)


LYRICS_CACHE = load_cache(LYRICS_CACHE_FILE)
LASTFM_CACHE = load_cache(LASTFM_CACHE_FILE)
ITUNES_CACHE = load_cache(ITUNES_CACHE_FILE)


def save_all_caches():
    save_cache(
        LYRICS_CACHE,
        LYRICS_CACHE_FILE,
    )
    save_cache(
        LASTFM_CACHE,
        LASTFM_CACHE_FILE,
    )
    save_cache(
        ITUNES_CACHE,
        ITUNES_CACHE_FILE,
    )


# GENERAL HELPERS

def safe_text(value):
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip()


def normalise_text(value):
    text = unicodedata.normalize(
        "NFKC",
        safe_text(value),
    ).casefold()

    text = re.sub(
        r"\(.*?\)|\[.*?\]",
        " ",
        text,
    )

    text = re.sub(
        r"[^\w\s\u0900-\u097f\u0a00-\u0a7f"
        r"\u3400-\u4dbf\u4e00-\u9fff"
        r"\u3040-\u30ff\uac00-\ud7af]",
        " ",
        text,
    )

    text = re.sub(
        r"_+",
        " ",
        text,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def similarity(left, right):
    left_n = normalise_text(left)
    right_n = normalise_text(right)

    if not left_n or not right_n:
        return 0.0

    if left_n == right_n:
        return 1.0

    return SequenceMatcher(
        None,
        left_n,
        right_n,
    ).ratio()


def split_values(value):
    text = safe_text(value)

    if not text:
        return []

    return [
        item.strip()
        for item in re.split(
            r"\s*[|;,]\s*",
            text,
        )
        if item.strip()
    ]


def has_phrase(text, phrase):
    text = safe_text(text).casefold()
    phrase = safe_text(phrase).casefold()

    if not text or not phrase:
        return False

    if re.search(r"[^\x00-\x7f]", phrase):
        return phrase in text

    pattern = (
        rf"(?<![a-z0-9])"
        rf"{re.escape(phrase)}"
        rf"(?![a-z0-9])"
    )

    return bool(
        re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        )
    )


def script_count(text, pattern):
    return len(
        re.findall(
            pattern,
            safe_text(text),
        )
    )


def contains_devanagari(text):
    return bool(
        re.search(
            r"[\u0900-\u097f]",
            safe_text(text),
        )
    )


def contains_gurmukhi(text):
    return bool(
        re.search(
            r"[\u0a00-\u0a7f]",
            safe_text(text),
        )
    )


def contains_han(text):
    return bool(
        re.search(
            r"[\u3400-\u4dbf\u4e00-\u9fff]",
            safe_text(text),
        )
    )


def contains_kana_or_hangul(text):
    return bool(
        re.search(
            r"[\u3040-\u30ff\uac00-\ud7af]",
            safe_text(text),
        )
    )


def source_is_exact_itunes(row):
    source = safe_text(
        row.get("source_dataset")
    ).casefold()

    url = safe_text(
        row.get("track_view_url")
    ).casefold()

    track_id = safe_text(
        row.get("source_track_id")
    )

    return bool(
        (
            "itunes" in source
            or "apple music" in source
            or "itunes.apple.com" in url
            or "music.apple.com" in url
        )
        and (
            track_id
            or url
            or safe_text(row.get("raw_genre"))
            or safe_text(row.get("collection"))
            or safe_text(row.get("album"))
        )
    )


# BAD TRACK FILTER

BAD_VERSION_PHRASES = [
    "karaoke",
    "instrumental",
    "cover version",
    "cover song",
    "tribute",
    "reaction",
    "tutorial",
    "podcast",
    "medley",
    "compilation",
    "full album",
    "concert",
    "live",
    "live version",
    "live at",
    "live from",
    "acoustic live",
    "dj mix",
    "dj remix",
    "remix",
    "mixtape",
    "mix tape",
    "mashup",
    "nonstop mix",
    "non stop mix",
    "sped up",
    "slowed",
    "nightcore",
    "8d audio",
    "lofi version",
    "lyrics video only",
    "backing track",
    "vocals only",
    "vocal only",
    "club mix",
    "recreated version",
    "recreation",
    "演唱会",
    "演唱會",
    "現場",
    "现场",
    "伴奏",
    "翻唱",
    "合集",
    "纯音乐",
]

PLAYLIST_PHRASES = [
    "playlist",
    "jukebox",
    "all songs",
    "top songs",
    "best songs",
    "hit songs",
    "superhit songs",
    "song collection",
    "hits collection",
    "non stop",
    "non-stop",
]


def bad_track_reason(row):
    title = safe_text(row.get("title"))
    artist = safe_text(row.get("artist"))
    album = safe_text(
        row.get("collection")
        or row.get("album")
    )

    title_album = f"{title} {album}"
    all_text = f"{title} {artist} {album}"

    for phrase in BAD_VERSION_PHRASES:
        if has_phrase(title_album, phrase):
            return (
                "reject_bad_version_"
                + normalise_text(phrase)
                .replace(" ", "_")
            )

    for phrase in PLAYLIST_PHRASES:
        if has_phrase(all_text, phrase):
            return "reject_playlist_or_compilation"

    if re.search(
        r"\btop\s*\d+\b.*\b(?:songs?|tracks?|hits?)\b",
        all_text.casefold(),
    ):
        return "reject_numbered_playlist"

    if re.match(
        r"^\s*\d{1,3}\s*[.)]\s+",
        title,
    ):
        return "reject_numbered_track_title"

    return ""


def _validate_bad_version_filter():
    reject_cases = [
        {
            "title": "Kahaan (Live)",
            "artist": "Rashmeet Kaur",
            "collection": "Live Session",
        },
        {
            "title": "Song Name (DJ Remix)",
            "artist": "Artist",
            "collection": "Single",
        },
        {
            "title": "Bollywood Mixtape",
            "artist": "Artist",
            "collection": "Mixtape",
        },
        {
            "title": "Song Mashup",
            "artist": "Artist",
            "collection": "Single",
        },
    ]

    for test_row in reject_cases:
        reason = bad_track_reason(test_row)

        if not reason:
            raise RuntimeError(
                "Cell 5B bad-version self-test failed: "
                f"{test_row['title']!r} was not rejected."
            )


_validate_bad_version_filter()
print("Bad-version filter: passed")

# GENRE MAPPING

GENRE_KEYWORDS = {
    "Hip-Hop/Rap": [
        "hip hop",
        "hip-hop",
        "rap",
        "trap",
        "drill",
        "grime",
        "说唱",
        "嘻哈",
    ],
    "R&B/Soul": [
        "r&b",
        "rnb",
        "rhythm and blues",
        "soul",
        "neo soul",
    ],
    "Folk/Acoustic": [
        "folk",
        "acoustic",
        "singer-songwriter",
        "singer songwriter",
        "unplugged",
        "ghazal",
        "sufi",
        "民谣",
        "民謠",
    ],
    "Rock": [
        "rock",
        "metal",
        "punk",
        "grunge",
        "alternative rock",
        "indie rock",
        "hard rock",
        "摇滚",
        "搖滾",
    ],
    "Pop": [
        "pop",
        "dance pop",
        "electropop",
        "synthpop",
        "indie pop",
        "mandopop",
        "cantopop",
        "c-pop",
        "chinese pop",
        "流行",
    ],
}

BROAD_GENRE_LABELS = [
    "bollywood",
    "filmi",
    "indian pop",
    "indipop",
    "regional indian",
    "indian music",
    "hindi",
    "hindi music",
    "soundtrack",
    "original soundtrack",
    "motion picture soundtrack",
    "stage & screen",
    "stage and screen",
    "world",
    "worldwide",
    "music",
]


def remove_broad_labels(value):
    text = safe_text(value).casefold()
    text = text.replace("&", " and ")

    for phrase in sorted(
        BROAD_GENRE_LABELS,
        key=len,
        reverse=True,
    ):
        text = re.sub(
            rf"(?<!\w){re.escape(phrase)}(?!\w)",
            " ",
            text,
        )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def map_genre(values):
    if isinstance(values, str):
        values = [values]

    scores = {
        genre: 0
        for genre in GENRES
    }

    for rank, value in enumerate(values):
        cleaned = remove_broad_labels(value)

        if not cleaned:
            continue

        weight = max(1, 20 - rank)

        for genre, keywords in GENRE_KEYWORDS.items():
            for keyword in keywords:
                if has_phrase(cleaned, keyword):
                    scores[genre] += weight

    best_genre = max(
        scores,
        key=scores.get,
    )

    if scores[best_genre] <= 0:
        return None, scores

    ranked_scores = sorted(
        scores.values(),
        reverse=True,
    )

    if (
        len(ranked_scores) > 1
        and ranked_scores[0] == ranked_scores[1]
    ):
        return None, scores

    return best_genre, scores


def original_or_seed_genre(row):
    candidates = [
        row.get("final_genre"),
        row.get("seed_genre"),
        row.get("planned_genre"),
        row.get("query_genre_group"),
    ]

    candidates.extend(
        split_values(
            row.get("query_genre_groups")
        )
    )

    for value in candidates:
        value = safe_text(value)

        if value in GENRES:
            return value

    return None


# SAFE HTTP

def request_json(
    url,
    *,
    params,
    service,
    sleep_after=0.1,
):
    for attempt in range(MAX_RETRIES):
        try:
            response = SESSION.get(
                url,
                params=params,
                timeout=HTTP_TIMEOUT,
            )

            API_CALLS[service] += 1

            if response.status_code == 429:
                retry_after = response.headers.get(
                    "Retry-After",
                    "",
                )

                try:
                    wait_seconds = float(retry_after)
                except Exception:
                    wait_seconds = 2 ** attempt

                if wait_seconds > MAX_429_WAIT_SECONDS:
                    return None, (
                        f"{service}_429_wait_too_long"
                    )

                time.sleep(
                    wait_seconds
                    + random.uniform(0.1, 0.3)
                )
                continue

            if response.status_code in {
                500,
                502,
                503,
                504,
            }:
                time.sleep(
                    min(
                        2 ** attempt,
                        8,
                    )
                )
                continue

            if response.status_code != 200:
                return None, (
                    f"{service}_http_"
                    f"{response.status_code}"
                )

            payload = response.json()
            time.sleep(sleep_after)
            return payload, "ok"

        except Exception as error:
            if attempt == MAX_RETRIES - 1:
                return None, (
                    f"{service}_"
                    f"{type(error).__name__}"
                )

            time.sleep(
                min(
                    2 ** attempt,
                    8,
                )
            )

    return None, f"{service}_failed"


# LYRICS LANGUAGE CHECK

HINDI_WORDS = {
    "aaj", "aaja", "aankh", "aankhon", "aao", "aashiq",
    "aashiqui", "apna", "apni", "baarish", "baat", "bata",
    "bewafa", "chaand", "chale", "channa", "dard", "deewana",
    "deewane", "dil", "door", "duniya", "faasle", "gaya",
    "gham", "hai", "hain", "hawayein", "ho", "hoon", "hua",
    "hum", "ishq", "jaan", "jaana", "jaane", "kabhi", "kaise",
    "kahan", "kahaan", "kasoor", "khabar", "khayaal", "khwab",
    "khwaab", "ki", "kiya", "kya", "kyun", "lage", "main",
    "maine", "mann", "mera", "mere", "meri", "mila", "mohabbat",
    "mujhe", "na", "nahi", "naina", "pyaar", "pyar", "raat",
    "raha", "rahe", "rang", "saath", "safar", "sahiba", "sajna",
    "sajana", "sanson", "sapna", "se", "tera", "tere", "teri",
    "toh", "tu", "tujhe", "tujhko", "tum", "tumse", "yaad",
    "yaar", "yaara", "yeh", "zara", "zindagi",
}

PUNJABI_WORDS = {
    "assi", "akh", "balle", "bhangra", "gabru", "gabhru",
    "jatt", "jatti", "karda", "karde", "kardi", "kudi",
    "lagda", "munda", "naal", "nachda", "nachdi", "ni",
    "nu", "pind", "punjab", "sardar", "sardaar", "sohni",
    "tainu", "tenu", "ve", "vich", "wala", "wali",
    "warga", "wargi",
}

OTHER_INDIC_PATTERNS = {
    "bengali": r"[\u0980-\u09ff]",
    "gujarati": r"[\u0a80-\u0aff]",
    "odia": r"[\u0b00-\u0b7f]",
    "tamil": r"[\u0b80-\u0bff]",
    "telugu": r"[\u0c00-\u0c7f]",
    "kannada": r"[\u0c80-\u0cff]",
    "malayalam": r"[\u0d00-\u0d7f]",
}


def clean_synced_lyrics(value):
    text = safe_text(value)

    text = re.sub(
        r"\[[^\]]+\]",
        " ",
        text,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def lyrics_cache_key(title, artist, album, duration):
    return "||".join(
        [
            normalise_text(title),
            normalise_text(artist),
            normalise_text(album),
            safe_text(duration),
        ]
    )


def fetch_lyrics(title, artist, album="", duration=""):
    if not USE_LYRICS_LANGUAGE_CHECK:
        return {
            "ok": False,
            "status": "lyrics_disabled",
        }

    key = lyrics_cache_key(
        title,
        artist,
        album,
        duration,
    )

    cached = LYRICS_CACHE.get(key)

    if isinstance(cached, dict):
        return cached

    params = {
        "track_name": title,
        "artist_name": artist,
    }

    if safe_text(album):
        params["album_name"] = safe_text(album)

    try:
        duration_int = int(
            round(float(duration))
        )

        if duration_int > 0:
            params["duration"] = duration_int
    except Exception:
        pass

    payload, status = request_json(
        LRCLIB_SEARCH_URL,
        params=params,
        service="LRCLIB",
        sleep_after=0.08,
    )

    if status != "ok" or not isinstance(
        payload,
        list,
    ):
        result = {
            "ok": False,
            "status": status,
        }
        LYRICS_CACHE[key] = result
        return result

    best = None
    best_score = -1.0

    for item in payload:
        if not isinstance(item, dict):
            continue

        title_score = similarity(
            title,
            item.get("trackName"),
        )

        artist_score = similarity(
            artist,
            item.get("artistName"),
        )

        if title_score < 0.72:
            continue

        if artist_score < 0.45:
            continue

        score = (
            title_score * 0.70
            + artist_score * 0.25
        )

        try:
            expected_duration = float(duration)
            found_duration = float(
                item.get("duration")
            )

            difference = abs(
                expected_duration
                - found_duration
            )

            if difference <= 20:
                score += 0.05
            elif difference > 45:
                score -= 0.10
        except Exception:
            difference = None

        if score > best_score:
            best = item
            best_score = score

    if best is None:
        result = {
            "ok": False,
            "status": "lyrics_no_matching_record",
        }
        LYRICS_CACHE[key] = result
        return result

    lyrics_text = safe_text(
        best.get("plainLyrics")
    )

    if not lyrics_text:
        lyrics_text = clean_synced_lyrics(
            best.get("syncedLyrics")
        )

    if not lyrics_text:
        result = {
            "ok": False,
            "status": "lyrics_record_has_no_text",
        }
        LYRICS_CACHE[key] = result
        return result

    devanagari_count = script_count(
        lyrics_text,
        r"[\u0900-\u097f]",
    )
    gurmukhi_count = script_count(
        lyrics_text,
        r"[\u0a00-\u0a7f]",
    )
    han_count = script_count(
        lyrics_text,
        r"[\u3400-\u4dbf\u4e00-\u9fff]",
    )
    kana_hangul_count = script_count(
        lyrics_text,
        r"[\u3040-\u30ff\uac00-\ud7af]",
    )

    other_indic_counts = {
        name: script_count(
            lyrics_text,
            pattern,
        )
        for name, pattern
        in OTHER_INDIC_PATTERNS.items()
    }

    roman_tokens = re.findall(
        r"[a-zA-Z']+",
        lyrics_text.casefold(),
    )

    hindi_hits = sum(
        token in HINDI_WORDS
        for token in roman_tokens
    )

    punjabi_hits = sum(
        token in PUNJABI_WORDS
        for token in roman_tokens
    )

    result = {
        "ok": True,
        "status": "lyrics_match_found",
        "provider": "LRCLIB",
        "record_id": safe_text(
            best.get("id")
        ),
        "matched_title": safe_text(
            best.get("trackName")
        ),
        "matched_artist": safe_text(
            best.get("artistName")
        ),
        "match_score": round(
            best_score,
            4,
        ),
        "lyrics_character_count": len(
            lyrics_text
        ),
        "devanagari_count": devanagari_count,
        "gurmukhi_count": gurmukhi_count,
        "han_count": han_count,
        "kana_hangul_count": kana_hangul_count,
        "other_indic_counts": other_indic_counts,
        "hindi_word_hits": hindi_hits,
        "punjabi_word_hits": punjabi_hits,
    }

    # The full lyrics are intentionally not cached.
    LYRICS_CACHE[key] = result
    return result


# LAST.FM TRACK GENRE


def lastfm_track_info(title, artist):
    if not LASTFM_ENABLED:
        return {
            "ok": False,
            "status": "lastfm_disabled",
            "tags": [],
            "mapped_genre": None,
            "scores": {},
        }

    key = (
        normalise_text(artist)
        + "||"
        + normalise_text(title)
    )

    cached = LASTFM_CACHE.get(key)

    if isinstance(cached, dict):
        return cached

    params = {
        "method": "track.getInfo",
        "artist": artist,
        "track": title,
        "api_key": LASTFM_API_KEY,
        "format": "json",
        "autocorrect": 1,
    }

    payload, status = request_json(
        LASTFM_URL,
        params=params,
        service="LastFM",
        sleep_after=0.10,
    )

    track = (
        payload.get("track", {})
        if status == "ok"
        and isinstance(payload, dict)
        else {}
    )

    if not isinstance(track, dict):
        track = {}

    found_title = safe_text(
        track.get("name")
    )

    artist_value = track.get(
        "artist",
        {},
    )

    found_artist = (
        safe_text(artist_value.get("name"))
        if isinstance(artist_value, dict)
        else safe_text(artist_value)
    )

    identity_ok = bool(
        found_title
        and found_artist
        and similarity(
            title,
            found_title,
        ) >= 0.72
        and similarity(
            artist,
            found_artist,
        ) >= 0.45
    )

    tag_items = (
        track.get("toptags", {})
        .get("tag", [])
        if track
        else []
    )

    if not isinstance(tag_items, list):
        tag_items = []

    tags = [
        safe_text(item.get("name")).casefold()
        for item in tag_items[:20]
        if isinstance(item, dict)
        and safe_text(item.get("name"))
    ]

    if identity_ok and not tags:
        payload2, status2 = request_json(
            LASTFM_URL,
            params={
                "method": "track.getTopTags",
                "artist": found_artist or artist,
                "track": found_title or title,
                "api_key": LASTFM_API_KEY,
                "format": "json",
                "autocorrect": 1,
            },
            service="LastFM",
            sleep_after=0.10,
        )

        if (
            status2 == "ok"
            and isinstance(payload2, dict)
        ):
            items = (
                payload2.get("toptags", {})
                .get("tag", [])
            )

            if isinstance(items, list):
                tags = [
                    safe_text(
                        item.get("name")
                    ).casefold()
                    for item in items[:20]
                    if isinstance(item, dict)
                    and safe_text(
                        item.get("name")
                    )
                ]

    mapped_genre, scores = map_genre(tags)

    result = {
        "ok": identity_ok,
        "status": (
            "lastfm_track_match"
            if identity_ok
            else status
            if status != "ok"
            else "lastfm_no_exact_track"
        ),
        "tags": tags,
        "mapped_genre": mapped_genre,
        "scores": scores,
        "canonical_title": found_title,
        "canonical_artist": found_artist,
    }

    LASTFM_CACHE[key] = result
    return result


# OPTIONAL iTUNES RECHECK

def verify_itunes_if_needed(row, language_name):
    if source_is_exact_itunes(row):
        raw_genre = safe_text(
            row.get("raw_genre")
        )
        mapped, scores = map_genre(
            split_values(raw_genre)
        )

        return {
            "ok": True,
            "status": "reused_cell4_itunes_row",
            "raw_genre": raw_genre,
            "mapped_genre": mapped,
            "scores": scores,
            "collection": safe_text(
                row.get("collection")
                or row.get("album")
            ),
            "country": safe_text(
                row.get("country")
            ),
        }

    if not RECHECK_ITUNES:
        return {
            "ok": False,
            "status": "itunes_recheck_disabled",
            "raw_genre": "",
            "mapped_genre": None,
            "scores": {},
            "collection": "",
            "country": "",
        }

    title = safe_text(row.get("title"))
    artist = safe_text(row.get("artist"))

    countries = (
        ["IN"]
        if language_name == "Hindi"
        else ["CN", "TW", "HK"]
    )

    key = (
        language_name
        + "||"
        + normalise_text(title)
        + "||"
        + normalise_text(artist)
    )

    cached = ITUNES_CACHE.get(key)

    if isinstance(cached, dict):
        return cached

    for country in countries:
        payload, status = request_json(
            ITUNES_SEARCH_URL,
            params={
                "term": f"{title} {artist}",
                "country": country,
                "media": "music",
                "entity": "song",
                "limit": 15,
            },
            service="iTunes",
            sleep_after=0.12,
        )

        if (
            status != "ok"
            or not isinstance(payload, dict)
        ):
            continue

        for item in payload.get(
            "results",
            [],
        ):
            if not isinstance(item, dict):
                continue

            if similarity(
                title,
                item.get("trackName"),
            ) < 0.72:
                continue

            if similarity(
                artist,
                item.get("artistName"),
            ) < 0.45:
                continue

            raw_genre = safe_text(
                item.get("primaryGenreName")
            )
            mapped, scores = map_genre(
                [raw_genre]
            )

            result = {
                "ok": True,
                "status": "itunes_track_match",
                "raw_genre": raw_genre,
                "mapped_genre": mapped,
                "scores": scores,
                "collection": safe_text(
                    item.get("collectionName")
                ),
                "country": country,
            }

            ITUNES_CACHE[key] = result
            return result

    result = {
        "ok": False,
        "status": "itunes_no_match",
        "raw_genre": "",
        "mapped_genre": None,
        "scores": {},
        "collection": "",
        "country": "",
    }

    ITUNES_CACHE[key] = result
    return result


# LANGUAGE DECISION

HINDI_ARTISTS = [
    "arijit singh",
    "shreya ghoshal",
    "sonu nigam",
    "udit narayan",
    "alka yagnik",
    "kumar sanu",
    "sunidhi chauhan",
    "shaan",
    "mohit chauhan",
    "jubin nautiyal",
    "vishal mishra",
    "javed ali",
    "benny dayal",
    "palak muchhal",
    "tulsi kumar",
    "asees kaur",
    "shilpa rao",
    "armaan malik",
    "amit trivedi",
    "pritam",
    "indian ocean",
    "the local train",
    "agnee",
    "euphoria",
    "divine",
    "raftaar",
    "emiway bantai",
    "kr$na",
    "krsna",
    "seedhe maut",
    "dino james",
    "muhfaad",
    "fotty seven",
    "harjas harjaayi",
    "prabh deep",
    "naezy",
    "mc altaf",
    "ikka",
    "badshah",
    "shah rule",
    "j trix",
    "karma",
    "qaab",
    "yungsta",
    "munawar faruqui",
]

PUNJABI_PRIMARY_ARTISTS = [
    "karan aujla",
    "ap dhillon",
    "shubh",
    "sidhu moose wala",
    "diljit dosanjh",
    "ammy virk",
    "gippy grewal",
    "arjan dhillon",
    "navaan sandhu",
    "jass manak",
    "jassie gill",
    "garry sandhu",
    "prem dhillon",
    "shinda kahlon",
    "the prophec",
]

HINDI_CONTEXT_MARKERS = [
    "hindi",
    "bollywood",
    "filmi",
    "hindi film",
    "hindi soundtrack",
]

CHINESE_CONTEXT_MARKERS = [
    "mandopop",
    "cantopop",
    "c-pop",
    "chinese pop",
    "chinese rock",
    "chinese rap",
    "chinese hip hop",
    "chinese r&b",
    "chinese folk",
    "mandarin",
    "cantonese",
    "华语",
    "國語",
    "中文",
    "流行",
    "摇滚",
    "搖滾",
    "说唱",
    "嘻哈",
    "民谣",
    "民謠",
]

# NOTE: expanded to explicitly cover Rajasthani-folk and Bengali-folk
# labels, since these are the two most common "false Hindi" sources
# (regional folk catalogued loosely as "Hindi"/"Indian" on iTunes/Last.fm).
NON_HINDI_MARKERS = [
    "punjabi",
    "bhangra",
    "pollywood",
    "haryanvi",
    "bhojpuri",
    "tamil",
    "telugu",
    "malayalam",
    "kannada",
    "marathi",
    "bengali",
    "bangla",
    "gujarati",
    "odia",
    "assamese",
    "rajasthani",
    "rajasthan folk",
    "rajasthani folk",
    "marwari",
    "manganiyar",
    "langa",
    "kutchi",
    "mewati",
    "bhajan",
    "aarti",
    "mantra",
    "chalisa",
    "devotional",
    "baul",
    "bengali folk",
    "rabindra sangeet",
    "nazrul geeti",
]

# Explicit "this specific release is the English version/mix" markers.
# Kept separate from NON_HINDI_MARKERS (which is a generic word-boundary
# scan over genre/notes/search_query/collection/album) because a bare
# "english" marker there was too broad — it can appear incidentally in
# unrelated metadata and was wrongly rejecting genuine Hindi/Hinglish
# hip-hop tracks (rap titles and metadata often mention "English" in
# passing, e.g. bios, feature credits, or dataset notes).
ENGLISH_VERSION_MARKERS = [
    "english version",
    "english mix",
    "(english)",
    "[english]",
    "english lyrics",
    "english translation",
]

# Hip-Hop/Rap and Pop are the two genres where Hindi/Hinglish tracks
# routinely have fully English titles (rap slang, Western-style track
# names) while the actual vocals are Hindi or heavy Hindi-English
# code-switching. Word-list-based title checks systematically miss these,
# so exact iTunes India-storefront rows in these genres get a genre-aware
# trust path below (still gated behind every negative signal above it).
HINGLISH_TITLE_GENRES = [
    "hip hop",
    "hip-hop",
    "rap",
    "pop",
]


def artist_in_list(artist, names):
    artist_n = normalise_text(artist)

    return any(
        normalise_text(name) == artist_n
        or (
            len(normalise_text(name)) >= 6
            and normalise_text(name)
            in artist_n
        )
        for name in names
    )


def title_word_counts(title):
    tokens = re.findall(
        r"[a-zA-Z']+",
        safe_text(title).casefold(),
    )

    return (
        sum(
            token in HINDI_WORDS
            for token in tokens
        ),
        sum(
            token in PUNJABI_WORDS
            for token in tokens
        ),
    )


def add_lyrics_columns(row, lyrics):
    row["lyrics_status"] = lyrics.get(
        "status",
        "",
    )
    row["lyrics_provider"] = lyrics.get(
        "provider",
        "",
    )
    row["lyrics_record_id"] = lyrics.get(
        "record_id",
        "",
    )
    row["lyrics_match_score"] = lyrics.get(
        "match_score",
        "",
    )
    row["lyrics_character_count"] = lyrics.get(
        "lyrics_character_count",
        0,
    )
    row["lyrics_devanagari_count"] = lyrics.get(
        "devanagari_count",
        0,
    )
    row["lyrics_gurmukhi_count"] = lyrics.get(
        "gurmukhi_count",
        0,
    )
    row["lyrics_han_count"] = lyrics.get(
        "han_count",
        0,
    )
    row["lyrics_kana_hangul_count"] = lyrics.get(
        "kana_hangul_count",
        0,
    )
    row["lyrics_hindi_word_hits"] = lyrics.get(
        "hindi_word_hits",
        0,
    )
    row["lyrics_punjabi_word_hits"] = lyrics.get(
        "punjabi_word_hits",
        0,
    )

    other_indic_counts = lyrics.get(
        "other_indic_counts",
        {},
    )

    for name in OTHER_INDIC_PATTERNS:
        row[f"lyrics_{name}_count"] = (
            other_indic_counts.get(name, 0)
            if isinstance(other_indic_counts, dict)
            else 0
        )

    row["lyrics_text_saved"] = False


def decide_hindi_language(
    row,
    itunes_result,
):
    title = safe_text(row.get("title"))
    artist = safe_text(row.get("artist"))
    album = safe_text(
        row.get("collection")
        or row.get("album")
        or itunes_result.get("collection")
    )

    title_artist = f"{title} {artist}"

    if contains_han(
        title_artist
    ) or contains_kana_or_hangul(
        title_artist
    ):
        return (
            False,
            "reject_non_hindi_script",
            "high",
            {},
        )

    if contains_gurmukhi(title_artist):
        return (
            False,
            "reject_gurmukhi_title",
            "high",
            {},
        )

    for name, pattern in OTHER_INDIC_PATTERNS.items():
        if re.search(pattern, title_artist):
            return (
                False,
                f"reject_{name}_title_script",
                "high",
                {},
            )

    title_artist_casefold = title_artist.casefold()

    if any(
        marker in title_artist_casefold
        for marker in ENGLISH_VERSION_MARKERS
    ):
        return (
            False,
            "reject_explicit_english_version_title",
            "high",
            {},
        )

    # NOTE: row's own collection/album is now included directly (not just
    # the iTunes-recheck collection), so album titles like "Rajasthani
    # Folk Collection" or "Bengali Folk Hits" get caught even when the
    # Cell-4 row was already treated as an exact iTunes match.
    metadata_text = " ".join(
        [
            safe_text(row.get("raw_genre")),
            safe_text(row.get("notes")),
            safe_text(row.get("search_query")),
            safe_text(row.get("collection")),
            safe_text(row.get("album")),
            safe_text(itunes_result.get("raw_genre")),
            safe_text(itunes_result.get("collection")),
        ]
    ).casefold()

    # Hard commercial gate: a non-commercial track (no iTunes store URL,
    # commercial_catalogue_score < 4) is only accepted if it has Devanagari
    # in the title — otherwise it is rejected regardless of other signals.
    _commercial_score_val = int(float(str(row.get("commercial_catalogue_score", 0) or 0)))
    _has_store_url = bool(safe_text(row.get("track_view_url")))
    _is_commercial = _has_store_url or _commercial_score_val >= 4

    title_hindi_hits, title_punjabi_hits = (
        title_word_counts(title)
    )

    if contains_devanagari(title):
        return (
            True,
            "accepted_hindi_devanagari_title",
            "high",
            {},
        )

    if not _is_commercial:
        return (
            False,
            "reject_non_commercial_no_devanagari",
            "high",
            {},
        )

    # Explicit non-Hindi metadata markers (Rajasthani/Bengali/Punjabi/
    # English/etc.) are checked BEFORE any acceptance path, not just before
    # the lyrics call, so they can never be skipped by a fast-path accept.
    explicit_non_hindi = any(
        marker in metadata_text
        for marker in NON_HINDI_MARKERS
    )

    if explicit_non_hindi:
        return (
            False,
            "reject_non_hindi_metadata",
            "moderate",
            {},
        )

    exact_itunes = itunes_result.get(
        "ok",
        False,
    )

    hindi_context = any(
        marker in metadata_text
        for marker in HINDI_CONTEXT_MARKERS
    )

    if (
        title_hindi_hits >= 2
        and exact_itunes
    ):
        return (
            True,
            "accepted_hindi_title_words",
            "moderate",
            {},
        )

    if title_punjabi_hits > title_hindi_hits:
        return (
            False,
            "reject_punjabi_title_words",
            "moderate",
            {},
        )

    if artist_in_list(
        artist,
        PUNJABI_PRIMARY_ARTISTS,
    ):
        return (
            False,
            "reject_punjabi_primary_artist_no_hindi_proof",
            "moderate",
            {},
        )

    # Known-artist trust.
    # By this point every negative signal has already been checked and
    # cleared: non-Hindi script, explicit English-version title markers,
    # explicit non-Hindi metadata markers (regional languages, devotional,
    # Rajasthani/Bengali folk, etc.), Punjabi title words, and known

    # (Previously this also required title_hindi_hits>=1 and exact_itunes,
    # which wrongly rejected genuine Hindi/Hinglish rap with English-word
    # titles — e.g. "Muqabla", "Company", "Backflip" — and underground
    # tracks that were never flagged as an exact iTunes catalogue match.)
    if artist_in_list(
        artist,
        HINDI_ARTISTS,
    ):
        return (
            True,
            "accepted_hindi_known_artist",
            "moderate",
            {},
        )

    # Genre-aware Hinglish trust path.
    # Desi Hip-Hop/Rap and Hindi Pop routinely ship with fully English
    # titles (rap slang, Western-style track names) while the actual
    # vocals are Hindi or heavy Hindi-English code-switching. Word-list
    # title checks (HINDI_WORDS) systematically miss these, and many
    # underground/independent rap tracks have no lyrics on LRCLIB at all,
    # so requiring title-word or lyrics evidence wrongly rejected a large
    # share of genuine Hindi hip-hop.
    #
 
    row_genre_text = " ".join(
        [
            safe_text(row.get("raw_genre")),
            safe_text(itunes_result.get("raw_genre")),
        ]
    ).casefold()

    is_hinglish_title_genre = any(
        has_phrase(row_genre_text, genre_word)
        for genre_word in HINGLISH_TITLE_GENRES
    )

    country_text_combined = " ".join(
        [
            safe_text(row.get("country")),
            safe_text(itunes_result.get("country")),
        ]
    ).upper()

    country_is_india = (
        has_phrase(country_text_combined, "IN")
        or "INDIA" in country_text_combined
    )

    if (
        is_hinglish_title_genre
        and country_is_india
    ):
        return (
            True,
            "accepted_hindi_itunes_india_hiphop_pop_trust",
            "moderate",
            {},
        )

    # Lyrics are used only for the remaining genuinely ambiguous rows.
    lyrics = fetch_lyrics(
        title,
        artist,
        album,
        row.get("duration_seconds", ""),
    )

    if lyrics.get("ok"):
        other_indic = lyrics.get(
            "other_indic_counts",
            {},
        )

        if lyrics.get("gurmukhi_count", 0) >= 3:
            return (
                False,
                "reject_punjabi_lyrics_script",
                "high",
                lyrics,
            )

        for name, count in other_indic.items():
            # Bengali/Odia/Assamese: even 1 script char is a strong signal
            # because Hindi lyrics are never in Bengali script.
            threshold = 1 if name in ("bengali", "odia", "assamese") else 2
            if count >= threshold:
                return (
                    False,
                    f"reject_{name}_lyrics_script",
                    "high",
                    lyrics,
                )

        if lyrics.get(
            "devanagari_count",
            0,
        ) >= 15:
            return (
                True,
                "accepted_hindi_lyrics_devanagari",
                "high",
                lyrics,
            )

        hindi_hits = lyrics.get(
            "hindi_word_hits",
            0,
        )
        punjabi_hits = lyrics.get(
            "punjabi_word_hits",
            0,
        )

        if (
            punjabi_hits >= 4
            and punjabi_hits
            > hindi_hits * 1.20
        ):
            return (
                False,
                "reject_punjabi_romanised_lyrics",
                "moderate",
                lyrics,
            )

        if (
            hindi_hits >= 6
            and hindi_hits
            >= punjabi_hits * 1.50
        ):
            return (
                True,
                "accepted_hindi_romanised_lyrics",
                "moderate",
                lyrics,
            )

    if title_punjabi_hits > title_hindi_hits:
        return (
            False,
            "reject_punjabi_title_words",
            "moderate",
            lyrics,
        )

    if artist_in_list(
        artist,
        PUNJABI_PRIMARY_ARTISTS,
    ):
        return (
            False,
            "reject_punjabi_primary_artist_no_hindi_proof",
            "moderate",
            lyrics,
        )

    # If lyrics were found but contain zero Hindi evidence
    # (no Devanagari, no Hindi roman words, no Hindi title words),
    # reject even if iTunes metadata says Hindi — the actual audio is wrong-language.
    lyrics_ok = lyrics.get("ok", False)
    lyrics_devanagari = lyrics.get("devanagari_count", 0) if lyrics_ok else -1
    lyrics_hindi_hits_local = lyrics.get("hindi_word_hits", 0) if lyrics_ok else -1
    if (
        lyrics_ok
        and lyrics_devanagari == 0
        and lyrics_hindi_hits_local == 0
        and title_hindi_hits == 0
        and not contains_devanagari(title)
    ):
        return (
            False,
            "reject_lyrics_found_zero_hindi_evidence",
            "high",
            lyrics,
        )

    # TIGHTENED (previously an OR of three weak signals — hindi_context,
    # artist_in_list, or title_hindi_hits alone — any one of which could
    # accept an English/Rajasthani/Bengali track that merely shared an
    # artist or a loosely-tagged genre with a real Hindi song).
    # Now requires actual title-level Hindi word evidence AND
    # (explicit Hindi context OR a known Hindi artist), and only applies
    # when lyrics could not be resolved either way (lyrics not found /
    # inconclusive) so we are not silently overriding a lyrics-based signal.
    if (
        exact_itunes
        and title_hindi_hits >= 1
        and (
            hindi_context
            or artist_in_list(
                artist,
                HINDI_ARTISTS,
            )
        )
    ):
        return (
            True,
            "accepted_hindi_exact_itunes_fallback",
            "low",
            lyrics,
        )

    return (
        False,
        "reject_no_hindi_language_evidence",
        "low",
        lyrics,
    )


def decide_chinese_language(
    row,
    itunes_result,
):
    title = safe_text(row.get("title"))
    artist = safe_text(row.get("artist"))
    album = safe_text(
        row.get("collection")
        or row.get("album")
        or itunes_result.get("collection")
    )

    title_artist = f"{title} {artist}"

    if (
        contains_kana_or_hangul(title_artist)
        and not contains_han(title_artist)
    ):
        return (
            False,
            "reject_japanese_or_korean_script",
            "high",
            {},
        )

    metadata_text = " ".join(
        [
            safe_text(row.get("raw_genre")),
            safe_text(row.get("notes")),
            safe_text(row.get("search_query")),
            safe_text(row.get("collection")),
            safe_text(row.get("album")),
            safe_text(itunes_result.get("raw_genre")),
            safe_text(itunes_result.get("collection")),
        ]
    ).casefold()

    country_text = " ".join(
        [
            safe_text(row.get("country")),
            safe_text(itunes_result.get("country")),
        ]
    ).upper()

    chinese_context = any(
        marker in metadata_text
        for marker in CHINESE_CONTEXT_MARKERS
    )

    chinese_market = any(
        country in country_text
        for country in [
            "CN",
            "TW",
            "HK",
            "SG",
            "MO",
        ]
    )

    if (
        contains_han(title_artist)
        and (
            chinese_context
            or chinese_market
        )
    ):
        return (
            True,
            "accepted_chinese_title_and_context",
            "high",
            {},
        )

    # Fast-accept: Cell 4 rows from Chinese storefronts that have Han
    # characters in the title or artist. Requiring actual CJK characters
    # prevents English-language songs released on HK/TW/CN storefronts
    # from being accepted on market-location alone.
    if chinese_market and contains_han(f"{title} {artist}") and not contains_kana_or_hangul(f"{title} {artist}"):
        return (
            True,
            "accepted_chinese_market_han_title_fast",
            "moderate",
            {},
        )

    lyrics = fetch_lyrics(
        title,
        artist,
        album,
        row.get("duration_seconds", ""),
    )

    if lyrics.get("ok"):
        han_count = lyrics.get(
            "han_count",
            0,
        )
        kana_hangul = lyrics.get(
            "kana_hangul_count",
            0,
        )

        if (
            kana_hangul >= 10
            and kana_hangul > han_count
        ):
            return (
                False,
                "reject_japanese_or_korean_lyrics",
                "high",
                lyrics,
            )

        if han_count >= 20:
            return (
                True,
                "accepted_chinese_lyrics",
                "high",
                lyrics,
            )

    # Metadata fallback: only accept if the title/artist actually
    # contain Han characters — prevents English songs with a Chinese
    # genre tag (e.g. 'cantopop') from passing on metadata alone.
    if chinese_context and chinese_market and contains_han(title_artist):
        return (
            True,
            "accepted_chinese_metadata_fallback",
            "low",
            lyrics,
        )

    return (
        False,
        "reject_no_chinese_language_evidence",
        "low",
        lyrics,
    )


# VERIFY ONE ROW

def choose_final_genre(
    row,
    itunes_result,
    lastfm_result,
):
    original = original_or_seed_genre(row)

    lastfm_genre = lastfm_result.get(
        "mapped_genre"
    )

    itunes_genre = itunes_result.get(
        "mapped_genre"
    )

    # Last.fm tags are matched at track level, so they get first priority.
    if lastfm_genre in GENRES:
        if (
            itunes_genre in GENRES
            and itunes_genre != lastfm_genre
        ):
            return (
                lastfm_genre,
                "lastfm_track_tags_over_conflicting_itunes",
                "moderate",
            )

        return (
            lastfm_genre,
            "lastfm_track_tags",
            "moderate",
        )

    if itunes_genre in GENRES:
        return (
            itunes_genre,
            "itunes_specific_genre_accepted",
            "moderate",
        )

    if original in GENRES:
        return (
            original,
            "cell4_provisional_genre",
            "provisional",
        )

    return (
        None,
        "no_usable_genre",
        "none",
    )


def verify_row(row, language_name):
    row_dict = row.to_dict()

    title = safe_text(
        row_dict.get("title")
    )
    artist = safe_text(
        row_dict.get("artist")
    )

    if not title or not artist:
        row_dict["final_filter_reason"] = (
            "reject_missing_title_or_artist"
        )
        return False, row_dict

    bad_reason = bad_track_reason(row_dict)

    if bad_reason:
        row_dict["final_filter_reason"] = bad_reason
        return False, row_dict

    itunes_result = verify_itunes_if_needed(
        row_dict,
        language_name,
    )

    if language_name == "Hindi":
        (
            language_ok,
            language_reason,
            language_confidence,
            lyrics_result,
        ) = decide_hindi_language(
            row_dict,
            itunes_result,
        )
    else:
        (
            language_ok,
            language_reason,
            language_confidence,
            lyrics_result,
        ) = decide_chinese_language(
            row_dict,
            itunes_result,
        )

    add_lyrics_columns(
        row_dict,
        lyrics_result,
    )

    row_dict["language_decision"] = (
        language_reason
    )
    row_dict["language_confidence"] = (
        language_confidence
    )

    if not language_ok:
        row_dict["final_filter_reason"] = (
            language_reason
        )
        row_dict["verification_status"] = (
            "rejected_language_check"
        )
        return False, row_dict

    existing_genre = original_or_seed_genre(
        row_dict
    )
    itunes_genre = itunes_result.get(
        "mapped_genre"
    )

    needs_lastfm = bool(
        LASTFM_ENABLED
        and (
            not LASTFM_ONLY_FOR_UNRESOLVED_GENRE
            or (
                existing_genre not in GENRES
                and itunes_genre not in GENRES
            )
        )
    )

    if needs_lastfm:
        lastfm_result = lastfm_track_info(
            title,
            artist,
        )
    else:
        lastfm_result = {
            "ok": False,
            "status": (
                "lastfm_skipped_genre_already_available"
                if LASTFM_ENABLED
                else "lastfm_disabled"
            ),
            "tags": [],
            "mapped_genre": None,
            "scores": {},
        }

    (
        final_genre,
        genre_reason,
        genre_confidence,
    ) = choose_final_genre(
        row_dict,
        itunes_result,
        lastfm_result,
    )

    row_dict["original_final_genre"] = safe_text(
        row_dict.get("final_genre")
    )

    row_dict["lastfm_track_status"] = (
        lastfm_result.get("status", "")
    )
    row_dict["lastfm_track_tags"] = ", ".join(
        lastfm_result.get("tags", [])
    )
    row_dict["lastfm_track_genre"] = (
        lastfm_result.get("mapped_genre")
        or ""
    )
    row_dict["lastfm_track_scores"] = json.dumps(
        lastfm_result.get("scores", {}),
        ensure_ascii=False,
        sort_keys=True,
    )

    row_dict["itunes_check_status"] = (
        itunes_result.get("status", "")
    )
    row_dict["itunes_mapped_genre"] = (
        itunes_result.get("mapped_genre")
        or ""
    )

    if final_genre not in GENRES:
        row_dict["final_filter_reason"] = (
            "reject_no_usable_genre"
        )
        row_dict["verification_status"] = (
            "rejected_genre_check"
        )
        return False, row_dict

    row_dict["final_genre"] = final_genre
    row_dict["genre_decision"] = genre_reason
    row_dict["genre_confidence"] = genre_confidence
    row_dict["language"] = language_name
    row_dict["language_code"] = (
        "HI"
        if language_name == "Hindi"
        else "ZH"
    )
    row_dict["label"] = (
        safe_text(row_dict.get("label"))
        or "Human"
    )
    row_dict["model"] = (
        safe_text(row_dict.get("model"))
        or "Human"
    )
    row_dict["verification_status"] = (
        "accepted_automatic_language_and_genre_check"
    )
    row_dict["final_filter_reason"] = (
        "accepted_"
        + language_reason
        + "__"
        + genre_reason
    )

    return True, row_dict


# PROCESS ONE LANGUAGE

def read_checkpoint_csv(path):
    path = Path(path)

    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()

    try:
        return pd.read_csv(
            path,
            low_memory=False,
        )
    except pd.errors.EmptyDataError:
        return pd.DataFrame()
    except Exception:
        return pd.DataFrame()


def row_key_from_values(title, artist):
    return (
        normalise_text(title)
        + "||"
        + normalise_text(artist)
    )


def dataframe_keys(df):
    if df is None or len(df) == 0:
        return set()

    if "title" not in df.columns or "artist" not in df.columns:
        return set()

    return {
        row_key_from_values(
            title,
            artist,
        )
        for title, artist in zip(
            df["title"],
            df["artist"],
        )
        if normalise_text(title)
        and normalise_text(artist)
    }


def _recheck_hindi_row_no_network(row_dict):
    """
    Re-run the Hindi language decision using ONLY the script-count and
    roman-word-hit columns that were already saved on the row by the
    previous Cell 5B run.  Zero network calls — fast enough for 2000+ rows.

    Returns (keep: bool, reason: str)
    """
    title  = safe_text(row_dict.get("title", ""))
    artist = safe_text(row_dict.get("artist", ""))
    metadata_text = " ".join([
        safe_text(row_dict.get("raw_genre", "")),
        safe_text(row_dict.get("notes", "")),
        safe_text(row_dict.get("collection", "")),
        safe_text(row_dict.get("album", "")),
    ]).casefold()

    title_artist = f"{title} {artist}"

    # Hard script rejections (title/artist)
    if contains_han(title_artist) or contains_kana_or_hangul(title_artist):
        return False, "recheck_reject_non_hindi_script"
    if contains_gurmukhi(title_artist):
        return False, "recheck_reject_gurmukhi_title"
    for name, pattern in OTHER_INDIC_PATTERNS.items():
        if re.search(pattern, title_artist):
            return False, f"recheck_reject_{name}_title_script"

    # Metadata non-Hindi markers
    for marker in NON_HINDI_MARKERS:
        if has_phrase(metadata_text, marker):
            return False, f"recheck_reject_non_hindi_marker:{marker}"

    # Read pre-computed lyrics columns (stored from previous run)
    def _int(col):
        try:
            v = row_dict.get(col, 0)
            return int(float(v)) if str(v).strip() not in ("", "nan", "None") else 0
        except Exception:
            return 0

    lyrics_ok        = safe_text(row_dict.get("lyrics_status", "")).startswith("lyrics_match")
    dev_count        = _int("lyrics_devanagari_count")
    gurm_count       = _int("lyrics_gurmukhi_count")
    hindi_hits       = _int("lyrics_hindi_word_hits")
    punjabi_hits     = _int("lyrics_punjabi_word_hits")
    other_indic      = {
        name: _int(f"lyrics_{name}_count")
        for name in OTHER_INDIC_PATTERNS
    }

    # Re-apply tightened Bengali/Odia/Assamese threshold (now 1, was 3)
    if gurm_count >= 1:
        return False, f"recheck_reject_punjabi_lyrics_script:{gurm_count}"
    for name, count in other_indic.items():
        threshold = 1 if name in ("bengali", "odia", "assamese") else 2
        if count >= threshold:
            return False, f"recheck_reject_{name}_lyrics_script:{count}"

    title_hindi_hits, title_punjabi_hits = title_word_counts(title)

    # Punjabi title/artist
    if title_punjabi_hits > title_hindi_hits:
        return False, "recheck_reject_punjabi_title_words"
    if artist_in_list(artist, PUNJABI_PRIMARY_ARTISTS):
        return False, "recheck_reject_punjabi_artist"

    # Zero-Hindi-evidence guard (new strict rule):
    # If lyrics were found but contain no Devanagari and no Hindi roman
    # words and the title has no Hindi words → reject.
    if (
        lyrics_ok
        and dev_count == 0
        and hindi_hits == 0
        and title_hindi_hits == 0
        and not contains_devanagari(title)
    ):
        return False, "recheck_reject_lyrics_zero_hindi_evidence"

    # If Punjabi roman evidence dominates
    if (
        punjabi_hits >= 4
        and punjabi_hits > hindi_hits * 1.20
    ):
        return False, f"recheck_reject_punjabi_roman:{punjabi_hits}>{hindi_hits}"

    return True, "recheck_pass"


def locally_clean_resumed_kept(
    kept_df,
    language_name=None,
):
    """
    Re-apply all current filters to old kept rows — NO network calls.

    1. bad_track_reason          — live/remix/karaoke/instrumental.
    2. _recheck_hindi_row_no_network — re-runs the Hindi language decision
       using script-count columns already stored on the row, so the new
       stricter rules (Bengali threshold, Rajasthani marker, zero-Hindi
       evidence guard) apply to cached rows without any API calls.
    """
    if kept_df is None or len(kept_df) == 0:
        return pd.DataFrame(), pd.DataFrame()

    still_kept = []
    newly_rejected = []

    for _, row in kept_df.iterrows():
        row_dict = row.to_dict()

        # Step 1: bad-version / playlist filter (fast, no network)
        reason = bad_track_reason(row_dict)
        if reason:
            row_dict["final_filter_reason"] = reason
            row_dict["verification_status"] = (
                "rejected_during_resume_local_recheck"
            )
            newly_rejected.append(row_dict)
            continue

        # Step 2: Hindi-only fast language recheck (no network)
        if language_name == "Hindi":
            keep, recheck_reason = _recheck_hindi_row_no_network(row_dict)
            if not keep:
                row_dict["final_filter_reason"] = recheck_reason
                row_dict["verification_status"] = (
                    "rejected_during_resume_language_recheck"
                )
                newly_rejected.append(row_dict)
                continue

        still_kept.append(row_dict)

    return (
        pd.DataFrame(still_kept),
        pd.DataFrame(newly_rejected),
    )


def combine_unique_rows(
    first,
    second,
):
    frames = [
        frame
        for frame in [first, second]
        if frame is not None
        and len(frame) > 0
    ]

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(
        frames,
        ignore_index=True,
        sort=False,
    )

    if (
        "title" not in combined.columns
        or "artist" not in combined.columns
    ):
        return combined

    combined["_resume_key"] = [
        row_key_from_values(
            title,
            artist,
        )
        for title, artist in zip(
            combined["title"],
            combined["artist"],
        )
    ]

    combined = (
        combined.drop_duplicates(
            subset=["_resume_key"],
            keep="last",
        )
        .drop(
            columns=["_resume_key"],
            errors="ignore",
        )
        .reset_index(drop=True)
    )

    return combined


def process_language(df, language_name):
    kept_csv = (
        MANIFEST_DIR
        / f"{language_name.lower()}_strict_verified_live.csv"
    )

    rejected_csv = (
        MANIFEST_DIR
        / f"{language_name.lower()}_strict_rejected_live.csv"
    )

    review_csv = (
        MANIFEST_DIR
        / f"{language_name.lower()}_strict_manual_review_live.csv"
    )

    work_df = df.copy()

    if len(work_df) == 0:
        pd.DataFrame().to_csv(
            kept_csv,
            index=False,
            encoding="utf-8-sig",
        )
        pd.DataFrame().to_csv(
            rejected_csv,
            index=False,
            encoding="utf-8-sig",
        )
        pd.DataFrame().to_csv(
            review_csv,
            index=False,
            encoding="utf-8-sig",
        )
        return pd.DataFrame(), pd.DataFrame()

    work_df["_key"] = (
        work_df["title"]
        .apply(normalise_text)
        + "||"
        + work_df["artist"]
        .apply(normalise_text)
    )

    work_df = (
        work_df.drop_duplicates(
            subset=["_key"],
            keep="first",
        )
        .reset_index(drop=True)
    )

    total_unique = len(work_df)

    existing_kept = pd.DataFrame()
    existing_rejected = pd.DataFrame()

    if RESUME_FROM_CHECKPOINTS:
        existing_kept = read_checkpoint_csv(
            kept_csv
        )
        existing_rejected = read_checkpoint_csv(
            rejected_csv
        )

        (
            existing_kept,
            invalid_old_kept,
        ) = locally_clean_resumed_kept(
            existing_kept,
            language_name=language_name,
        )

        existing_rejected = combine_unique_rows(
            existing_rejected,
            invalid_old_kept,
        )

    processed_keys = (
        dataframe_keys(existing_kept)
        | dataframe_keys(existing_rejected)
    )

    remaining_df = work_df[
        ~work_df["_key"].isin(
            processed_keys
        )
    ].copy()

    remaining_df = (
        remaining_df.drop(
            columns=["_key"],
            errors="ignore",
        )
        .reset_index(drop=True)
    )

    kept_rows = (
        existing_kept.to_dict("records")
        if len(existing_kept)
        else []
    )

    rejected_rows = (
        existing_rejected.to_dict("records")
        if len(existing_rejected)
        else []
    )

    already_done = len(processed_keys)

    print(
        f"\nChecking {language_name}: "
        f"{total_unique} unique tracks"
    )
    print(
        f"Resume: {already_done} already processed | "
        f"{len(remaining_df)} remaining"
    )

    if len(remaining_df) == 0:
        kept_df = combine_unique_rows(
            pd.DataFrame(kept_rows),
            pd.DataFrame(),
        )
        rejected_df = combine_unique_rows(
            pd.DataFrame(rejected_rows),
            pd.DataFrame(),
        )

        print(
            f"{language_name} already complete: "
            f"kept {len(kept_df)}, "
            f"rejected {len(rejected_df)}"
        )

        return kept_df, rejected_df

    progress_bar = tqdm(
        total=total_unique,
        initial=min(already_done, total_unique),
        desc=language_name,
        unit="track",
        dynamic_ncols=True,
    )

    newly_processed = 0

    # Use a thread pool to overlap I/O waits (lyrics + Last.fm calls).
    # verify_row is I/O-bound so threads give a real speedup even with GIL.
    from concurrent.futures import ThreadPoolExecutor, as_completed
    _VERIFY_WORKERS = 8

    rows_list = list(remaining_df.iterrows())

    def _safe_verify(idx_row):
        _, row = idx_row
        try:
            return verify_row(row, language_name)
        except Exception as error:
            checked = row.to_dict()
            checked["final_filter_reason"] = (
                "reject_processing_error_" + type(error).__name__
            )
            checked["processing_error"] = str(error)
            return False, checked

    with ThreadPoolExecutor(max_workers=_VERIFY_WORKERS) as _pool:
        futures = {_pool.submit(_safe_verify, idx_row): idx_row for idx_row in rows_list}
        for future in as_completed(futures):
            ok, checked_row = future.result()
            current_title = safe_text(checked_row.get("title"))
            current_artist = safe_text(checked_row.get("artist"))

            progress_bar.set_postfix_str(
                f"{current_title[:28]} - {current_artist[:22]}",
                refresh=False,
            )
            progress_bar.update(1)

            if ok:
                kept_rows.append(checked_row)
            else:
                rejected_rows.append(checked_row)

            newly_processed += 1
            checked_total = already_done + newly_processed

            if (
                PRINT_PROGRESS_EVERY
                and newly_processed % PRINT_PROGRESS_EVERY == 0
            ):
                tqdm.write(
                    f"{language_name} "
                    f"{checked_total}/{total_unique} | "
                    f"{'KEEP' if ok else 'REJECT'} | "
                    f"{current_title} - {current_artist} | "
                    f"kept {len(kept_rows)} | "
                    f"rejected {len(rejected_rows)}"
                )

            if (
                CHECKPOINT_EVERY
                and newly_processed % CHECKPOINT_EVERY == 0
            ):
                kept_checkpoint = combine_unique_rows(
                    pd.DataFrame(kept_rows), pd.DataFrame()
                )
                rejected_checkpoint = combine_unique_rows(
                    pd.DataFrame(rejected_rows), pd.DataFrame()
                )
                kept_checkpoint.to_csv(kept_csv, index=False, encoding="utf-8-sig")
                rejected_checkpoint.to_csv(rejected_csv, index=False, encoding="utf-8-sig")
                save_all_caches()

    kept_df = combine_unique_rows(
        pd.DataFrame(kept_rows),
        pd.DataFrame(),
    )

    rejected_df = combine_unique_rows(
        pd.DataFrame(rejected_rows),
        pd.DataFrame(),
    )

    # A row must exist in only one final file.
    rejected_keys = dataframe_keys(
        rejected_df
    )

    if len(kept_df):
        kept_df["_final_key"] = [
            row_key_from_values(
                title,
                artist,
            )
            for title, artist in zip(
                kept_df["title"],
                kept_df["artist"],
            )
        ]

        kept_df = (
            kept_df[
                ~kept_df["_final_key"].isin(
                    rejected_keys
                )
            ]
            .drop(
                columns=["_final_key"],
                errors="ignore",
            )
            .reset_index(drop=True)
        )

    kept_df.to_csv(
        kept_csv,
        index=False,
        encoding="utf-8-sig",
    )

    rejected_df.to_csv(
        rejected_csv,
        index=False,
        encoding="utf-8-sig",
    )

    pd.DataFrame().to_csv(
        review_csv,
        index=False,
        encoding="utf-8-sig",
    )

    save_all_caches()

    print(
        f"{language_name} complete: "
        f"kept {len(kept_df)}, "
        f"rejected {len(rejected_df)}"
    )

    if len(kept_df):
        print(
            kept_df["final_genre"]
            .value_counts()
            .reindex(
                GENRES,
                fill_value=0,
            )
        )

        print(
            f"\nSample kept {language_name} songs:"
        )

        display(
            kept_df[
                [
                    "title",
                    "artist",
                    "final_genre",
                    "language_decision",
                    "genre_decision",
                ]
            ].head(20)
        )

    return kept_df, rejected_df


# RUN

print(
    "Cell 5B fast resume: local checks first, "
    "lyrics only when needed, Last.fm only for unresolved genre"
)
print(
    "Resume checkpoints:",
    RESUME_FROM_CHECKPOINTS,
)

hindi_verified, hindi_rejected = process_language(
    hindi_itunes_candidates,
    "Hindi",
)

chinese_verified, chinese_rejected = process_language(
    chinese_itunes_candidates,
    "Chinese",
)

# Empty variables are retained for compatibility.
hindi_manual_review = pd.DataFrame()
chinese_manual_review = pd.DataFrame()

# Same variable names used by Cell 6.
hindi_itunes_candidates = hindi_verified
chinese_itunes_candidates = chinese_verified

print(
    "\nReady for Cell 6."
)

print(
    "API calls:",
    dict(API_CALLS),
)

## Strict Genre Balancing

> **Methodology Note:** To prevent model bias during training, the dataset must be perfectly balanced. This step takes the strictly verified candidate pools and randomly samples exactly 60 tracks per genre for each language. It also assigns the final, standardized `song_id` (e.g., `HUMAN_HI_POP_001`) to each track.

In [ ]:
# CELL 6: DETERMINISTIC GENRE BALANCING (PER-GENRE ARTIST CAP)

import pandas as pd

TARGET_PER_GENRE = 60

# Default cap for most genres. Override per-genre below for any
# genre whose artist pool is thin (commonly Folk/Acoustic).
# Default cap used when no language-specific override exists.
STRICT_MAX_SONGS_PER_ARTIST = 2

# Per-language, per-genre caps.
#
# English: cap=2 — pool is smaller (~56 distinct Pop artists).
# Hindi:   cap=1 — pool is very large (150+ artists per genre);
#                   cap=1 gives exactly 60 distinct artists per genre.
# Chinese: cap=2 — pool size similar to English.
#
# Folk/Acoustic is relaxed for all languages (thinner pool).
LANGUAGE_ARTIST_CAPS = {
    "English": {
        "Pop": 2,
        "Rock": 2,
        "Hip-Hop/Rap": 2,
        "R&B/Soul": 2,
        "Folk/Acoustic": 2,
    },
    "Hindi": {
        "Pop": 1,
        "Rock": 2,
        "Hip-Hop/Rap": 1,
        "R&B/Soul": 1,
        "Folk/Acoustic": 2,
    },
    "Chinese": {
        "Pop": 2,
        "Rock": 2,
        "Hip-Hop/Rap": 2,
        "R&B/Soul": 2,
        "Folk/Acoustic": 2,
    },
}

# Legacy flat dict kept for reference; overridden by LANGUAGE_ARTIST_CAPS.
STRICT_MAX_SONGS_PER_ARTIST_BY_GENRE = {
    "Pop": 2,
    "Rock": 2,
    "Hip-Hop/Rap": 2,
    "R&B/Soul": 2,
    "Folk/Acoustic": 2,
}


def clean_artist_name(value):
    """Return the primary (lead) artist name, casefolded.

    Strips everything after the first "&", ",", "feat.", "ft.",
    "featuring", "with", or whole-word "x" so that
    "Abhijeet Srivastava & Tony Kakkar" and
    "Abhijeet Srivastava & Shloke Lal" both count toward the
    same per-artist cap as "Abhijeet Srivastava".
    """
    text = str(value).strip()
    primary = re.split(
        r'[&,]|\bfeat\.?\b|\bft\.?\b|\bfeaturing\b|\bwith\b|\bx\b',
        text,
        maxsplit=1,
        flags=re.IGNORECASE,
    )[0].strip()
    return (primary or text).casefold()


def clean_song_key(title, artist):
    return normalise_for_match(title) + " | " + normalise_for_match(artist)


def round_robin_artist_sample(part, target, artist_cap, random_state=42):
    """Maximise artist diversity before taking additional tracks."""
    grouped_rows = {
        artist_name: (
            artist_frame.sample(frac=1, random_state=random_state)
            .to_dict("records")
        )
        for artist_name, artist_frame in part.groupby(
            "artist_clean",
            sort=True,
        )
    }

    selected = []

    for round_number in range(artist_cap):
        for artist_name in sorted(grouped_rows):
            rows = grouped_rows[artist_name]
            if round_number < len(rows):
                selected.append(rows[round_number])

            if len(selected) == target:
                return pd.DataFrame(selected)

    return pd.DataFrame(selected)


def balance_candidates_to_60(df, language):
    if df is None or len(df) == 0:
        raise ValueError(f"No candidate data found for {language}.")

    frame = df.copy()
    language_code = LANGUAGE_CODES[language]

    required_columns = {"title", "artist", "final_genre"}
    missing_columns = sorted(required_columns - set(frame.columns))

    if missing_columns:
        raise ValueError(
            f"{language}: missing required columns: "
            + ", ".join(missing_columns)
        )

    frame["song_match_key"] = frame.apply(
        lambda row: clean_song_key(
            row.get("title", ""),
            row.get("artist", ""),
        ),
        axis=1,
    )

    before_duplicates = len(frame)
    frame = frame.drop_duplicates(
        subset=["song_match_key"],
        keep="first",
    ).copy()

    print(
        f"\n{language}: duplicate songs removed = "
        f"{before_duplicates - len(frame)}"
    )

    frame["artist_clean"] = frame["artist"].apply(clean_artist_name)

    balanced_parts = []
    balance_report = []

    print(f"\nCandidate counts for {language}:")
    print(
        frame["final_genre"]
        .value_counts()
        .reindex(GENRES, fill_value=0)
    )

    for genre in GENRES:
        part = frame[frame["final_genre"] == genre].copy()

        _lang_caps = LANGUAGE_ARTIST_CAPS.get(language, {})
        genre_cap = _lang_caps.get(
            genre,
            STRICT_MAX_SONGS_PER_ARTIST_BY_GENRE.get(
                genre,
                STRICT_MAX_SONGS_PER_ARTIST,
            ),
        )

        if len(part) < TARGET_PER_GENRE:
            raise ValueError(
                f"{language} {genre} has only {len(part)} unique "
                f"candidates. Need at least {TARGET_PER_GENRE}."
            )

        # STRICT CAP LOGIC: check if it's mathematically possible to
        # hit the target using this genre's artist cap.
        # If the preferred cap is too tight, keep raising it by 1 until
        # the target is reachable or we exhaust the full pool. This uses
        # the strictest cap that actually works rather than hard-failing.
        artist_sizes = part.groupby("artist_clean").size().astype(int)
        possible_with_strict_cap = int(
            artist_sizes.clip(upper=genre_cap).sum()
        )

        if possible_with_strict_cap < TARGET_PER_GENRE:
            unique_artists = int(artist_sizes.shape[0])
            max_possible = int(artist_sizes.sum())  # all songs, no cap

            if max_possible < TARGET_PER_GENRE:
                raise ValueError(
                    f"STRICT CAP FAILURE: {language} {genre} has only "
                    f"{max_possible} songs across {unique_artists} artists "
                    f"in total — cannot reach {TARGET_PER_GENRE} "
                    f"regardless of cap. Add more candidates for this genre."
                )

            # Increment cap until target is reachable.
            trial_cap = genre_cap + 1
            while int(artist_sizes.clip(upper=trial_cap).sum()) < TARGET_PER_GENRE:
                trial_cap += 1

            print(
                f"\n[AUTO-FALLBACK] {language} {genre}: "
                f"preferred cap={genre_cap} can only reach "
                f"{possible_with_strict_cap} "
                f"({unique_artists} distinct primary artists). "
                f"Using cap={trial_cap} to reach {TARGET_PER_GENRE}."
            )
            genre_cap = trial_cap

        print(f"OK | {language} | {genre}: artist cap = {genre_cap}.")

        selected = round_robin_artist_sample(
            part,
            TARGET_PER_GENRE,
            genre_cap,
            random_state=RANDOM_STATE,
        )

        if len(selected) != TARGET_PER_GENRE:
            raise RuntimeError(
                f"{language} {genre}: selected {len(selected)}, "
                f"expected {TARGET_PER_GENRE}."
            )

        balance_report.append(
            {
                "language": language,
                "genre": genre,
                "available_candidates": len(part),
                "unique_artists_selected": int(
                    selected["artist_clean"].nunique()
                ),
                "artist_cap_used": genre_cap,
                "maximum_selected_per_artist": int(
                    selected["artist_clean"].value_counts().max()
                ),
            }
        )

        balanced_parts.append(selected)

    output = pd.concat(
        balanced_parts,
        ignore_index=True,
        sort=False,
    )

    final_rows = []

    for genre in GENRES:
        genre_rows = (
            output[output["final_genre"] == genre]
            .copy()
            .reset_index(drop=True)
        )

        for sequence, (_, row) in enumerate(
            genre_rows.iterrows(),
            start=1,
        ):
            row = row.copy()
            row["song_id"] = make_song_id(
                language_code,
                "Human",
                genre,
                sequence,
            )
            row["language"] = language
            row["language_code"] = language_code
            row["label"] = "Human"
            row["model"] = "Human"
            final_rows.append(row)

    output = pd.DataFrame(final_rows)
    output = output.drop(
        columns=["song_match_key", "artist_clean"],
        errors="ignore",
    )
    output = standardise_schema(output)

    report_path = (
        MANIFEST_DIR
        / f"{language.casefold()}_artist_balance_report.csv"
    )
    pd.DataFrame(balance_report).to_csv(
        report_path,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"\nArtist balance report saved: {report_path}")
    print(
        "Duplicate title + artist rows:",
        int(output.duplicated(subset=["title", "artist"]).sum()),
    )

    check_counts(output, f"{language} Balanced Human Metadata")
    return output

### Generate the three balanced manifests

A successful language must contain exactly 60 rows in each of the five genres and 300 rows in total.


In [ ]:
# Load fresh from disk to guarantee we're balancing the LATEST saved
# strict candidates file (not a stale in-memory variable, and not
# accidentally a backup_*.csv file).
english_candidates = pd.read_csv(
    MANIFEST_DIR / "english_real_lastfm_itunes_strict_candidates.csv"
)

print("Loaded rows:", len(english_candidates))
print(english_candidates["final_genre"].value_counts())

english_human_metadata = balance_candidates_to_60(english_candidates, "English")

english_human_metadata.to_csv(
    MANIFEST_DIR / "english_human_metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
hindi_human_metadata = balance_candidates_to_60(hindi_itunes_candidates, "Hindi")

hindi_human_metadata.to_csv(
    MANIFEST_DIR / "hindi_human_metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
chinese_human_metadata = balance_candidates_to_60(chinese_itunes_candidates, "Chinese")

chinese_human_metadata.to_csv(
    MANIFEST_DIR / "chinese_human_metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

## 8. Final metadata confirmation and repair

The corrected version uses `count_genres_frame()` so a later pandas Series named `genre_counts` cannot overwrite the function.


In [ ]:
# FINAL 300+300 CONFIRMATION AND AUTOMATIC REPAIR


import json
import os
import random
import re
import shutil
import time
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime
from difflib import SequenceMatcher
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm


# SETTINGS

MANIFEST_DIR = Path(
    globals().get(
        "MANIFEST_DIR",
        Path("dataset_collection_clean") / "manifests",
    )
)

CACHE_DIR = Path(
    globals().get(
        "CACHE_DIR",
        Path("dataset_collection_clean") / "api_cache",
    )
)

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

GENRES = [
    "Pop",
    "Rock",
    "Hip-Hop/Rap",
    "R&B/Soul",
    "Folk/Acoustic",
]

USE_LYRICS_LANGUAGE_CHECK = True
USE_LASTFM_GENRE_CHECK = True

# Cell 4 already contains exact iTunes/Apple rows.
# Network iTunes rechecking is therefore unnecessary here.
RECHECK_ITUNES = True

# Resume only checkpoints created by this exact strict version.
RESUME_FROM_CHECKPOINTS = True
CELL5B_VERSION = "cell7_final_hindi_chinese_repair_v5_known_errors"

# This cell runs after Cell 6.
# It audits the final 300 Hindi + 300 Chinese rows first.
# Older CSV candidates are checked only to replace invalid rows.
TARGET_PER_GENRE = 60
ENFORCE_ARTIST_LIMIT_IN_FINAL_CONFIRMATION = False

# Ignore local-audio-only rows while rescanning previous metadata.
EXCLUDE_LOCAL_AUDIO_ROWS = True

# iTunes alone is not enough because its catalogue genre can be broad.
# Track-level Last.fm tags are preferred. When unavailable, cached
# Last.fm artist tags may support the existing/iTunes genre.
REQUIRE_LASTFM_GENRE_EVIDENCE = True
USE_LASTFM_ARTIST_FALLBACK = True
REJECT_ITUNES_LASTFM_CONFLICT = False

# Spotify is an additional catalogue/identity signal.
# Spotify artist genres are secondary evidence only.
USE_SPOTIFY_SUPPORT = True
SPOTIFY_SEARCH_LIMIT = 10

SPOTIFY_MARKETS = {
    "Hindi": ["IN"],
    "Chinese": ["CN", "TW", "HK"],
}

# The two final metadata CSVs are overwritten only when both
# languages pass every final check and contain exactly 5 x 60 rows.
OVERWRITE_ONLY_WHEN_BOTH_PERFECT = True
CREATE_TIMESTAMPED_BACKUPS = True

# Search these locations for older metadata/candidate CSVs.
CANDIDATE_SEARCH_DIRS = [
    Path.cwd(),
    MANIFEST_DIR,
    Path("dataset_collection_clean"),
    Path("dataset_collection_clean") / "manifests",
    Path("Notebook"),
    Path("/mnt/data"),
]

# Language optimisation:
#   clear language evidence is accepted locally
#   lyrics are used for genuinely uncertain tracks
# Genre verification:
#   every language-accepted track is checked on Last.fm
#   non-iTunes catalogue rows are searched on iTunes
FAST_MODE = True
LASTFM_ONLY_FOR_UNRESOLVED_GENRE = False

CHECKPOINT_EVERY = 5
PRINT_PROGRESS_EVERY = 10

# Do not wait repeatedly on missing lyrics or temporary API failures.
HTTP_TIMEOUT = 8
MAX_RETRIES = 1
MAX_429_WAIT_SECONDS = 5

LRCLIB_SEARCH_URL = "https://lrclib.net/api/search"
LASTFM_URL = "https://ws.audioscrobbler.com/2.0/"
ITUNES_SEARCH_URL = "https://itunes.apple.com/search"

SPOTIFY_TOKEN_URL = "https://accounts.spotify.com/api/token"
SPOTIFY_SEARCH_URL = "https://api.spotify.com/v1/search"
SPOTIFY_ARTIST_URL = "https://api.spotify.com/v1/artists/{artist_id}"

LYRICS_CACHE_FILE = CACHE_DIR / "cell5b_lrclib_language_cache.json"
LASTFM_CACHE_FILE = CACHE_DIR / "cell5b_lastfm_track_cache.json"
LASTFM_ARTIST_CACHE_FILE = CACHE_DIR / "cell5b_lastfm_artist_cache.json"
ITUNES_CACHE_FILE = CACHE_DIR / "cell5b_itunes_verify_cache.json"
SPOTIFY_TRACK_CACHE_FILE = CACHE_DIR / "cell7_spotify_track_cache.json"
SPOTIFY_ARTIST_CACHE_FILE = CACHE_DIR / "cell7_spotify_artist_cache.json"

LASTFM_API_KEY = str(
    globals().get("LASTFM_API_KEY", "")
    or os.getenv("LASTFM_API_KEY", "")
).strip()

if USE_LASTFM_GENRE_CHECK and not LASTFM_API_KEY:
    LASTFM_API_KEY = getpass(
        "Enter Last.fm API key or press Enter to skip: "
    ).strip()

LASTFM_ENABLED = bool(
    USE_LASTFM_GENRE_CHECK and LASTFM_API_KEY
)

SPOTIFY_CLIENT_ID = str(
    globals().get("SPOTIFY_CLIENT_ID", "")
    or globals().get("SPOTIPY_CLIENT_ID", "")
    or os.getenv("SPOTIFY_CLIENT_ID", "")
    or os.getenv("SPOTIPY_CLIENT_ID", "")
).strip()

SPOTIFY_CLIENT_SECRET = str(
    globals().get("SPOTIFY_CLIENT_SECRET", "")
    or globals().get("SPOTIPY_CLIENT_SECRET", "")
    or os.getenv("SPOTIFY_CLIENT_SECRET", "")
    or os.getenv("SPOTIPY_CLIENT_SECRET", "")
).strip()

SPOTIFY_ACCESS_TOKEN = str(
    globals().get("SPOTIFY_ACCESS_TOKEN", "")
    or globals().get("SPOTIPY_ACCESS_TOKEN", "")
    or os.getenv("SPOTIFY_ACCESS_TOKEN", "")
    or os.getenv("SPOTIPY_ACCESS_TOKEN", "")
).strip()

# Spotify account email/password is never required.
SPOTIFY_ENABLED = bool(
    USE_SPOTIFY_SUPPORT
    and (
        SPOTIFY_ACCESS_TOKEN
        or (
            SPOTIFY_CLIENT_ID
            and SPOTIFY_CLIENT_SECRET
        )
    )
)

SPOTIFY_TOKEN_STATE = {
    "access_token": SPOTIFY_ACCESS_TOKEN,
    "expires_at": (
        float("inf")
        if SPOTIFY_ACCESS_TOKEN
        else 0.0
    ),
}

SESSION = requests.Session()
SESSION.headers.update(
    {
        "User-Agent": (
            "MScMusicDataset/1.0 "
            "student-research-language-check"
        ),
        "Accept": "application/json",
    }
)

API_CALLS = defaultdict(int)


# CACHE

def load_cache(path):
    path = Path(path)

    if not path.exists():
        return {}

    try:
        value = json.loads(
            path.read_text(encoding="utf-8")
        )
        return value if isinstance(value, dict) else {}
    except Exception:
        return {}


def save_cache(cache, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary.write_text(
        json.dumps(
            cache,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    temporary.replace(path)


LYRICS_CACHE = load_cache(LYRICS_CACHE_FILE)
LASTFM_CACHE = load_cache(LASTFM_CACHE_FILE)
LASTFM_ARTIST_CACHE = load_cache(
    LASTFM_ARTIST_CACHE_FILE
)
ITUNES_CACHE = load_cache(ITUNES_CACHE_FILE)
SPOTIFY_TRACK_CACHE = load_cache(
    SPOTIFY_TRACK_CACHE_FILE
)
SPOTIFY_ARTIST_CACHE = load_cache(
    SPOTIFY_ARTIST_CACHE_FILE
)


def save_all_caches():
    save_cache(
        LYRICS_CACHE,
        LYRICS_CACHE_FILE,
    )
    save_cache(
        LASTFM_CACHE,
        LASTFM_CACHE_FILE,
    )
    save_cache(
        LASTFM_ARTIST_CACHE,
        LASTFM_ARTIST_CACHE_FILE,
    )
    save_cache(
        ITUNES_CACHE,
        ITUNES_CACHE_FILE,
    )
    save_cache(
        SPOTIFY_TRACK_CACHE,
        SPOTIFY_TRACK_CACHE_FILE,
    )
    save_cache(
        SPOTIFY_ARTIST_CACHE,
        SPOTIFY_ARTIST_CACHE_FILE,
    )


# GENERAL HELPERS

def safe_text(value):
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip()


def normalise_text(value):
    text = unicodedata.normalize(
        "NFKC",
        safe_text(value),
    ).casefold()

    text = re.sub(
        r"\(.*?\)|\[.*?\]",
        " ",
        text,
    )

    text = re.sub(
        r"[^\w\s\u0900-\u097f\u0a00-\u0a7f"
        r"\u3400-\u4dbf\u4e00-\u9fff"
        r"\u3040-\u30ff\uac00-\ud7af]",
        " ",
        text,
    )

    text = re.sub(
        r"_+",
        " ",
        text,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def similarity(left, right):
    left_n = normalise_text(left)
    right_n = normalise_text(right)

    if not left_n or not right_n:
        return 0.0

    if left_n == right_n:
        return 1.0

    return SequenceMatcher(
        None,
        left_n,
        right_n,
    ).ratio()


def split_values(value):
    text = safe_text(value)

    if not text:
        return []

    return [
        item.strip()
        for item in re.split(
            r"\s*[|;,]\s*",
            text,
        )
        if item.strip()
    ]


def has_phrase(text, phrase):
    text = safe_text(text).casefold()
    phrase = safe_text(phrase).casefold()

    if not text or not phrase:
        return False

    if re.search(r"[^\x00-\x7f]", phrase):
        return phrase in text

    pattern = (
        rf"(?<![a-z0-9])"
        rf"{re.escape(phrase)}"
        rf"(?![a-z0-9])"
    )

    return bool(
        re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        )
    )


def script_count(text, pattern):
    return len(
        re.findall(
            pattern,
            safe_text(text),
        )
    )


def contains_devanagari(text):
    return bool(
        re.search(
            r"[\u0900-\u097f]",
            safe_text(text),
        )
    )


def contains_gurmukhi(text):
    return bool(
        re.search(
            r"[\u0a00-\u0a7f]",
            safe_text(text),
        )
    )


def contains_han(text):
    return bool(
        re.search(
            r"[\u3400-\u4dbf\u4e00-\u9fff]",
            safe_text(text),
        )
    )


def contains_kana_or_hangul(text):
    return bool(
        re.search(
            r"[\u3040-\u30ff\uac00-\ud7af]",
            safe_text(text),
        )
    )


def source_is_exact_itunes(row):
    source = safe_text(
        row.get("source_dataset")
    ).casefold()

    url = safe_text(
        row.get("track_view_url")
    ).casefold()

    track_id = safe_text(
        row.get("source_track_id")
    )

    return bool(
        (
            "itunes" in source
            or "apple music" in source
            or "itunes.apple.com" in url
            or "music.apple.com" in url
        )
        and (
            track_id
            or url
            or safe_text(row.get("raw_genre"))
            or safe_text(row.get("collection"))
            or safe_text(row.get("album"))
        )
    )


# BAD TRACK FILTER

BAD_VERSION_PHRASES = [
    "karaoke",
    "instrumental",
    "cover version",
    "cover song",
    "tribute",
    "reaction",
    "tutorial",
    "podcast",
    "medley",
    "compilation",
    "full album",
    "concert",
    "live",
    "live version",
    "live at",
    "live from",
    "acoustic live",
    "dj mix",
    "dj remix",
    "remix",
    "mixtape",
    "mix tape",
    "mashup",
    "nonstop mix",
    "non stop mix",
    "sped up",
    "slowed",
    "nightcore",
    "8d audio",
    "lofi version",
    "lyrics video only",
    "backing track",
    "vocals only",
    "vocal only",
    "club mix",
    "remix",
    "recreated version",
    "recreation",
    "演唱会",
    "演唱會",
    "現場",
    "现场",
    "伴奏",
    "翻唱",
    "合集",
    "纯音乐",
]

PLAYLIST_PHRASES = [
    "playlist",
    "jukebox",
    "all songs",
    "top songs",
    "best songs",
    "hit songs",
    "superhit songs",
    "song collection",
    "hits collection",
    "non stop",
    "non-stop",
]


def bad_track_reason(row):
    title = safe_text(row.get("title"))
    artist = safe_text(row.get("artist"))
    album = safe_text(
        row.get("collection")
        or row.get("album")
    )

    title_album = f"{title} {album}"
    all_text = f"{title} {artist} {album}"

    for phrase in BAD_VERSION_PHRASES:
        if has_phrase(title_album, phrase):
            return (
                "reject_bad_version_"
                + normalise_text(phrase)
                .replace(" ", "_")
            )

    for phrase in PLAYLIST_PHRASES:
        if has_phrase(all_text, phrase):
            return "reject_playlist_or_compilation"

    if re.search(
        r"\btop\s*\d+\b.*\b(?:songs?|tracks?|hits?)\b",
        all_text.casefold(),
    ):
        return "reject_numbered_playlist"

    if re.match(
        r"^\s*\d{1,3}\s*[.)]\s+",
        title,
    ):
        return "reject_numbered_track_title"

    return ""


def _validate_bad_version_filter():
    reject_cases = [
        {
            "title": "Kahaan (Live)",
            "artist": "Rashmeet Kaur",
            "collection": "Live Session",
        },
        {
            "title": "Song Name (DJ Remix)",
            "artist": "Artist",
            "collection": "Single",
        },
        {
            "title": "Bollywood Mixtape",
            "artist": "Artist",
            "collection": "Mixtape",
        },
        {
            "title": "Song Mashup",
            "artist": "Artist",
            "collection": "Single",
        },
    ]

    for test_row in reject_cases:
        reason = bad_track_reason(test_row)

        if not reason:
            raise RuntimeError(
                "Cell 5B bad-version self-test failed: "
                f"{test_row['title']!r} was not rejected."
            )


_validate_bad_version_filter()
print("Bad-version filter: passed")

# GENRE MAPPING

GENRE_KEYWORDS = {
    "Hip-Hop/Rap": [
        "hip hop",
        "hip-hop",
        "rap",
        "trap",
        "drill",
        "grime",
        "说唱",
        "嘻哈",
    ],
    "R&B/Soul": [
        "r&b",
        "rnb",
        "rhythm and blues",
        "soul",
        "neo soul",
    ],
    "Folk/Acoustic": [
        "folk",
        "acoustic",
        "singer-songwriter",
        "singer songwriter",
        "unplugged",
        "ghazal",
        "sufi",
        "民谣",
        "民謠",
    ],
    "Rock": [
        "rock",
        "metal",
        "punk",
        "grunge",
        "alternative rock",
        "indie rock",
        "hard rock",
        "摇滚",
        "搖滾",
    ],
    "Pop": [
        "pop",
        "dance pop",
        "electropop",
        "synthpop",
        "indie pop",
        "mandopop",
        "cantopop",
        "c-pop",
        "chinese pop",
        "流行",
    ],
}

BROAD_GENRE_LABELS = [
    "bollywood",
    "filmi",
    "indian pop",
    "indipop",
    "regional indian",
    "indian music",
    "hindi",
    "hindi music",
    "soundtrack",
    "original soundtrack",
    "motion picture soundtrack",
    "stage & screen",
    "stage and screen",
    "world",
    "worldwide",
    "music",
]


def remove_broad_labels(value):
    text = safe_text(value).casefold()
    text = text.replace("&", " and ")

    for phrase in sorted(
        BROAD_GENRE_LABELS,
        key=len,
        reverse=True,
    ):
        text = re.sub(
            rf"(?<!\w){re.escape(phrase)}(?!\w)",
            " ",
            text,
        )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def map_genre(values):
    if isinstance(values, str):
        values = [values]

    scores = {
        genre: 0
        for genre in GENRES
    }

    for rank, value in enumerate(values):
        cleaned = remove_broad_labels(value)

        if not cleaned:
            continue

        weight = max(1, 20 - rank)

        for genre, keywords in GENRE_KEYWORDS.items():
            for keyword in keywords:
                if has_phrase(cleaned, keyword):
                    scores[genre] += weight

    best_genre = max(
        scores,
        key=scores.get,
    )

    if scores[best_genre] <= 0:
        return None, scores

    ranked_scores = sorted(
        scores.values(),
        reverse=True,
    )

    if (
        len(ranked_scores) > 1
        and ranked_scores[0] == ranked_scores[1]
    ):
        return None, scores

    return best_genre, scores


def original_or_seed_genre(row):
    candidates = [
        row.get("final_genre"),
        row.get("seed_genre"),
        row.get("planned_genre"),
        row.get("query_genre_group"),
    ]

    candidates.extend(
        split_values(
            row.get("query_genre_groups")
        )
    )

    for value in candidates:
        value = safe_text(value)

        if value in GENRES:
            return value

    return None


# SAFE HTTP

def request_json(
    url,
    *,
    params,
    service,
    sleep_after=0.1,
):
    for attempt in range(MAX_RETRIES):
        try:
            response = SESSION.get(
                url,
                params=params,
                timeout=HTTP_TIMEOUT,
            )

            API_CALLS[service] += 1

            if response.status_code == 429:
                retry_after = response.headers.get(
                    "Retry-After",
                    "",
                )

                try:
                    wait_seconds = float(retry_after)
                except Exception:
                    wait_seconds = 2 ** attempt

                if wait_seconds > MAX_429_WAIT_SECONDS:
                    return None, (
                        f"{service}_429_wait_too_long"
                    )

                time.sleep(
                    wait_seconds
                    + random.uniform(0.1, 0.3)
                )
                continue

            if response.status_code in {
                500,
                502,
                503,
                504,
            }:
                time.sleep(
                    min(
                        2 ** attempt,
                        8,
                    )
                )
                continue

            if response.status_code != 200:
                return None, (
                    f"{service}_http_"
                    f"{response.status_code}"
                )

            payload = response.json()
            time.sleep(sleep_after)
            return payload, "ok"

        except Exception as error:
            if attempt == MAX_RETRIES - 1:
                return None, (
                    f"{service}_"
                    f"{type(error).__name__}"
                )

            time.sleep(
                min(
                    2 ** attempt,
                    8,
                )
            )

    return None, f"{service}_failed"


# LYRICS LANGUAGE CHECK


HINDI_WORDS = {
    "aaj", "aaja", "aankh", "aankhon", "aao", "aashiq",
    "aashiqui", "apna", "apni", "baarish", "baat", "bata",
    "bewafa", "chaand", "chale", "channa", "dard", "deewana",
    "deewane", "dil", "door", "duniya", "faasle", "gaya",
    "gham", "hai", "hain", "hawayein", "ho", "hoon", "hua",
    "hum", "ishq", "jaan", "jaana", "jaane", "kabhi", "kaise",
    "kahan", "kahaan", "kasoor", "khabar", "khayaal", "khwab",
    "khwaab", "ki", "kiya", "kya", "kyun", "lage", "main",
    "maine", "mann", "mera", "mere", "meri", "mila", "mohabbat",
    "mujhe", "na", "nahi", "naina", "pyaar", "pyar", "raat",
    "raha", "rahe", "rang", "saath", "safar", "sahiba", "sajna",
    "sajana", "sanson", "sapna", "se", "tera", "tere", "teri",
    "toh", "tu", "tujhe", "tujhko", "tum", "tumse", "yaad",
    "yaar", "yaara", "yeh", "zara", "zindagi",
}

PUNJABI_WORDS = {
    "assi", "akh", "balle", "bhangra", "gabru", "gabhru",
    "jatt", "jatti", "karda", "karde", "kardi", "kudi",
    "lagda", "munda", "naal", "nachda", "nachdi", "ni",
    "nu", "pind", "punjab", "sardar", "sardaar", "sohni",
    "tainu", "tenu", "ve", "vich", "wala", "wali",
    "warga", "wargi",
}

OTHER_INDIC_PATTERNS = {
    "bengali": r"[\u0980-\u09ff]",
    "gujarati": r"[\u0a80-\u0aff]",
    "odia": r"[\u0b00-\u0b7f]",
    "tamil": r"[\u0b80-\u0bff]",
    "telugu": r"[\u0c00-\u0c7f]",
    "kannada": r"[\u0c80-\u0cff]",
    "malayalam": r"[\u0d00-\u0d7f]",
}


def clean_synced_lyrics(value):
    text = safe_text(value)

    text = re.sub(
        r"\[[^\]]+\]",
        " ",
        text,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def lyrics_cache_key(title, artist, album, duration):
    return "||".join(
        [
            normalise_text(title),
            normalise_text(artist),
            normalise_text(album),
            safe_text(duration),
        ]
    )


def fetch_lyrics(title, artist, album="", duration=""):
    if not USE_LYRICS_LANGUAGE_CHECK:
        return {
            "ok": False,
            "status": "lyrics_disabled",
        }

    key = lyrics_cache_key(
        title,
        artist,
        album,
        duration,
    )

    cached = LYRICS_CACHE.get(key)

    if isinstance(cached, dict):
        return cached

    params = {
        "track_name": title,
        "artist_name": artist,
    }

    if safe_text(album):
        params["album_name"] = safe_text(album)

    try:
        duration_int = int(
            round(float(duration))
        )

        if duration_int > 0:
            params["duration"] = duration_int
    except Exception:
        pass

    payload, status = request_json(
        LRCLIB_SEARCH_URL,
        params=params,
        service="LRCLIB",
        sleep_after=0.08,
    )

    if status != "ok" or not isinstance(
        payload,
        list,
    ):
        result = {
            "ok": False,
            "status": status,
        }
        LYRICS_CACHE[key] = result
        return result

    best = None
    best_score = -1.0

    for item in payload:
        if not isinstance(item, dict):
            continue

        title_score = similarity(
            title,
            item.get("trackName"),
        )

        artist_score = similarity(
            artist,
            item.get("artistName"),
        )

        if title_score < 0.72:
            continue

        if artist_score < 0.45:
            continue

        score = (
            title_score * 0.70
            + artist_score * 0.25
        )

        try:
            expected_duration = float(duration)
            found_duration = float(
                item.get("duration")
            )

            difference = abs(
                expected_duration
                - found_duration
            )

            if difference <= 20:
                score += 0.05
            elif difference > 45:
                score -= 0.10
        except Exception:
            difference = None

        if score > best_score:
            best = item
            best_score = score

    if best is None:
        result = {
            "ok": False,
            "status": "lyrics_no_matching_record",
        }
        LYRICS_CACHE[key] = result
        return result

    lyrics_text = safe_text(
        best.get("plainLyrics")
    )

    if not lyrics_text:
        lyrics_text = clean_synced_lyrics(
            best.get("syncedLyrics")
        )

    if not lyrics_text:
        result = {
            "ok": False,
            "status": "lyrics_record_has_no_text",
        }
        LYRICS_CACHE[key] = result
        return result

    devanagari_count = script_count(
        lyrics_text,
        r"[\u0900-\u097f]",
    )
    gurmukhi_count = script_count(
        lyrics_text,
        r"[\u0a00-\u0a7f]",
    )
    han_count = script_count(
        lyrics_text,
        r"[\u3400-\u4dbf\u4e00-\u9fff]",
    )
    kana_hangul_count = script_count(
        lyrics_text,
        r"[\u3040-\u30ff\uac00-\ud7af]",
    )

    other_indic_counts = {
        name: script_count(
            lyrics_text,
            pattern,
        )
        for name, pattern
        in OTHER_INDIC_PATTERNS.items()
    }

    roman_tokens = re.findall(
        r"[a-zA-Z']+",
        lyrics_text.casefold(),
    )

    hindi_hits = sum(
        token in HINDI_WORDS
        for token in roman_tokens
    )

    punjabi_hits = sum(
        token in PUNJABI_WORDS
        for token in roman_tokens
    )

    result = {
        "ok": True,
        "status": "lyrics_match_found",
        "provider": "LRCLIB",
        "record_id": safe_text(
            best.get("id")
        ),
        "matched_title": safe_text(
            best.get("trackName")
        ),
        "matched_artist": safe_text(
            best.get("artistName")
        ),
        "match_score": round(
            best_score,
            4,
        ),
        "lyrics_character_count": len(
            lyrics_text
        ),
        "devanagari_count": devanagari_count,
        "gurmukhi_count": gurmukhi_count,
        "han_count": han_count,
        "kana_hangul_count": kana_hangul_count,
        "other_indic_counts": other_indic_counts,
        "hindi_word_hits": hindi_hits,
        "punjabi_word_hits": punjabi_hits,
    }

    # The full lyrics are intentionally not cached.
    LYRICS_CACHE[key] = result
    return result


# LAST.FM TRACK GENRE

def lastfm_track_info(title, artist):
    if not LASTFM_ENABLED:
        return {
            "ok": False,
            "status": "lastfm_disabled",
            "tags": [],
            "mapped_genre": None,
            "scores": {},
        }

    key = (
        normalise_text(artist)
        + "||"
        + normalise_text(title)
    )

    cached = LASTFM_CACHE.get(key)

    if isinstance(cached, dict):
        return cached

    params = {
        "method": "track.getInfo",
        "artist": artist,
        "track": title,
        "api_key": LASTFM_API_KEY,
        "format": "json",
        "autocorrect": 1,
    }

    payload, status = request_json(
        LASTFM_URL,
        params=params,
        service="LastFM",
        sleep_after=0.02,
    )

    track = (
        payload.get("track", {})
        if status == "ok"
        and isinstance(payload, dict)
        else {}
    )

    if not isinstance(track, dict):
        track = {}

    found_title = safe_text(
        track.get("name")
    )

    artist_value = track.get(
        "artist",
        {},
    )

    found_artist = (
        safe_text(artist_value.get("name"))
        if isinstance(artist_value, dict)
        else safe_text(artist_value)
    )

    identity_ok = bool(
        found_title
        and found_artist
        and similarity(
            title,
            found_title,
        ) >= 0.72
        and similarity(
            artist,
            found_artist,
        ) >= 0.45
    )

    tag_items = (
        track.get("toptags", {})
        .get("tag", [])
        if track
        else []
    )

    if not isinstance(tag_items, list):
        tag_items = []

    tags = [
        safe_text(item.get("name")).casefold()
        for item in tag_items[:20]
        if isinstance(item, dict)
        and safe_text(item.get("name"))
    ]

    # One Last.fm request per track only.
    # When track.getInfo has no tags, treat it as no track-level
    # genre evidence instead of making a second slow request.

    mapped_genre, scores = map_genre(tags)

    result = {
        "ok": identity_ok,
        "status": (
            "lastfm_track_match"
            if identity_ok and tags
            else "lastfm_exact_track_no_tags"
            if identity_ok
            else status
            if status != "ok"
            else "lastfm_no_exact_track"
        ),
        "tags": tags,
        "mapped_genre": mapped_genre,
        "scores": scores,
        "canonical_title": found_title,
        "canonical_artist": found_artist,
    }

    LASTFM_CACHE[key] = result
    return result


# LAST.FM ARTIST GENRE FALLBACK

def lastfm_artist_info(artist):
    """
    Artist-level fallback used only when exact-track Last.fm tags
    are missing. Results are cached per artist, so several songs
    by the same artist do not cause repeated requests.
    """
    if (
        not LASTFM_ENABLED
        or not USE_LASTFM_ARTIST_FALLBACK
    ):
        return {
            "ok": False,
            "status": "lastfm_artist_fallback_disabled",
            "tags": [],
            "mapped_genre": None,
            "scores": {},
        }

    key = normalise_text(artist)

    cached = LASTFM_ARTIST_CACHE.get(key)

    if isinstance(cached, dict):
        return cached

    payload, status = request_json(
        LASTFM_URL,
        params={
            "method": "artist.getTopTags",
            "artist": artist,
            "api_key": LASTFM_API_KEY,
            "format": "json",
            "autocorrect": 1,
        },
        service="LastFMArtist",
        sleep_after=0.02,
    )

    tag_items = (
        payload.get("toptags", {})
        .get("tag", [])
        if status == "ok"
        and isinstance(payload, dict)
        else []
    )

    if not isinstance(tag_items, list):
        tag_items = []

    tags = [
        safe_text(
            item.get("name")
        ).casefold()
        for item in tag_items[:20]
        if isinstance(item, dict)
        and safe_text(
            item.get("name")
        )
    ]

    mapped_genre, scores = map_genre(
        tags
    )

    result = {
        "ok": bool(tags),
        "status": (
            "lastfm_artist_tags"
            if tags
            else status
            if status != "ok"
            else "lastfm_artist_no_tags"
        ),
        "tags": tags,
        "mapped_genre": mapped_genre,
        "scores": scores,
    }

    LASTFM_ARTIST_CACHE[key] = result
    return result


# SPOTIFY SUPPORTING METADATA

def spotify_access_token(
    force_refresh=False,
):
    """
    Obtain a Spotify app token using Client Credentials.

    Existing SPOTIFY_ACCESS_TOKEN / SPOTIPY_ACCESS_TOKEN is reused
    when supplied. Client ID and Client Secret are preferred because
    the code can refresh the app token automatically.
    """
    if not SPOTIFY_ENABLED:
        return "", "spotify_disabled"

    current_token = safe_text(
        SPOTIFY_TOKEN_STATE.get(
            "access_token"
        )
    )

    expires_at = float(
        SPOTIFY_TOKEN_STATE.get(
            "expires_at",
            0.0,
        )
        or 0.0
    )

    if (
        not force_refresh
        and current_token
        and time.time() < expires_at - 60
    ):
        return current_token, "spotify_token_cached"

    if (
        not SPOTIFY_CLIENT_ID
        or not SPOTIFY_CLIENT_SECRET
    ):
        if current_token and not force_refresh:
            return current_token, "spotify_static_token"

        return "", "spotify_credentials_missing"

    try:
        response = SESSION.post(
            SPOTIFY_TOKEN_URL,
            data={
                "grant_type": "client_credentials",
            },
            auth=(
                SPOTIFY_CLIENT_ID,
                SPOTIFY_CLIENT_SECRET,
            ),
            timeout=HTTP_TIMEOUT,
        )

        API_CALLS["SpotifyToken"] += 1

        if response.status_code != 200:
            return "", (
                "spotify_token_http_"
                + str(response.status_code)
            )

        payload = response.json()

        token = safe_text(
            payload.get("access_token")
        )

        expires_in = float(
            payload.get("expires_in", 3600)
            or 3600
        )

        if not token:
            return "", "spotify_token_missing"

        SPOTIFY_TOKEN_STATE[
            "access_token"
        ] = token

        SPOTIFY_TOKEN_STATE[
            "expires_at"
        ] = time.time() + expires_in

        return token, "spotify_token_refreshed"

    except Exception as error:
        return "", (
            "spotify_token_"
            + type(error).__name__
        )


def spotify_get_json(
    url,
    *,
    params=None,
    service="Spotify",
):
    """
    Spotify GET with token refresh and safe 429 handling.
    """
    for token_attempt in range(2):
        token, token_status = (
            spotify_access_token(
                force_refresh=(
                    token_attempt > 0
                )
            )
        )

        if not token:
            return None, token_status

        try:
            response = SESSION.get(
                url,
                params=params or {},
                headers={
                    "Authorization": (
                        f"Bearer {token}"
                    ),
                    "Accept": "application/json",
                },
                timeout=HTTP_TIMEOUT,
            )

            API_CALLS[service] += 1

            if response.status_code == 401:
                # Refresh once when Client Credentials are available.
                if (
                    token_attempt == 0
                    and SPOTIFY_CLIENT_ID
                    and SPOTIFY_CLIENT_SECRET
                ):
                    continue

                return None, "spotify_http_401"

            if response.status_code == 429:
                retry_after = response.headers.get(
                    "Retry-After",
                    "",
                )

                try:
                    wait_seconds = float(
                        retry_after
                    )
                except Exception:
                    wait_seconds = 1.0

                if (
                    wait_seconds
                    > MAX_429_WAIT_SECONDS
                ):
                    return None, (
                        "spotify_429_wait_too_long"
                    )

                time.sleep(
                    wait_seconds
                    + random.uniform(
                        0.1,
                        0.3,
                    )
                )

                continue

            if response.status_code != 200:
                return None, (
                    "spotify_http_"
                    + str(
                        response.status_code
                    )
                )

            return response.json(), "ok"

        except Exception as error:
            return None, (
                "spotify_"
                + type(error).__name__
            )

    return None, "spotify_failed"


def spotify_artist_genres(
    artist_id,
):
    """
    Spotify artist genres are supporting evidence only.
    The endpoint may return an empty/deprecated genre field.
    """
    artist_id = safe_text(
        artist_id
    )

    if not artist_id:
        return {
            "ok": False,
            "status": "spotify_artist_id_missing",
            "genres": [],
            "mapped_genre": None,
            "scores": {},
        }

    cached = SPOTIFY_ARTIST_CACHE.get(
        artist_id
    )

    if isinstance(cached, dict):
        return cached

    payload, status = spotify_get_json(
        SPOTIFY_ARTIST_URL.format(
            artist_id=artist_id
        ),
        service="SpotifyArtist",
    )

    if (
        status != "ok"
        or not isinstance(payload, dict)
    ):
        result = {
            "ok": False,
            "status": status,
            "genres": [],
            "mapped_genre": None,
            "scores": {},
        }

        SPOTIFY_ARTIST_CACHE[
            artist_id
        ] = result

        return result

    genres = [
        safe_text(value).casefold()
        for value in payload.get(
            "genres",
            [],
        )
        if safe_text(value)
    ]

    mapped_genre, scores = map_genre(
        genres
    )

    result = {
        "ok": bool(genres),
        "status": (
            "spotify_artist_genres"
            if genres
            else "spotify_artist_genres_empty"
        ),
        "genres": genres,
        "mapped_genre": mapped_genre,
        "scores": scores,
        "artist_name": safe_text(
            payload.get("name")
        ),
    }

    SPOTIFY_ARTIST_CACHE[
        artist_id
    ] = result

    return result


def spotify_track_info(
    title,
    artist,
    language_name,
):
    """
    Find the exact Spotify catalogue track.

    This uses metadata only:
      - title/artist identity
      - track ID and URL
      - album/release date
      - duration/ISRC
      - artist genres when Spotify returns them

    No Spotify audio is downloaded or used.
    """
    if not SPOTIFY_ENABLED:
        return {
            "ok": False,
            "status": "spotify_disabled",
            "mapped_genre": None,
            "artist_genres": [],
            "scores": {},
        }

    key = "||".join(
        [
            language_name,
            normalise_text(title),
            normalise_text(artist),
        ]
    )

    cached = SPOTIFY_TRACK_CACHE.get(
        key
    )

    if isinstance(cached, dict):
        return cached

    markets = SPOTIFY_MARKETS.get(
        language_name,
        [],
    )

    if not markets:
        markets = ["IN"]

    best_item = None
    best_market = ""
    best_score = -1.0

    for market in markets:
        payload, status = spotify_get_json(
            SPOTIFY_SEARCH_URL,
            params={
                "q": (
                    f'track:"{title}" '
                    f'artist:"{artist}"'
                ),
                "type": "track",
                "market": market,
                "limit": min(
                    int(
                        SPOTIFY_SEARCH_LIMIT
                    ),
                    10,
                ),
            },
            service="SpotifySearch",
        )

        if (
            status != "ok"
            or not isinstance(
                payload,
                dict,
            )
        ):
            continue

        items = (
            payload.get("tracks", {})
            .get("items", [])
        )

        if not isinstance(items, list):
            continue

        for item in items:
            if not isinstance(item, dict):
                continue

            found_title = safe_text(
                item.get("name")
            )

            artists = item.get(
                "artists",
                [],
            )

            if not isinstance(
                artists,
                list,
            ):
                artists = []

            found_artist = ", ".join(
                safe_text(
                    value.get("name")
                )
                for value in artists
                if isinstance(value, dict)
                and safe_text(
                    value.get("name")
                )
            )

            title_score = similarity(
                title,
                found_title,
            )

            artist_score = similarity(
                artist,
                found_artist,
            )

            if title_score < 0.72:
                continue

            if artist_score < 0.45:
                continue

            score = (
                title_score * 0.72
                + artist_score * 0.28
            )

            if score > best_score:
                best_item = item
                best_market = market
                best_score = score

    if best_item is None:
        result = {
            "ok": False,
            "status": "spotify_no_exact_track",
            "mapped_genre": None,
            "artist_genres": [],
            "scores": {},
        }

        SPOTIFY_TRACK_CACHE[key] = result
        return result

    artists = best_item.get(
        "artists",
        [],
    )

    if not isinstance(artists, list):
        artists = []

    primary_artist = (
        artists[0]
        if artists
        and isinstance(artists[0], dict)
        else {}
    )

    artist_result = spotify_artist_genres(
        primary_artist.get("id")
    )

    external_urls = best_item.get(
        "external_urls",
        {},
    )

    if not isinstance(
        external_urls,
        dict,
    ):
        external_urls = {}

    album = best_item.get(
        "album",
        {},
    )

    if not isinstance(album, dict):
        album = {}

    external_ids = best_item.get(
        "external_ids",
        {},
    )

    if not isinstance(
        external_ids,
        dict,
    ):
        external_ids = {}

    result = {
        "ok": True,
        "status": "spotify_track_match",
        "match_score": round(
            best_score,
            4,
        ),
        "matched_title": safe_text(
            best_item.get("name")
        ),
        "matched_artist": ", ".join(
            safe_text(
                value.get("name")
            )
            for value in artists
            if isinstance(value, dict)
            and safe_text(
                value.get("name")
            )
        ),
        "track_id": safe_text(
            best_item.get("id")
        ),
        "track_url": safe_text(
            external_urls.get("spotify")
        ),
        "album": safe_text(
            album.get("name")
        ),
        "release_date": safe_text(
            album.get("release_date")
        ),
        "duration_seconds": (
            float(
                best_item.get(
                    "duration_ms",
                    0,
                )
                or 0
            )
            / 1000.0
        ),
        "isrc": safe_text(
            external_ids.get("isrc")
        ),
        "market": best_market,
        "artist_id": safe_text(
            primary_artist.get("id")
        ),
        "artist_genres": (
            artist_result.get(
                "genres",
                []
            )
        ),
        "artist_genre_status": (
            artist_result.get(
                "status",
                ""
            )
        ),
        "mapped_genre": (
            artist_result.get(
                "mapped_genre"
            )
        ),
        "scores": (
            artist_result.get(
                "scores",
                {}
            )
        ),
        "genre_scope": "artist",
    }

    SPOTIFY_TRACK_CACHE[key] = result
    return result


# OPTIONAL iTUNES RECHECK

def verify_itunes_if_needed(row, language_name):
    if source_is_exact_itunes(row):
        raw_genre = safe_text(
            row.get("raw_genre")
        )
        mapped, scores = map_genre(
            split_values(raw_genre)
        )

        return {
            "ok": True,
            "status": "reused_cell4_itunes_row",
            "raw_genre": raw_genre,
            "mapped_genre": mapped,
            "scores": scores,
            "collection": safe_text(
                row.get("collection")
                or row.get("album")
            ),
            "country": safe_text(
                row.get("country")
            ),
        }

    if not RECHECK_ITUNES:
        return {
            "ok": False,
            "status": "itunes_recheck_disabled",
            "raw_genre": "",
            "mapped_genre": None,
            "scores": {},
            "collection": "",
            "country": "",
        }

    title = safe_text(row.get("title"))
    artist = safe_text(row.get("artist"))

    countries = (
        ["IN"]
        if language_name == "Hindi"
        else ["CN", "TW", "HK"]
    )

    key = (
        language_name
        + "||"
        + normalise_text(title)
        + "||"
        + normalise_text(artist)
    )

    cached = ITUNES_CACHE.get(key)

    if isinstance(cached, dict):
        return cached

    for country in countries:
        payload, status = request_json(
            ITUNES_SEARCH_URL,
            params={
                "term": f"{title} {artist}",
                "country": country,
                "media": "music",
                "entity": "song",
                "limit": 15,
            },
            service="iTunes",
            sleep_after=0.12,
        )

        if (
            status != "ok"
            or not isinstance(payload, dict)
        ):
            continue

        for item in payload.get(
            "results",
            [],
        ):
            if not isinstance(item, dict):
                continue

            if similarity(
                title,
                item.get("trackName"),
            ) < 0.72:
                continue

            if similarity(
                artist,
                item.get("artistName"),
            ) < 0.45:
                continue

            raw_genre = safe_text(
                item.get("primaryGenreName")
            )
            mapped, scores = map_genre(
                [raw_genre]
            )

            result = {
                "ok": True,
                "status": "itunes_track_match",
                "raw_genre": raw_genre,
                "mapped_genre": mapped,
                "scores": scores,
                "collection": safe_text(
                    item.get("collectionName")
                ),
                "country": country,
            }

            ITUNES_CACHE[key] = result
            return result

    result = {
        "ok": False,
        "status": "itunes_no_match",
        "raw_genre": "",
        "mapped_genre": None,
        "scores": {},
        "collection": "",
        "country": "",
    }

    ITUNES_CACHE[key] = result
    return result


# LANGUAGE DECISION

HINDI_ARTISTS = [
    "arijit singh",
    "shreya ghoshal",
    "sonu nigam",
    "udit narayan",
    "alka yagnik",
    "kumar sanu",
    "sunidhi chauhan",
    "shaan",
    "mohit chauhan",
    "jubin nautiyal",
    "vishal mishra",
    "javed ali",
    "benny dayal",
    "palak muchhal",
    "tulsi kumar",
    "asees kaur",
    "shilpa rao",
    "armaan malik",
    "amit trivedi",
    "pritam",
    "indian ocean",
    "the local train",
    "agnee",
    "euphoria",
    "divine",
    "raftaar",
    "emiway bantai",
    "kr$na",
    "krsna",
    "seedhe maut",
    "dino james",
]

PUNJABI_PRIMARY_ARTISTS = [
    "karan aujla",
    "ap dhillon",
    "shubh",
    "sidhu moose wala",
    "diljit dosanjh",
    "ammy virk",
    "gippy grewal",
    "arjan dhillon",
    "navaan sandhu",
    "jass manak",
    "jassie gill",
    "garry sandhu",
    "prem dhillon",
    "shinda kahlon",
]

HINDI_CONTEXT_MARKERS = [
    "hindi",
    "bollywood",
    "filmi",
    "hindi film",
    "hindi soundtrack",
]

CHINESE_CONTEXT_MARKERS = [
    "mandopop",
    "cantopop",
    "c-pop",
    "chinese pop",
    "chinese rock",
    "chinese rap",
    "chinese hip hop",
    "chinese r&b",
    "chinese folk",
    "mandarin",
    "cantonese",
    "华语",
    "國語",
    "中文",
    "流行",
    "摇滚",
    "搖滾",
    "说唱",
    "嘻哈",
    "民谣",
    "民謠",
]

NON_HINDI_MARKERS = [
    "punjabi",
    "bhangra",
    "pollywood",
    "haryanvi",
    "bhojpuri",
    "tamil",
    "telugu",
    "malayalam",
    "kannada",
    "marathi",
    "bengali",
    "bangla",
    "gujarati",
    "odia",
    "assamese",
    "bhajan",
    "aarti",
    "mantra",
    "chalisa",
    "devotional",
]


def artist_in_list(artist, names):
    artist_n = normalise_text(artist)

    return any(
        normalise_text(name) == artist_n
        or (
            len(normalise_text(name)) >= 6
            and normalise_text(name)
            in artist_n
        )
        for name in names
    )


def title_word_counts(title):
    tokens = re.findall(
        r"[a-zA-Z']+",
        safe_text(title).casefold(),
    )

    return (
        sum(
            token in HINDI_WORDS
            for token in tokens
        ),
        sum(
            token in PUNJABI_WORDS
            for token in tokens
        ),
    )


def add_lyrics_columns(row, lyrics):
    row["lyrics_status"] = lyrics.get(
        "status",
        "",
    )
    row["lyrics_provider"] = lyrics.get(
        "provider",
        "",
    )
    row["lyrics_record_id"] = lyrics.get(
        "record_id",
        "",
    )
    row["lyrics_match_score"] = lyrics.get(
        "match_score",
        "",
    )
    row["lyrics_character_count"] = lyrics.get(
        "lyrics_character_count",
        0,
    )
    row["lyrics_devanagari_count"] = lyrics.get(
        "devanagari_count",
        0,
    )
    row["lyrics_gurmukhi_count"] = lyrics.get(
        "gurmukhi_count",
        0,
    )
    row["lyrics_han_count"] = lyrics.get(
        "han_count",
        0,
    )
    row["lyrics_kana_hangul_count"] = lyrics.get(
        "kana_hangul_count",
        0,
    )
    row["lyrics_hindi_word_hits"] = lyrics.get(
        "hindi_word_hits",
        0,
    )
    row["lyrics_punjabi_word_hits"] = lyrics.get(
        "punjabi_word_hits",
        0,
    )
    row["lyrics_text_saved"] = False


def decide_hindi_language(
    row,
    itunes_result,
):
    title = safe_text(row.get("title"))
    artist = safe_text(row.get("artist"))
    album = safe_text(
        row.get("collection")
        or row.get("album")
        or itunes_result.get("collection")
    )

    title_artist = f"{title} {artist}"

    if contains_han(
        title_artist
    ) or contains_kana_or_hangul(
        title_artist
    ):
        return (
            False,
            "reject_non_hindi_script",
            "high",
            {},
        )

    if contains_gurmukhi(title_artist):
        return (
            False,
            "reject_gurmukhi_title",
            "high",
            {},
        )

    for name, pattern in OTHER_INDIC_PATTERNS.items():
        if re.search(pattern, title_artist):
            return (
                False,
                f"reject_{name}_title_script",
                "high",
                {},
            )

    metadata_text = " ".join(
        [
            safe_text(row.get("raw_genre")),
            safe_text(row.get("notes")),
            safe_text(itunes_result.get("raw_genre")),
            safe_text(itunes_result.get("collection")),
        ]
    ).casefold()

    title_hindi_hits, title_punjabi_hits = (
        title_word_counts(title)
    )

    if contains_devanagari(title):
        return (
            True,
            "accepted_hindi_devanagari_title",
            "high",
            {},
        )

    exact_itunes = itunes_result.get(
        "ok",
        False,
    )

    hindi_context = any(
        marker in metadata_text
        for marker in HINDI_CONTEXT_MARKERS
    )

    if (
        title_hindi_hits >= 2
        and exact_itunes
    ):
        return (
            True,
            "accepted_hindi_title_words",
            "moderate",
            {},
        )

    # Reject obvious non-Hindi rows before making a slow lyrics request.
    explicit_non_hindi_before_lyrics = any(
        marker in metadata_text
        for marker in NON_HINDI_MARKERS
    )

    if explicit_non_hindi_before_lyrics:
        return (
            False,
            "reject_non_hindi_metadata",
            "moderate",
            {},
        )

    if title_punjabi_hits > title_hindi_hits:
        return (
            False,
            "reject_punjabi_title_words",
            "moderate",
            {},
        )

    if artist_in_list(
        artist,
        PUNJABI_PRIMARY_ARTISTS,
    ):
        return (
            False,
            "reject_punjabi_primary_artist_no_hindi_proof",
            "moderate",
            {},
        )

    # Fast acceptance for exact iTunes catalogue rows from established
    # Hindi artists or with at least one Hindi title word.
    if (
        FAST_MODE
        and exact_itunes
        and (
            artist_in_list(
                artist,
                HINDI_ARTISTS,
            )
            or title_hindi_hits >= 1
        )
    ):
        return (
            True,
            "accepted_hindi_exact_itunes_fast_fallback",
            "low",
            {},
        )

    # Lyrics are used only for the remaining genuinely ambiguous rows.
    lyrics = fetch_lyrics(
        title,
        artist,
        album,
        row.get("duration_seconds", ""),
    )

    if lyrics.get("ok"):
        other_indic = lyrics.get(
            "other_indic_counts",
            {},
        )

        if lyrics.get("gurmukhi_count", 0) >= 3:
            return (
                False,
                "reject_punjabi_lyrics_script",
                "high",
                lyrics,
            )

        for name, count in other_indic.items():
            if count >= 3:
                return (
                    False,
                    f"reject_{name}_lyrics_script",
                    "high",
                    lyrics,
                )

        if lyrics.get(
            "devanagari_count",
            0,
        ) >= 15:
            return (
                True,
                "accepted_hindi_lyrics_devanagari",
                "high",
                lyrics,
            )

        hindi_hits = lyrics.get(
            "hindi_word_hits",
            0,
        )
        punjabi_hits = lyrics.get(
            "punjabi_word_hits",
            0,
        )

        if (
            punjabi_hits >= 4
            and punjabi_hits
            > hindi_hits * 1.20
        ):
            return (
                False,
                "reject_punjabi_romanised_lyrics",
                "moderate",
                lyrics,
            )

        if (
            hindi_hits >= 6
            and hindi_hits
            >= punjabi_hits * 1.50
        ):
            return (
                True,
                "accepted_hindi_romanised_lyrics",
                "moderate",
                lyrics,
            )

    explicit_non_hindi = any(
        marker in metadata_text
        for marker in NON_HINDI_MARKERS
    )

    if explicit_non_hindi:
        return (
            False,
            "reject_non_hindi_metadata",
            "moderate",
            lyrics,
        )

    if title_punjabi_hits > title_hindi_hits:
        return (
            False,
            "reject_punjabi_title_words",
            "moderate",
            lyrics,
        )

    if artist_in_list(
        artist,
        PUNJABI_PRIMARY_ARTISTS,
    ):
        return (
            False,
            "reject_punjabi_primary_artist_no_hindi_proof",
            "moderate",
            lyrics,
        )

    if (
        exact_itunes
        and (
            hindi_context
            or artist_in_list(
                artist,
                HINDI_ARTISTS,
            )
            or title_hindi_hits >= 1
        )
    ):
        return (
            True,
            "accepted_hindi_exact_itunes_fallback",
            "low",
            lyrics,
        )

    return (
        False,
        "reject_no_hindi_language_evidence",
        "low",
        lyrics,
    )


def decide_chinese_language(
    row,
    itunes_result,
):
    title = safe_text(row.get("title"))
    artist = safe_text(row.get("artist"))
    album = safe_text(
        row.get("collection")
        or row.get("album")
        or itunes_result.get("collection")
    )

    title_artist = f"{title} {artist}"

    if (
        contains_kana_or_hangul(title_artist)
        and not contains_han(title_artist)
    ):
        return (
            False,
            "reject_japanese_or_korean_script",
            "high",
            {},
        )

    metadata_text = " ".join(
        [
            safe_text(row.get("raw_genre")),
            safe_text(row.get("notes")),
            safe_text(itunes_result.get("raw_genre")),
            safe_text(itunes_result.get("collection")),
        ]
    ).casefold()

    country_text = " ".join(
        [
            safe_text(row.get("country")),
            safe_text(itunes_result.get("country")),
        ]
    ).upper()

    chinese_context = any(
        marker in metadata_text
        for marker in CHINESE_CONTEXT_MARKERS
    )

    chinese_market = any(
        country in country_text
        for country in [
            "CN",
            "TW",
            "HK",
            "SG",
            "MO",
        ]
    )

    if (
        contains_han(title_artist)
        and (
            chinese_context
            or chinese_market
        )
    ):
        return (
            True,
            "accepted_chinese_title_and_context",
            "high",
            {},
        )

    lyrics = fetch_lyrics(
        title,
        artist,
        album,
        row.get("duration_seconds", ""),
    )

    if lyrics.get("ok"):
        han_count = lyrics.get(
            "han_count",
            0,
        )
        kana_hangul = lyrics.get(
            "kana_hangul_count",
            0,
        )

        if (
            kana_hangul >= 10
            and kana_hangul > han_count
        ):
            return (
                False,
                "reject_japanese_or_korean_lyrics",
                "high",
                lyrics,
            )

        if han_count >= 20:
            return (
                True,
                "accepted_chinese_lyrics",
                "high",
                lyrics,
            )

    if chinese_context and chinese_market:
        return (
            True,
            "accepted_chinese_metadata_fallback",
            "low",
            lyrics,
        )

    return (
        False,
        "reject_no_chinese_language_evidence",
        "low",
        lyrics,
    )


# VERIFY ONE ROW

def choose_final_genre(
    row,
    itunes_result,
    lastfm_result,
    artist_result,
):
    """
    Genre evidence order:

      1. Exact-track Last.fm genre.
      2. Artist-level Last.fm genre only when it agrees with either
         the existing provisional genre or the specific iTunes genre.
      3. iTunes alone never decides the final research genre.

    Because Apple catalogue labels can be broad, an iTunes conflict
    does not automatically reject a specific exact-track Last.fm tag.
    """
    original_genre = original_or_seed_genre(
        row
    )

    track_genre = lastfm_result.get(
        "mapped_genre"
    )

    artist_genre = artist_result.get(
        "mapped_genre"
    )

    itunes_genre = itunes_result.get(
        "mapped_genre"
    )

    # Exact-track Last.fm is strongest.
    if track_genre in GENRES:
        if (
            itunes_genre in GENRES
            and itunes_genre == track_genre
        ):
            return (
                track_genre,
                "itunes_lastfm_track_genre_agreement",
                "high",
            )

        if (
            itunes_genre in GENRES
            and itunes_genre != track_genre
        ):
            if REJECT_ITUNES_LASTFM_CONFLICT:
                return (
                    None,
                    "reject_itunes_lastfm_track_genre_conflict",
                    "conflict",
                )

            return (
                track_genre,
                "lastfm_track_genre_over_itunes",
                "moderate",
            )

        return (
            track_genre,
            "lastfm_track_genre_only",
            "moderate",
        )

    # Track tags are often absent for Hindi/Chinese catalogue rows.
    # Artist tags are accepted only with a second supporting signal.
    if artist_genre in GENRES:
        if (
            itunes_genre in GENRES
            and artist_genre == itunes_genre
        ):
            return (
                artist_genre,
                "lastfm_artist_itunes_genre_agreement",
                "moderate",
            )

        if (
            original_genre in GENRES
            and artist_genre == original_genre
        ):
            return (
                artist_genre,
                "lastfm_artist_existing_genre_agreement",
                "moderate",
            )

    if itunes_genre in GENRES:
        return (
            None,
            "reject_itunes_genre_without_lastfm_support",
            "none",
        )

    if artist_genre in GENRES:
        return (
            None,
            "reject_lastfm_artist_genre_without_second_signal",
            "none",
        )

    return (
        None,
        "reject_no_external_genre_evidence",
        "none",
    )


def verify_row(row, language_name):
    row_dict = row.to_dict()
    row_dict["cell5b_version"] = CELL5B_VERSION

    title = safe_text(
        row_dict.get("title")
    )
    artist = safe_text(
        row_dict.get("artist")
    )

    normalised_title = normalise_text(
        title
    )

    if normalised_title in CONFIRMED_REMOVE_TITLES.get(
        language_name,
        set(),
    ):
        row_dict["final_filter_reason"] = (
            "reject_confirmed_wrong_language_or_genre"
        )

        row_dict["verification_status"] = (
            "rejected_confirmed_final_audit_error"
        )

        row_dict["confirmed_remove"] = True
        row_dict["cell5b_version"] = (
            CELL5B_VERSION
        )

        return False, row_dict

    row_dict["confirmed_remove"] = False
    row_dict["needs_final_manual_review"] = bool(
        normalised_title
        in REVIEW_TITLES.get(
            language_name,
            set(),
        )
    )

    if not title or not artist:
        row_dict["final_filter_reason"] = (
            "reject_missing_title_or_artist"
        )
        return False, row_dict

    bad_reason = bad_track_reason(row_dict)

    if bad_reason:
        row_dict["final_filter_reason"] = bad_reason
        return False, row_dict

    itunes_result = verify_itunes_if_needed(
        row_dict,
        language_name,
    )

    if language_name == "Hindi":
        (
            language_ok,
            language_reason,
            language_confidence,
            lyrics_result,
        ) = decide_hindi_language(
            row_dict,
            itunes_result,
        )
    else:
        (
            language_ok,
            language_reason,
            language_confidence,
            lyrics_result,
        ) = decide_chinese_language(
            row_dict,
            itunes_result,
        )

    add_lyrics_columns(
        row_dict,
        lyrics_result,
    )

    row_dict["language_decision"] = (
        language_reason
    )
    row_dict["language_confidence"] = (
        language_confidence
    )

    if not language_ok:
        row_dict["final_filter_reason"] = (
            language_reason
        )
        row_dict["verification_status"] = (
            "rejected_language_check"
        )
        return False, row_dict

    existing_genre = original_or_seed_genre(
        row_dict
    )
    itunes_genre = itunes_result.get(
        "mapped_genre"
    )

    needs_lastfm = bool(
        LASTFM_ENABLED
        and (
            not LASTFM_ONLY_FOR_UNRESOLVED_GENRE
            or (
                existing_genre not in GENRES
                and itunes_genre not in GENRES
            )
        )
    )

    if needs_lastfm:
        lastfm_result = lastfm_track_info(
            title,
            artist,
        )
    else:
        lastfm_result = {
            "ok": False,
            "status": (
                "lastfm_skipped_genre_already_available"
                if LASTFM_ENABLED
                else "lastfm_disabled"
            ),
            "tags": [],
            "mapped_genre": None,
            "scores": {},
        }

    if (
        lastfm_result.get("mapped_genre")
        not in GENRES
    ):
        artist_result = lastfm_artist_info(
            artist
        )
    else:
        artist_result = {
            "ok": False,
            "status": "artist_fallback_not_needed",
            "tags": [],
            "mapped_genre": None,
            "scores": {},
        }

    (
        final_genre,
        genre_reason,
        genre_confidence,
    ) = choose_final_genre(
        row_dict,
        itunes_result,
        lastfm_result,
        artist_result,
    )

    row_dict["original_final_genre"] = safe_text(
        row_dict.get("final_genre")
    )

    row_dict["lastfm_track_status"] = (
        lastfm_result.get("status", "")
    )
    row_dict["lastfm_track_tags"] = ", ".join(
        lastfm_result.get("tags", [])
    )
    row_dict["lastfm_track_genre"] = (
        lastfm_result.get("mapped_genre")
        or ""
    )
    row_dict["lastfm_track_scores"] = json.dumps(
        lastfm_result.get("scores", {}),
        ensure_ascii=False,
        sort_keys=True,
    )

    row_dict["lastfm_artist_status"] = (
        artist_result.get("status", "")
    )
    row_dict["lastfm_artist_tags"] = ", ".join(
        artist_result.get("tags", [])
    )
    row_dict["lastfm_artist_genre"] = (
        artist_result.get("mapped_genre")
        or ""
    )
    row_dict["lastfm_artist_scores"] = json.dumps(
        artist_result.get("scores", {}),
        ensure_ascii=False,
        sort_keys=True,
    )

    row_dict["itunes_check_status"] = (
        itunes_result.get("status", "")
    )
    row_dict["itunes_mapped_genre"] = (
        itunes_result.get("mapped_genre")
        or ""
    )

    if final_genre not in GENRES:
        row_dict["genre_decision"] = genre_reason
        row_dict["genre_confidence"] = genre_confidence
        row_dict["final_filter_reason"] = genre_reason
        row_dict["verification_status"] = (
            "rejected_genre_check"
        )
        return False, row_dict

    row_dict["final_genre"] = final_genre
    row_dict["genre_decision"] = genre_reason
    row_dict["genre_confidence"] = genre_confidence
    row_dict["language"] = language_name
    row_dict["language_code"] = (
        "HI"
        if language_name == "Hindi"
        else "ZH"
    )
    row_dict["label"] = (
        safe_text(row_dict.get("label"))
        or "Human"
    )
    row_dict["model"] = (
        safe_text(row_dict.get("model"))
        or "Human"
    )
    row_dict["verification_status"] = (
        "accepted_automatic_language_and_genre_check"
    )
    row_dict["final_filter_reason"] = (
        "accepted_"
        + language_reason
        + "__"
        + genre_reason
    )

    return True, row_dict


# PREVIOUS CSV DISCOVERY AND POOL BUILDING

ORIGINAL_METADATA_NAMES = {
    "Hindi": [
        "hindi_human_metadata(1).csv",
        "hindi_human_metadata.csv",
    ],
    "Chinese": [
        "chinese_human_metadata(1).csv",
        "chinese_human_metadata.csv",
    ],
}

LANGUAGE_FILE_TOKENS = {
    "Hindi": [
        "hindi",
        "bollywood",
    ],
    "Chinese": [
        "chinese",
        "mandarin",
        "cantopop",
        "mandopop",
        "c_pop",
        "c-pop",
    ],
}

INCLUDE_FILE_TOKENS = [
    "human_metadata",
    "candidate",
    "candidates",
    "verified",
    "itunes",
    "spotify",
    "balanced",
    "manifest",
]

EXCLUDE_FILE_TOKENS = [
    "rejected",
    "manual_review",
    "review",
    "failed",
    "failure",
    "audit",
    "audited",
    "clean",
    "conservative",
    "remove",
    "backup",
    "conversion",
    "checkpoint",
    "strict_manual",
    "ai_",
    "suno",
    "muse",
    "cmi_",
]

IN_MEMORY_POOL_NAMES = {
    "Hindi": [
        "hindi_human_metadata",
        "hindi_final",
        "hindi_verified",
        "hindi_itunes_candidates",
        "hindi_candidates",
        "hindi_shortlist",
    ],
    "Chinese": [
        "chinese_human_metadata",
        "chinese_final",
        "chinese_verified",
        "chinese_itunes_candidates",
        "chinese_candidates",
        "chinese_shortlist",
    ],
}


def existing_search_dirs():
    output = []
    seen = set()

    for folder in CANDIDATE_SEARCH_DIRS:
        folder = Path(folder)

        try:
            resolved = folder.resolve()
        except Exception:
            resolved = folder

        key = str(resolved).casefold()

        if key in seen or not folder.exists():
            continue

        seen.add(key)
        output.append(folder)

    return output


def find_original_metadata_csv(language_name):
    for folder in existing_search_dirs():
        for filename in ORIGINAL_METADATA_NAMES[
            language_name
        ]:
            candidate = folder / filename

            if candidate.exists():
                return candidate

    language_token = language_name.casefold()
    matches = []

    for folder in existing_search_dirs():
        try:
            files = folder.glob("*.csv")
        except Exception:
            continue

        for path in files:
            name = path.name.casefold()

            if (
                language_token in name
                and "human_metadata" in name
                and not any(
                    token in name
                    for token in EXCLUDE_FILE_TOKENS
                )
            ):
                matches.append(path)

    if not matches:
        raise FileNotFoundError(
            f"Could not find the original "
            f"{language_name} human metadata CSV."
        )

    matches.sort(
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    return matches[0]


def candidate_csv_files(
    language_name,
    original_csv,
):
    language_tokens = LANGUAGE_FILE_TOKENS[
        language_name
    ]

    files = []
    seen = set()

    def add_path(path):
        path = Path(path)

        if not path.exists() or path.suffix.casefold() != ".csv":
            return

        try:
            resolved = path.resolve()
        except Exception:
            resolved = path

        key = str(resolved).casefold()

        if key in seen:
            return

        seen.add(key)
        files.append(path)

    # Original metadata gets first priority.
    add_path(original_csv)

    for folder in existing_search_dirs():
        try:
            direct_files = list(
                folder.glob("*.csv")
            )
        except Exception:
            direct_files = []

        # Recurse only through project folders, not the whole drive.
        recursive_files = []

        folder_text = str(folder).casefold()

        if (
            "dataset_collection_clean" in folder_text
            or folder.name.casefold() == "manifests"
            or folder.name.casefold() == "notebook"
        ):
            try:
                recursive_files = list(
                    folder.rglob("*.csv")
                )
            except Exception:
                recursive_files = []

        for path in [
            *direct_files,
            *recursive_files,
        ]:
            name = path.name.casefold()

            if not any(
                token in name
                for token in language_tokens
            ):
                continue

            if not any(
                token in name
                for token in INCLUDE_FILE_TOKENS
            ):
                continue

            if any(
                token in name
                for token in EXCLUDE_FILE_TOKENS
            ):
                continue

            add_path(path)

    return files


def first_existing_column(df, names):
    for name in names:
        if name in df.columns:
            return name

    return None


def copy_alias_column(
    df,
    target,
    aliases,
):
    if target in df.columns:
        return

    source = first_existing_column(
        df,
        aliases,
    )

    if source:
        df[target] = df[source]


def standardise_candidate_frame(
    frame,
    language_name,
    source_label,
    source_priority,
):
    if frame is None or len(frame) == 0:
        return pd.DataFrame()

    work = frame.copy()

    copy_alias_column(
        work,
        "title",
        [
            "track_name",
            "trackName",
            "song",
            "song_name",
            "name",
        ],
    )

    copy_alias_column(
        work,
        "artist",
        [
            "artist_name",
            "artistName",
            "singer",
            "artists",
        ],
    )

    copy_alias_column(
        work,
        "final_genre",
        [
            "genre",
            "genre_group",
            "mapped_genre",
            "planned_genre",
            "seed_genre",
            "query_genre_group",
        ],
    )

    copy_alias_column(
        work,
        "raw_genre",
        [
            "primaryGenreName",
            "itunes_genre",
            "source_genre",
        ],
    )

    copy_alias_column(
        work,
        "track_view_url",
        [
            "track_url",
            "url",
            "apple_music_url",
            "itunes_url",
        ],
    )

    copy_alias_column(
        work,
        "source_track_id",
        [
            "track_id",
            "trackId",
            "id",
        ],
    )

    copy_alias_column(
        work,
        "collection",
        [
            "album",
            "collectionName",
            "album_name",
        ],
    )

    copy_alias_column(
        work,
        "preview_url",
        [
            "previewUrl",
            "audio_preview_url",
        ],
    )

    if "duration_seconds" not in work.columns:
        duration_ms_col = first_existing_column(
            work,
            [
                "duration_ms",
                "trackTimeMillis",
            ],
        )

        if duration_ms_col:
            work["duration_seconds"] = (
                pd.to_numeric(
                    work[duration_ms_col],
                    errors="coerce",
                )
                / 1000.0
            )

    if (
        "title" not in work.columns
        or "artist" not in work.columns
    ):
        return pd.DataFrame()

    work = work[
        work["title"].apply(
            lambda value: bool(
                safe_text(value)
            )
        )
        & work["artist"].apply(
            lambda value: bool(
                safe_text(value)
            )
        )
    ].copy()

    if "source_dataset" not in work.columns:
        work["source_dataset"] = source_label
    else:
        work["source_dataset"] = (
            work["source_dataset"]
            .fillna("")
            .astype(str)
        )

        empty_source = (
            work["source_dataset"]
            .str.strip()
            .eq("")
        )

        work.loc[
            empty_source,
            "source_dataset",
        ] = source_label

    work["candidate_source_file"] = (
        source_label
    )

    work["candidate_source_priority"] = (
        source_priority
    )

    work["language"] = language_name
    work["language_code"] = (
        "HI"
        if language_name == "Hindi"
        else "ZH"
    )

    if "label" not in work.columns:
        work["label"] = "Human"

    if "model" not in work.columns:
        work["model"] = "Human"

    # Remove clearly AI-generated rows from old mixed manifests.
    ai_text = (
        work["label"].fillna("").astype(str)
        + " "
        + work["model"].fillna("").astype(str)
        + " "
        + work["source_dataset"].fillna("").astype(str)
    ).str.casefold()

    work = work[
        ~ai_text.str.contains(
            r"\b(?:suno|udio|ai generated|"
            r"artificial intelligence|muse|cmi)\b",
            regex=True,
            na=False,
        )
    ].copy()

    if EXCLUDE_LOCAL_AUDIO_ROWS:
        if "local_audio_path" in work.columns:
            local_path_present = (
                work["local_audio_path"]
                .fillna("")
                .astype(str)
                .str.strip()
                .ne("")
            )
        else:
            local_path_present = pd.Series(
                False,
                index=work.index,
            )

        source_text = (
            work["source_dataset"]
            .fillna("")
            .astype(str)
            .str.casefold()
        )

        local_source_marker = (
            source_text.str.contains(
                "local audio",
                regex=False,
                na=False,
            )
        )

        work = work[
            ~(
                local_path_present
                | local_source_marker
            )
        ].copy()

    for column in [
        "final_genre",
        "seed_genre",
        "planned_genre",
        "query_genre_group",
        "raw_genre",
        "track_view_url",
        "source_track_id",
        "collection",
        "preview_url",
        "country",
        "notes",
    ]:
        if column not in work.columns:
            work[column] = ""

    return work


def provisional_genre_from_row(row):
    for column in [
        "final_genre",
        "seed_genre",
        "planned_genre",
        "query_genre_group",
    ]:
        value = safe_text(
            row.get(column)
        )

        if value in GENRES:
            return value

    mapped, _ = map_genre(
        split_values(
            row.get("raw_genre")
        )
    )

    return (
        mapped
        if mapped in GENRES
        else "Unknown"
    )


def candidate_quality_score(row):
    score = 0.0

    priority = row.get(
        "candidate_source_priority",
        999,
    )

    try:
        score += max(
            0,
            100 - float(priority) * 10,
        )
    except Exception:
        pass

    source_text = (
        safe_text(
            row.get("source_dataset")
        )
        + " "
        + safe_text(
            row.get("candidate_source_file")
        )
    ).casefold()

    if (
        "human_metadata" in source_text
        or "human metadata" in source_text
    ):
        score += 30

    if "verified" in source_text:
        score += 20

    if (
        "itunes" in source_text
        or "apple" in source_text
    ):
        score += 10

    if safe_text(
        row.get("track_view_url")
    ):
        score += 5

    if safe_text(
        row.get("source_track_id")
    ):
        score += 5

    if safe_text(
        row.get("preview_url")
    ):
        score += 2

    if safe_text(
        row.get("language_confidence")
    ).casefold() == "high":
        score += 8

    if safe_text(
        row.get("genre_confidence")
    ).casefold() == "high":
        score += 8

    return score


def build_previous_csv_candidate_pool(
    language_name,
    original_csv,
):
    frames = []
    inventory_rows = []
    priority = 0

    for path in candidate_csv_files(
        language_name,
        original_csv,
    ):
        try:
            frame = pd.read_csv(
                path,
                low_memory=False,
            )
        except Exception as error:
            inventory_rows.append(
                {
                    "language": language_name,
                    "source": str(path),
                    "status": (
                        "read_error_"
                        + type(error).__name__
                    ),
                    "rows_loaded": 0,
                }
            )
            continue

        standardised = standardise_candidate_frame(
            frame,
            language_name,
            str(path),
            priority,
        )

        inventory_rows.append(
            {
                "language": language_name,
                "source": str(path),
                "status": "loaded",
                "rows_loaded": len(
                    standardised
                ),
            }
        )

        if len(standardised):
            frames.append(
                standardised
            )
            priority += 1

    # Also include useful DataFrames already in notebook memory.
    for variable_name in IN_MEMORY_POOL_NAMES[
        language_name
    ]:
        value = globals().get(
            variable_name
        )

        if not isinstance(
            value,
            pd.DataFrame,
        ):
            continue

        standardised = standardise_candidate_frame(
            value,
            language_name,
            f"in_memory:{variable_name}",
            priority,
        )

        inventory_rows.append(
            {
                "language": language_name,
                "source": (
                    f"in_memory:{variable_name}"
                ),
                "status": "loaded",
                "rows_loaded": len(
                    standardised
                ),
            }
        )

        if len(standardised):
            frames.append(
                standardised
            )
            priority += 1

    if not frames:
        raise ValueError(
            f"No usable previous CSV candidates "
            f"found for {language_name}."
        )

    all_columns = sorted(
        set().union(
            *[
                set(frame.columns)
                for frame in frames
            ]
        )
    )

    pool = pd.concat(
        [
            frame.reindex(
                columns=all_columns
            )
            for frame in frames
        ],
        ignore_index=True,
        sort=False,
    )

    # Defragment once before adding helper columns.
    pool = pool.copy()

    pool["_song_key"] = [
        row_key_from_values(
            title,
            artist,
        )
        for title, artist in zip(
            pool["title"],
            pool["artist"],
        )
    ]

    pool["_track_id_key"] = (
        pool["source_track_id"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
    )

    pool["_provisional_genre"] = (
        pool.apply(
            provisional_genre_from_row,
            axis=1,
        )
    )

    pool["_candidate_quality"] = (
        pool.apply(
            candidate_quality_score,
            axis=1,
        )
    )

    pool = pool.sort_values(
        by=[
            "candidate_source_priority",
            "_candidate_quality",
        ],
        ascending=[
            True,
            False,
        ],
        kind="stable",
    )

    # Same Apple/Spotify track ID may appear with different script/title.
    with_track_id = pool[
        pool["_track_id_key"].ne("")
    ].drop_duplicates(
        subset=["_track_id_key"],
        keep="first",
    )

    without_track_id = pool[
        pool["_track_id_key"].eq("")
    ]

    pool = pd.concat(
        [
            with_track_id,
            without_track_id,
        ],
        ignore_index=True,
        sort=False,
    )

    pool = (
        pool.drop_duplicates(
            subset=["_song_key"],
            keep="first",
        )
        .reset_index(drop=True)
    )

    inventory_df = pd.DataFrame(
        inventory_rows
    )

    return pool, inventory_df


# PROCESS ONE LANGUAGE

def read_checkpoint_csv(path):
    path = Path(path)

    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()

    try:
        return pd.read_csv(
            path,
            low_memory=False,
        )
    except pd.errors.EmptyDataError:
        return pd.DataFrame()
    except Exception:
        return pd.DataFrame()


def row_key_from_values(title, artist):
    return (
        normalise_text(title)
        + "||"
        + normalise_text(artist)
    )


def dataframe_keys(df):
    if df is None or len(df) == 0:
        return set()

    if "title" not in df.columns or "artist" not in df.columns:
        return set()

    return {
        row_key_from_values(
            title,
            artist,
        )
        for title, artist in zip(
            df["title"],
            df["artist"],
        )
        if normalise_text(title)
        and normalise_text(artist)
    }


def locally_clean_resumed_kept(
    kept_df,
):
    """
    Apply the current live/DJ/remix/mixtape filter to old kept rows.

    This prevents a previously accepted live/remix track from being
    preserved only because resume mode skipped it.
    """
    if kept_df is None or len(kept_df) == 0:
        return pd.DataFrame(), pd.DataFrame()

    still_kept = []
    newly_rejected = []

    for _, row in kept_df.iterrows():
        row_dict = row.to_dict()
        reason = bad_track_reason(row_dict)

        if reason:
            row_dict["final_filter_reason"] = reason
            row_dict["verification_status"] = (
                "rejected_during_resume_local_recheck"
            )
            newly_rejected.append(row_dict)
        else:
            still_kept.append(row_dict)

    return (
        pd.DataFrame(still_kept),
        pd.DataFrame(newly_rejected),
    )


def combine_unique_rows(
    first,
    second,
):
    frames = [
        frame
        for frame in [first, second]
        if frame is not None
        and len(frame) > 0
    ]

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(
        frames,
        ignore_index=True,
        sort=False,
    )

    # Defragment before adding helper columns.
    combined = combined.copy()

    if (
        "title" not in combined.columns
        or "artist" not in combined.columns
    ):
        return combined

    combined["_resume_key"] = [
        row_key_from_values(
            title,
            artist,
        )
        for title, artist in zip(
            combined["title"],
            combined["artist"],
        )
    ]

    combined = (
        combined.drop_duplicates(
            subset=["_resume_key"],
            keep="last",
        )
        .drop(
            columns=["_resume_key"],
            errors="ignore",
        )
        .reset_index(drop=True)
    )

    return combined



# FINAL HINDI + CHINESE AUDIT AND TARGETED REPAIR

FINAL_AUDIT_CHECKPOINT_EVERY = 5

# Confirmed rows from the final audit that must not remain in the
# Hindi/Chinese language datasets. Titles are normalised before matching.
CONFIRMED_REMOVE_TITLES = {
    "Hindi": {
        normalise_text(
            'Pyar Suneya Si (From "Vich Bolunga Tere")'
        ),
        normalise_text("Afsanay"),
    },
    "Chinese": {
        normalise_text("GANADARA (feat. IU)"),
        normalise_text("Blue"),
        normalise_text("SWIM"),
        normalise_text(
            "Arabian Nights (2019) [Japanese Version]"
        ),
        normalise_text(
            'You’ve Got a Friend in Me '
            '(From "Toy Story") '
            '[feat. Plínio Fernandes]'
        ),
        normalise_text(
            'Can You Feel the Love Tonight '
            '(From "The Lion King")'
        ),
        normalise_text("Maria"),
        normalise_text("Arabian Nights (2019)"),
        normalise_text("I’m Lookin’ out for You"),
        normalise_text(
            "H.S.K.T. (feat. Wonstein)"
        ),
        normalise_text(
            'The Phantom Of The Opera '
            '(1988 Japanese Cast Recording Of '
            '"The Phantom Of The Opera")'
        ),
        normalise_text(
            'Loyal Brave True (From "Mulan")'
        ),
        normalise_text("Why"),
        normalise_text(
            'Beauty and the Beast '
            '(From "Beauty and the Beast")'
        ),
        normalise_text("Nobody Like U"),
        normalise_text("Dance Monkey (Acoustic)"),
        normalise_text(
            'A Whole New World (From "Aladdin")'
        ),
        normalise_text(
            "The Christmas Song "
            "(Chestnuts Roasting On An Open Fire)"
        ),
    },
}

# These rows are not proven wrong, but their language or exact five-class
# genre needs stronger evidence. They are retained unless the APIs/lyrics
# provide explicit contradictory evidence, and are exported separately.
REVIEW_TITLES = {
    "Hindi": {
        normalise_text(value)
        for value in [
            "Kudi Meri",
            "Ni Tu Baar Baar",
            "Laal Pungaad Meri Jaan",
            "Kahan Kahan",
            "Rhythm 'n' Blues (feat. Usha Uthup)",
            "Bailadila Jaadoo Bastar",
            "Inside Out",
            "Subah (feat. Pavithra)",
            "Tu Dass Kivein",
            "Gaye Chari Gala Baawara Insaan",
            "K.",
            "My Pain Is Constant and Sharp",
            "Iya Kaye Hajaari Mere Jaisa Hi",
            "Aarzu (feat. JANI, Nabeel Akbar & Talha Anjum)",
            "On My Way",
            "Iron",
            "Gone Girl",
            "F.G.H.M",
            "BANNED (feat. Abhijay Sharma)",
            "MATRABHASHA",
            "ROBBERY",
            "Custom Fabric",
            "Peep The Tareeka (feat. FINSTA & Lil Kabeer)",
            "Backbone",
            "Wanna Say",
            "Blossom",
            "KAMU ITU BAHAGIA",
            "Dil Roya Ve",
            "Manave",
            "Channa",
            'Oh Baale (From "Animal")',
            "Nain Ta Heere - Lisa",
            "Haq",
            "Mehrma",
            "For You",
            "Stay",
            "Saturday",
            "Landfill",
            "Patola (feat. Gayatri Bhardwaj & Sahil Anand)",
            "Munde Da Character",
            "Vibe Cute Badi",
            "Fresh",
        ]
    },
    "Chinese": {
        normalise_text(value)
        for value in [
            "Take Me to Your Heart",
            "Stray",
            "9999",
            "再度重相逢",
            "曾经的你",
            "Waiting For You",
            "Hey Aries",
            "141.44",
            "Blessed Mother",
            "repeating",
            "98",
            "Aphrodite",
            "sa ye",
            "Better Dayz (feat. aMEI)",
            "Don't Worry About Me",
            "Cold World",
            "1234567",
            "Singalongsong",
            "每个人都会 (Song for Cartier [Love Project])",
            "面面",
            "XZMHXDXH",
            "爱在",
            "just a phase",
            "Ghost",
            "MAMA SAYS",
            "Life Like This",
            "2020 (feat. Lu1)",
            "Real Lonely",
            "马儿啊,你慢些走",
            "Goldberg of the Wind",
            "Sai Ma",
            "Dublin Rain",
            "美麗的草原我的家",
            "Chinese New Year Song 2020",
            "Lunar New Year Music",
        ]
    },
}


# Existing final rows are not rejected merely because an API has
# no lyrics, no Last.fm tags, or no precise iTunes genre.
UNCERTAIN_LANGUAGE_REASONS = {
    "reject_no_hindi_language_evidence",
    "reject_no_chinese_language_evidence",
}

EXPLICIT_WRONG_LANGUAGE_PREFIXES = (
    "reject_non_hindi_",
    "reject_gurmukhi_",
    "reject_punjabi_",
    "reject_bengali_",
    "reject_gujarati_",
    "reject_odia_",
    "reject_tamil_",
    "reject_telugu_",
    "reject_kannada_",
    "reject_malayalam_",
    "reject_japanese_",
    "reject_korean_",
)


def add_external_genre_columns(
    row_dict,
    itunes_result,
    lastfm_result,
    artist_result,
    spotify_result,
):
    row_dict["lastfm_track_status"] = (
        lastfm_result.get("status", "")
    )

    row_dict["lastfm_track_tags"] = ", ".join(
        lastfm_result.get("tags", [])
    )

    row_dict["lastfm_track_genre"] = (
        lastfm_result.get("mapped_genre")
        or ""
    )

    row_dict["lastfm_track_scores"] = json.dumps(
        lastfm_result.get("scores", {}),
        ensure_ascii=False,
        sort_keys=True,
    )

    row_dict["lastfm_artist_status"] = (
        artist_result.get("status", "")
    )

    row_dict["lastfm_artist_tags"] = ", ".join(
        artist_result.get("tags", [])
    )

    row_dict["lastfm_artist_genre"] = (
        artist_result.get("mapped_genre")
        or ""
    )

    row_dict["lastfm_artist_scores"] = json.dumps(
        artist_result.get("scores", {}),
        ensure_ascii=False,
        sort_keys=True,
    )

    row_dict["itunes_check_status"] = (
        itunes_result.get("status", "")
    )

    row_dict["itunes_mapped_genre"] = (
        itunes_result.get("mapped_genre")
        or ""
    )

    row_dict["itunes_raw_genre_checked"] = (
        itunes_result.get("raw_genre", "")
    )

    row_dict["spotify_check_status"] = (
        spotify_result.get("status", "")
    )
    row_dict["spotify_match_score"] = (
        spotify_result.get(
            "match_score",
            "",
        )
    )
    row_dict["spotify_track_id"] = (
        spotify_result.get("track_id", "")
    )
    row_dict["spotify_track_url"] = (
        spotify_result.get("track_url", "")
    )
    row_dict["spotify_album"] = (
        spotify_result.get("album", "")
    )
    row_dict["spotify_release_date"] = (
        spotify_result.get(
            "release_date",
            "",
        )
    )
    row_dict["spotify_isrc"] = (
        spotify_result.get("isrc", "")
    )
    row_dict["spotify_market"] = (
        spotify_result.get("market", "")
    )
    row_dict["spotify_artist_genres"] = ", ".join(
        spotify_result.get(
            "artist_genres",
            [],
        )
    )
    row_dict["spotify_mapped_genre"] = (
        spotify_result.get(
            "mapped_genre"
        )
        or ""
    )
    row_dict["spotify_genre_scope"] = (
        spotify_result.get(
            "genre_scope",
            "",
        )
    )
    row_dict["spotify_genre_scores"] = json.dumps(
        spotify_result.get(
            "scores",
            {},
        ),
        ensure_ascii=False,
        sort_keys=True,
    )


def final_confirmation_verify_row(
    row,
    language_name,
    *,
    preserve_existing_final,
):
    """
    Existing final row:
      Keep its current genre unless clear contradictory evidence exists.
      Missing lyrics/tags are not treated as proof that it is wrong.

    Replacement candidate:
      Require positive language and genre support before acceptance.
    """
    row_dict = row.to_dict()
    row_dict["cell5b_version"] = CELL5B_VERSION

    title = safe_text(
        row_dict.get("title")
    )

    artist = safe_text(
        row_dict.get("artist")
    )

    if not title or not artist:
        row_dict["final_filter_reason"] = (
            "reject_missing_title_or_artist"
        )

        row_dict["verification_status"] = (
            "rejected_missing_metadata"
        )

        return False, row_dict

    bad_reason = bad_track_reason(
        row_dict
    )

    if bad_reason:
        row_dict["final_filter_reason"] = (
            bad_reason
        )

        row_dict["verification_status"] = (
            "rejected_bad_version"
        )

        return False, row_dict

    itunes_result = verify_itunes_if_needed(
        row_dict,
        language_name,
    )

    if language_name == "Hindi":
        (
            language_ok,
            language_reason,
            language_confidence,
            lyrics_result,
        ) = decide_hindi_language(
            row_dict,
            itunes_result,
        )
    else:
        (
            language_ok,
            language_reason,
            language_confidence,
            lyrics_result,
        ) = decide_chinese_language(
            row_dict,
            itunes_result,
        )

    add_lyrics_columns(
        row_dict,
        lyrics_result,
    )

    row_dict["language_decision"] = (
        language_reason
    )

    row_dict["language_confidence"] = (
        language_confidence
    )

    if not language_ok:
        if (
            preserve_existing_final
            and language_reason
            in UNCERTAIN_LANGUAGE_REASONS
        ):
            language_ok = True

            row_dict["language_decision"] = (
                "kept_existing_language_"
                "because_no_contradictory_evidence"
            )

            row_dict["language_confidence"] = (
                "unconfirmed"
            )
        else:
            row_dict["final_filter_reason"] = (
                language_reason
            )

            row_dict["verification_status"] = (
                "rejected_language_check"
            )

            return False, row_dict

    original_genre = original_or_seed_genre(
        row_dict
    )

    if original_genre not in GENRES:
        row_dict["final_filter_reason"] = (
            "reject_missing_valid_original_genre"
        )

        row_dict["verification_status"] = (
            "rejected_genre_check"
        )

        return False, row_dict

    lastfm_result = lastfm_track_info(
        title,
        artist,
    )

    if (
        lastfm_result.get("mapped_genre")
        not in GENRES
    ):
        artist_result = lastfm_artist_info(
            artist
        )
    else:
        artist_result = {
            "ok": False,
            "status": "artist_fallback_not_needed",
            "tags": [],
            "mapped_genre": None,
            "scores": {},
        }

    spotify_result = spotify_track_info(
        title,
        artist,
        language_name,
    )

    add_external_genre_columns(
        row_dict,
        itunes_result,
        lastfm_result,
        artist_result,
        spotify_result,
    )

    track_genre = lastfm_result.get(
        "mapped_genre"
    )

    artist_genre = artist_result.get(
        "mapped_genre"
    )

    itunes_genre = itunes_result.get(
        "mapped_genre"
    )

    spotify_genre = spotify_result.get(
        "mapped_genre"
    )

    spotify_match = bool(
        spotify_result.get("ok")
    )

    row_dict["original_final_genre"] = (
        original_genre
    )

    # EXISTING FINAL ROW
    
    if preserve_existing_final:
        final_genre = original_genre
        genre_reason = (
            "kept_original_genre_no_contradiction"
        )
        genre_confidence = "retained"

        # Reclassify an existing final song only when exact-track
        # Last.fm and iTunes independently agree on the same different
        # genre. Artist tags alone never move an existing song.
        if (
            track_genre in GENRES
            and itunes_genre in GENRES
            and track_genre == itunes_genre
        ):
            if track_genre == original_genre:
                genre_reason = (
                    "itunes_lastfm_track_support_original_genre"
                )
                genre_confidence = "high"
            else:
                final_genre = track_genre
                genre_reason = (
                    "reclassified_exact_track_itunes_agreement"
                )
                genre_confidence = "high"

        elif track_genre == original_genre:
            genre_reason = (
                "lastfm_track_supports_original_genre"
            )
            genre_confidence = "moderate"

        elif itunes_genre == original_genre:
            genre_reason = (
                "itunes_supports_original_genre"
            )
            genre_confidence = "moderate"

        elif artist_genre == original_genre:
            genre_reason = (
                "lastfm_artist_tags_support_original_genre"
            )
            genre_confidence = "moderate"

        elif (
            spotify_match
            and spotify_genre == original_genre
        ):
            genre_reason = (
                "spotify_artist_genre_supports_original_genre"
            )
            genre_confidence = "moderate"

        elif (
            track_genre in GENRES
            and spotify_genre == track_genre
        ):
            # Spotify artist genre supports the Last.fm track conflict,
            # but without iTunes agreement we still retain the existing
            # final genre.
            genre_reason = (
                "kept_original_genre_"
                "lastfm_spotify_conflict_without_itunes"
            )
            genre_confidence = "retained"

        elif track_genre in GENRES:
            genre_reason = (
                "kept_original_genre_single_lastfm_conflict"
            )
            genre_confidence = "retained"

        elif itunes_genre in GENRES:
            genre_reason = (
                "kept_original_genre_single_itunes_conflict"
            )
            genre_confidence = "retained"

        row_dict["final_genre"] = (
            final_genre
        )

        row_dict["genre_decision"] = (
            genre_reason
        )

        row_dict["genre_confidence"] = (
            genre_confidence
        )

        row_dict["language"] = language_name
        row_dict["language_code"] = (
            "HI"
            if language_name == "Hindi"
            else "ZH"
        )

        row_dict["label"] = (
            safe_text(
                row_dict.get("label")
            )
            or "Human"
        )

        row_dict["model"] = (
            safe_text(
                row_dict.get("model")
            )
            or "Human"
        )

        row_dict["verification_status"] = (
            "accepted_existing_final_confirmation"
        )

        row_dict["final_filter_reason"] = (
            "accepted_"
            + row_dict["language_decision"]
            + "__"
            + genre_reason
        )

        return True, row_dict

    # REPLACEMENT CANDIDATE
    # Replacements remain strict because they are being introduced
    # into the final dataset for the first time.
    if track_genre in GENRES:
        final_genre = track_genre

        supporting_sources = sum(
            [
                itunes_genre == track_genre,
                (
                    spotify_match
                    and spotify_genre
                    == track_genre
                ),
            ]
        )

        if supporting_sources >= 1:
            genre_reason = (
                "replacement_lastfm_track_"
                "supported_by_catalogue"
            )
            genre_confidence = "high"
        else:
            genre_reason = (
                "replacement_lastfm_track_genre"
            )
            genre_confidence = "moderate"

    elif (
        artist_genre in GENRES
        and (
            artist_genre == original_genre
            or artist_genre == itunes_genre
            or (
                spotify_match
                and artist_genre
                == spotify_genre
            )
        )
    ):
        final_genre = artist_genre
        genre_reason = (
            "replacement_lastfm_artist_supported_genre"
        )
        genre_confidence = "moderate"

    elif (
        spotify_match
        and spotify_genre in GENRES
        and itunes_genre == spotify_genre
    ):
        final_genre = spotify_genre
        genre_reason = (
            "replacement_spotify_itunes_genre_agreement"
        )
        genre_confidence = "moderate"

    elif (
        itunes_genre in GENRES
        and itunes_genre == original_genre
    ):
        final_genre = original_genre
        genre_reason = (
            "replacement_itunes_existing_genre_agreement"
        )
        genre_confidence = "moderate"

    else:
        row_dict["final_filter_reason"] = (
            "reject_replacement_without_supported_genre"
        )

        row_dict["verification_status"] = (
            "rejected_genre_check"
        )

        return False, row_dict

    row_dict["final_genre"] = (
        final_genre
    )

    row_dict["genre_decision"] = (
        genre_reason
    )

    row_dict["genre_confidence"] = (
        genre_confidence
    )

    row_dict["language"] = language_name
    row_dict["language_code"] = (
        "HI"
        if language_name == "Hindi"
        else "ZH"
    )

    row_dict["label"] = "Human"
    row_dict["model"] = "Human"

    row_dict["verification_status"] = (
        "accepted_replacement_confirmation"
    )

    row_dict["final_filter_reason"] = (
        "accepted_"
        + row_dict["language_decision"]
        + "__"
        + genre_reason
    )

    return True, row_dict


def primary_artist_key(value):
    """
    Use the first credited artist for the Cell 6 diversity rule.
    """
    text = safe_text(value)

    first = re.split(
        r"\s*(?:,|&|/|\+|"
        r"\bfeat\.?\b|\bfeaturing\b|"
        r"\bft\.?\b|\bx\b)\s*",
        text,
        maxsplit=1,
        flags=re.IGNORECASE,
    )[0]

    return normalise_text(first)


def source_track_key(row):
    for column in [
        "source_track_id",
        "track_id",
        "trackId",
    ]:
        value = safe_text(
            row.get(column)
        )

        if value:
            return value.casefold()

    url = safe_text(
        row.get("track_view_url")
    )

    match = re.search(
        r"[?&]i=(\d+)",
        url,
    )

    return (
        match.group(1)
        if match
        else ""
    )


def current_version_rows(path):
    frame = read_checkpoint_csv(path)

    if (
        len(frame) == 0
        or "cell5b_version"
        not in frame.columns
    ):
        return pd.DataFrame()

    return frame[
        frame["cell5b_version"]
        == CELL5B_VERSION
    ].copy()


def checkpoint_key_set(frame):
    return dataframe_keys(frame)


def save_csv(frame, path):
    Path(path).parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    frame.to_csv(
        path,
        index=False,
        encoding="utf-8-sig",
    )


def save_final_audit_checkpoint(
    checked_rows,
    replacement_rows,
    checked_csv,
    replacement_csv,
):
    checked_df = combine_unique_rows(
        pd.DataFrame(checked_rows),
        pd.DataFrame(),
    )

    replacement_df = combine_unique_rows(
        pd.DataFrame(replacement_rows),
        pd.DataFrame(),
    )

    save_csv(
        checked_df,
        checked_csv,
    )

    save_csv(
        replacement_df,
        replacement_csv,
    )

    save_all_caches()


def audit_final_300_rows(
    final_df,
    language_name,
    checked_csv,
):
    """
    Recheck only the already-balanced final 300 rows.
    """
    work = standardise_candidate_frame(
        final_df,
        language_name,
        source_label=(
            f"final_{language_name.lower()}_metadata"
        ),
        source_priority=-1000,
    )

    if len(work) == 0:
        raise ValueError(
            f"No usable rows found in final "
            f"{language_name} metadata."
        )

    work = work.copy()

    work["_final_original_order"] = range(
        len(work)
    )

    work["_key"] = [
        row_key_from_values(
            title,
            artist,
        )
        for title, artist in zip(
            work["title"],
            work["artist"],
        )
    ]

    work = (
        work.drop_duplicates(
            subset=["_key"],
            keep="first",
        )
        .reset_index(drop=True)
    )

    existing = current_version_rows(
        checked_csv
    )

    current_keys = set(
        work["_key"]
    )

    if len(existing):
        existing["_checkpoint_key"] = [
            row_key_from_values(
                title,
                artist,
            )
            for title, artist in zip(
                existing["title"],
                existing["artist"],
            )
        ]

        existing = (
            existing[
                existing["_checkpoint_key"]
                .isin(current_keys)
            ]
            .drop(
                columns=["_checkpoint_key"],
                errors="ignore",
            )
            .reset_index(drop=True)
        )

    done_keys = checkpoint_key_set(
        existing
    )

    remaining = (
        work[
            ~work["_key"].isin(
                done_keys
            )
        ]
        .drop(
            columns=["_key"],
            errors="ignore",
        )
        .reset_index(drop=True)
    )

    checked_rows = (
        existing.to_dict("records")
        if len(existing)
        else []
    )

    print(
        f"\n{language_name}: auditing final rows"
    )

    print(
        f"Final rows loaded: {len(work)}"
    )

    print(
        f"Already audited: {len(done_keys)}"
    )

    print(
        f"Remaining final rows: {len(remaining)}"
    )

    progress = tqdm(
        remaining.iterrows(),
        total=len(work),
        initial=min(
            len(done_keys),
            len(work),
        ),
        desc=f"{language_name} final audit",
        unit="track",
        dynamic_ncols=True,
    )

    new_count = 0

    try:
        for _, row in progress:
            title = safe_text(
                row.get("title")
            )

            artist = safe_text(
                row.get("artist")
            )

            progress.set_postfix_str(
                f"{title[:28]} - {artist[:22]}",
                refresh=False,
            )

            try:
                accepted, checked = (
                    final_confirmation_verify_row(
                        row,
                        language_name,
                        preserve_existing_final=True,
                    )
                )
            except Exception as error:
                checked = row.to_dict()
                checked["cell5b_version"] = (
                    CELL5B_VERSION
                )
                checked[
                    "verification_status"
                ] = "rejected_processing_error"
                checked[
                    "final_filter_reason"
                ] = (
                    "reject_processing_error_"
                    + type(error).__name__
                )
                checked["processing_error"] = (
                    str(error)
                )
                accepted = False

            checked["cell5b_version"] = (
                CELL5B_VERSION
            )

            checked["final_audit_origin"] = (
                "existing_final_300"
            )

            checked["final_audit_pass"] = bool(
                accepted
            )

            checked_rows.append(
                checked
            )

            new_count += 1

            if (
                new_count
                % FINAL_AUDIT_CHECKPOINT_EVERY
                == 0
            ):
                save_csv(
                    combine_unique_rows(
                        pd.DataFrame(
                            checked_rows
                        ),
                        pd.DataFrame(),
                    ),
                    checked_csv,
                )

                save_all_caches()

    except KeyboardInterrupt:
        save_csv(
            combine_unique_rows(
                pd.DataFrame(checked_rows),
                pd.DataFrame(),
            ),
            checked_csv,
        )

        save_all_caches()

        print(
            f"\n{language_name}: final-audit "
            "progress saved."
        )

        print(
            "Run this cell again to resume."
        )

        raise

    checked_df = combine_unique_rows(
        pd.DataFrame(checked_rows),
        pd.DataFrame(),
    )

    if len(checked_df):
        print(
            f"\n{language_name}: final-audit decisions"
        )

        print(
            checked_df[
                "verification_status"
            ].value_counts(
                dropna=False
            )
        )

        print(
            f"\n{language_name}: most common reasons"
        )

        print(
            checked_df[
                "final_filter_reason"
            ].value_counts(
                dropna=False
            ).head(15)
        )

    save_csv(
        checked_df,
        checked_csv,
    )

    save_all_caches()

    return checked_df


def row_selection_score(row):
    """
    Existing final rows are always preferred over replacements.
    Within the same origin, prefer stronger verification evidence.
    """
    origin = safe_text(
        row.get("final_audit_origin")
    )

    score = (
        100000
        if origin == "existing_final_300"
        else 0
    )

    language_confidence = safe_text(
        row.get("language_confidence")
    ).casefold()

    genre_confidence = safe_text(
        row.get("genre_confidence")
    ).casefold()

    confidence_points = {
        "high": 300,
        "moderate": 200,
        "provisional": 50,
        "low": 25,
    }

    score += confidence_points.get(
        language_confidence,
        0,
    )

    score += confidence_points.get(
        genre_confidence,
        0,
    )

    try:
        score += float(
            row.get(
                "_candidate_quality",
                0,
            )
        )
    except Exception:
        pass

    try:
        original_order = int(
            row.get(
                "_final_original_order",
                999999,
            )
        )

        score += max(
            0,
            10000 - original_order,
        )
    except Exception:
        pass

    return score


def select_exact_balanced_rows(
    verified_rows,
):
    """
    Select exactly 60 per genre while enforcing:
      - unique title + artist
      - unique catalogue track ID when present

    Artist limits are not reapplied here because Cell 6 already handled
    diversity. Reapplying them would incorrectly remove valid final rows.
    """
    if verified_rows is None or len(
        verified_rows
    ) == 0:
        return (
            pd.DataFrame(),
            pd.DataFrame(),
        )

    work = verified_rows.copy()

    work["_selection_score"] = (
        work.apply(
            row_selection_score,
            axis=1,
        )
    )

    work = work.sort_values(
        by="_selection_score",
        ascending=False,
        kind="stable",
    )

    selected = []
    overflow = []

    counts = {
        genre: 0
        for genre in GENRES
    }

    used_song_keys = set()
    used_track_keys = set()

    for _, row in work.iterrows():
        row_dict = row.to_dict()

        genre = safe_text(
            row_dict.get("final_genre")
        )

        reason = ""

        if genre not in GENRES:
            reason = "overflow_invalid_final_genre"

        song_key = row_key_from_values(
            row_dict.get("title"),
            row_dict.get("artist"),
        )

        track_key = source_track_key(
            row_dict
        )

        if not reason and song_key in used_song_keys:
            reason = (
                "overflow_duplicate_title_artist"
            )

        if (
            not reason
            and track_key
            and track_key in used_track_keys
        ):
            reason = (
                "overflow_duplicate_catalogue_track"
            )

        if (
            not reason
            and counts[genre]
            >= TARGET_PER_GENRE
        ):
            reason = "overflow_genre_quota_full"

        if reason:
            row_dict[
                "final_selection_status"
            ] = reason

            overflow.append(
                row_dict
            )

            continue

        row_dict[
            "final_selection_status"
        ] = "selected_final_dataset"

        selected.append(
            row_dict
        )

        counts[genre] += 1

        used_song_keys.add(
            song_key
        )

        if track_key:
            used_track_keys.add(
                track_key
            )

    selected_df = pd.DataFrame(
        selected
    )

    overflow_df = pd.DataFrame(
        overflow
    )

    return selected_df, overflow_df


def genre_counts(frame):
    if frame is None or len(frame) == 0:
        return pd.Series(
            0,
            index=GENRES,
            dtype=int,
        )

    return (
        frame["final_genre"]
        .value_counts()
        .reindex(
            GENRES,
            fill_value=0,
        )
        .astype(int)
    )


def remaining_deficits(frame):
    counts = genre_counts(
        frame
    )

    return {
        genre: max(
            0,
            TARGET_PER_GENRE
            - int(counts[genre]),
        )
        for genre in GENRES
    }


def is_perfect_final(frame):
    counts = genre_counts(
        frame
    )

    return bool(
        len(frame)
        == TARGET_PER_GENRE * len(GENRES)
        and all(
            int(counts[genre])
            == TARGET_PER_GENRE
            for genre in GENRES
        )
        and frame.duplicated(
            subset=["title", "artist"]
        ).sum() == 0
    )


def replacement_scan_order(
    candidate_pool,
    missing_genres,
):
    """
    First inspect candidates whose old/provisional genre matches
    a missing genre. Only use other candidates as a fallback.
    """
    pool = candidate_pool.copy()

    pool["_target_priority"] = (
        pool["_provisional_genre"]
        .isin(missing_genres)
        .astype(int)
    )

    return (
        pool.sort_values(
            by=[
                "_target_priority",
                "candidate_source_priority",
                "_candidate_quality",
            ],
            ascending=[
                False,
                True,
                False,
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )


def refill_only_missing_slots(
    language_name,
    candidate_pool,
    audited_final,
    replacement_csv,
):
    """
    Use older CSV candidates only until all deficits have been filled.
    Every replacement passes the same language, lyrics, bad-version
    and genre verification as the existing final songs.
    """
    passed_final = audited_final[
        audited_final[
            "final_audit_pass"
        ].fillna(False).astype(bool)
    ].copy()

    passed_final[
        "final_audit_origin"
    ] = "existing_final_300"

    existing_replacements = (
        current_version_rows(
            replacement_csv
        )
    )

    accepted_existing_replacements = (
        existing_replacements[
            existing_replacements[
                "replacement_check_pass"
            ].fillna(False).astype(bool)
        ].copy()
        if len(existing_replacements)
        and "replacement_check_pass"
        in existing_replacements.columns
        else pd.DataFrame()
    )

    verified_pool = combine_unique_rows(
        passed_final,
        accepted_existing_replacements,
    )

    selected, overflow = (
        select_exact_balanced_rows(
            verified_pool
        )
    )

    deficits = remaining_deficits(
        selected
    )

    print(
        f"\n{language_name}: valid rows after "
        "final audit"
    )

    print(
        genre_counts(selected)
    )

    print(
        f"{language_name}: replacement deficits"
    )

    print(deficits)

    if all(
        value == 0
        for value in deficits.values()
    ):
        return (
            selected,
            overflow,
            existing_replacements,
        )

    final_original_keys = dataframe_keys(
        audited_final
    )

    replacement_done_keys = dataframe_keys(
        existing_replacements
    )

    pool = candidate_pool.copy()

    pool["_normalised_title"] = (
        pool["title"]
        .apply(normalise_text)
    )

    pool = pool[
        ~pool["_normalised_title"].isin(
            CONFIRMED_REMOVE_TITLES.get(
                language_name,
                set(),
            )
        )
    ].copy()

    pool["_key"] = [
        row_key_from_values(
            title,
            artist,
        )
        for title, artist in zip(
            pool["title"],
            pool["artist"],
        )
    ]

    pool = pool[
        ~pool["_key"].isin(
            final_original_keys
            | replacement_done_keys
        )
    ].copy()

    pool = replacement_scan_order(
        pool,
        {
            genre
            for genre, value
            in deficits.items()
            if value > 0
        },
    )

    checked_replacements = (
        existing_replacements.to_dict(
            "records"
        )
        if len(existing_replacements)
        else []
    )

    newly_checked = 0

    progress = tqdm(
        pool.iterrows(),
        total=len(pool),
        desc=f"{language_name} replacements",
        unit="candidate",
        dynamic_ncols=True,
    )

    try:
        for _, row in progress:
            deficits = remaining_deficits(
                selected
            )

            if all(
                value == 0
                for value in deficits.values()
            ):
                break

            provisional = safe_text(
                row.get(
                    "_provisional_genre"
                )
            )

            # Skip API calls for clearly unrelated genres while a
            # matching provisional candidate is still available.
            if (
                provisional in GENRES
                and deficits.get(
                    provisional,
                    0,
                ) <= 0
            ):
                continue

            title = safe_text(
                row.get("title")
            )

            artist = safe_text(
                row.get("artist")
            )

            progress.set_postfix_str(
                (
                    f"need {deficits} | "
                    f"{title[:22]}"
                ),
                refresh=False,
            )

            try:
                accepted, checked = (
                    final_confirmation_verify_row(
                        row,
                        language_name,
                        preserve_existing_final=False,
                    )
                )
            except Exception as error:
                checked = row.to_dict()
                checked["cell5b_version"] = (
                    CELL5B_VERSION
                )
                checked[
                    "verification_status"
                ] = "rejected_processing_error"
                checked[
                    "final_filter_reason"
                ] = (
                    "reject_processing_error_"
                    + type(error).__name__
                )
                checked["processing_error"] = (
                    str(error)
                )
                accepted = False

            checked["cell5b_version"] = (
                CELL5B_VERSION
            )

            checked["final_audit_origin"] = (
                "replacement_from_previous_csv"
            )

            checked[
                "replacement_check_pass"
            ] = bool(accepted)

            checked_replacements.append(
                checked
            )

            newly_checked += 1

            if accepted:
                verified_pool = combine_unique_rows(
                    verified_pool,
                    pd.DataFrame(
                        [checked]
                    ),
                )

                selected, overflow = (
                    select_exact_balanced_rows(
                        verified_pool
                    )
                )

            if (
                newly_checked
                % FINAL_AUDIT_CHECKPOINT_EVERY
                == 0
            ):
                save_csv(
                    combine_unique_rows(
                        pd.DataFrame(
                            checked_replacements
                        ),
                        pd.DataFrame(),
                    ),
                    replacement_csv,
                )

                save_all_caches()

    except KeyboardInterrupt:
        save_csv(
            combine_unique_rows(
                pd.DataFrame(
                    checked_replacements
                ),
                pd.DataFrame(),
            ),
            replacement_csv,
        )

        save_all_caches()

        print(
            f"\n{language_name}: replacement "
            "progress saved."
        )

        print(
            "Run this cell again to resume."
        )

        raise

    replacement_df = combine_unique_rows(
        pd.DataFrame(
            checked_replacements
        ),
        pd.DataFrame(),
    )

    save_csv(
        replacement_df,
        replacement_csv,
    )

    save_all_caches()

    return (
        selected,
        overflow,
        replacement_df,
    )


def final_song_id(
    language_name,
    genre,
    number,
):
    language_code = (
        "HI"
        if language_name == "Hindi"
        else "ZH"
    )

    genre_code = {
        "Pop": "POP",
        "Rock": "ROCK",
        "Hip-Hop/Rap": "HIPHOP",
        "R&B/Soul": "RNB",
        "Folk/Acoustic": "FOLK",
    }[genre]

    return (
        f"{language_code}_HUMAN_"
        f"{genre_code}_{number:03d}"
    )


def organise_final_rows(
    frame,
    language_name,
):
    final_rows = []

    for genre in GENRES:
        part = (
            frame[
                frame["final_genre"]
                == genre
            ]
            .copy()
            .head(TARGET_PER_GENRE)
            .reset_index(drop=True)
        )

        for number, (_, row) in enumerate(
            part.iterrows(),
            start=1,
        ):
            row_dict = row.to_dict()

            row_dict["song_id"] = (
                final_song_id(
                    language_name,
                    genre,
                    number,
                )
            )

            row_dict["language"] = (
                language_name
            )

            row_dict["language_code"] = (
                "HI"
                if language_name == "Hindi"
                else "ZH"
            )

            row_dict["label"] = "Human"
            row_dict["model"] = "Human"

            final_rows.append(
                row_dict
            )

    return pd.DataFrame(
        final_rows
    )


def timestamped_backup(path):
    path = Path(path)

    backup_folder = (
        path.parent
        / "metadata_backups"
    )

    backup_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    backup_path = (
        backup_folder
        / (
            path.stem
            + "_before_final_confirmation_"
            + timestamp
            + path.suffix
        )
    )

    shutil.copy2(
        path,
        backup_path,
    )

    return backup_path


def write_using_original_schema(
    frame,
    output_path,
    original_columns,
):
    work = frame.copy()

    for column in original_columns:
        if column not in work.columns:
            work[column] = ""

    work = work.reindex(
        columns=original_columns
    )

    work.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )


def audit_and_repair_language(
    final_csv,
    language_name,
):
    final_csv = Path(final_csv)

    original_final = pd.read_csv(
        final_csv,
        low_memory=False,
    )

    original_columns = list(
        original_final.columns
    )

    checked_csv = (
        MANIFEST_DIR
        / (
            f"{language_name.lower()}"
            "_final_300_audit_checked_live.csv"
        )
    )

    replacement_csv = (
        MANIFEST_DIR
        / (
            f"{language_name.lower()}"
            "_final_300_replacement_checks_live.csv"
        )
    )

    rejected_csv = (
        MANIFEST_DIR
        / (
            f"{language_name.lower()}"
            "_final_300_rejected_rows.csv"
        )
    )

    overflow_csv = (
        MANIFEST_DIR
        / (
            f"{language_name.lower()}"
            "_final_300_verified_overflow.csv"
        )
    )

    proposed_csv = (
        MANIFEST_DIR
        / (
            f"{language_name.lower()}"
            "_final_300_repaired_proposed.csv"
        )
    )

    audited_final = audit_final_300_rows(
        original_final,
        language_name,
        checked_csv,
    )

    rejected_final = audited_final[
        ~audited_final[
            "final_audit_pass"
        ].fillna(False).astype(bool)
    ].copy()

    save_csv(
        rejected_final,
        rejected_csv,
    )

    (
        selected,
        overflow,
        replacement_checks,
    ) = refill_only_missing_slots(
        language_name,
        candidate_pool=(
            hindi_candidate_pool
            if language_name == "Hindi"
            else chinese_candidate_pool
        ),
        audited_final=audited_final,
        replacement_csv=replacement_csv,
    )

    selected = organise_final_rows(
        selected,
        language_name,
    )

    save_csv(
        overflow,
        overflow_csv,
    )

    save_csv(
        selected,
        proposed_csv,
    )

    review_queue_csv = (
        MANIFEST_DIR
        / (
            f"{language_name.lower()}"
            "_final_unresolved_review.csv"
        )
    )

    if len(selected):
        selected_review_mask = (
            selected["title"]
            .apply(normalise_text)
            .isin(
                REVIEW_TITLES.get(
                    language_name,
                    set(),
                )
            )
        )

        unresolved_review = (
            selected[
                selected_review_mask
            ]
            .copy()
            .reset_index(drop=True)
        )
    else:
        unresolved_review = pd.DataFrame()

    save_csv(
        unresolved_review,
        review_queue_csv,
    )

    perfect = is_perfect_final(
        selected
    )

    print(
        f"\n{language_name} proposed final counts:"
    )

    print(
        genre_counts(
            selected
        )
    )

    print(
        f"{language_name} rows: "
        f"{len(selected)}"
    )

    print(
        f"{language_name} perfect: "
        f"{perfect}"
    )

    return {
        "language": language_name,
        "final_csv": final_csv,
        "original_columns": (
            original_columns
        ),
        "proposed": selected,
        "perfect": perfect,
        "audited_final": audited_final,
        "rejected_final": rejected_final,
        "replacement_checks": (
            replacement_checks
        ),
        "proposed_csv": proposed_csv,
        "unresolved_review": (
            unresolved_review
        ),
        "review_queue_csv": (
            review_queue_csv
        ),
    }


# RUN

print(
    "Final Hindi + Chinese repair: audit the existing "
    "300 + 300 rows, remove confirmed errors, and refill deficits."
)

print(
    "English metadata is not read or modified by this cell."
)

print(
    "Older candidate CSVs are checked only "
    "when invalid rows create a genre deficit."
)

print(
    "Checks: explicit wrong language, uncertain-row lyrics, "
    "bad versions, duplicate tracks and strongly supported "
    "genre conflicts."
)

print(
    "Artist limits are not reapplied here; Cell 6 already handled "
    "artist diversity."
)

print(
    "Existing final songs are not rejected merely because "
    "Last.fm or lyrics evidence is missing."
)

print(
    "Last.fm enabled:",
    LASTFM_ENABLED,
)

print(
    "Spotify enabled:",
    SPOTIFY_ENABLED,
)

if SPOTIFY_ENABLED:
    print(
        "Spotify role: exact-track catalogue confirmation "
        "and secondary artist-genre support."
    )
else:
    print(
        "Spotify credentials were not detected; "
        "the audit will continue without Spotify."
    )

if not LASTFM_ENABLED:
    raise ValueError(
        "Last.fm API key is required for final "
        "genre confirmation."
    )


HINDI_FINAL_CSV = (
    find_original_metadata_csv(
        "Hindi"
    )
)

CHINESE_FINAL_CSV = (
    find_original_metadata_csv(
        "Chinese"
    )
)


(
    hindi_candidate_pool,
    hindi_source_inventory,
) = build_previous_csv_candidate_pool(
    "Hindi",
    HINDI_FINAL_CSV,
)

(
    chinese_candidate_pool,
    chinese_source_inventory,
) = build_previous_csv_candidate_pool(
    "Chinese",
    CHINESE_FINAL_CSV,
)


SOURCE_INVENTORY_CSV = (
    MANIFEST_DIR
    / "final_repair_candidate_source_inventory.csv"
)

pd.concat(
    [
        hindi_source_inventory,
        chinese_source_inventory,
    ],
    ignore_index=True,
    sort=False,
).to_csv(
    SOURCE_INVENTORY_CSV,
    index=False,
    encoding="utf-8-sig",
)


print(
    "\nFinal Hindi CSV:",
    HINDI_FINAL_CSV,
)

print(
    "Final Chinese CSV:",
    CHINESE_FINAL_CSV,
)

print(
    "Hindi replacement pool available:",
    len(hindi_candidate_pool),
)

print(
    "Chinese replacement pool available:",
    len(chinese_candidate_pool),
)


hindi_result = audit_and_repair_language(
    HINDI_FINAL_CSV,
    "Hindi",
)

chinese_result = audit_and_repair_language(
    CHINESE_FINAL_CSV,
    "Chinese",
)


both_perfect = bool(
    hindi_result["perfect"]
    and chinese_result["perfect"]
)


if both_perfect:
    if CREATE_TIMESTAMPED_BACKUPS:
        hindi_backup = timestamped_backup(
            HINDI_FINAL_CSV
        )

        chinese_backup = timestamped_backup(
            CHINESE_FINAL_CSV
        )

        print(
            "\nHindi backup:",
            hindi_backup,
        )

        print(
            "Chinese backup:",
            chinese_backup,
        )

    write_using_original_schema(
        hindi_result["proposed"],
        HINDI_FINAL_CSV,
        hindi_result["original_columns"],
    )

    write_using_original_schema(
        chinese_result["proposed"],
        CHINESE_FINAL_CSV,
        chinese_result[
            "original_columns"
        ],
    )

    print(
        "\nSUCCESS: both original final CSVs "
        "have been replaced with verified "
        "perfectly balanced 300-row datasets."
    )

else:
    print(
        "\nOriginal final CSVs were NOT overwritten."
    )

    print(
        "Hindi remaining deficits:",
        remaining_deficits(
            hindi_result["proposed"]
        ),
    )

    print(
        "Chinese remaining deficits:",
        remaining_deficits(
            chinese_result["proposed"]
        ),
    )

    print(
        "Add more older candidate CSVs and rerun. "
        "Saved checkpoints will resume automatically."
    )


# Final compatibility variables.
hindi_human_metadata = (
    hindi_result["proposed"].copy()
)

chinese_human_metadata = (
    chinese_result["proposed"].copy()
)

hindi_final = (
    hindi_human_metadata.copy()
)

chinese_final = (
    chinese_human_metadata.copy()
)

print(
    "\nHindi unresolved review rows:",
    len(
        hindi_result[
            "unresolved_review"
        ]
    ),
)

print(
    "Hindi review CSV:",
    hindi_result[
        "review_queue_csv"
    ],
)

print(
    "\nChinese unresolved review rows:",
    len(
        chinese_result[
            "unresolved_review"
        ]
    ),
)

print(
    "Chinese review CSV:",
    chinese_result[
        "review_queue_csv"
    ],
)

print(
    "\nAPI calls:",
    dict(API_CALLS),
)

In [ ]:
# Create the missing genre counts frame required by Cell 9
all_metadata = pd.concat([hindi_human_metadata, chinese_human_metadata], ignore_index=True)
count_genres_frame = all_metadata["final_genre"].value_counts().reset_index()
count_genres_frame.columns = ["genre", "count"]

print("Variable 'count_genres_frame' created successfully.")
print(count_genres_frame)

In [ ]:
#DEEP REVIEW OF UNRESOLVED HINDI + CHINESE ROWS


# Confirmed wrong rows are removed and replaced automatically.
# Strongly supported genre corrections are applied automatically.
# Still-ambiguous rows remain in the final dataset and are exported
# for manual listening; they are not guessed or deleted.

import re
import json
import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


# REQUIRE CELL 7

REQUIRED_CELL7_NAMES = [
    "MANIFEST_DIR",
    "GENRES",
    "TARGET_PER_GENRE",
    "safe_text",
    "normalise_text",
    "similarity",
    "script_count",
    "clean_synced_lyrics",
    "request_json",
    "verify_itunes_if_needed",
    "lastfm_track_info",
    "lastfm_artist_info",
    "spotify_track_info",
    "bad_track_reason",
    "row_key_from_values",
    "combine_unique_rows",
    "select_exact_balanced_rows",
    "organise_final_rows",
    "is_perfect_final",
    "remaining_deficits",
    "count_genres_frame",
    "find_original_metadata_csv",
    "build_previous_csv_candidate_pool",
    "refill_only_missing_slots",
    "write_using_original_schema",
    "timestamped_backup",
    "save_all_caches",
    "HINDI_WORDS",
    "PUNJABI_WORDS",
    "OTHER_INDIC_PATTERNS",
    "LRCLIB_SEARCH_URL",
]

missing_names = [
    name
    for name in REQUIRED_CELL7_NAMES
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Run the latest Cell 7 first. Missing names: "
        + ", ".join(missing_names)
    )


CELL8_VERSION = "cell8_deep_review_hindi_chinese_v2_empty_resume_safe"

DEEP_LYRICS_MIN_TITLE_MATCH = 0.65
DEEP_LYRICS_MIN_ARTIST_MATCH = 0.35
DEEP_LYRICS_MAX_QUERIES_PER_TRACK = 4

AUTO_APPLY_CONFIRMED_DECISIONS = True
OVERWRITE_ONLY_IF_BOTH_PERFECT = True


# PATHS

HINDI_REVIEW_CSV = (
    MANIFEST_DIR
    / "hindi_final_unresolved_review.csv"
)

CHINESE_REVIEW_CSV = (
    MANIFEST_DIR
    / "chinese_final_unresolved_review.csv"
)

HINDI_DEEP_RESULTS_CSV = (
    MANIFEST_DIR
    / "hindi_deep_review_results.csv"
)

CHINESE_DEEP_RESULTS_CSV = (
    MANIFEST_DIR
    / "chinese_deep_review_results.csv"
)

HINDI_STILL_MANUAL_CSV = (
    MANIFEST_DIR
    / "hindi_still_manual_review.csv"
)

CHINESE_STILL_MANUAL_CSV = (
    MANIFEST_DIR
    / "chinese_still_manual_review.csv"
)

HINDI_DEEP_REPLACEMENT_CSV = (
    MANIFEST_DIR
    / "hindi_deep_review_replacement_checks.csv"
)

CHINESE_DEEP_REPLACEMENT_CSV = (
    MANIFEST_DIR
    / "chinese_deep_review_replacement_checks.csv"
)


# TEXT / QUERY HELPERS

def simplify_title_for_lyrics(value):
    text = safe_text(value)

    text = re.sub(
        r"\([^)]*\)|\[[^\]]*\]",
        " ",
        text,
    )

    text = re.sub(
        r"\b(?:feat|featuring|ft)\.?\b.*$",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\bfrom\s+[\"“].*?[\"”]",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip(" -–—")


def primary_artist_for_lyrics(value):
    text = safe_text(value)

    first = re.split(
        r"\s*(?:,|&|/|\+|"
        r"\bfeat\.?\b|\bfeaturing\b|"
        r"\bft\.?\b|\bx\b)\s*",
        text,
        maxsplit=1,
        flags=re.IGNORECASE,
    )[0]

    return first.strip()


def deep_query_variants(
    row,
    spotify_result,
):
    title = safe_text(
        row.get("title")
    )

    artist = safe_text(
        row.get("artist")
    )

    simplified_title = (
        simplify_title_for_lyrics(
            title
        )
    )

    primary_artist = (
        primary_artist_for_lyrics(
            artist
        )
    )

    spotify_title = safe_text(
        spotify_result.get(
            "matched_title"
        )
    )

    spotify_artist = safe_text(
        spotify_result.get(
            "matched_artist"
        )
    )

    candidates = [
        (title, artist),
        (
            simplified_title,
            artist,
        ),
        (
            simplified_title,
            primary_artist,
        ),
        (
            spotify_title,
            spotify_artist,
        ),
    ]

    output = []
    seen = set()

    for candidate_title, candidate_artist in candidates:
        candidate_title = safe_text(
            candidate_title
        )

        candidate_artist = safe_text(
            candidate_artist
        )

        if not candidate_title:
            continue

        key = (
            normalise_text(
                candidate_title
            )
            + "||"
            + normalise_text(
                candidate_artist
            )
        )

        if key in seen:
            continue

        seen.add(key)

        output.append(
            (
                candidate_title,
                candidate_artist,
            )
        )

        if len(output) >= (
            DEEP_LYRICS_MAX_QUERIES_PER_TRACK
        ):
            break

    return output


# DEEP LRCLIB SEARCH

def analyse_lyrics_text(
    lyrics_text,
):
    devanagari_count = script_count(
        lyrics_text,
        r"[\u0900-\u097f]",
    )

    gurmukhi_count = script_count(
        lyrics_text,
        r"[\u0a00-\u0a7f]",
    )

    han_count = script_count(
        lyrics_text,
        r"[\u3400-\u4dbf\u4e00-\u9fff]",
    )

    kana_count = script_count(
        lyrics_text,
        r"[\u3040-\u30ff]",
    )

    hangul_count = script_count(
        lyrics_text,
        r"[\uac00-\ud7af]",
    )

    other_indic_counts = {
        name: script_count(
            lyrics_text,
            pattern,
        )
        for name, pattern
        in OTHER_INDIC_PATTERNS.items()
    }

    roman_tokens = re.findall(
        r"[a-zA-Z']+",
        safe_text(
            lyrics_text
        ).casefold(),
    )

    hindi_hits = sum(
        token in HINDI_WORDS
        for token in roman_tokens
    )

    punjabi_hits = sum(
        token in PUNJABI_WORDS
        for token in roman_tokens
    )

    return {
        "lyrics_character_count": len(
            safe_text(lyrics_text)
        ),
        "devanagari_count": (
            devanagari_count
        ),
        "gurmukhi_count": (
            gurmukhi_count
        ),
        "han_count": han_count,
        "kana_count": kana_count,
        "hangul_count": hangul_count,
        "kana_hangul_count": (
            kana_count
            + hangul_count
        ),
        "other_indic_counts": (
            other_indic_counts
        ),
        "hindi_word_hits": (
            hindi_hits
        ),
        "punjabi_word_hits": (
            punjabi_hits
        ),
    }


def deep_lrclib_search(
    row,
    language_name,
    spotify_result,
):
    best = None
    best_score = -1.0
    query_log = []

    original_title = safe_text(
        row.get("title")
    )

    original_artist = safe_text(
        row.get("artist")
    )

    duration = row.get(
        "duration_seconds",
        "",
    )

    for query_title, query_artist in (
        deep_query_variants(
            row,
            spotify_result,
        )
    ):
        params = {
            "track_name": query_title,
        }

        if query_artist:
            params[
                "artist_name"
            ] = query_artist

        payload, status = request_json(
            LRCLIB_SEARCH_URL,
            params=params,
            service="LRCLIBDeep",
            sleep_after=0.03,
        )

        query_log.append(
            {
                "query_title": query_title,
                "query_artist": query_artist,
                "status": status,
            }
        )

        if (
            status != "ok"
            or not isinstance(
                payload,
                list,
            )
        ):
            continue

        for item in payload:
            if not isinstance(
                item,
                dict,
            ):
                continue

            found_title = safe_text(
                item.get("trackName")
            )

            found_artist = safe_text(
                item.get("artistName")
            )

            title_score = max(
                similarity(
                    original_title,
                    found_title,
                ),
                similarity(
                    query_title,
                    found_title,
                ),
            )

            artist_score = max(
                similarity(
                    original_artist,
                    found_artist,
                ),
                similarity(
                    query_artist,
                    found_artist,
                ),
            )

            if (
                title_score
                < DEEP_LYRICS_MIN_TITLE_MATCH
            ):
                continue

            if (
                query_artist
                and artist_score
                < DEEP_LYRICS_MIN_ARTIST_MATCH
            ):
                continue

            score = (
                title_score * 0.72
                + artist_score * 0.23
            )

            try:
                expected_duration = float(
                    duration
                )

                found_duration = float(
                    item.get("duration")
                )

                difference = abs(
                    expected_duration
                    - found_duration
                )

                if difference <= 15:
                    score += 0.05
                elif difference > 50:
                    score -= 0.10
            except Exception:
                difference = None

            lyrics_text = safe_text(
                item.get(
                    "plainLyrics"
                )
            )

            if not lyrics_text:
                lyrics_text = (
                    clean_synced_lyrics(
                        item.get(
                            "syncedLyrics"
                        )
                    )
                )

            if not lyrics_text:
                continue

            if score > best_score:
                best = {
                    "record_id": safe_text(
                        item.get("id")
                    ),
                    "matched_title": (
                        found_title
                    ),
                    "matched_artist": (
                        found_artist
                    ),
                    "match_score": round(
                        score,
                        4,
                    ),
                    "query_title": (
                        query_title
                    ),
                    "query_artist": (
                        query_artist
                    ),
                    **analyse_lyrics_text(
                        lyrics_text
                    ),
                }

                best_score = score

    if best is None:
        return {
            "ok": False,
            "status": (
                "deep_lyrics_no_match"
            ),
            "query_log": query_log,
        }

    return {
        "ok": True,
        "status": (
            "deep_lyrics_match"
        ),
        "provider": "LRCLIB",
        "query_log": query_log,
        **best,
    }


# LANGUAGE DECISION

def deep_language_decision(
    row,
    language_name,
    lyrics,
    itunes_result,
    spotify_result,
):
    raw_genre = " ".join(
        [
            safe_text(
                row.get("raw_genre")
            ),
            safe_text(
                itunes_result.get(
                    "raw_genre"
                )
            ),
        ]
    ).casefold()

    title_artist = (
        safe_text(
            row.get("title")
        )
        + " "
        + safe_text(
            row.get("artist")
        )
    )

    if language_name == "Hindi":
        explicit_wrong_markers = [
            "punjabi",
            "urdu",
            "telugu",
            "tamil",
            "kannada",
            "malayalam",
            "bengali",
            "bangla",
            "gujarati",
            "marathi",
            "bhojpuri",
            "haryanvi",
        ]

        for marker in explicit_wrong_markers:
            if marker in raw_genre:
                return (
                    "CONFIRMED_REMOVE",
                    (
                        "explicit_non_hindi_"
                        + marker
                        + "_metadata"
                    ),
                    "high",
                )

        if lyrics.get("ok"):
            if (
                lyrics.get(
                    "gurmukhi_count",
                    0,
                )
                >= 3
            ):
                return (
                    "CONFIRMED_REMOVE",
                    "punjabi_gurmukhi_lyrics",
                    "high",
                )

            for name, count in (
                lyrics.get(
                    "other_indic_counts",
                    {},
                ).items()
            ):
                if count >= 3:
                    return (
                        "CONFIRMED_REMOVE",
                        (
                            "non_hindi_"
                            + name
                            + "_lyrics"
                        ),
                        "high",
                    )

            if (
                lyrics.get(
                    "devanagari_count",
                    0,
                )
                >= 15
            ):
                return (
                    "CONFIRMED_KEEP",
                    "hindi_devanagari_lyrics",
                    "high",
                )

            hindi_hits = lyrics.get(
                "hindi_word_hits",
                0,
            )

            punjabi_hits = lyrics.get(
                "punjabi_word_hits",
                0,
            )

            if (
                punjabi_hits >= 4
                and punjabi_hits
                > hindi_hits * 1.20
            ):
                return (
                    "CONFIRMED_REMOVE",
                    "punjabi_romanised_lyrics",
                    "moderate",
                )

            if (
                hindi_hits >= 6
                and hindi_hits
                >= punjabi_hits * 1.50
            ):
                return (
                    "CONFIRMED_KEEP",
                    "hindi_romanised_lyrics",
                    "moderate",
                )

        if re.search(
            r"[\u0900-\u097f]",
            title_artist,
        ):
            return (
                "CONFIRMED_KEEP",
                "hindi_devanagari_title",
                "moderate",
            )

        return (
            "STILL_MANUAL",
            "insufficient_hindi_language_evidence",
            "low",
        )

    # Chinese
    if any(
        marker in raw_genre
        for marker in [
            "k-pop",
            "j-pop",
            "japanese",
            "korean",
        ]
    ):
        return (
            "CONFIRMED_REMOVE",
            "explicit_non_chinese_metadata",
            "high",
        )

    if lyrics.get("ok"):
        han_count = lyrics.get(
            "han_count",
            0,
        )

        kana_hangul_count = (
            lyrics.get(
                "kana_hangul_count",
                0,
            )
        )

        if (
            kana_hangul_count >= 10
            and kana_hangul_count
            > han_count
        ):
            return (
                "CONFIRMED_REMOVE",
                "japanese_or_korean_lyrics",
                "high",
            )

        if han_count >= 20:
            return (
                "CONFIRMED_KEEP",
                "chinese_han_lyrics",
                "high",
            )

    title_has_han = bool(
        re.search(
            r"[\u3400-\u4dbf\u4e00-\u9fff]",
            title_artist,
        )
    )

    chinese_context = any(
        marker in raw_genre
        for marker in [
            "mandopop",
            "cantopop",
            "chinese",
            "華語",
            "国语",
            "國語",
            "中文",
        ]
    )

    spotify_market = safe_text(
        spotify_result.get(
            "market"
        )
    ).upper()

    if (
        title_has_han
        and (
            chinese_context
            or spotify_market
            in {"CN", "TW", "HK"}
        )
    ):
        return (
            "CONFIRMED_KEEP",
            "chinese_title_and_market_context",
            "moderate",
        )

    return (
        "STILL_MANUAL",
        "insufficient_chinese_language_evidence",
        "low",
    )


# GENRE DECISION

def deep_genre_decision(
    row,
    language_status,
    itunes_result,
    lastfm_result,
    spotify_result,
):
    original_genre = safe_text(
        row.get("final_genre")
    )

    itunes_genre = (
        itunes_result.get(
            "mapped_genre"
        )
    )

    lastfm_track_genre = (
        lastfm_result.get(
            "mapped_genre"
        )
    )

    spotify_artist_genre = (
        spotify_result.get(
            "mapped_genre"
        )
    )

    if original_genre not in GENRES:
        return (
            "STILL_MANUAL",
            "",
            "missing_valid_original_genre",
            "low",
        )

    if language_status == (
        "CONFIRMED_REMOVE"
    ):
        return (
            "CONFIRMED_REMOVE",
            original_genre,
            "language_failed_before_genre",
            "high",
        )

    # Strong automatic reclassification:
    # exact-track Last.fm and exact iTunes match agree.
    if (
        lastfm_track_genre in GENRES
        and itunes_genre in GENRES
        and lastfm_track_genre
        == itunes_genre
        and lastfm_track_genre
        != original_genre
    ):
        return (
            "RECLASSIFY",
            lastfm_track_genre,
            "lastfm_track_itunes_agree_different_genre",
            "high",
        )

    support_original = []

    if (
        lastfm_track_genre
        == original_genre
    ):
        support_original.append(
            "lastfm_track"
        )

    if itunes_genre == original_genre:
        support_original.append(
            "itunes"
        )

    if (
        spotify_result.get("ok")
        and spotify_artist_genre
        == original_genre
    ):
        support_original.append(
            "spotify_artist"
        )

    if support_original:
        return (
            "CONFIRMED_KEEP",
            original_genre,
            (
                "original_genre_supported_by_"
                + "_".join(
                    support_original
                )
            ),
            (
                "high"
                if len(
                    support_original
                ) >= 2
                else "moderate"
            ),
        )

    # One exact-track Last.fm conflict is useful but not enough to
    # automatically move an already-curated final row.
    if (
        lastfm_track_genre in GENRES
        and lastfm_track_genre
        != original_genre
    ):
        return (
            "STILL_MANUAL",
            original_genre,
            (
                "single_lastfm_track_conflict_"
                + lastfm_track_genre
            ),
            "moderate",
        )

    if (
        itunes_genre in GENRES
        and spotify_artist_genre
        in GENRES
        and itunes_genre
        == spotify_artist_genre
        and itunes_genre
        != original_genre
    ):
        return (
            "STILL_MANUAL",
            original_genre,
            (
                "itunes_spotify_artist_agree_"
                "but_no_lastfm_track_support"
            ),
            "moderate",
        )

    return (
        "STILL_MANUAL",
        original_genre,
        "insufficient_specific_genre_evidence",
        "low",
    )


# AUDIT ONE REVIEW ROW

def deep_review_one_row(
    row,
    language_name,
):
    row_dict = row.to_dict()

    title = safe_text(
        row_dict.get("title")
    )

    artist = safe_text(
        row_dict.get("artist")
    )

    bad_reason = bad_track_reason(
        row_dict
    )

    if bad_reason:
        row_dict.update(
            {
                "deep_review_version": (
                    CELL8_VERSION
                ),
                "deep_review_status": (
                    "CONFIRMED_REMOVE"
                ),
                "deep_language_status": (
                    "NOT_CHECKED"
                ),
                "deep_language_reason": (
                    bad_reason
                ),
                "deep_genre_status": (
                    "NOT_CHECKED"
                ),
                "deep_genre_reason": (
                    bad_reason
                ),
                "deep_recommended_genre": (
                    safe_text(
                        row_dict.get(
                            "final_genre"
                        )
                    )
                ),
                "deep_review_confidence": (
                    "high"
                ),
            }
        )

        return row_dict

    itunes_result = (
        verify_itunes_if_needed(
            row_dict,
            language_name,
        )
    )

    lastfm_result = lastfm_track_info(
        title,
        artist,
    )

    spotify_result = spotify_track_info(
        title,
        artist,
        language_name,
    )

    lyrics_result = deep_lrclib_search(
        row_dict,
        language_name,
        spotify_result,
    )

    (
        language_status,
        language_reason,
        language_confidence,
    ) = deep_language_decision(
        row_dict,
        language_name,
        lyrics_result,
        itunes_result,
        spotify_result,
    )

    (
        genre_status,
        recommended_genre,
        genre_reason,
        genre_confidence,
    ) = deep_genre_decision(
        row_dict,
        language_status,
        itunes_result,
        lastfm_result,
        spotify_result,
    )

    if language_status == (
        "CONFIRMED_REMOVE"
    ):
        final_status = (
            "CONFIRMED_REMOVE"
        )

    elif genre_status == "RECLASSIFY":
        final_status = "RECLASSIFY"

    elif (
        language_status
        == "CONFIRMED_KEEP"
        and genre_status
        == "CONFIRMED_KEEP"
    ):
        final_status = (
            "CONFIRMED_KEEP"
        )

    else:
        final_status = "STILL_MANUAL"

    confidence_order = {
        "low": 1,
        "moderate": 2,
        "high": 3,
    }

    final_confidence = min(
        [
            language_confidence,
            genre_confidence,
        ],
        key=lambda value: (
            confidence_order.get(
                value,
                0,
            )
        ),
    )

    row_dict.update(
        {
            "deep_review_version": (
                CELL8_VERSION
            ),
            "deep_review_status": (
                final_status
            ),
            "deep_language_status": (
                language_status
            ),
            "deep_language_reason": (
                language_reason
            ),
            "deep_genre_status": (
                genre_status
            ),
            "deep_genre_reason": (
                genre_reason
            ),
            "deep_recommended_genre": (
                recommended_genre
            ),
            "deep_review_confidence": (
                final_confidence
            ),
            "deep_lyrics_status": (
                lyrics_result.get(
                    "status",
                    "",
                )
            ),
            "deep_lyrics_match_score": (
                lyrics_result.get(
                    "match_score",
                    "",
                )
            ),
            "deep_lyrics_matched_title": (
                lyrics_result.get(
                    "matched_title",
                    "",
                )
            ),
            "deep_lyrics_matched_artist": (
                lyrics_result.get(
                    "matched_artist",
                    "",
                )
            ),
            "deep_lyrics_devanagari_count": (
                lyrics_result.get(
                    "devanagari_count",
                    0,
                )
            ),
            "deep_lyrics_gurmukhi_count": (
                lyrics_result.get(
                    "gurmukhi_count",
                    0,
                )
            ),
            "deep_lyrics_han_count": (
                lyrics_result.get(
                    "han_count",
                    0,
                )
            ),
            "deep_lyrics_kana_hangul_count": (
                lyrics_result.get(
                    "kana_hangul_count",
                    0,
                )
            ),
            "deep_lyrics_hindi_word_hits": (
                lyrics_result.get(
                    "hindi_word_hits",
                    0,
                )
            ),
            "deep_lyrics_punjabi_word_hits": (
                lyrics_result.get(
                    "punjabi_word_hits",
                    0,
                )
            ),
            "deep_lastfm_track_status": (
                lastfm_result.get(
                    "status",
                    "",
                )
            ),
            "deep_lastfm_track_genre": (
                lastfm_result.get(
                    "mapped_genre"
                )
                or ""
            ),
            "deep_lastfm_track_tags": (
                ", ".join(
                    lastfm_result.get(
                        "tags",
                        [],
                    )
                )
            ),
            "deep_itunes_status": (
                itunes_result.get(
                    "status",
                    "",
                )
            ),
            "deep_itunes_genre": (
                itunes_result.get(
                    "mapped_genre"
                )
                or ""
            ),
            "deep_itunes_raw_genre": (
                itunes_result.get(
                    "raw_genre",
                    "",
                )
            ),
            "deep_spotify_status": (
                spotify_result.get(
                    "status",
                    "",
                )
            ),
            "deep_spotify_artist_genre": (
                spotify_result.get(
                    "mapped_genre"
                )
                or ""
            ),
            "deep_spotify_artist_genres": (
                ", ".join(
                    spotify_result.get(
                        "artist_genres",
                        [],
                    )
                )
            ),
            "deep_spotify_track_id": (
                spotify_result.get(
                    "track_id",
                    "",
                )
            ),
        }
    )

    return row_dict


# PROCESS REVIEW QUEUE

DEEP_REVIEW_OUTPUT_COLUMNS = [
    "title",
    "artist",
    "final_genre",
    "deep_review_version",
    "deep_review_status",
    "deep_language_status",
    "deep_language_reason",
    "deep_genre_status",
    "deep_genre_reason",
    "deep_recommended_genre",
    "deep_review_confidence",
    "deep_lyrics_status",
    "deep_lyrics_match_score",
    "deep_lyrics_matched_title",
    "deep_lyrics_matched_artist",
    "deep_lyrics_devanagari_count",
    "deep_lyrics_gurmukhi_count",
    "deep_lyrics_han_count",
    "deep_lyrics_kana_hangul_count",
    "deep_lyrics_hindi_word_hits",
    "deep_lyrics_punjabi_word_hits",
    "deep_lastfm_track_status",
    "deep_lastfm_track_genre",
    "deep_lastfm_track_tags",
    "deep_itunes_status",
    "deep_itunes_genre",
    "deep_itunes_raw_genre",
    "deep_spotify_status",
    "deep_spotify_artist_genre",
    "deep_spotify_artist_genres",
    "deep_spotify_track_id",
]


def make_structured_empty_deep_results(
    review_df,
):
    base_columns = list(
        review_df.columns
    )

    all_columns = list(
        dict.fromkeys(
            base_columns
            + DEEP_REVIEW_OUTPUT_COLUMNS
        )
    )

    return pd.DataFrame(
        columns=all_columns
    )


def safe_read_csv_or_empty(
    path,
):
    path = Path(path)

    if not path.exists():
        return pd.DataFrame()

    try:
        return pd.read_csv(
            path,
            low_memory=False,
        )

    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def process_review_queue(
    review_csv,
    output_csv,
    manual_csv,
    language_name,
):
    review_csv = Path(
        review_csv
    )

    output_csv = Path(
        output_csv
    )

    manual_csv = Path(
        manual_csv
    )

    output_csv.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    manual_csv.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if not review_csv.exists():
        raise FileNotFoundError(
            f"Review CSV not found: "
            f"{review_csv}"
        )

    review_df = safe_read_csv_or_empty(
        review_csv
    )

    print(
        f"\n{language_name} deep review rows:",
        len(review_df),
    )

    # EMPTY QUEUE:
    # Return a correctly structured empty DataFrame instead of
    # creating a no-column DataFrame and crashing on
    # results_df["deep_review_status"].
    if review_df.empty:
        results_df = (
            make_structured_empty_deep_results(
                review_df
            )
        )

        results_df.to_csv(
            output_csv,
            index=False,
            encoding="utf-8-sig",
        )

        results_df.to_csv(
            manual_csv,
            index=False,
            encoding="utf-8-sig",
        )

        print(
            f"{language_name}: review queue is empty. "
            "No rows require deep review."
        )

        print(
            f"{language_name}: empty structured outputs saved."
        )

        return results_df

    required_columns = {
        "title",
        "artist",
    }

    missing_columns = (
        required_columns
        - set(review_df.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{language_name} review CSV is missing: "
            + ", ".join(
                sorted(missing_columns)
            )
        )

    review_df = review_df.copy()

    review_df["_deep_review_key"] = [
        row_key_from_values(
            title,
            artist,
        )
        for title, artist in zip(
            review_df["title"],
            review_df["artist"],
        )
    ]

    review_df = (
        review_df.drop_duplicates(
            subset=["_deep_review_key"],
            keep="first",
        )
        .reset_index(drop=True)
    )

    review_keys = list(
        review_df["_deep_review_key"]
    )

    current_key_set = set(
        review_keys
    )

    # RESUME:
    # Reuse completed rows from an existing partial result file,
    # but only when they still belong to the current review queue.
    existing_df = safe_read_csv_or_empty(
        output_csv
    )

    completed_map = {}

    if (
        not existing_df.empty
        and {
            "title",
            "artist",
            "deep_review_status",
        }.issubset(
            existing_df.columns
        )
    ):
        existing_df = existing_df.copy()

        existing_df["_deep_review_key"] = [
            row_key_from_values(
                title,
                artist,
            )
            for title, artist in zip(
                existing_df["title"],
                existing_df["artist"],
            )
        ]

        existing_df = (
            existing_df[
                existing_df[
                    "_deep_review_key"
                ].isin(
                    current_key_set
                )
            ]
            .drop_duplicates(
                subset=["_deep_review_key"],
                keep="last",
            )
        )

        for _, existing_row in (
            existing_df.iterrows()
        ):
            status = safe_text(
                existing_row.get(
                    "deep_review_status"
                )
            )

            if status:
                completed_map[
                    existing_row[
                        "_deep_review_key"
                    ]
                ] = (
                    existing_row.drop(
                        labels=[
                            "_deep_review_key"
                        ],
                        errors="ignore",
                    ).to_dict()
                )

    print(
        f"{language_name}: previously completed rows reused:",
        len(completed_map),
    )

    pending_df = review_df[
        ~review_df[
            "_deep_review_key"
        ].isin(
            completed_map.keys()
        )
    ].copy()

    print(
        f"{language_name}: rows remaining to process:",
        len(pending_df),
    )

    result_map = dict(
        completed_map
    )

    newly_processed = 0

    def build_ordered_results():
        ordered_rows = []

        for key in review_keys:
            result = result_map.get(
                key
            )

            if result is not None:
                ordered_rows.append(
                    result
                )

        if not ordered_rows:
            return (
                make_structured_empty_deep_results(
                    review_df.drop(
                        columns=[
                            "_deep_review_key"
                        ],
                        errors="ignore",
                    )
                )
            )

        frame = pd.DataFrame(
            ordered_rows
        )

        for column in (
            DEEP_REVIEW_OUTPUT_COLUMNS
        ):
            if column not in frame.columns:
                frame[column] = ""

        return frame

    try:
        for _, row in tqdm(
            pending_df.iterrows(),
            total=len(pending_df),
            desc=f"{language_name} deep review",
            unit="track",
            dynamic_ncols=True,
        ):
            key = row[
                "_deep_review_key"
            ]

            clean_row = row.drop(
                labels=[
                    "_deep_review_key"
                ],
                errors="ignore",
            )

            result_map[key] = (
                deep_review_one_row(
                    clean_row,
                    language_name,
                )
            )

            newly_processed += 1

            if newly_processed % 5 == 0:
                checkpoint_df = (
                    build_ordered_results()
                )

                checkpoint_df.to_csv(
                    output_csv,
                    index=False,
                    encoding="utf-8-sig",
                )

                save_all_caches()

    except KeyboardInterrupt:
        checkpoint_df = (
            build_ordered_results()
        )

        checkpoint_df.to_csv(
            output_csv,
            index=False,
            encoding="utf-8-sig",
        )

        save_all_caches()

        print(
            f"\n{language_name}: interrupted. "
            "Partial deep-review results saved."
        )

        raise

    results_df = (
        build_ordered_results()
    )

    # Required column must always exist, even for unusual or
    # partially structured result rows.
    if (
        "deep_review_status"
        not in results_df.columns
    ):
        results_df[
            "deep_review_status"
        ] = pd.Series(
            dtype="object"
        )

    results_df.to_csv(
        output_csv,
        index=False,
        encoding="utf-8-sig",
    )

    manual_mask = (
        results_df[
            "deep_review_status"
        ]
        .astype(str)
        .str.strip()
        .eq("STILL_MANUAL")
    )

    manual_df = results_df[
        manual_mask
    ].copy()

    manual_df.to_csv(
        manual_csv,
        index=False,
        encoding="utf-8-sig",
    )

    save_all_caches()

    print(
        f"\n{language_name} deep-review decisions:"
    )

    decision_counts = (
        results_df[
            "deep_review_status"
        ]
        .astype(str)
        .replace(
            {
                "": "MISSING_STATUS",
                "nan": "MISSING_STATUS",
            }
        )
        .value_counts(
            dropna=False
        )
    )

    print(
        decision_counts
    )

    print(
        f"{language_name}: newly processed:",
        newly_processed,
    )

    print(
        f"{language_name}: still manual:",
        len(manual_df),
    )

    return results_df


# APPLY CONFIRMED DECISIONS TO FINAL CSV

def apply_deep_decisions(
    final_csv,
    deep_results,
    language_name,
):
    final_csv = Path(
        final_csv
    )

    final_df = pd.read_csv(
        final_csv,
        low_memory=False,
    )

    original_columns = list(
        final_df.columns
    )

    final_df["_key"] = [
        row_key_from_values(
            title,
            artist,
        )
        for title, artist in zip(
            final_df["title"],
            final_df["artist"],
        )
    ]

    decision_map = {}

    for _, row in (
        deep_results.iterrows()
    ):
        key = row_key_from_values(
            row.get("title"),
            row.get("artist"),
        )

        decision_map[key] = (
            row.to_dict()
        )

    audited_rows = []

    for _, row in final_df.iterrows():
        row_dict = row.to_dict()

        key = row_dict["_key"]

        decision = decision_map.get(
            key
        )

        row_dict["final_audit_origin"] = (
            "existing_final_300"
        )

        row_dict["final_audit_pass"] = True

        if decision:
            status = safe_text(
                decision.get(
                    "deep_review_status"
                )
            )

            row_dict[
                "deep_review_status"
            ] = status

            row_dict[
                "deep_language_reason"
            ] = safe_text(
                decision.get(
                    "deep_language_reason"
                )
            )

            row_dict[
                "deep_genre_reason"
            ] = safe_text(
                decision.get(
                    "deep_genre_reason"
                )
            )

            if status == (
                "CONFIRMED_REMOVE"
            ):
                row_dict[
                    "final_audit_pass"
                ] = False

            elif status == "RECLASSIFY":
                recommended = safe_text(
                    decision.get(
                        "deep_recommended_genre"
                    )
                )

                if recommended in GENRES:
                    row_dict[
                        "final_genre"
                    ] = recommended

        audited_rows.append(
            row_dict
        )

    audited_final = pd.DataFrame(
        audited_rows
    ).drop(
        columns=["_key"],
        errors="ignore",
    )

    return (
        audited_final,
        original_columns,
    )


def ensure_candidate_pools():
    global hindi_candidate_pool
    global chinese_candidate_pool

    hindi_final_csv = (
        find_original_metadata_csv(
            "Hindi"
        )
    )

    chinese_final_csv = (
        find_original_metadata_csv(
            "Chinese"
        )
    )

    if (
        "hindi_candidate_pool"
        not in globals()
        or not isinstance(
            globals().get(
                "hindi_candidate_pool"
            ),
            pd.DataFrame,
        )
    ):
        (
            hindi_candidate_pool,
            _,
        ) = build_previous_csv_candidate_pool(
            "Hindi",
            hindi_final_csv,
        )

    if (
        "chinese_candidate_pool"
        not in globals()
        or not isinstance(
            globals().get(
                "chinese_candidate_pool"
            ),
            pd.DataFrame,
        )
    ):
        (
            chinese_candidate_pool,
            _,
        ) = build_previous_csv_candidate_pool(
            "Chinese",
            chinese_final_csv,
        )

    return (
        hindi_final_csv,
        chinese_final_csv,
    )


def repair_after_deep_review(
    final_csv,
    deep_results,
    language_name,
    replacement_csv,
):
    (
        audited_final,
        original_columns,
    ) = apply_deep_decisions(
        final_csv,
        deep_results,
        language_name,
    )

    candidate_pool = (
        hindi_candidate_pool
        if language_name == "Hindi"
        else chinese_candidate_pool
    )

    selected, overflow, replacement_checks = (
        refill_only_missing_slots(
            language_name,
            candidate_pool,
            audited_final,
            replacement_csv,
        )
    )

    selected = organise_final_rows(
        selected,
        language_name,
    )

    perfect = is_perfect_final(
        selected
    )

    print(
        f"\n{language_name} post-deep-review counts:"
    )

    print(
        genre_counts(
            selected
        )
    )

    print(
        f"{language_name} perfect:",
        perfect,
    )

    return {
        "selected": selected,
        "perfect": perfect,
        "original_columns": (
            original_columns
        ),
        "final_csv": Path(
            final_csv
        ),
        "overflow": overflow,
        "replacement_checks": (
            replacement_checks
        ),
    }


# RUN

print(
    "Cell 8: deeper review of unresolved "
    "Hindi and Chinese rows only."
)

print(
    "English is not read or modified."
)

print(
    "Rows without enough evidence remain in "
    "STILL_MANUAL; they are not guessed."
)


hindi_deep_results = process_review_queue(
    HINDI_REVIEW_CSV,
    HINDI_DEEP_RESULTS_CSV,
    HINDI_STILL_MANUAL_CSV,
    "Hindi",
)

chinese_deep_results = process_review_queue(
    CHINESE_REVIEW_CSV,
    CHINESE_DEEP_RESULTS_CSV,
    CHINESE_STILL_MANUAL_CSV,
    "Chinese",
)


if AUTO_APPLY_CONFIRMED_DECISIONS:
    (
        HINDI_FINAL_CSV,
        CHINESE_FINAL_CSV,
    ) = ensure_candidate_pools()

    # Use a separate replacement checkpoint version for Cell 8.
    CELL5B_VERSION = CELL8_VERSION

    hindi_repair = repair_after_deep_review(
        HINDI_FINAL_CSV,
        hindi_deep_results,
        "Hindi",
        HINDI_DEEP_REPLACEMENT_CSV,
    )

    chinese_repair = repair_after_deep_review(
        CHINESE_FINAL_CSV,
        chinese_deep_results,
        "Chinese",
        CHINESE_DEEP_REPLACEMENT_CSV,
    )

    both_perfect = bool(
        hindi_repair["perfect"]
        and chinese_repair["perfect"]
    )

    if (
        both_perfect
        or not OVERWRITE_ONLY_IF_BOTH_PERFECT
    ):
        hindi_backup = timestamped_backup(
            HINDI_FINAL_CSV
        )

        chinese_backup = timestamped_backup(
            CHINESE_FINAL_CSV
        )

        write_using_original_schema(
            hindi_repair["selected"],
            HINDI_FINAL_CSV,
            hindi_repair[
                "original_columns"
            ],
        )

        write_using_original_schema(
            chinese_repair["selected"],
            CHINESE_FINAL_CSV,
            chinese_repair[
                "original_columns"
            ],
        )

        print(
            "\nSUCCESS: confirmed Cell 8 decisions "
            "were applied and both final CSVs remain "
            "perfectly balanced."
        )

        print(
            "Hindi backup:",
            hindi_backup,
        )

        print(
            "Chinese backup:",
            chinese_backup,
        )

    else:
        print(
            "\nFinal CSVs were not overwritten because "
            "both languages did not reach perfect balance."
        )

        print(
            "Hindi deficits:",
            remaining_deficits(
                hindi_repair[
                    "selected"
                ]
            ),
        )

        print(
            "Chinese deficits:",
            remaining_deficits(
                chinese_repair[
                    "selected"
                ]
            ),
        )


print(
    "\nHindi deep results:",
    HINDI_DEEP_RESULTS_CSV,
)

print(
    "Hindi still manual:",
    HINDI_STILL_MANUAL_CSV,
)

print(
    "\nChinese deep results:",
    CHINESE_DEEP_RESULTS_CSV,
)

print(
    "Chinese still manual:",
    CHINESE_STILL_MANUAL_CSV,
)

print(
    "\nAPI calls:",
    dict(API_CALLS),
)

In [ ]:
# REPLACE ALL STILL_MANUAL HINDI + CHINESE ROWS
#
# This cell:
#   - removes every STILL_MANUAL row
#   - removes every CONFIRMED_REMOVE row
#   - applies RECLASSIFY decisions
#   - keeps CONFIRMED_KEEP rows
#   - refills only the resulting genre deficits
#   - preserves exactly 60 songs per genre
#   - does not read or modify English

from pathlib import Path
import pandas as pd


# REQUIRE PREVIOUS CELLS

REQUIRED_NAMES = [
    "MANIFEST_DIR",
    "GENRES",
    "TARGET_PER_GENRE",
    "safe_text",
    "row_key_from_values",
    "find_original_metadata_csv",
    "build_previous_csv_candidate_pool",
    "refill_only_missing_slots",
    "organise_final_rows",
    "is_perfect_final",
    "count_genres_frame",
    "remaining_deficits",
    "timestamped_backup",
    "write_using_original_schema",
]

missing = [
    name
    for name in REQUIRED_NAMES
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run the latest Cell 7 and Cell 8 first. Missing: "
        + ", ".join(missing)
    )


CELL9_VERSION = (
    "cell9_replace_all_still_manual_v1"
)

# Replacement checks use a fresh version so old checkpoints
# from Cell 7/8 are not mixed into this run.
CELL5B_VERSION = CELL9_VERSION


# INPUT FILES

HINDI_DEEP_RESULTS_CSV = (
    MANIFEST_DIR
    / "hindi_deep_review_results.csv"
)

CHINESE_DEEP_RESULTS_CSV = (
    MANIFEST_DIR
    / "chinese_deep_review_results.csv"
)

HINDI_REPLACEMENT_CHECKS_CSV = (
    MANIFEST_DIR
    / "hindi_replace_all_manual_checks.csv"
)

CHINESE_REPLACEMENT_CHECKS_CSV = (
    MANIFEST_DIR
    / "chinese_replace_all_manual_checks.csv"
)

HINDI_REMOVED_MANUAL_CSV = (
    MANIFEST_DIR
    / "hindi_removed_all_manual_rows.csv"
)

CHINESE_REMOVED_MANUAL_CSV = (
    MANIFEST_DIR
    / "chinese_removed_all_manual_rows.csv"
)

HINDI_PROPOSED_CSV = (
    MANIFEST_DIR
    / "hindi_after_replace_all_manual_proposed.csv"
)

CHINESE_PROPOSED_CSV = (
    MANIFEST_DIR
    / "chinese_after_replace_all_manual_proposed.csv"
)


# LOAD / APPLY DECISIONS

REMOVE_STATUSES = {
    "STILL_MANUAL",
    "CONFIRMED_REMOVE",
}

KEEP_STATUSES = {
    "CONFIRMED_KEEP",
}


def load_deep_results(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Deep-review result file not found: {path}. "
            "Let Cell 8 finish first."
        )

    frame = pd.read_csv(
        path,
        low_memory=False,
    )

    required_columns = {
        "title",
        "artist",
        "deep_review_status",
    }

    missing_columns = (
        required_columns
        - set(frame.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{path.name} is missing columns: "
            + ", ".join(
                sorted(missing_columns)
            )
        )

    return frame


def apply_replace_all_manual_decisions(
    final_csv,
    deep_results,
    language_name,
    removed_csv,
):
    final_csv = Path(final_csv)

    final_df = pd.read_csv(
        final_csv,
        low_memory=False,
    )

    original_columns = list(
        final_df.columns
    )

    final_df["_decision_key"] = [
        row_key_from_values(
            title,
            artist,
        )
        for title, artist in zip(
            final_df["title"],
            final_df["artist"],
        )
    ]

    decision_map = {}

    for _, row in deep_results.iterrows():
        key = row_key_from_values(
            row.get("title"),
            row.get("artist"),
        )

        decision_map[key] = row.to_dict()

    audited_rows = []
    removed_rows = []

    for _, row in final_df.iterrows():
        row_dict = row.to_dict()
        key = row_dict[
            "_decision_key"
        ]

        decision = decision_map.get(
            key
        )

        row_dict["final_audit_origin"] = (
            "existing_final_300"
        )

        row_dict["final_audit_pass"] = True
        row_dict["cell9_decision"] = (
            "KEEP_NOT_IN_REVIEW_QUEUE"
        )

        if decision:
            status = safe_text(
                decision.get(
                    "deep_review_status"
                )
            )

            row_dict[
                "deep_review_status"
            ] = status

            row_dict[
                "deep_language_reason"
            ] = safe_text(
                decision.get(
                    "deep_language_reason"
                )
            )

            row_dict[
                "deep_genre_reason"
            ] = safe_text(
                decision.get(
                    "deep_genre_reason"
                )
            )

            if status in REMOVE_STATUSES:
                row_dict[
                    "final_audit_pass"
                ] = False

                row_dict[
                    "cell9_decision"
                ] = (
                    "REMOVE_ALL_UNRESOLVED"
                )

                removed_rows.append(
                    row_dict.copy()
                )

            elif status == "RECLASSIFY":
                recommended = safe_text(
                    decision.get(
                        "deep_recommended_genre"
                    )
                )

                if recommended in GENRES:
                    row_dict[
                        "final_genre"
                    ] = recommended

                    row_dict[
                        "cell9_decision"
                    ] = (
                        "RECLASSIFY_CONFIRMED"
                    )
                else:
                    row_dict[
                        "final_audit_pass"
                    ] = False

                    row_dict[
                        "cell9_decision"
                    ] = (
                        "REMOVE_INVALID_RECLASSIFICATION"
                    )

                    removed_rows.append(
                        row_dict.copy()
                    )

            elif status in KEEP_STATUSES:
                row_dict[
                    "cell9_decision"
                ] = (
                    "KEEP_CONFIRMED"
                )

            else:
                # Any unexpected review status is removed rather than guessed.
                row_dict[
                    "final_audit_pass"
                ] = False

                row_dict[
                    "cell9_decision"
                ] = (
                    "REMOVE_UNKNOWN_REVIEW_STATUS"
                )

                removed_rows.append(
                    row_dict.copy()
                )

        audited_rows.append(
            row_dict
        )

    audited_df = pd.DataFrame(
        audited_rows
    ).drop(
        columns=["_decision_key"],
        errors="ignore",
    )

    removed_df = pd.DataFrame(
        removed_rows
    ).drop(
        columns=["_decision_key"],
        errors="ignore",
    )

    removed_df.to_csv(
        removed_csv,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"\n{language_name}: rows removed "
        f"from review queue: {len(removed_df)}"
    )

    if len(removed_df):
        print(
            removed_df[
                "final_genre"
            ].value_counts(
                dropna=False
            )
        )

    return (
        audited_df,
        original_columns,
        removed_df,
    )


# CANDIDATE POOLS

def get_candidate_pools(
    hindi_final_csv,
    chinese_final_csv,
):
    global hindi_candidate_pool
    global chinese_candidate_pool

    if (
        "hindi_candidate_pool"
        not in globals()
        or not isinstance(
            globals().get(
                "hindi_candidate_pool"
            ),
            pd.DataFrame,
        )
    ):
        (
            hindi_candidate_pool,
            _,
        ) = build_previous_csv_candidate_pool(
            "Hindi",
            hindi_final_csv,
        )

    if (
        "chinese_candidate_pool"
        not in globals()
        or not isinstance(
            globals().get(
                "chinese_candidate_pool"
            ),
            pd.DataFrame,
        )
    ):
        (
            chinese_candidate_pool,
            _,
        ) = build_previous_csv_candidate_pool(
            "Chinese",
            chinese_final_csv,
        )


# REPAIR ONE LANGUAGE

def replace_all_manual_for_language(
    final_csv,
    deep_results,
    language_name,
    candidate_pool,
    replacement_csv,
    removed_csv,
    proposed_csv,
):
    (
        audited_final,
        original_columns,
        removed_df,
    ) = apply_replace_all_manual_decisions(
        final_csv,
        deep_results,
        language_name,
        removed_csv,
    )

    (
        selected,
        overflow,
        replacement_checks,
    ) = refill_only_missing_slots(
        language_name,
        candidate_pool,
        audited_final,
        replacement_csv,
    )

    selected = organise_final_rows(
        selected,
        language_name,
    )

    selected.to_csv(
        proposed_csv,
        index=False,
        encoding="utf-8-sig",
    )

    perfect = is_perfect_final(
        selected
    )

    print(
        f"\n{language_name} proposed counts:"
    )

    print(
        genre_counts(
            selected
        )
    )

    print(
        f"{language_name} rows:",
        len(selected),
    )

    print(
        f"{language_name} perfect:",
        perfect,
    )

    return {
        "final_csv": Path(final_csv),
        "selected": selected,
        "perfect": perfect,
        "original_columns": (
            original_columns
        ),
        "removed": removed_df,
        "overflow": overflow,
        "replacement_checks": (
            replacement_checks
        ),
        "proposed_csv": Path(
            proposed_csv
        ),
    }


# RUN

print(
    "Cell 9: replace every STILL_MANUAL Hindi and Chinese row."
)

print(
    "English is not read or modified."
)

print(
    "Only confirmed/reverified rows will remain in the final datasets."
)


hindi_deep_results = load_deep_results(
    HINDI_DEEP_RESULTS_CSV
)

chinese_deep_results = load_deep_results(
    CHINESE_DEEP_RESULTS_CSV
)


HINDI_FINAL_CSV = (
    find_original_metadata_csv(
        "Hindi"
    )
)

CHINESE_FINAL_CSV = (
    find_original_metadata_csv(
        "Chinese"
    )
)


get_candidate_pools(
    HINDI_FINAL_CSV,
    CHINESE_FINAL_CSV,
)


hindi_result = (
    replace_all_manual_for_language(
        HINDI_FINAL_CSV,
        hindi_deep_results,
        "Hindi",
        hindi_candidate_pool,
        HINDI_REPLACEMENT_CHECKS_CSV,
        HINDI_REMOVED_MANUAL_CSV,
        HINDI_PROPOSED_CSV,
    )
)

chinese_result = (
    replace_all_manual_for_language(
        CHINESE_FINAL_CSV,
        chinese_deep_results,
        "Chinese",
        chinese_candidate_pool,
        CHINESE_REPLACEMENT_CHECKS_CSV,
        CHINESE_REMOVED_MANUAL_CSV,
        CHINESE_PROPOSED_CSV,
    )
)


both_perfect = bool(
    hindi_result["perfect"]
    and chinese_result["perfect"]
)


if both_perfect:
    hindi_backup = timestamped_backup(
        HINDI_FINAL_CSV
    )

    chinese_backup = timestamped_backup(
        CHINESE_FINAL_CSV
    )

    write_using_original_schema(
        hindi_result["selected"],
        HINDI_FINAL_CSV,
        hindi_result[
            "original_columns"
        ],
    )

    write_using_original_schema(
        chinese_result["selected"],
        CHINESE_FINAL_CSV,
        chinese_result[
            "original_columns"
        ],
    )

    print(
        "\nSUCCESS: all STILL_MANUAL rows were removed, "
        "replacements were verified, and both final CSVs "
        "remain exactly 300 rows."
    )

    print(
        "Hindi backup:",
        hindi_backup,
    )

    print(
        "Chinese backup:",
        chinese_backup,
    )

else:
    print(
        "\nOriginal final CSVs were NOT overwritten."
    )

    print(
        "Hindi remaining deficits:",
        remaining_deficits(
            hindi_result[
                "selected"
            ]
        ),
    )

    print(
        "Chinese remaining deficits:",
        remaining_deficits(
            chinese_result[
                "selected"
            ]
        ),
    )

    print(
        "The proposed partial files and replacement checkpoints "
        "have been saved. Add more candidate CSVs and rerun."
    )


print(
    "\nHindi removed rows:",
    HINDI_REMOVED_MANUAL_CSV,
)

print(
    "Chinese removed rows:",
    CHINESE_REMOVED_MANUAL_CSV,
)

print(
    "\nHindi proposed:",
    HINDI_PROPOSED_CSV,
)

print(
    "Chinese proposed:",
    CHINESE_PROPOSED_CSV,
)

## 9. Optional Chinese platform review

Run this branch only when the Chinese unresolved-review CSV still contains rows after the main audit.


In [ ]:
# ============================================================
# LIVE CHINESE MUSICBRAINZ + PLATFORM VERIFICATION (FIXED)
# ============================================================

import pandas as pd
import requests
import time
import re
from urllib.parse import quote_plus
from IPython.display import display, clear_output

MUSICBRAINZ_RECORDING_SEARCH_URL = "https://musicbrainz.org/ws/2/recording/"
MUSICBRAINZ_RELEASE_LOOKUP_URL = "https://musicbrainz.org/ws/2/release/"

START_AT = 0
MAX_ROWS = None
SAVE_EVERY = 10

_release_lang_cache = {}  # avoid re-fetching the same release


def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


# ============================================================
# SCRIPT DETECTION (unchanged)
# ============================================================

def classify_script(text):
    text = safe_text(text)
    has_han    = bool(re.search(r"[\u3400-\u4DBF\u4E00-\u9FFF\uF900-\uFAFF]", text))
    has_kana   = bool(re.search(r"[\u3040-\u30FF]", text))
    has_hangul = bool(re.search(r"[\uAC00-\uD7AF]", text))
    if has_kana or has_hangul:
        return "mixed_cjk" if has_han else "japanese_or_korean"
    if has_han:
        return "chinese"
    return "latin_only"


def has_chinese_chars(text):
    return classify_script(text) == "chinese"


def is_japanese_or_korean(text):
    return classify_script(text) in {"japanese_or_korean", "mixed_cjk"}


def norm_text(x):
    x = safe_text(x).lower()
    x = re.sub(r"\(.*?\)|\[.*?\]", " ", x)
    x = re.sub(r"[^a-z0-9\u4e00-\u9fff]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def make_links(title, artist):
    q = quote_plus(f"{safe_text(title)} {safe_text(artist)}")
    return {
        "netease_search_url": f"https://music.163.com/#/search/m/?s={q}&type=1",
        "qq_search_url": f"https://y.qq.com/n/ryqq/search?w={q}",
        "kugou_search_url": f"https://www.kugou.com/yy/html/search.html#searchType=song&searchKeyWord={q}",
        "youtube_search_url": f"https://www.youtube.com/results?search_query={q}+official+audio",
    }


# ============================================================
# LANGUAGES actually confirmed as Chinese (used for the strict
# "latin-only title" override rule)
# ============================================================
_CHINESE_MB_LANGUAGES = {"cmn", "zho", "yue", "chi", "nan", "hak"}

_NON_CHINESE_MB_LANGUAGES = {
    "jpn", "kor", "eng", "fra", "deu", "spa", "por",
    "ita", "rus", "ara", "hin", "vie", "tha", "ind",
}


def fetch_release_language(release_id):
    """Follow-up lookup — recording SEARCH results don't carry
    text-representation, so we fetch the release directly."""
    if not release_id:
        return ""
    if release_id in _release_lang_cache:
        return _release_lang_cache[release_id]

    lang = ""
    try:
        r = requests.get(
            f"{MUSICBRAINZ_RELEASE_LOOKUP_URL}{release_id}",
            params={"fmt": "json"},
            headers={"User-Agent": "HullMScMusicQualityResearch/1.0 (student project)"},
            timeout=15,
        )
        if r.status_code == 200:
            tr = r.json().get("text-representation")
            if isinstance(tr, dict) and tr.get("language"):
                lang = str(tr["language"]).strip().lower()
    except Exception:
        lang = ""

    time.sleep(1.1)  # stay polite to MB rate limits
    _release_lang_cache[release_id] = lang
    return lang


def search_musicbrainz_live(title, artist):
    query = f'recording:"{safe_text(title)}" AND artist:"{safe_text(artist)}"'

    try:
        r = requests.get(
            MUSICBRAINZ_RECORDING_SEARCH_URL,
            params={"query": query, "fmt": "json", "limit": 3},
            headers={"User-Agent": "HullMScMusicQualityResearch/1.0 (student project)"},
            timeout=20,
        )

        empty = {
            "musicbrainz_ok": False, "musicbrainz_score": 0,
            "musicbrainz_title": "", "musicbrainz_artist": "",
            "musicbrainz_status": "", "musicbrainz_language": "",
            "musicbrainz_area": "", "musicbrainz_non_chinese_flag": False,
        }

        if r.status_code != 200:
            empty["musicbrainz_status"] = f"http_{r.status_code}"
            return empty

        recordings = r.json().get("recordings", [])
        if not recordings:
            empty["musicbrainz_status"] = "no_match"
            return empty

        best = recordings[0]
        mb_title = best.get("title", "")
        mb_score = int(best.get("score", 0))

        mb_artist_parts = [a.get("name", "") for a in best.get("artist-credit", []) if isinstance(a, dict)]
        mb_artist = ", ".join(x for x in mb_artist_parts if x)

        # area — informational only now, NOT used to reject
        mb_area = ""
        releases = best.get("releases", []) or []
        for rel in releases:
            if isinstance(rel, dict):
                area_events = rel.get("release-events", [{}])
                if isinstance(area_events, list) and area_events:
                    area_name = area_events[0].get("area", {}).get("name", "")
                    if area_name:
                        mb_area = area_name.strip().lower()
                        break

        # language — actual follow-up lookup on the first release id
        mb_language = ""
        for rel in releases:
            if isinstance(rel, dict) and rel.get("id"):
                mb_language = fetch_release_language(rel["id"])
                if mb_language:
                    break

        non_chinese_flag = bool(mb_language and mb_language in _NON_CHINESE_MB_LANGUAGES)
        chinese_confirmed = bool(mb_language and mb_language in _CHINESE_MB_LANGUAGES)

        title_match = (norm_text(title) in norm_text(mb_title) or norm_text(mb_title) in norm_text(title))
        artist_match = (norm_text(artist) in norm_text(mb_artist) or norm_text(mb_artist) in norm_text(artist))
        score_ok = mb_score >= 70 and (title_match or artist_match)

        ok = score_ok and not non_chinese_flag
        status = "verified" if ok else ("non_chinese_language" if non_chinese_flag else "weak_match")

        return {
            "musicbrainz_ok": ok,
            "musicbrainz_score": mb_score,
            "musicbrainz_title": mb_title,
            "musicbrainz_artist": mb_artist,
            "musicbrainz_status": status,
            "musicbrainz_language": mb_language,
            "musicbrainz_area": mb_area,
            "musicbrainz_non_chinese_flag": non_chinese_flag,
            "musicbrainz_chinese_confirmed": chinese_confirmed,
        }

    except Exception as e:
        return {
            "musicbrainz_ok": False, "musicbrainz_score": 0,
            "musicbrainz_title": "", "musicbrainz_artist": "",
            "musicbrainz_status": f"error_{type(e).__name__}",
            "musicbrainz_language": "", "musicbrainz_area": "",
            "musicbrainz_non_chinese_flag": False,
            "musicbrainz_chinese_confirmed": False,
        }


# ============================================================
# REJECT REASONS — checked in priority order (FIXED)
# ============================================================

def get_reject_reason(title, artist, raw_genre, search_query, mb, title_artist_script):
    # 1. Kana or Hangul in title/artist → definite non-Chinese
    if title_artist_script in ("japanese_or_korean", "mixed_cjk"):
        return "reject_japanese_or_korean_script_in_title_or_artist"

    # 2. MusicBrainz explicitly confirms a non-Chinese LANGUAGE
    #    (not area — area is where a release was registered, not
    #    what language the song is in)
    if mb.get("musicbrainz_non_chinese_flag"):
        return f"reject_mb_non_chinese_language_{mb.get('musicbrainz_language','')}"

    # 3. Title/artist has NO Chinese script at all — this is the
    #    strongest per-song signal. Only survives if MusicBrainz
    #    positively confirms Chinese language; a Han-character
    #    raw_genre/search_query field is NOT enough on its own.
    if title_artist_script == "latin_only":
        if mb.get("musicbrainz_chinese_confirmed"):
            return ""  # genuinely confirmed despite Latin title
        return "reject_no_chinese_evidence_in_title_or_artist"

    # 4. No Chinese evidence anywhere in any metadata field
    all_meta = f"{title} {artist} {raw_genre} {search_query}"
    if classify_script(all_meta) == "latin_only":
        return "reject_no_chinese_script_evidence"

    # 5. Kana/hangul contamination in genre/search fields only
    if is_japanese_or_korean(f"{raw_genre} {search_query}"):
        return "reject_japanese_or_korean_script_in_metadata"

    return ""


# ============================================================
# RUN LIVE
# ============================================================

chinese_csv = MANIFEST_DIR / "chinese_human_metadata.csv"
df = pd.read_csv(chinese_csv)

end_at = len(df) if MAX_ROWS is None else min(len(df), START_AT + MAX_ROWS)
rows = []
out_csv = MANIFEST_DIR / "chinese_live_platform_verification_report.csv"

for i in range(START_AT, end_at):
    row = df.iloc[i]

    song_id      = safe_text(row.get("song_id", ""))
    title        = safe_text(row.get("title", ""))
    artist       = safe_text(row.get("artist", ""))
    genre        = safe_text(row.get("final_genre", ""))
    raw_genre    = safe_text(row.get("raw_genre", ""))
    search_query = safe_text(row.get("search_query", ""))

    clear_output(wait=True)
    print("=" * 90, flush=True)
    print(f"Checking row {i+1}/{len(df)}", flush=True)
    print("=" * 90, flush=True)
    print("Song ID:", song_id, "| Title:", title, "| Artist:", artist)

    mb    = search_musicbrainz_live(title, artist)
    links = make_links(title, artist)

    title_artist_script = classify_script(f"{title} {artist}")
    chinese_evidence = has_chinese_chars(f"{title} {artist} {raw_genre} {search_query}")

    reject_reason = get_reject_reason(title, artist, raw_genre, search_query, mb, title_artist_script)

    if reject_reason:
        final_status = "rejected_non_chinese"
    elif mb["musicbrainz_ok"]:
        final_status = "auto_verified_musicbrainz"
    elif chinese_evidence:
        final_status = "manual_check_chinese_platforms"
    else:
        final_status = "high_risk_no_chinese_evidence"

    result = {
        "row_number": i + 1, "song_id": song_id, "final_genre": genre,
        "title": title, "artist": artist, "raw_genre": raw_genre,
        "search_query": search_query, "country": row.get("country", ""),
        "title_artist_script": title_artist_script,
        "chinese_evidence": chinese_evidence,
        "reject_reason": reject_reason,
        "platform_verification_status": final_status,
        "musicbrainz_ok": mb["musicbrainz_ok"],
        "musicbrainz_score": mb["musicbrainz_score"],
        "musicbrainz_title": mb["musicbrainz_title"],
        "musicbrainz_artist": mb["musicbrainz_artist"],
        "musicbrainz_status": mb["musicbrainz_status"],
        "musicbrainz_language": mb["musicbrainz_language"],
        "musicbrainz_area": mb["musicbrainz_area"],
        "musicbrainz_non_chinese_flag": mb["musicbrainz_non_chinese_flag"],
        "musicbrainz_chinese_confirmed": mb.get("musicbrainz_chinese_confirmed", False),
        **links,
    }
    rows.append(result)

    print("Script:", title_artist_script, "| Result:", final_status,
          "| Reject reason:", reject_reason or "—")
    print("MB:", mb["musicbrainz_status"], "| Lang:", mb["musicbrainz_language"] or "—",
          "| Area:", mb["musicbrainz_area"] or "—")
    display(pd.DataFrame([result]))

    if len(rows) % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8-sig")
        print("\nProgress saved:", out_csv, flush=True)

verify_df = pd.DataFrame(rows)
verify_df.to_csv(out_csv, index=False, encoding="utf-8-sig")

clear_output(wait=True)
print("LIVE VERIFICATION FINISHED (fixed version)")
print("Saved:", out_csv)
print("\nStatus counts:")
print(verify_df["platform_verification_status"].value_counts(dropna=False))
print("\nMusicBrainz language counts:")
print(verify_df["musicbrainz_language"].value_counts(dropna=False).head(15))

if "rejected_non_chinese" in verify_df["platform_verification_status"].values:
    print("\n--- REJECTED NON-CHINESE ROWS ---")
    display(verify_df[verify_df["platform_verification_status"] == "rejected_non_chinese"]
            [["row_number", "title", "artist", "raw_genre", "reject_reason", "musicbrainz_language"]])

In [ ]:
# AUTO-DECIDE CHINESE VERIFICATION
# Converts manual_check_chinese_platforms into automatic decisions
# Metadata only, no download

import pandas as pd
import re

report_csv = MANIFEST_DIR / "chinese_live_platform_verification_report.csv"
metadata_csv = MANIFEST_DIR / "chinese_human_metadata.csv"

verify_df = pd.read_csv(report_csv)
metadata_df = pd.read_csv(metadata_csv)


def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def classify_script(text):
    """
    Returns 'chinese', 'japanese_or_korean', 'latin_only', or 'mixed_cjk'.
    Han ideographs are shared between Chinese and Japanese, so kana/hangul
    presence is checked first to avoid misclassifying Japanese as Chinese.
    """
    import re as _re
    text = safe_text(text)
    has_han    = bool(_re.search(r"[\u3400-\u4DBF\u4E00-\u9FFF\uF900-\uFAFF]", text))
    has_kana   = bool(_re.search(r"[\u3040-\u30FF]", text))
    has_hangul = bool(_re.search(r"[\uAC00-\uD7AF]", text))
    if has_kana or has_hangul:
        return "mixed_cjk" if has_han else "japanese_or_korean"
    if has_han:
        return "chinese"
    return "latin_only"


def has_chinese_chars(text):
    """True only when Han chars are present AND no kana/hangul contamination."""
    return classify_script(text) == "chinese"


def is_japanese_or_korean(text):
    return classify_script(text) in {"japanese_or_korean", "mixed_cjk"}



def extract_duration_sec_from_notes(notes):
    notes = safe_text(notes)
    m = re.search(r"duration_sec=([0-9.]+)", notes)
    if not m:
        return None
    try:
        return float(m.group(1))
    except Exception:
        return None


def auto_decide_chinese_row(row):
    song_id      = safe_text(row.get("song_id", ""))
    title        = safe_text(row.get("title", ""))
    artist       = safe_text(row.get("artist", ""))
    raw_genre    = safe_text(row.get("raw_genre", ""))
    search_query = safe_text(row.get("search_query", ""))
    country      = safe_text(row.get("country", ""))
    mb_ok        = bool(row.get("musicbrainz_ok", False))

    
    # platform_verification_status == "rejected_non_chinese" means Cell 37
    # already confirmed Japanese / Korean / English via script or MB language tag.
    cell37_status  = safe_text(row.get("platform_verification_status", ""))
    cell37_reason  = safe_text(row.get("reject_reason", ""))
    mb_non_chinese = bool(row.get("musicbrainz_non_chinese_flag", False))
    mb_language    = safe_text(row.get("musicbrainz_language", ""))

    # Fast-reject: Cell 37 already classified this as non-Chinese.
    if cell37_status == "rejected_non_chinese" and cell37_reason:
        return "REJECT_REPLACE", cell37_reason

    combined = f"{title} {artist} {raw_genre} {search_query}".lower()

    chinese_markers = [
        "mandopop", "cantopop", "c-pop",
        "chinese pop", "chinese rock", "chinese rap",
        "chinese hip hop", "chinese r&b", "chinese folk",
        "mandarin", "\u534e\u8bed", "\u570b\u8a9e", "\u4e2d\u6587",
        "\u4e2d\u56fd", "\u4e2d\u570b",
        "\u6d41\u884c", "\u6447\u6eda", "\u6416\u6ebb",
        "\u8bf4\u5531", "\u563b\u54c8", "\u6c11\u8c23", "\u6c11\u8b20",
    ]

    bad_words = [
        "moana", "frozen", "peppa", "disney",
        "children", "lullaby", "christmas",
        "whitney houston", "martin garrix", "dua lipa",
        "blake shelton", "daft punk",
        "classical", "jazz", "gospel", "country",
        "instrumental", "karaoke", "cover", "remix",
        "live", "concert",
        "\u6f14\u5531\u4f1a", "\u73fe\u5834", "\u73b0\u573a",
        "compilation", "full album", "\u5408\u96c6",
    ]

    if any(b in combined for b in bad_words):
        return "REJECT_REPLACE", "bad_keyword_or_wrong_source"

    # Script check on title + artist (kana/hangul = Japanese or Korean)
    title_artist_script = classify_script(f"{title} {artist}")
    if title_artist_script in {"japanese_or_korean", "mixed_cjk"}:
        return "REJECT_REPLACE", "reject_japanese_or_korean_script_in_title_or_artist"

    # MusicBrainz non-Chinese language flag
    if mb_non_chinese:
        return "REJECT_REPLACE", f"reject_mb_non_chinese_language_{mb_language or 'unknown'}"

    # Duration check from metadata notes
    meta_match = metadata_df[metadata_df["song_id"].astype(str) == song_id]
    duration_sec = None
    if len(meta_match) > 0:
        duration_sec = extract_duration_sec_from_notes(
            meta_match.iloc[0].get("notes", "")
        )
    if duration_sec is not None and not (90 <= duration_sec <= 480):
        return "REJECT_REPLACE", "duration_not_normal_song"

    has_cn     = has_chinese_chars(combined)
    has_marker = any(m in combined for m in chinese_markers)
    good_country = country in ["CN", "HK", "TW", "SG", "MO"]

    if mb_ok:
        return "AUTO_VERIFIED", "musicbrainz_verified"

    if has_cn and good_country:
        return "AUTO_VERIFIED", "chinese_script_plus_chinese_store"

    if has_cn and has_marker:
        return "AUTO_VERIFIED", "chinese_script_plus_chinese_marker"

    if has_cn:
        return "AUTO_VERIFIED_WEAK", "chinese_script_only"

    if has_marker and good_country:
        return "AUTO_VERIFIED_WEAK", "chinese_marker_plus_chinese_store"

    return "REJECT_REPLACE", "no_strong_chinese_evidence"


# APPLY AUTO DECISION

decisions = verify_df.apply(auto_decide_chinese_row, axis=1)

verify_df["auto_verification_decision"] = [d[0] for d in decisions]
verify_df["auto_verification_reason"] = [d[1] for d in decisions]

print("Auto decisions:")
print(verify_df["auto_verification_decision"].value_counts(dropna=False))

print("\nReasons:")
print(verify_df["auto_verification_reason"].value_counts(dropna=False))

display(
    verify_df[
        verify_df["auto_verification_decision"].str.contains("REJECT", case=False, na=False)
    ][[
        "song_id",
        "final_genre",
        "title",
        "artist",
        "raw_genre",
        "search_query",
        "country",
        "platform_verification_status",
        "auto_verification_decision",
        "auto_verification_reason"
    ]]
)

# Save auto decision report
out_csv = MANIFEST_DIR / "chinese_auto_verification_decision_report.csv"
verify_df.to_csv(out_csv, index=False, encoding="utf-8-sig")

print("\nSaved:", out_csv)

In [ ]:
# MARK AUTO-REJECTED CHINESE ROWS FOR REPLACEMENT

import pandas as pd

decision_csv = MANIFEST_DIR / "chinese_auto_verification_decision_report.csv"
metadata_csv = MANIFEST_DIR / "chinese_human_metadata.csv"

decision_df = pd.read_csv(decision_csv)
metadata_df = pd.read_csv(metadata_csv)

# Normalise song_id to stripped strings on both sides so float IDs
# (e.g. 1.0 stored as float by pandas) still match.
decision_df["_sid"] = decision_df["song_id"].astype(str).str.strip()
metadata_df["_sid"] = metadata_df["song_id"].astype(str).str.strip()

# Build a dict: song_id -> reject reason from Cell 6E
reject_map = dict(
    zip(
        decision_df.loc[
            decision_df["auto_verification_decision"]
            .astype(str).str.contains("REJECT", case=False, na=False),
            "_sid",
        ],
        decision_df.loc[
            decision_df["auto_verification_decision"]
            .astype(str).str.contains("REJECT", case=False, na=False),
            "auto_verification_reason",
        ].astype(str),
    )
)

reject_ids = list(reject_map.keys())

print("Rows to replace:", len(reject_ids))
print(reject_ids)

# Mark each rejected row individually so notes get the specific reason.
for sid, reason in reject_map.items():
    mask = metadata_df["_sid"] == sid
    metadata_df.loc[mask, "verification_status"] = "auto_rejected_replace_needed"
    # Append reason to existing notes only for the matched rows.
    metadata_df.loc[mask, "notes"] = (
        metadata_df.loc[mask, "notes"].astype(str)
        + f"; auto_rejected_replace_needed; reject_reason={reason}"
    )

metadata_df = metadata_df.drop(columns=["_sid"], errors="ignore")
decision_df = decision_df.drop(columns=["_sid"], errors="ignore")

metadata_df.to_csv(metadata_csv, index=False, encoding="utf-8-sig")

print("Updated:", metadata_csv)

display(
    metadata_df[
        metadata_df["song_id"].astype(str).str.strip().isin(reject_ids)
    ][[
        "song_id", "final_genre", "title", "artist",
        "raw_genre", "search_query", "verification_status", "notes",
    ]]
)

In [ ]:
# REPLACE AUTO-REJECTED CHINESE METADATA ROWS
# Fixes pandas LossySetitemError / dtype issue
# Same genre replacement, keeps same song_id
# Metadata only, no download

import pandas as pd
import re
import shutil
from pathlib import Path

metadata_csv = MANIFEST_DIR / "chinese_human_metadata.csv"
backup_csv = MANIFEST_DIR / "chinese_human_metadata_backup_before_auto_reject_fix.csv"


# 0. RESTORE BACKUP IF PREVIOUS RUN PARTLY FAILED

if backup_csv.exists():
    shutil.copy(str(backup_csv), str(metadata_csv))
    print("Restored safe backup before fixing:", backup_csv)
else:
    print("No previous backup found. Using current file:", metadata_csv)


# HELPERS

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def classify_script(text):
    """
    Returns 'chinese', 'japanese_or_korean', 'latin_only', or 'mixed_cjk'.
    Han ideographs are shared between Chinese and Japanese, so kana/hangul
    presence is checked first to avoid misclassifying Japanese as Chinese.
    """
    import re as _re
    text = safe_text(text)
    has_han    = bool(_re.search(r"[\u3400-\u4DBF\u4E00-\u9FFF\uF900-\uFAFF]", text))
    has_kana   = bool(_re.search(r"[\u3040-\u30FF]", text))
    has_hangul = bool(_re.search(r"[\uAC00-\uD7AF]", text))
    if has_kana or has_hangul:
        return "mixed_cjk" if has_han else "japanese_or_korean"
    if has_han:
        return "chinese"
    return "latin_only"


def has_chinese_chars(text):
    """True only when Han chars are present AND no kana/hangul contamination."""
    return classify_script(text) == "chinese"


def is_japanese_or_korean(text):
    return classify_script(text) in {"japanese_or_korean", "mixed_cjk"}


def norm_key(x):
    x = safe_text(x).lower()
    x = re.sub(r"\(.*?\)|\[.*?\]", " ", x)
    x = re.sub(r"[^a-z0-9\u4e00-\u9fff]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def make_key(title, artist):
    return norm_key(title) + " | " + norm_key(artist)


def extract_duration_sec(notes):
    notes = safe_text(notes)
    m = re.search(r"duration_sec=([0-9.]+)", notes)

    if not m:
        return None

    try:
        return float(m.group(1))
    except Exception:
        return None


def force_schema_object_dtype(df):
    """
    Prevent pandas LossySetitemError by making all schema columns object dtype.
    """
    df = df.copy()

    for col in FINAL_SCHEMA:
        if col not in df.columns:
            df[col] = ""

        df[col] = df[col].astype("object")

    return df


def is_clean_chinese_candidate(row):
    title = safe_text(row.get("title", ""))
    artist = safe_text(row.get("artist", ""))
    raw_genre = safe_text(row.get("raw_genre", ""))
    search_query = safe_text(row.get("search_query", ""))
    country = safe_text(row.get("country", ""))
    notes = safe_text(row.get("notes", ""))
    verification_status = safe_text(row.get("verification_status", ""))

    text = f"{title} {artist} {raw_genre} {search_query} {country} {notes} {verification_status}".lower()

    bad_words = [
        "auto_rejected_replace_needed",
        "moana", "frozen", "peppa", "disney",
        "children", "lullaby", "christmas",
        "whitney houston", "martin garrix", "dua lipa",
        "blake shelton", "daft punk",
        "classical", "jazz", "gospel", "country",
        "instrumental", "karaoke", "cover", "remix",
        "live", "concert", "演唱会", "演唱會", "現場", "现场",
        "compilation", "full album", "合集",
        "head shoulders", "knees and toes", "itsy bitsy",
        "twinkle", "little star", "under the sea", "part of your world",
    ]

    if any(bad in text for bad in bad_words):
        return False

    # Must have Chinese evidence Han chars without kana/hangul contamination
    combined_meta = f"{title} {artist} {raw_genre} {search_query}"
    if is_japanese_or_korean(combined_meta):
        return False  # Japanese or Korean song, not Chinese
    if not has_chinese_chars(combined_meta):
        return False

    # Prefer Chinese-region iTunes stores
    if country not in ["CN", "HK", "TW", "SG", "MO", "US"]:
        return False

    # Normal full-song duration if available
    dur = extract_duration_sec(notes)
    if dur is not None:
        if dur < 90 or dur > 480:
            return False

    return True


def load_clean_chinese_pool():
    pools = []

    # Memory variables
    if "chinese_itunes_candidates" in globals():
        try:
            if len(chinese_itunes_candidates) > 0:
                print("Using chinese_itunes_candidates from memory")
                pools.append(chinese_itunes_candidates.copy())
        except Exception:
            pass

    if "chinese_verified" in globals():
        try:
            if len(chinese_verified) > 0:
                print("Using chinese_verified from memory")
                pools.append(chinese_verified.copy())
        except Exception:
            pass

    # Saved candidate files
    possible_files = [
        MANIFEST_DIR / "chinese_itunes_candidates_updated.csv",
        MANIFEST_DIR / "chinese_itunes_commercial_candidates.csv",
        MANIFEST_DIR / "chinese_strict_verified_live.csv",
    ]

    for f in possible_files:
        if f.exists():
            print("Using pool file:", f)
            pools.append(pd.read_csv(f))

    if len(pools) == 0:
        raise FileNotFoundError(
            "No Chinese candidate pool found. Run Cell 4 + Cell 5B first."
        )

    pool = pd.concat(pools, ignore_index=True)

    pool = force_schema_object_dtype(pool)
    pool = pool[FINAL_SCHEMA].copy()

    pool["match_key"] = pool.apply(
        lambda r: make_key(r["title"], r["artist"]),
        axis=1
    )

    pool = pool.drop_duplicates(subset=["match_key"], keep="first").reset_index(drop=True)

    clean_pool = pool[pool.apply(is_clean_chinese_candidate, axis=1)].copy()
    clean_pool = clean_pool[clean_pool["final_genre"].isin(GENRES)].copy()

    clean_pool["_country_score"] = clean_pool["country"].map({
        "CN": 5,
        "HK": 4,
        "TW": 4,
        "SG": 3,
        "MO": 3,
        "US": 1,
    }).fillna(0)

    clean_pool["_has_chinese_title"] = clean_pool["title"].apply(
        lambda x: 1 if has_chinese_chars(x) else 0
    )

    clean_pool = clean_pool.sort_values(
        ["final_genre", "_has_chinese_title", "_country_score"],
        ascending=[True, False, False]
    ).reset_index(drop=True)

    return clean_pool


# 1. LOAD CURRENT METADATA SAFELY

df = pd.read_csv(metadata_csv)
df = force_schema_object_dtype(df)

# Save fresh backup before this fixed run
fixed_run_backup = MANIFEST_DIR / "chinese_human_metadata_backup_before_fixed_6F_run.csv"
df.to_csv(fixed_run_backup, index=False, encoding="utf-8-sig")
print("Fresh backup saved:", fixed_run_backup)

df["match_key"] = df.apply(lambda r: make_key(r["title"], r["artist"]), axis=1)

reject_mask = df["verification_status"].astype(str).str.contains(
    "auto_rejected_replace_needed",
    case=False,
    na=False
)

reject_idx = df[reject_mask].index.tolist()

print("\nRows marked for replacement:", len(reject_idx))

display(
    df.loc[
        reject_idx,
        ["song_id", "final_genre", "title", "artist", "raw_genre", "search_query", "country", "verification_status"]
    ]
)


# 2. LOAD CLEAN POOL

pool = load_clean_chinese_pool()
pool = force_schema_object_dtype(pool)

# Exclude existing good rows from replacement pool
existing_good_keys = set(df.drop(index=reject_idx)["match_key"].astype(str))

pool = pool[~pool["match_key"].isin(existing_good_keys)].copy()

print("\nClean pool counts:")
print(pool["final_genre"].value_counts(dropna=False))


# 3. REPLACE BAD ROWS SAME GENRE

replace_log = []
used_keys = set(existing_good_keys)

for idx in reject_idx:
    old = df.loc[idx].copy()
    old_song_id = safe_text(old["song_id"])
    genre = safe_text(old["final_genre"])

    candidates = pool[
        (pool["final_genre"] == genre)
        & (~pool["match_key"].isin(used_keys))
    ].copy()

    if len(candidates) == 0:
        print(f"NO REPLACEMENT FOUND: {old_song_id} | {genre} | {old['title']} - {old['artist']}")
        continue

    cand = candidates.iloc[0].copy()

    # Capture the old rejection notes BEFORE overwriting with candidate metadata.
    old_notes = safe_text(old.get("notes", ""))

    # Replace metadata safely but keep same song_id + genre
    for col in FINAL_SCHEMA:
        if col in cand.index and col != "song_id":
            value = cand[col]

            if pd.isna(value):
                value = ""

            df.at[idx, col] = value

    df.at[idx, "song_id"] = old_song_id
    df.at[idx, "language"] = "Chinese"
    df.at[idx, "language_code"] = LANGUAGE_CODES["Chinese"]
    df.at[idx, "label"] = "Human"
    df.at[idx, "model"] = "Human"
    df.at[idx, "final_genre"] = genre
    df.at[idx, "verification_status"] = "auto_replaced_clean_chinese_metadata"
    # Append to the OLD rejection notes, not the newly written candidate notes.
    df.at[idx, "notes"] = old_notes + "; replaced_auto_rejected_row"

    new_key = make_key(df.at[idx, "title"], df.at[idx, "artist"])
    used_keys.add(new_key)

    replace_log.append({
        "song_id": old_song_id,
        "genre": genre,
        "old_title": old["title"],
        "old_artist": old["artist"],
        "old_raw_genre": old["raw_genre"],
        "new_title": df.at[idx, "title"],
        "new_artist": df.at[idx, "artist"],
        "new_raw_genre": df.at[idx, "raw_genre"],
        "new_country": df.at[idx, "country"],
    })

    print(f"REPLACED {old_song_id}: {old['title']} -> {df.at[idx, 'title']}")


# 4. SAVE CLEANED FILE

df = df.drop(columns=["match_key"], errors="ignore")
df = standardise_schema(df)

df.to_csv(metadata_csv, index=False, encoding="utf-8-sig")

log_csv = MANIFEST_DIR / "chinese_auto_rejected_replacement_log.csv"
pd.DataFrame(replace_log).to_csv(log_csv, index=False, encoding="utf-8-sig")

print("\nSaved fixed Chinese metadata:", metadata_csv)
print("Saved replacement log:", log_csv)

print("\nGenre counts:")
print(df["final_genre"].value_counts(dropna=False))

print("\nDuplicate title+artist:")
print(df.duplicated(subset=["title", "artist"]).sum())

print("\nStill marked auto_rejected_replace_needed:")
print(
    df["verification_status"].astype(str).str.contains(
        "auto_rejected_replace_needed",
        case=False,
        na=False
    ).sum()
)

display(df.tail(30))

## YouTube URL Matching
Uses the YouTube Data API to find the best official audio match for Hindi and Chinese tracks. (English tracks already retain their SONICS YouTube IDs).

In [ ]:
# YOUTUBE URL MATCHING — SAFE RESUME WITH LATEST METADATA
# Resume + live save | Metadata URL only | No audio download
#
# REQUIREMENT:
# Run the updated Cell 1 first so youtube_api_get() and the
# five YouTube API keys are available.

import re
import time
from pathlib import Path
from difflib import SequenceMatcher

import pandas as pd
from tqdm.auto import tqdm


# CONFIG

YOUTUBE_SEARCH_RESULTS = 5
SAVE_AFTER_EVERY_ROW = True
REQUEST_DELAY_SECONDS = 1.25

# Retry rows with these previous statuses when the cell is rerun.
RETRY_NO_ACCEPTABLE_MATCH_ON_RERUN = False

RETRYABLE_STATUSES = {
    "",
    "nan",
    "none",
    "null",
    "no_result",
    "no_video_id",
    "request_failed",
    "all_youtube_keys_unavailable",
    "all_youtube_keys_temporarily_unavailable",
}

if RETRY_NO_ACCEPTABLE_MATCH_ON_RERUN:
    RETRYABLE_STATUSES.add(
        "no_acceptable_match"
    )

# Reject likely alternate/non-original versions unless the source
# song title itself contains the same marker.
BAD_VERSION_MARKERS = {
    "live",
    "remix",
    "karaoke",
    "cover",
    "instrumental",
    "nightcore",
    "slowed",
    "reverb",
    "sped up",
    "8d audio",
    "reaction",
    "tutorial",
    "dance practice",
}


# REQUIRE UPDATED CELL 1

if "youtube_api_get" not in globals():
    raise RuntimeError(
        "Run the updated Cell 1 first. "
        "youtube_api_get() is not available."
    )

if not globals().get("YOUTUBE_API_KEYS"):
    raise RuntimeError(
        "No YouTube API keys are configured. "
        "Run Cell 1 and enter your keys."
    )


# TEXT HELPERS

def is_missing_url(value):
    text = str(value).strip()

    return (
        text == ""
        or text.casefold()
        in {"nan", "none", "null"}
    )


def normalise_youtube_text(value):
    text = str(value or "").casefold()

    text = re.sub(
        r"\([^)]*\)|\[[^\]]*\]",
        " ",
        text,
    )

    text = re.sub(
        r"[^a-z0-9\u0900-\u097f"
        r"\u3400-\u4dbf\u4e00-\u9fff"
        r"\u3040-\u30ff\uac00-\ud7af]+",
        " ",
        text,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def text_similarity(left, right):
    left_normalised = normalise_youtube_text(left)
    right_normalised = normalise_youtube_text(right)

    if not left_normalised or not right_normalised:
        return 0.0

    sequence_score = SequenceMatcher(
        None,
        left_normalised,
        right_normalised,
    ).ratio()

    left_tokens = set(
        left_normalised.split()
    )

    right_tokens = set(
        right_normalised.split()
    )

    if not left_tokens or not right_tokens:
        token_score = 0.0
    else:
        token_score = (
            len(left_tokens & right_tokens)
            / len(left_tokens | right_tokens)
        )

    return (
        sequence_score * 0.65
        + token_score * 0.35
    )


def contains_bad_version_marker(
    candidate_title,
    source_title,
):
    candidate_normalised = (
        normalise_youtube_text(
            candidate_title
        )
    )

    source_normalised = (
        normalise_youtube_text(
            source_title
        )
    )

    for marker in BAD_VERSION_MARKERS:
        if (
            marker in candidate_normalised
            and marker not in source_normalised
        ):
            return marker

    return ""


# RESULT SCORING

def score_youtube_candidate(
    source_title,
    source_artist,
    item,
):
    snippet = item.get(
        "snippet",
        {},
    )

    candidate_title = snippet.get(
        "title",
        "",
    )

    channel_title = snippet.get(
        "channelTitle",
        "",
    )

    title_score = text_similarity(
        source_title,
        candidate_title,
    )

    artist_score = max(
        text_similarity(
            source_artist,
            candidate_title,
        ),
        text_similarity(
            source_artist,
            channel_title,
        ),
    )

    bad_marker = contains_bad_version_marker(
        candidate_title,
        source_title,
    )

    score = (
        title_score * 0.72
        + artist_score * 0.28
    )

    combined_text = normalise_youtube_text(
        candidate_title
        + " "
        + channel_title
    )

    if "official audio" in combined_text:
        score += 0.10

    if "official video" in combined_text:
        score += 0.07

    if "topic" in normalise_youtube_text(
        channel_title
    ):
        score += 0.06

    if bad_marker:
        score -= 0.35

    return {
        "score": score,
        "title_score": title_score,
        "artist_score": artist_score,
        "bad_marker": bad_marker,
        "candidate_title": candidate_title,
        "channel_title": channel_title,
    }


# YOUTUBE SEARCH

def youtube_search_params(
    title,
    artist,
    language,
):
    query = (
        f"{title} {artist} official audio"
    )

    params = {
        "part": "snippet",
        "q": query,
        "type": "video",
        "maxResults": (
            YOUTUBE_SEARCH_RESULTS
        ),
        "safeSearch": "none",
    }

    if language == "Hindi":
        params["regionCode"] = "IN"
        params["relevanceLanguage"] = "hi"

    elif language == "Chinese":
        params["regionCode"] = "TW"
        params["relevanceLanguage"] = "zh"

    return params


def find_best_youtube_match(
    title,
    artist,
    language="",
):
    title = str(title or "").strip()
    artist = str(artist or "").strip()

    if not title or not artist:
        return {
            "url": "",
            "video_id": "",
            "status": (
                "missing_title_or_artist"
            ),
            "match_score": 0.0,
            "matched_title": "",
            "matched_channel": "",
        }

    try:
        data = youtube_api_get(
            "search",
            youtube_search_params(
                title,
                artist,
                language,
            ),
            timeout=20,
        )

    except RuntimeError as error:
        message = str(error).casefold()

        if (
            "every configured youtube api key"
            in message
            or "all configured youtube api keys"
            in message
            or "temporarily rate-limited"
            in message
        ):
            return {
                "url": "",
                "video_id": "",
                "status": (
                    "all_youtube_keys_temporarily_unavailable"
                ),
                "match_score": 0.0,
                "matched_title": "",
                "matched_channel": "",
            }

        return {
            "url": "",
            "video_id": "",
            "status": (
                "youtube_api_runtime_error"
            ),
            "match_score": 0.0,
            "matched_title": "",
            "matched_channel": "",
        }

    except Exception as error:
        return {
            "url": "",
            "video_id": "",
            "status": (
                "request_failed_"
                + type(error).__name__
            ),
            "match_score": 0.0,
            "matched_title": "",
            "matched_channel": "",
        }

    items = data.get(
        "items",
        [],
    )

    if not items:
        return {
            "url": "",
            "video_id": "",
            "status": "no_result",
            "match_score": 0.0,
            "matched_title": "",
            "matched_channel": "",
        }

    scored_items = []

    for item in items:
        video_id = (
            item.get("id", {})
            .get("videoId", "")
        )

        if not video_id:
            continue

        score_data = (
            score_youtube_candidate(
                title,
                artist,
                item,
            )
        )

        scored_items.append(
            {
                "video_id": video_id,
                **score_data,
            }
        )

    if not scored_items:
        return {
            "url": "",
            "video_id": "",
            "status": "no_video_id",
            "match_score": 0.0,
            "matched_title": "",
            "matched_channel": "",
        }

    best = max(
        scored_items,
        key=lambda row: row["score"],
    )
    
    
    # Require a reasonable title match and at least some artist evidence.
    acceptable = (
        best["title_score"] >= 0.48
        and (
            best["artist_score"] >= 0.22
            or best["score"] >= 0.70
        )
        and not best["bad_marker"]
    )

    if not acceptable:
        return {
            "url": "",
            "video_id": "",
            "status": (
                "no_acceptable_match"
            ),
            "match_score": round(
                best["score"],
                4,
            ),
            "matched_title": (
                best["candidate_title"]
            ),
            "matched_channel": (
                best["channel_title"]
            ),
        }

    video_id = best[
        "video_id"
    ]

    return {
        "url": (
            "https://www.youtube.com/"
            f"watch?v={video_id}"
        ),
        "video_id": video_id,
        "status": (
            "youtube_match_found"
        ),
        "match_score": round(
            best["score"],
            4,
        ),
        "matched_title": (
            best["candidate_title"]
        ),
        "matched_channel": (
            best["channel_title"]
        ),
    }


# RESUME + LIVE SAVE

def should_retry_youtube_row(row):
    current_url = row.get(
        "youtube_url",
        "",
    )

    if not is_missing_url(
        current_url
    ):
        return False

    status = str(
        row.get(
            "youtube_match_status",
            "",
        )
    ).strip().casefold()

    retry_prefixes = (
        "request_failed_",
        "stop_youtube_api_error_",
        "youtube_api_error_",
        "youtube_api_runtime_error",
        "all_youtube_keys_unavailable",
    )

    if status.startswith(
        retry_prefixes
    ):
        return True

    return status in RETRYABLE_STATUSES


YOUTUBE_RESULT_COLUMNS = [
    "youtube_url",
    "youtube_video_id",
    "youtube_match_status",
    "youtube_match_score",
    "youtube_matched_title",
    "youtube_matched_channel",
]


def ensure_youtube_columns(frame):
    frame = frame.copy()

    missing_columns = [
        column
        for column in YOUTUBE_RESULT_COLUMNS
        if column not in frame.columns
    ]

    if missing_columns:
        additions = pd.DataFrame(
            {
                column: ""
                for column in missing_columns
            },
            index=frame.index,
        )

        frame = pd.concat(
            [frame, additions],
            axis=1,
        ).copy()

    for column in YOUTUBE_RESULT_COLUMNS:
        frame[column] = (
            frame[column]
            .astype("object")
        )

    return frame


def youtube_resume_key(
    title,
    artist,
):
    return (
        normalise_youtube_text(title)
        + "||"
        + normalise_youtube_text(artist)
    )


def sync_resume_with_current_metadata(
    current_df,
    saved_df,
    language,
):
    """
    Current metadata is always the source of truth.

    Saved YouTube fields are reused only when title+artist still match.
    Rows that exist only in the old resume CSV are discarded.
    """
    current = ensure_youtube_columns(
        current_df.copy()
    )

    saved = ensure_youtube_columns(
        saved_df.copy()
    )

    current["_youtube_resume_key"] = [
        youtube_resume_key(
            title,
            artist,
        )
        for title, artist in zip(
            current["title"],
            current["artist"],
        )
    ]

    saved["_youtube_resume_key"] = [
        youtube_resume_key(
            title,
            artist,
        )
        for title, artist in zip(
            saved["title"],
            saved["artist"],
        )
    ]

    current = (
        current.drop_duplicates(
            subset=["_youtube_resume_key"],
            keep="first",
        )
        .reset_index(drop=True)
    )

    saved = (
        saved.drop_duplicates(
            subset=["_youtube_resume_key"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    saved_keys = set(
        saved["_youtube_resume_key"]
    )

    current_keys = set(
        current["_youtube_resume_key"]
    )

    stale_saved_rows = len(
        saved_keys - current_keys
    )

    reusable_rows = len(
        saved_keys & current_keys
    )

    saved_lookup = (
        saved.set_index(
            "_youtube_resume_key"
        )[YOUTUBE_RESULT_COLUMNS]
        .to_dict("index")
    )

    reused_urls = 0

    for index, row in current.iterrows():
        key = row[
            "_youtube_resume_key"
        ]

        previous = saved_lookup.get(
            key
        )

        if not previous:
            continue

        # Reuse saved result fields for the same title+artist only.
        for column in YOUTUBE_RESULT_COLUMNS:
            current.at[
                index,
                column,
            ] = previous.get(
                column,
                "",
            )

        if not is_missing_url(
            previous.get(
                "youtube_url",
                "",
            )
        ):
            reused_urls += 1

    current = current.drop(
        columns=["_youtube_resume_key"],
        errors="ignore",
    )

    print(
        f"{language}: matching resume rows reused = "
        f"{reusable_rows}"
    )

    print(
        f"{language}: saved URLs reused = "
        f"{reused_urls}"
    )

    print(
        f"{language}: stale resume rows dropped = "
        f"{stale_saved_rows}"
    )

    print(
        f"{language}: latest metadata rows = "
        f"{len(current)}"
    )

    return current


def add_youtube_urls_resume(
    df,
    output_csv,
    language,
):
    output_csv = Path(
        output_csv
    )

    output_csv.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    required_columns = {
        "title",
        "artist",
    }

    missing_columns = (
        required_columns
        - set(df.columns)
        if df is not None
        else required_columns
    )

    if missing_columns:
        raise ValueError(
            f"{language} metadata is missing: "
            + ", ".join(
                sorted(missing_columns)
            )
        )

    if df is None or len(df) == 0:
        raise ValueError(
            f"No current metadata supplied for {language}."
        )

    current_df = df.copy()

    if output_csv.exists():
        print(
            "Resume file found. Synchronising with latest metadata:",
            output_csv,
        )

        saved_df = pd.read_csv(
            output_csv,
            low_memory=False,
        )

        work_df = sync_resume_with_current_metadata(
            current_df,
            saved_df,
            language,
        )

        # Save immediately so stale rows disappear from the resume file.
        work_df.to_csv(
            output_csv,
            index=False,
            encoding="utf-8-sig",
        )

    else:
        print(
            "Starting fresh from latest metadata:",
            output_csv,
        )

        work_df = ensure_youtube_columns(
            current_df
        )

    # Interleave rows by artist so consecutive API requests come from
    # different artists. This prevents a single artist block from
    # exhausting one YouTube API key before it can recover.
    # The resume key is title+artist, not row position, so this is safe.
    work_df = (
        work_df
        .assign(_artist_rank=work_df.groupby("artist").cumcount())
        .sort_values(["_artist_rank", "artist"])
        .drop(columns=["_artist_rank"])
        .reset_index(drop=True)
    )

    total = len(work_df)

    completed_mask = ~work_df[
        "youtube_url"
    ].apply(is_missing_url)

    already_done = int(
        completed_mask.sum()
    )

    remaining = int(
        work_df.apply(
            should_retry_youtube_row,
            axis=1,
        ).sum()
    )

    print(
        f"{language}: total rows = {total}"
    )

    print(
        f"{language}: already completed = "
        f"{already_done}"
    )

    print(
        f"{language}: remaining/retryable = "
        f"{remaining}"
    )

    progress = tqdm(
        work_df.iterrows(),
        total=len(work_df),
        desc=(
            f"YouTube URLs {language}"
        ),
        unit="track",
        dynamic_ncols=True,
    )

    try:
        for index, row in progress:
            if not should_retry_youtube_row(
                row
            ):
                continue

            title = str(
                row.get("title", "")
            ).strip()

            artist = str(
                row.get("artist", "")
            ).strip()

            progress.set_postfix_str(
                (
                    f"{title[:28]} - "
                    f"{artist[:20]}"
                ),
                refresh=False,
            )

            result = find_best_youtube_match(
                title,
                artist,
                language=language,
            )

            work_df.at[
                index,
                "youtube_url",
            ] = result["url"]

            work_df.at[
                index,
                "youtube_video_id",
            ] = result["video_id"]

            work_df.at[
                index,
                "youtube_match_status",
            ] = result["status"]

            work_df.at[
                index,
                "youtube_match_score",
            ] = result[
                "match_score"
            ]

            work_df.at[
                index,
                "youtube_matched_title",
            ] = result[
                "matched_title"
            ]

            work_df.at[
                index,
                "youtube_matched_channel",
            ] = result[
                "matched_channel"
            ]

            tqdm.write(
                f"{index + 1}/{total} | "
                f"{result['status']} | "
                f"{title} - {artist}"
            )

            if SAVE_AFTER_EVERY_ROW:
                work_df.to_csv(
                    output_csv,
                    index=False,
                    encoding="utf-8-sig",
                )

            if (
                result["status"]
                in {
                    "all_youtube_keys_unavailable",
                    "all_youtube_keys_temporarily_unavailable",
                }
            ):
                print(
                    "\nStopped because all configured "
                    "YouTube API keys are temporarily unavailable "
                    "or quota-limited."
                )

                print(
                    "The CSV has been saved. "
                    "Reset/replenish the keys and rerun "
                    "this function to resume."
                )

                return work_df

            time.sleep(
                REQUEST_DELAY_SECONDS
            )

    except KeyboardInterrupt:
        work_df.to_csv(
            output_csv,
            index=False,
            encoding="utf-8-sig",
        )

        print(
            "\nInterrupted. Progress saved:"
        )

        print(
            output_csv
        )

        raise

    work_df.to_csv(
        output_csv,
        index=False,
        encoding="utf-8-sig",
    )

    completed = int(
        (
            ~work_df["youtube_url"]
            .apply(is_missing_url)
        ).sum()
    )

    print(
        "\nCompleted:",
        output_csv,
    )

    print(
        f"{language}: URLs found = "
        f"{completed}/{total}"
    )

    print(
        f"{language}: status summary:"
    )

    print(
        work_df[
            "youtube_match_status"
        ].value_counts(
            dropna=False
        )
    )

    return work_df

In [ ]:
# English already has YouTube IDs from SONICS, so just save
english_human_metadata.to_csv(
    MANIFEST_DIR / "english_human_with_youtube_urls.csv",
    index=False,
    encoding="utf-8-sig"
)

# Hindi with resume
hindi_human_yt = add_youtube_urls_resume(
    hindi_human_metadata,
    MANIFEST_DIR / "hindi_human_with_youtube_urls.csv",
    "Hindi"
)

# Chinese with resume
chinese_human_yt = add_youtube_urls_resume(
    chinese_human_metadata,
    MANIFEST_DIR / "chinese_human_with_youtube_urls.csv",
    "Chinese"
)

In [ ]:
# YT-DLP FALLBACK + FULL AUTO-REPLACEMENT WORKFLOW

import re
import time
import random
from pathlib import Path
from difflib import SequenceMatcher
from urllib.parse import quote_plus

import pandas as pd
from tqdm.auto import tqdm

try:
    import yt_dlp
except ImportError as error:
    raise ImportError(
        "yt-dlp is not installed. Run: pip install -U yt-dlp"
    ) from error


# CONFIGURATION

YTDLP_SEARCH_RESULTS = 5
MAX_CANDIDATES_PER_REPLACEMENT = 25  # How many alternative songs to try before giving up
GENRES = ["Pop", "Rock", "Hip-Hop/Rap", "R&B/Soul", "Folk/Acoustic"]

# Slow metadata-only requests to reduce temporary blocking.
REQUEST_DELAY_MIN_SECONDS = 2.5
REQUEST_DELAY_MAX_SECONDS = 4.0

# None means process every unresolved row.
MAX_ROWS_PER_RUN = None

# Minimum score required for automatic acceptance.
MIN_ACCEPT_SCORE = 0.66
MIN_TITLE_SCORE = 0.48
MIN_ARTIST_SCORE = 0.20

# Retry these statuses when URL is still missing.
YTDLP_RETRYABLE_STATUSES = {
    "",
    "nan",
    "none",
    "null",
    "no_result",
    "no_video_id",
    "no_acceptable_match",
    "youtube_api_runtime_error",
    "all_youtube_keys_unavailable",
    "all_youtube_keys_temporarily_unavailable",
}

BAD_VERSION_MARKERS = {
    "live", "karaoke", "instrumental", "cover", "backing track",
    "piano version", "nightcore", "slowed", "reverb", "sped up",
    "8d audio", "reaction", "tutorial", "dance practice", "lyrics cover",
}

if "MANIFEST_DIR" not in globals():
    MANIFEST_DIR = Path("dataset_collection_clean") / "manifests"
MANIFEST_DIR = Path(MANIFEST_DIR)


# TEXT & GENRE HELPERS

def is_missing_value(value):
    text = str(value).strip()
    return text == "" or text.casefold() in {"nan", "none", "null"}


def normalise_match_text(value):
    text = str(value or "").casefold()
    text = re.sub(r"\([^)]*\)|\[[^\]]*\]", " ", text)
    text = re.sub(r"[^a-z0-9\u0900-\u097f\u3400-\u4dbf\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def text_similarity(left, right):
    left_text = normalise_match_text(left)
    right_text = normalise_match_text(right)

    if not left_text or not right_text:
        return 0.0

    sequence_score = SequenceMatcher(None, left_text, right_text).ratio()
    left_tokens = set(left_text.split())
    right_tokens = set(right_text.split())
    token_score = (len(left_tokens & right_tokens) / len(left_tokens | right_tokens)
                   if left_tokens and right_tokens else 0.0)

    return sequence_score * 0.65 + token_score * 0.35


def primary_artist(artist):
    artist = str(artist or "").strip()
    artist = re.sub(r'^(.+?)\s*-\s*\1$', r'\1', artist, flags=re.IGNORECASE).strip()
    parts = re.split(r"\s*(?:,|&| feat\.?| featuring | x )\s*", artist, maxsplit=1, flags=re.IGNORECASE)
    return parts[0].strip() if parts else artist


def clean_genre(value):
    value = str(value or "").strip()
    mapping = {
        "pop": "Pop", "rock": "Rock", "hip hop": "Hip-Hop/Rap", "hip-hop": "Hip-Hop/Rap",
        "hiphop": "Hip-Hop/Rap", "rap": "Hip-Hop/Rap", "hip-hop/rap": "Hip-Hop/Rap",
        "r&b": "R&B/Soul", "rnb": "R&B/Soul", "soul": "R&B/Soul", "r&b/soul": "R&B/Soul",
        "folk": "Folk/Acoustic", "acoustic": "Folk/Acoustic", "folk/acoustic": "Folk/Acoustic",
    }
    return mapping.get(value.casefold(), value)


def song_key(title, artist):
    return normalise_match_text(title) + "||" + normalise_match_text(artist)


def find_bad_version_marker(candidate_title, source_title):
    candidate_text = normalise_match_text(candidate_title)
    source_text = normalise_match_text(source_title)
    for marker in BAD_VERSION_MARKERS:
        if marker in candidate_text and marker not in source_text:
            return marker
    return ""


# CANDIDATE POOL LOADER (For Auto-Replacement)

def first_existing_column(frame, names):
    for name in names:
        if name in frame.columns:
            return name
    return None

def standardise_candidate_frame(frame):
    if frame is None or frame.empty:
        return pd.DataFrame()
    frame = frame.copy()
    title_col = first_existing_column(frame, ["title", "track_name", "trackName", "name", "song_title", "song"])
    artist_col = first_existing_column(frame, ["artist", "artist_name", "artistName", "artists", "primary_artist", "singer"])
    genre_col = first_existing_column(frame, ["final_genre", "genre", "target_genre", "mapped_genre", "genre_name"])
    
    if not all([title_col, artist_col, genre_col]):
        return pd.DataFrame()

    frame["title"] = frame[title_col].astype(str).str.strip()
    frame["artist"] = frame[artist_col].astype(str).str.strip()
    frame["final_genre"] = frame[genre_col].apply(clean_genre)
    frame["_song_key"] = [song_key(t, a) for t, a in zip(frame["title"], frame["artist"])]

    frame = frame[
        ~frame["title"].apply(is_missing_value) & 
        ~frame["artist"].apply(is_missing_value) & 
        frame["final_genre"].isin(GENRES)
    ].copy()
    return frame.drop_duplicates(subset=["_song_key"], keep="first").reset_index(drop=True)

def build_candidate_pool(language):
    frames = []
    prefix = language.casefold()

    # Look for any candidate csv matching the language
    blocked_terms = {"with_youtube", "manual", "review", "report", "audit", "removed", "backup"}
    preferred_terms = {"candidate", "itunes", "spotify", "merged", "pool", "verified"}
    
    for path in sorted(MANIFEST_DIR.glob("*.csv")):
        name = path.name.casefold()
        if prefix not in name or any(t in name for t in blocked_terms) or not any(t in name for t in preferred_terms):
            continue
        try:
            raw = pd.read_csv(path, low_memory=False)
            standardised = standardise_candidate_frame(raw)
            if not standardised.empty:
                frames.append(standardised)
        except Exception:
            continue

    if not frames:
        return pd.DataFrame(columns=["title", "artist", "final_genre", "_song_key"])

    pool = pd.concat(frames, ignore_index=True, sort=False)
    pool = pool.drop_duplicates(subset=["_song_key"], keep="first").reset_index(drop=True)
    return pool


# YT-DLP METADATA-ONLY SEARCH

YTDLP_OPTIONS = {
    "quiet": True,
    "no_warnings": True,
    "skip_download": True,
    "extract_flat": True,
    "noplaylist": True,
    "socket_timeout": 30,
    "retries": 2,
    "fragment_retries": 1,
}

def build_search_queries(title, artist):
    main_artist = primary_artist(artist)
    import re as _re
    bare_title = _re.sub(r'\s*[\(\[].*?[\)\]]\s*', ' ', title).strip()
    bare_title = _re.sub(r'\s*-\s*(from|feat\.?|ft\.?).*$', '', bare_title, flags=_re.IGNORECASE).strip() or title

    queries = [
        f"{bare_title} {main_artist} official audio",
        f"{bare_title} {main_artist} Topic",
        f"{bare_title} {main_artist}",
        f"{title} {main_artist}",
        f'"{bare_title}" "{main_artist}"',
    ]

    def _clean_query(q):
        q = q.replace("\u2018", "").replace("\u2019", "").replace("\u02bb", "")
        q = re.sub(r"[\x00-\x1f]", " ", q)
        return re.sub(r" {2,}", " ", q).strip()

    return list(dict.fromkeys(_clean_query(q) for q in queries if q.strip()))


def extract_video_id(entry):
    video_id = str(entry.get("id", "") or "").strip()
    if video_id:
        return video_id
    url = str(entry.get("url", "") or entry.get("webpage_url", "") or "").strip()
    match = re.search(r"(?:v=|youtu\.be/)([A-Za-z0-9_-]{11})", url)
    return match.group(1) if match else ""


def score_ytdlp_entry(source_title, source_artist, entry):
    candidate_title = str(entry.get("title", "") or "").strip()
    uploader = str(entry.get("uploader", "") or entry.get("channel", "") or entry.get("channel_name", "") or "").strip()

    title_score = text_similarity(source_title, candidate_title)
    artist_score = max(
        text_similarity(source_artist, candidate_title),
        text_similarity(source_artist, uploader),
        text_similarity(primary_artist(source_artist), candidate_title),
        text_similarity(primary_artist(source_artist), uploader),
    )

    bad_marker = find_bad_version_marker(candidate_title, source_title)
    score = title_score * 0.72 + artist_score * 0.28
    combined = normalise_match_text(candidate_title + " " + uploader)

    if "official audio" in combined: score += 0.10
    if "official video" in combined: score += 0.07
    if "topic" in normalise_match_text(uploader): score += 0.06
    if bad_marker: score -= 0.35

    return {
        "score": float(score),
        "title_score": float(title_score),
        "artist_score": float(artist_score),
        "bad_marker": bad_marker,
        "candidate_title": candidate_title,
        "candidate_uploader": uploader,
    }


def search_with_ytdlp(title, artist):
    candidates = []
    last_error = ""

    with yt_dlp.YoutubeDL(YTDLP_OPTIONS) as ydl:
        for query in build_search_queries(title, artist):
            try:
                info = ydl.extract_info(f"ytsearch{YTDLP_SEARCH_RESULTS}:{query}", download=False)
            except Exception as error:
                last_error = f"{type(error).__name__}: {str(error)[:250]}"
                continue

            entries = info.get("entries", []) if isinstance(info, dict) else []
            for entry in entries:
                if not isinstance(entry, dict): continue
                video_id = extract_video_id(entry)
                if not video_id: continue

                score_data = score_ytdlp_entry(title, artist, entry)
                candidates.append({"video_id": video_id, "query_used": query, **score_data})

            if candidates:
                best_so_far = max(candidates, key=lambda row: row["score"])
                if (best_so_far["score"] >= MIN_ACCEPT_SCORE and 
                    best_so_far["title_score"] >= MIN_TITLE_SCORE and 
                    (best_so_far["artist_score"] >= MIN_ARTIST_SCORE or best_so_far["score"] >= 0.75) and 
                    not best_so_far["bad_marker"]):
                    break

    if not candidates:
        return {
            "youtube_url": "", "youtube_video_id": "",
            "youtube_match_status": "ytdlp_search_error" if last_error else "ytdlp_no_result",
            "youtube_match_score": 0.0, "youtube_matched_title": "", "youtube_matched_channel": "",
            "youtube_query_used": "", "youtube_search_error": last_error,
        }

    best_by_video = {}
    for candidate in candidates:
        video_id = candidate["video_id"]
        current = best_by_video.get(video_id)
        if current is None or candidate["score"] > current["score"]:
            best_by_video[video_id] = candidate

    best = max(best_by_video.values(), key=lambda row: row["score"])
    title_word_count = len(str(title or "").split())
    effective_min_score = MIN_ACCEPT_SCORE - 0.12 if title_word_count <= 5 and best["artist_score"] >= 0.30 else MIN_ACCEPT_SCORE

    acceptable = (
        best["score"] >= effective_min_score and 
        best["title_score"] >= MIN_TITLE_SCORE and 
        (best["artist_score"] >= MIN_ARTIST_SCORE or best["score"] >= 0.75) and 
        not best["bad_marker"]
    )

    if not acceptable:
        return {
            "youtube_url": "", "youtube_video_id": "",
            "youtube_match_status": "ytdlp_no_acceptable_match",
            "youtube_match_score": round(best["score"], 4),
            "youtube_matched_title": best["candidate_title"], "youtube_matched_channel": best["candidate_uploader"],
            "youtube_query_used": best["query_used"], "youtube_search_error": "",
        }

    return {
        "youtube_url": f"https://www.youtube.com/watch?v={best['video_id']}",
        "youtube_video_id": best["video_id"],
        "youtube_match_status": "youtube_match_found_ytdlp",
        "youtube_match_score": round(best["score"], 4),
        "youtube_matched_title": best["candidate_title"],
        "youtube_matched_channel": best["candidate_uploader"],
        "youtube_query_used": best["query_used"],
        "youtube_search_error": "",
    }



# DATAFRAME HELPERS

YOUTUBE_COLUMNS = {
    "youtube_url": "", "youtube_video_id": "", "youtube_match_status": "",
    "youtube_match_score": "", "youtube_matched_title": "", "youtube_matched_channel": "",
    "youtube_query_used": "", "youtube_search_error": "",
}

def ensure_columns(frame):
    frame = frame.copy()
    missing = {c: d for c, d in YOUTUBE_COLUMNS.items() if c not in frame.columns}
    if missing:
        frame = pd.concat([frame, pd.DataFrame(missing, index=frame.index)], axis=1).copy()
    for column in YOUTUBE_COLUMNS:
        if column in frame.columns:
            frame[column] = frame[column].astype(object)
    return frame

def should_try_ytdlp(row):
    if not is_missing_value(row.get("youtube_url", "")):
        return False
    status = str(row.get("youtube_match_status", "")).strip().casefold()
    if status.startswith(("stop_youtube_api_error_", "youtube_api_error_", "request_failed_")):
        return True
    return status in YTDLP_RETRYABLE_STATUSES or status.startswith("ytdlp_")


# AUTOMATIC YT-DLP FALLBACK WITH AUTO-REPLACEMENT

def auto_fallback_and_replace(input_csv, language):
    input_csv = Path(input_csv)
    if not input_csv.exists():
        print(f"File not found, skipping: {input_csv}")
        return None

    work_df = pd.read_csv(input_csv, low_memory=False)
    work_df = ensure_columns(work_df)

    unresolved_indexes = [idx for idx, row in work_df.iterrows() if should_try_ytdlp(row)]
    if MAX_ROWS_PER_RUN is not None:
        unresolved_indexes = unresolved_indexes[:MAX_ROWS_PER_RUN]

    print(f"\n{language}: rows eligible for yt-dlp fallback / replacement = {len(unresolved_indexes)}")
    if not unresolved_indexes:
        return work_df

    candidate_pool = build_candidate_pool(language)
    existing_keys = {song_key(t, a) for t, a in zip(work_df["title"], work_df["artist"])}
    attempted_keys = set()
    replace_log = []
    processed = 0

    try:
        for index in tqdm(unresolved_indexes, desc=f"yt-dlp auto-search {language}", unit="track"):
            row = work_df.loc[index]
            original_title = str(row.get("title", "")).strip()
            original_artist = str(row.get("artist", "")).strip()

            # 1. Search for original track
            result = search_with_ytdlp(original_title, original_artist)

            if result["youtube_match_status"] == "youtube_match_found_ytdlp":
                # Success! Update the row.
                for column, value in result.items():
                    work_df.at[index, column] = value
                tqdm.write(f"FOUND (Original): {original_title} - {original_artist}")
            else:
                # 2. Track not found. AUTO-REPLACE logic.
                target_genre = clean_genre(row.get("final_genre", ""))
                candidates = candidate_pool[
                    (candidate_pool["final_genre"] == target_genre) & 
                    (~candidate_pool["_song_key"].isin(existing_keys | attempted_keys))
                ]

                if candidates.empty:
                    # Give up, write the failed result
                    for column, value in result.items():
                        work_df.at[index, column] = value
                    tqdm.write(f"FAILED (No replacement candidates left): {original_title} - {original_artist}")
                else:
                    candidates = candidates.sample(frac=1).head(MAX_CANDIDATES_PER_REPLACEMENT)
                    replaced = False

                    for _, cand in candidates.iterrows():
                        cand_title, cand_artist = cand["title"], cand["artist"]
                        cand_key = cand["_song_key"]
                        attempted_keys.add(cand_key)

                        cand_result = search_with_ytdlp(cand_title, cand_artist)
                        if cand_result["youtube_match_status"] == "youtube_match_found_ytdlp":
                            # Successful replacement!
                            work_df.at[index, "title"] = cand_title
                            work_df.at[index, "artist"] = cand_artist
                            for column, value in cand_result.items():
                                work_df.at[index, column] = value
                            
                            existing_keys.add(cand_key)
                            replaced = True
                            
                            replace_log.append({
                                "old_title": original_title, "old_artist": original_artist,
                                "new_title": cand_title, "new_artist": cand_artist,
                                "genre": target_genre, "youtube_url": cand_result["youtube_url"]
                            })
                            tqdm.write(f"AUTO-REPLACED: {original_title} -> {cand_title} - {cand_artist}")
                            break
                        
                        time.sleep(random.uniform(1.0, 1.5)) # Brief pause between candidate checks
                    
                    if not replaced:
                        # Give up, write the original failed result
                        for column, value in result.items():
                            work_df.at[index, column] = value
                        tqdm.write(f"FAILED (Exhausted candidates): {original_title} - {original_artist}")

            processed += 1
            work_df.to_csv(input_csv, index=False, encoding="utf-8-sig")
            time.sleep(random.uniform(REQUEST_DELAY_MIN_SECONDS, REQUEST_DELAY_MAX_SECONDS))

    except KeyboardInterrupt:
        work_df.to_csv(input_csv, index=False, encoding="utf-8-sig")
        print("\nInterrupted. Progress saved.")
        raise

    work_df.to_csv(input_csv, index=False, encoding="utf-8-sig")

    if replace_log:
        pd.DataFrame(replace_log).to_csv(
            MANIFEST_DIR / f"{language.casefold()}_ytdlp_auto_replacements.csv", 
            index=False, encoding="utf-8-sig"
        )

    found_count = int((~work_df["youtube_url"].apply(is_missing_value)).sum())
    print(f"\n{language}: processed this run = {processed}")
    print(f"{language}: total URLs now = {found_count}/{len(work_df)}")
    print(work_df["youtube_match_status"].value_counts(dropna=False))

    return work_df


# RUN ALL LANGUAGES

for language_name, csv_name in [
    ("English", "english_human_with_youtube_urls.csv"),
    ("Hindi", "hindi_human_with_youtube_urls.csv"),
    ("Chinese", "chinese_human_with_youtube_urls.csv"),
]:
    target_csv = MANIFEST_DIR / csv_name
    if target_csv.exists():
        auto_fallback_and_replace(target_csv, language_name)
    else:
        print(f"\nSkipping {language_name}: {csv_name} not found.")

print("\nCell 8 complete! All unresolved tracks auto-searched and auto-replaced where necessary.")

## 11. Final contamination and structural audits

Run these advanced repair cells in order. Inspect the proposed/checkpoint CSVs before allowing a main manifest to be overwritten.


In [ ]:
# SYNC CORRECTED ROWS FROM *_with_youtube_urls.csv
#                    INTO *_metadata.csv so the two files stay in sync.
#
# A row is considered changed when its title or artist in
# with_youtube_urls.csv differs from what is in metadata.csv
# for the same song_id.  Only those rows are overwritten.
#
# A timestamped backup of each metadata.csv is written once
# before any rows are changed.

import pandas as pd
from pathlib import Path

if "MANIFEST_DIR" not in globals():
    MANIFEST_DIR = Path("dataset_collection_clean") / "manifests"
MANIFEST_DIR = Path(MANIFEST_DIR)


def sync_corrected_rows_to_metadata(language, with_youtube_csv, metadata_csv):
    """
    Copy rows that were changed by verification/replacement from
    with_youtube_csv into metadata_csv, keyed by song_id.

    Only rows where title or artist differs between the two files
    are overwritten, so unchanged verified-good rows are left alone.
    """
    with_youtube_csv = Path(with_youtube_csv)
    metadata_csv = Path(metadata_csv)

    if not with_youtube_csv.exists():
        print(f"{language}: {with_youtube_csv.name} not found — nothing to sync.")
        return None
    if not metadata_csv.exists():
        print(f"{language}: {metadata_csv.name} not found — nothing to sync into.")
        return None

    source_df = pd.read_csv(with_youtube_csv, low_memory=False)
    target_df = pd.read_csv(metadata_csv, low_memory=False).astype("object")

    if "song_id" not in source_df.columns or "song_id" not in target_df.columns:
        print(f"{language}: song_id column missing from one of the files — skipping.")
        return None

    # Write a one-time backup before touching the file
    backup = metadata_csv.with_name(metadata_csv.stem + "_before_sync.csv")
    if not backup.exists():
        target_df.to_csv(backup, index=False, encoding="utf-8-sig")
        print(f"{language}: backup saved -> {backup.name}")

    # Columns to copy across when a row has changed
    sync_cols = [
        c for c in ["title", "artist", "final_genre", "youtube_url",
                     "youtube_video_id", "raw_genre", "search_query", "country"]
        if c in target_df.columns and c in source_df.columns
    ]

    target_index = {str(sid): idx for idx, sid in target_df["song_id"].items()}
    synced_count = 0

    for _, src_row in source_df.iterrows():
        sid = str(src_row.get("song_id", ""))
        if not sid or sid not in target_index:
            continue
        t_idx = target_index[sid]

        # Only overwrite rows where something actually changed
        if (str(target_df.at[t_idx, "title"]).strip() == str(src_row["title"]).strip() and
                str(target_df.at[t_idx, "artist"]).strip() == str(src_row["artist"]).strip()):
            continue

        for col in sync_cols:
            target_df.at[t_idx, col] = src_row[col]

        if "verification_status" in target_df.columns:
            target_df.at[t_idx, "verification_status"] = (
                f"synced_from_combined_verification:"
                f"{src_row.get('combined_verdict', 'REPLACED')}"
            )
        if "notes" in target_df.columns:
            existing_notes = str(target_df.at[t_idx, "notes"] or "")
            new_note = f"synced:{src_row.get('combined_reason', 'purge_and_replace')}"
            target_df.at[t_idx, "notes"] = (existing_notes + "; " + new_note).strip("; ")

        synced_count += 1

    target_df.to_csv(metadata_csv, index=False, encoding="utf-8-sig")
    print(f"{language}: synced {synced_count} corrected rows into {metadata_csv.name}")
    return target_df


# Run for both languages that have separate metadata files.
# English metadata is generated directly from SONICS so no sync needed.
sync_corrected_rows_to_metadata(
    "Hindi",
    MANIFEST_DIR / "hindi_human_with_youtube_urls.csv",
    MANIFEST_DIR / "hindi_human_metadata.csv",
)

sync_corrected_rows_to_metadata(
    "Chinese",
    MANIFEST_DIR / "chinese_human_with_youtube_urls.csv",
    MANIFEST_DIR / "chinese_human_metadata.csv",
)

print("\nSync complete.")

In [ ]:
#  MANUAL MANIFEST REPAIR
# Removes confirmed wrong-language rows and replaces them from
# the local verified pool. Safe to re-run: creates backups first.

import re
import random
import shutil
from pathlib import Path

import pandas as pd

_FIX_MANIFEST_DIR = Path("manifests")
random.seed(42)


def _fm_norm(s):
    return re.sub(
        r"[^\w\s\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af\u0900-\u097f]",
        "", str(s or "").casefold(),
    ).strip()

def _fm_song_key(t, a):
    return _fm_norm(t) + "||" + _fm_norm(a)


# Artists that must NEVER appear in the final CSVs
HINDI_BLOCKED_FINAL = {
    # Pakistani Urdu rappers
    "talha anjum", "talhah yunus", "jokhay", "dhanji", "qaab",
    # English-vocal Indian bands
    "brodha v", "krsna", "half step down", "menwhopause", "inner sanctum",
    "superfuzz", "nicotine", "underground authority", "thermal and a quarter",
    "parikrama", "myndsnare", "aswekeepsearching", "brahma", "the down troddence",
    # Punjabi primary artists
    "karan aujla", "ap dhillon", "shubh", "sidhu moose wala", "diljit dosanjh",
    "ammy virk", "gippy grewal", "arjan dhillon", "navaan sandhu", "jass manak",
    "jassie gill", "garry sandhu", "prem dhillon", "shinda kahlon", "the prophec",
    "babbal rai", "ninja", "sharry mann", "mankirt aulakh", "guru randhawa",
    "hardy sandhu", "harrdy sandhu", "kulwinder billa", "jasmine sandlas",
    "nimrat khaira", "gurnam bhullar", "dilpreet dhillon", "r nait", "sukhe",
    "elly mangat", "ranjit bawa", "satinder sartaaj", "jazzy b", "miss pooja",
    "bohemia", "b praak", "jaani", "yo yo honey singh", "badshah",
    # Old / classic Bollywood playback singers (pre-digital era)
    # Their catalogues appear on streaming with modern release dates,
    # causing them to survive the MIN_RELEASE_YEAR = 2019 check.
    "kishore kumar", "kishor kumar", "mohammad rafi", "mohammed rafi",
    "asha bhosle", "lata mangeshkar", "mukesh", "hemant kumar",
    "manna dey", "geeta dutt", "talat mahmood", "shamshad begum",
    # Semi-classic playback singers (1990s–2000s Bollywood; now mostly archive releases)
    "kumar sanu", "alka yagnik", "udit narayan", "kavita krishnamurthy",
    "abhijeet bhattacharya", "sadhana sargam", "anuradha paudwal",
    "nitin mukesh", "vinod rathod", "sudesh bhosle",
    # Specifically blocked
    "ali sethi",      # Pakistani ghazal singer
    "bhayanak maut",  # Indian death metal, English vocals
    "vinod rathod",   # Old Bollywood playback singer
}

CHINESE_BLOCKED_FINAL = {
    "\u6734\u5bb0\u7bc4", "\u6734\u5bb0\u8303", "\u6734\u5bf3\u528d",
    "\u4e2d\u5c71\u5c06", "\u4e45\u77f3\u8b93", "\u4e45\u77f3\u8ba9",
    "\u4e45\u4fdd\u7530\u5229\u4f38",
    "jay park", "joe hisaishi", "nakayama sho", "toshinobu kubota",
    "hyukoh", "vaundy", "tatsuya kitani", "msasuke", "88rising", "4town",
    "beyond", "jackson wang",
}

HINDI_BLOCKED_POOL   = HINDI_BLOCKED_FINAL.copy()
CHINESE_BLOCKED_POOL = CHINESE_BLOCKED_FINAL.copy()


def _fm_artist_blocked(artist, blocked_set):
    an = _fm_norm(artist)
    for name in blocked_set:
        bn = _fm_norm(name)
        if not bn:
            continue
        if bn == an:
            return True
        if len(bn) >= 6 and re.search(r"(?<!\w)" + re.escape(bn) + r"(?!\w)", an):
            return True
    return False


# Explicit remove lists 
HINDI_REMOVE = [
    ("khabar e tahayyur e ishq",  "ali sethi"),
    ("rocket launcher",           "ali sethi"),
    ("dil mein ab",               "ali sethi"),
    ("more equal",                "thermal and a quarter"),
    ("ae mere pyare watan",       "vinod rathod"),
    ("way back",                  "jokhay"),
    ("way back",                  "talha anjum"),
    ("keep it casual",            "qaab"),
    ("downers at dusk",           "talha anjum"),
    ("just a metal kid",          "bhayanak maut"),
    ("teri mitti",                "b praak"),
]

CHINESE_REMOVE = [
    ("ai",                  "jackson wang"),
    ("yesterday",           "\u6734\u5bb0\u7bc4"),
    ("love is ugly",        "\u6734\u5bb0\u8303"),
    ("wanli",               "hyukoh"),
    ("goldberg of the wind","\u4e2d\u5c71\u5c06"),
    ("take over",           "league of legends"),
]

# False-positive protections ─
HINDI_KEEP_OVERRIDE = {
    ("main hoon hi kya",        "akanksha bhandari"),
    ("mann ke manjeere",        "shubha mudgal"),
    ("kahan shuru kahan khatam","dhvani bhanushali"),
}


def _fm_matches_remove(row, remove_list):
    t, a = _fm_norm(row.get("title", "")), _fm_norm(row.get("artist", ""))
    return any(rt in t and ra in a for rt, ra in remove_list)

def _fm_in_keep(row, keep_set):
    t, a = _fm_norm(row.get("title", "")), _fm_norm(row.get("artist", ""))
    return any(kt in t and ka in a for kt, ka in keep_set)


def _fm_load_pool(language, exclude_keys):
    prefix  = language.casefold()
    blocked = HINDI_BLOCKED_POOL if language == "Hindi" else CHINESE_BLOCKED_POOL
    pool_files = [
        _FIX_MANIFEST_DIR / f"{prefix}_strict_verified_live.csv",
        _FIX_MANIFEST_DIR / f"{prefix}_human_metadata.csv",
        _FIX_MANIFEST_DIR / f"{prefix}_candidate_pool.csv",
    ]
    frames = []
    for p in pool_files:
        if p.exists():
            try:
                df = pd.read_csv(p, low_memory=False, encoding="utf-8-sig")
                frames.append(df)
                print(f"    loaded pool {p.name}: {len(df)} rows")
            except Exception as e:
                print(f"    WARN: could not read {p}: {e}")
    if not frames:
        return pd.DataFrame()
    pool = pd.concat(frames, ignore_index=True, sort=False)
    for gcol in ("final_genre", "genre"):
        if gcol in pool.columns:
            pool["final_genre"] = pool[gcol]
            break
    pool = pool.dropna(subset=["title", "artist", "final_genre"])
    pool["_key"] = [_fm_song_key(t, a) for t, a in zip(pool["title"], pool["artist"])]
    pool = pool.drop_duplicates(subset=["_key"])
    pool = pool[~pool["artist"].apply(lambda a: _fm_artist_blocked(a, blocked))]
    pool = pool[~pool["_key"].isin(exclude_keys)].reset_index(drop=True)
    print(f"    pool after blocking+dedup: {len(pool)} candidates")
    return pool


def _fm_scan_blocked(df, blocked, keep_ov):
    bad_mask = df.apply(
        lambda r: _fm_artist_blocked(r.get("artist", ""), blocked)
                  and not _fm_in_keep(r, keep_ov),
        axis=1,
    )
    return df[bad_mask]


def _fm_fix_language(language, csv_name, remove_list, blocked, keep_ov):
    csv_path = _FIX_MANIFEST_DIR / csv_name
    if not csv_path.exists():
        print(f"{language}: {csv_name} not found — skipping")
        return

    df           = pd.read_csv(csv_path, low_memory=False, encoding="utf-8-sig")
    original_len = len(df)
    print(f"\n{'='*60}")
    print(f"{language}: {original_len} rows in {csv_name}")

    remove_mask  = df.apply(lambda r: _fm_matches_remove(r, remove_list), axis=1)
    blocked_rows = _fm_scan_blocked(df[~remove_mask], blocked, keep_ov)
    blocked_mask = df.index.isin(blocked_rows.index)

    bad_df  = df[remove_mask | blocked_mask].copy()
    good_df = df[~(remove_mask | blocked_mask)].copy().reset_index(drop=True)

    print(f"  Removing {len(bad_df)} bad rows:")
    for _, r in bad_df.iterrows():
        print(f"    - [{r.get('final_genre', '')}] {r['title']} — {r['artist']}")

    if bad_df.empty:
        print("  Nothing to remove — CSV is already clean.")
        return

    existing_keys = {_fm_song_key(r["title"], r["artist"]) for _, r in good_df.iterrows()}
    pool = _fm_load_pool(language, existing_keys)

    if pool.empty:
        print("  WARN: pool empty — saving with removals only.")
        bak = csv_path.with_suffix(".csv.bak")
        shutil.copy(csv_path, bak)
        good_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        return

    replacement_log, new_rows = [], []
    for _, bad_row in bad_df.iterrows():
        genre = str(bad_row.get("final_genre", "")).strip()
        avail = pool[(pool["final_genre"] == genre) & (~pool["_key"].isin(existing_keys))]
        if avail.empty:
            print(f"  FAILED: no pool candidates for genre '{genre}' — leaving gap")
            continue
        pick = avail.sample(1, random_state=random.randint(0, 9999)).iloc[0]
        existing_keys.add(pick["_key"])
        rep = bad_row.copy()
        rep["title"] = pick["title"]
        rep["artist"] = pick["artist"]
        rep["final_genre"] = genre
        for col in [
            "youtube_url", "youtube_video_id", "youtube_match_status",
            "isrc", "spotify_id", "mb_id", "deezer_id",
            "language_decision", "language_verified", "language_confidence",
            "lyrics_devanagari_count", "lyrics_han_count",
            "raw_genre", "collection", "release_year", "duration_seconds",
        ]:
            if col in pick.index and pd.notna(pick[col]):
                rep[col] = pick[col]
        rep["cell11_replaced_from"]   = bad_row["title"]
        rep["cell11_replaced_artist"] = bad_row["artist"]
        new_rows.append(rep)
        replacement_log.append({
            "removed_title": bad_row["title"], "removed_artist": bad_row["artist"],
            "new_title": pick["title"],         "new_artist": pick["artist"],
            "genre": genre,
        })
        print(f"  REPLACED: '{bad_row['title']}' ({bad_row['artist']})"
              f" -> '{pick['title']}' ({pick['artist']})  [{genre}]")

    final_df = pd.concat([good_df, pd.DataFrame(new_rows)], ignore_index=True, sort=False) \
               if new_rows else good_df
    bak = csv_path.with_suffix(".csv.bak")
    shutil.copy(csv_path, bak)
    final_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"\n  {language}: {original_len} -> {len(final_df)} rows  (backup: {bak.name})")
    if "final_genre" in final_df.columns:
        print("  Genre distribution:", final_df["final_genre"].value_counts().to_dict())
    if replacement_log:
        log_path = _FIX_MANIFEST_DIR / f"{language.casefold()}_fix_manifest_log.csv"
        pd.DataFrame(replacement_log).to_csv(log_path, index=False, encoding="utf-8-sig")
        print(f"  Log: {log_path.name}")


def _fm_clean_pool(language):
    prefix  = language.casefold()
    blocked = HINDI_BLOCKED_POOL if language == "Hindi" else CHINESE_BLOCKED_POOL
    keep_ov = HINDI_KEEP_OVERRIDE if language == "Hindi" else set()
    pool_path = _FIX_MANIFEST_DIR / f"{prefix}_strict_verified_live.csv"
    if not pool_path.exists():
        return
    df  = pd.read_csv(pool_path, low_memory=False, encoding="utf-8-sig")
    bad = _fm_scan_blocked(df, blocked, keep_ov)
    if bad.empty:
        print(f"  Pool {pool_path.name}: already clean ({len(df)} rows)")
        return
    print(f"  Pool {pool_path.name}: removing {len(bad)} blocked artists")
    for _, r in bad.iterrows():
        print(f"    - {r.get('title', '')} — {r.get('artist', '')}")
    clean = df.drop(index=bad.index).reset_index(drop=True)
    bak   = pool_path.with_suffix(".csv.pool.bak")
    shutil.copy(pool_path, bak)
    clean.to_csv(pool_path, index=False, encoding="utf-8-sig")
    print(f"    {len(df)} -> {len(clean)} rows  (backup: {bak.name})")


#  Run 
print("Cell 11B — manual manifest repair")

_fm_fix_language(
    "Hindi", "hindi_human_with_youtube_urls.csv",
    HINDI_REMOVE, HINDI_BLOCKED_FINAL, HINDI_KEEP_OVERRIDE,
)
_fm_fix_language(
    "Chinese", "chinese_human_with_youtube_urls.csv",
    CHINESE_REMOVE, CHINESE_BLOCKED_FINAL, set(),
)

print("\nCleaning verified-pool CSVs...")
_fm_clean_pool("Hindi")
_fm_clean_pool("Chinese")

print("\nDone. Quick check:")
for _lang in ("hindi", "chinese"):
    _p = _FIX_MANIFEST_DIR / f"{_lang}_human_with_youtube_urls.csv"
    if _p.exists():
        _df = pd.read_csv(_p, encoding="utf-8-sig")
        print(f"  {_lang}: {len(_df)} rows | genres: {dict(_df['final_genre'].value_counts())}")


## Archived media-repair implementation...do not run

This duplicate version used `yt-dlp bilisearch` and could produce repeated Bilibili HTTP 412 errors. The next cell contains the current public-API fallback and matching fixes.


## 12. Current media validation: YouTube first, Bilibili for Chinese

For the current Chinese repair, keep `MEDIA_LANGUAGES_TO_RUN = ["Chinese"]`. For a complete fresh audit, include all three languages.


In [ ]:
MEDIA_LANGUAGES_TO_RUN = [
    "Chinese",
]

# Full fresh audit:
# MEDIA_LANGUAGES_TO_RUN = [
#     "English",
#     "Hindi",
#     "Chinese",
# ]


In [ ]:
# YOUTUBE CHECK + CHINESE BILIBILI FALLBACK USING yt-dlp
#
# Metadata only. No audio/video download.
# English + Hindi + Chinese in one cell.
# No YouTube Data API key required.
#

import random
import re
import time
from difflib import SequenceMatcher
from pathlib import Path
from urllib.parse import urlparse, parse_qs

import pandas as pd
from tqdm.auto import tqdm

try:
    import yt_dlp
except ImportError as error:
    raise ImportError(
        "yt-dlp is required. Install it with: pip install -U yt-dlp"
    ) from error


# CONFIGURATION

if "MANIFEST_DIR" not in globals():
    MANIFEST_DIR = (
        Path("dataset_collection_clean")
        / "manifests"
    )

MANIFEST_DIR = Path(MANIFEST_DIR)
MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LANGUAGE_YOUTUBE_FILES = [
    ("English", "english_human_with_youtube_urls.csv"),
    ("Hindi", "hindi_human_with_youtube_urls.csv"),
    ("Chinese", "chinese_human_with_youtube_urls.csv"),
]

# Number of search results examined per query.
YTDLP_SEARCH_RESULTS = 8

# Search result acceptance thresholds.
MIN_TOTAL_SCORE = {
    "English": 0.70,
    "Hindi": 0.66,
    "Chinese": 0.63,
}

MIN_TITLE_SCORE = {
    "English": 0.62,
    "Hindi": 0.56,
    "Chinese": 0.55,
}

MIN_ARTIST_SCORE = {
    "English": 0.18,
    "Hindi": 0.14,
    "Chinese": 0.08,
}

# Request spacing.
CHECK_DELAY_MIN = 0.8
CHECK_DELAY_MAX = 1.5

SEARCH_DELAY_MIN = 1.8
SEARCH_DELAY_MAX = 3.0

# False = rows already marked available_public are skipped.
FORCE_RECHECK_ALL = False

# Retry these prior statuses when rerunning.
RETRY_STATUSES = {
    "",
    "nan",
    "none",
    "null",
    "missing_url",
    "invalid_youtube_url",
    "video_unavailable_or_deleted",
    "video_private",
    "video_not_public",
    "verification_error_transient",
    "verification_error_unknown",
    "no_replacement_url_found",
    "ytdlp_no_result",
    "ytdlp_no_acceptable_match",
    "ytdlp_search_error",
}

# A transient extraction/network failure must not trigger replacement.
TRANSIENT_ERROR_MARKERS = [
    "timed out",
    "timeout",
    "temporary failure",
    "temporarily unavailable",
    "connection reset",
    "connection aborted",
    "remote end closed",
    "http error 429",
    "too many requests",
    "http error 500",
    "http error 502",
    "http error 503",
    "http error 504",
    "network is unreachable",
    "unable to download webpage",
    "sign in to confirm you’re not a bot",
    "sign in to confirm you're not a bot",
    "confirm you’re not a bot",
    "confirm you're not a bot",
]

DEFINITE_UNAVAILABLE_MARKERS = [
    "video unavailable",
    "this video is unavailable",
    "private video",
    "video is private",
    "has been removed",
    "copyright claim",
    "account associated with this video has been terminated",
    "this video is no longer available",
    "requested format is not available",
]

BAD_VERSION_MARKERS = [
    "cover",
    "live",
    "concert",
    "karaoke",
    "instrumental",
    "remix",
    "mashup",
    "nightcore",
    "sped up",
    "sped-up",
    "slowed",
    "reverb",
    "8d audio",
    "reaction",
    "tutorial",
    "lyrics meaning",
    "translation",
    "fanmade",
    "fan made",
    "edit audio",
    "shorts",
    "performance",
    "acoustic",
    "piano version",
    "music box",
    "伴奏",
    "純音樂",
    "纯音乐",
    "鋼琴版",
    "钢琴版",
    "演奏版",
    "翻唱",
    "現場",
    "现场",
]

OFFICIAL_MARKERS = [
    "official audio",
    "official video",
    "official music video",
    "provided to youtube",
    "topic",
    "vevo",
]


# yt-dlp OPTIONS

COMMON_YDL_OPTIONS = {
    "quiet": True,
    "no_warnings": True,
    "skip_download": True,
    "noplaylist": True,
    "extract_flat": False,
    "socket_timeout": 20,
    "retries": 2,
    "fragment_retries": 2,
    "ignoreerrors": False,
}

SEARCH_YDL_OPTIONS = {
    **COMMON_YDL_OPTIONS,
    "extract_flat": True,
    "ignoreerrors": True,
    "playlistend": YTDLP_SEARCH_RESULTS,
}


# BASIC HELPERS

def is_missing(value):
    text = str(value).strip()

    return (
        text == ""
        or text.casefold()
        in {"nan", "none", "null"}
    )


def normalise_text(value):
    text = str(value or "").casefold()

    text = re.sub(
        r"\([^)]*\)|\[[^\]]*\]",
        " ",
        text,
    )

    text = re.sub(
        r"[^a-z0-9\u0600-\u06ff"
        r"\u0900-\u0d7f"
        r"\u3040-\u30ff"
        r"\u3400-\u4dbf"
        r"\u4e00-\u9fff"
        r"\uac00-\ud7af]+",
        " ",
        text,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def similarity(left, right):
    left = normalise_text(left)
    right = normalise_text(right)

    if not left or not right:
        return 0.0

    if left == right:
        return 1.0

    return SequenceMatcher(
        None,
        left,
        right,
    ).ratio()


def primary_artist(value):
    text = str(value or "").strip()

    parts = re.split(
        r"\s*(?:,|&|/|\+|"
        r"\bfeat\.?\b|\bfeaturing\b|"
        r"\bft\.?\b|\bx\b)\s*",
        text,
        maxsplit=1,
        flags=re.IGNORECASE,
    )

    return (
        parts[0].strip()
        if parts
        else text
    )


def extract_video_id(url):
    url = str(url or "").strip()

    if is_missing(url):
        return ""

    try:
        parsed = urlparse(url)
        host = parsed.netloc.casefold()

        if "youtu.be" in host:
            candidate = (
                parsed.path
                .strip("/")
                .split("/")[0]
            )
            return (
                candidate
                if re.fullmatch(
                    r"[A-Za-z0-9_-]{11}",
                    candidate,
                )
                else ""
            )

        if (
            "youtube.com" in host
            or "music.youtube.com" in host
        ):
            query_id = (
                parse_qs(
                    parsed.query
                )
                .get(
                    "v",
                    [""],
                )[0]
            )

            if re.fullmatch(
                r"[A-Za-z0-9_-]{11}",
                query_id,
            ):
                return query_id

            parts = [
                part
                for part in parsed.path.split("/")
                if part
            ]

            for marker in [
                "shorts",
                "embed",
                "live",
            ]:
                if marker in parts:
                    index = parts.index(marker)

                    if (
                        index + 1
                        < len(parts)
                        and re.fullmatch(
                            r"[A-Za-z0-9_-]{11}",
                            parts[index + 1],
                        )
                    ):
                        return parts[index + 1]

    except Exception:
        return ""

    return ""


def canonical_youtube_url(video_id):
    video_id = str(video_id or "").strip()

    if not re.fullmatch(
        r"[A-Za-z0-9_-]{11}",
        video_id,
    ):
        return ""

    return (
        "https://www.youtube.com/watch?v="
        + video_id
    )



# BILIBILI HELPERS

def extract_bilibili_video_id(url):
    """
    Returns a stable Bilibili media ID:
      - BVxxxxxxxxxx when present
      - av123456 when only an AV/AID is present
    """

    text = str(url or "").strip()

    if is_missing(text):
        return ""

    bvid_match = re.search(
        r"(BV[A-Za-z0-9]{10})",
        text,
        flags=re.IGNORECASE,
    )

    if bvid_match:
        value = bvid_match.group(1)
        return "BV" + value[2:]

    av_match = re.search(
        r"(?:/video/|^)(?:av)?(\d+)",
        text,
        flags=re.IGNORECASE,
    )

    if av_match:
        return "av" + av_match.group(1)

    return ""


def canonical_bilibili_url(video_id):
    video_id = str(video_id or "").strip()

    if re.fullmatch(
        r"BV[A-Za-z0-9]{10}",
        video_id,
        flags=re.IGNORECASE,
    ):
        return (
            "https://www.bilibili.com/video/"
            + video_id
        )

    if re.fullmatch(
        r"av\d+",
        video_id,
        flags=re.IGNORECASE,
    ):
        return (
            "https://www.bilibili.com/video/"
            + video_id
        )

    return ""


def contains_japanese_kana(value):
    return bool(
        re.search(
            r"[\u3040-\u30ff\u31f0-\u31ff]",
            str(value or ""),
        )
    )


def contains_korean_hangul(value):
    return bool(
        re.search(
            r"[\u1100-\u11ff\uac00-\ud7af]",
            str(value or ""),
        )
    )


def bilibili_result_has_wrong_language(
    title,
    uploader="",
    description="",
):
    """
    Han characters are shared by Chinese and Japanese.
    Therefore Han alone is not rejected, but Japanese Kana and
    Korean Hangul are strong conflicting-script evidence.
    """

    combined = (
        str(title or "")
        + " "
        + str(uploader or "")
        + " "
        + str(description or "")
    )

    return (
        contains_japanese_kana(combined)
        or contains_korean_hangul(combined)
    )


def check_bilibili_url_with_ytdlp(url):
    url = str(url or "").strip()

    if is_missing(url):
        return {
            "ok": False,
            "replaceable": True,
            "status": "missing_bilibili_url",
            "video_id": "",
            "matched_title": "",
            "matched_uploader": "",
            "error": "",
        }

    video_id = extract_bilibili_video_id(
        url
    )

    if not video_id:
        return {
            "ok": False,
            "replaceable": True,
            "status": "invalid_bilibili_url",
            "video_id": "",
            "matched_title": "",
            "matched_uploader": "",
            "error": "",
        }

    canonical_url = canonical_bilibili_url(
        video_id
    )

    try:
        with yt_dlp.YoutubeDL(
            COMMON_YDL_OPTIONS
        ) as ydl:
            info = ydl.extract_info(
                canonical_url,
                download=False,
            )

        if not isinstance(
            info,
            dict,
        ):
            return {
                "ok": False,
                "replaceable": False,
                "status": "bilibili_verification_error_unknown",
                "video_id": video_id,
                "matched_title": "",
                "matched_uploader": "",
                "error": "yt_dlp_returned_no_metadata",
            }

        matched_title = str(
            info.get(
                "title",
                "",
            )
            or ""
        )

        matched_uploader = str(
            info.get(
                "uploader",
                info.get(
                    "channel",
                    "",
                ),
            )
            or ""
        )

        description = str(
            info.get(
                "description",
                "",
            )
            or ""
        )

        if bilibili_result_has_wrong_language(
            matched_title,
            matched_uploader,
            description,
        ):
            return {
                "ok": False,
                "replaceable": True,
                "status": "bilibili_wrong_language_japanese_or_korean",
                "video_id": video_id,
                "matched_title": matched_title,
                "matched_uploader": matched_uploader,
                "error": "conflicting_kana_or_hangul_script",
            }

        live_status = str(
            info.get(
                "live_status",
                "",
            )
            or ""
        ).strip().casefold()

        if live_status in {
            "is_live",
            "is_upcoming",
        }:
            return {
                "ok": False,
                "replaceable": True,
                "status": "bilibili_live_or_upcoming_video",
                "video_id": video_id,
                "matched_title": matched_title,
                "matched_uploader": matched_uploader,
                "error": live_status,
            }

        duration = info.get(
            "duration",
            None,
        )

        try:
            duration = float(duration)

            if duration < 90 or duration > 1200:
                return {
                    "ok": False,
                    "replaceable": True,
                    "status": "bilibili_bad_duration",
                    "video_id": video_id,
                    "matched_title": matched_title,
                    "matched_uploader": matched_uploader,
                    "error": str(duration),
                }
        except Exception:
            pass

        return {
            "ok": True,
            "replaceable": False,
            "status": "available_public_bilibili",
            "video_id": (
                extract_bilibili_video_id(
                    info.get(
                        "webpage_url",
                        canonical_url,
                    )
                )
                or video_id
            ),
            "matched_title": matched_title,
            "matched_uploader": matched_uploader,
            "error": "",
        }

    except Exception as error:
        message = str(error)
        message_lower = message.casefold()

        if any(
            marker in message_lower
            for marker in TRANSIENT_ERROR_MARKERS
        ):
            return {
                "ok": False,
                "replaceable": False,
                "status": "bilibili_verification_error_transient",
                "video_id": video_id,
                "matched_title": "",
                "matched_uploader": "",
                "error": message[:500],
            }

        return {
            "ok": False,
            "replaceable": True,
            "status": "bilibili_unavailable_or_extraction_failed",
            "video_id": video_id,
            "matched_title": "",
            "matched_uploader": "",
            "error": message[:500],
        }


def source_title_allows_marker(
    source_title,
    marker,
):
    return (
        marker
        in normalise_text(
            source_title
        )
    )


def contains_disallowed_version(
    source_title,
    candidate_title,
):
    candidate_lower = normalise_text(
        candidate_title
    )

    for marker in BAD_VERSION_MARKERS:
        marker_normalised = normalise_text(
            marker
        )

        if (
            marker_normalised
            and marker_normalised
            in candidate_lower
            and not source_title_allows_marker(
                source_title,
                marker,
            )
        ):
            return marker

    return ""



# CHECK EXISTING URL USING yt-dlp

def check_youtube_url_with_ytdlp(url):
    url = str(url or "").strip()

    if is_missing(url):
        return {
            "ok": False,
            "replaceable": True,
            "status": "missing_url",
            "video_id": "",
            "matched_title": "",
            "matched_channel": "",
            "error": "",
        }

    video_id = extract_video_id(
        url
    )

    if not video_id:
        return {
            "ok": False,
            "replaceable": True,
            "status": "invalid_youtube_url",
            "video_id": "",
            "matched_title": "",
            "matched_channel": "",
            "error": "",
        }

    canonical_url = canonical_youtube_url(
        video_id
    )

    try:
        with yt_dlp.YoutubeDL(
            COMMON_YDL_OPTIONS
        ) as ydl:
            info = ydl.extract_info(
                canonical_url,
                download=False,
            )

        if not isinstance(
            info,
            dict,
        ):
            return {
                "ok": False,
                "replaceable": False,
                "status": "verification_error_unknown",
                "video_id": video_id,
                "matched_title": "",
                "matched_channel": "",
                "error": "yt_dlp_returned_no_metadata",
            }

        availability = str(
            info.get(
                "availability",
                "",
            )
            or ""
        ).strip().casefold()

        live_status = str(
            info.get(
                "live_status",
                "",
            )
            or ""
        ).strip().casefold()

        if availability in {
            "private",
            "premium_only",
            "subscriber_only",
            "needs_auth",
        }:
            return {
                "ok": False,
                "replaceable": True,
                "status": (
                    "video_private"
                    if availability == "private"
                    else "video_not_public"
                ),
                "video_id": video_id,
                "matched_title": str(
                    info.get(
                        "title",
                        "",
                    )
                    or ""
                ),
                "matched_channel": str(
                    info.get(
                        "channel",
                        info.get(
                            "uploader",
                            "",
                        ),
                    )
                    or ""
                ),
                "error": availability,
            }

        if live_status in {
            "is_live",
            "is_upcoming",
        }:
            # A live stream may still be public, but is unsuitable
            # as a stable song-audio URL.
            return {
                "ok": False,
                "replaceable": True,
                "status": (
                    "live_or_upcoming_video"
                ),
                "video_id": video_id,
                "matched_title": str(
                    info.get(
                        "title",
                        "",
                    )
                    or ""
                ),
                "matched_channel": str(
                    info.get(
                        "channel",
                        info.get(
                            "uploader",
                            "",
                        ),
                    )
                    or ""
                ),
                "error": live_status,
            }

        return {
            "ok": True,
            "replaceable": False,
            "status": "available_public",
            "video_id": str(
                info.get(
                    "id",
                    video_id,
                )
                or video_id
            ),
            "matched_title": str(
                info.get(
                    "title",
                    "",
                )
                or ""
            ),
            "matched_channel": str(
                info.get(
                    "channel",
                    info.get(
                        "uploader",
                        "",
                    ),
                )
                or ""
            ),
            "error": "",
        }

    except Exception as error:
        message = str(error)
        message_lower = message.casefold()

        if any(
            marker in message_lower
            for marker in DEFINITE_UNAVAILABLE_MARKERS
        ):
            status = (
                "video_private"
                if "private" in message_lower
                else "video_unavailable_or_deleted"
            )

            return {
                "ok": False,
                "replaceable": True,
                "status": status,
                "video_id": video_id,
                "matched_title": "",
                "matched_channel": "",
                "error": message[:500],
            }

        if any(
            marker in message_lower
            for marker in TRANSIENT_ERROR_MARKERS
        ):
            return {
                "ok": False,
                "replaceable": False,
                "status": "verification_error_transient",
                "video_id": video_id,
                "matched_title": "",
                "matched_channel": "",
                "error": message[:500],
            }

        return {
            "ok": False,
            "replaceable": False,
            "status": "verification_error_unknown",
            "video_id": video_id,
            "matched_title": "",
            "matched_channel": "",
            "error": message[:500],
        }


# SEARCH QUERIES

def query_variants(
    title,
    artist,
    language,
):
    title = str(title or "").strip()
    artist = str(artist or "").strip()
    first_artist = primary_artist(
        artist
    )

    queries = [
        f'"{title}" "{artist}" official audio',
        f'"{title}" "{first_artist}"',
        f'{title} {first_artist} Topic',
        f'"{title}" "{artist}"',
    ]

    if language == "Chinese":
        queries.extend(
            [
                f'"{title}" official audio',
                f'{title} {first_artist}',
            ]
        )

    output = []
    seen = set()

    for query in queries:
        query = re.sub(
            r"\s+",
            " ",
            query,
        ).strip()

        key = query.casefold()

        if (
            query
            and key not in seen
        ):
            seen.add(
                key
            )
            output.append(
                query
            )

    return output


# SCORE SEARCH RESULT

def score_youtube_candidate(
    source_title,
    source_artist,
    result,
    language,
):
    found_title = str(
        result.get(
            "title",
            "",
        )
        or ""
    )

    found_channel = str(
        result.get(
            "channel",
            result.get(
                "uploader",
                "",
            ),
        )
        or ""
    )

    found_artist = str(
        result.get(
            "artist",
            result.get(
                "creator",
                "",
            ),
        )
        or ""
    )

    found_description = str(
        result.get(
            "description",
            "",
        )
        or ""
    )

    bad_marker = contains_disallowed_version(
        source_title,
        found_title,
    )

    if bad_marker:
        return {
            "accepted": False,
            "score": 0.0,
            "title_score": 0.0,
            "artist_score": 0.0,
            "reason": (
                "rejected_bad_version_"
                + normalise_text(
                    bad_marker
                ).replace(
                    " ",
                    "_",
                )
            ),
        }

    title_score = similarity(
        source_title,
        found_title,
    )

    artist_text = (
        found_artist
        + " "
        + found_channel
        + " "
        + found_description[:250]
    )

    artist_score = max(
        similarity(
            source_artist,
            artist_text,
        ),
        similarity(
            primary_artist(
                source_artist
            ),
            artist_text,
        ),
        similarity(
            source_artist,
            found_title,
        ),
    )

    found_lower = (
        found_title
        + " "
        + found_channel
    ).casefold()

    official_bonus = 0.0

    if any(
        marker in found_lower
        for marker in OFFICIAL_MARKERS
    ):
        official_bonus += 0.10

    if (
        str(
            result.get(
                "channel_is_verified",
                "",
            )
        ).casefold()
        == "true"
    ):
        official_bonus += 0.04

    duration_bonus = 0.0

    try:
        duration = float(
            result.get(
                "duration",
                0,
            )
            or 0
        )

        if 90 <= duration <= 600:
            duration_bonus = 0.03
        elif duration > 1200:
            duration_bonus = -0.06
    except Exception:
        duration_bonus = 0.0

    if language == "Chinese":
        total_score = (
            title_score * 0.72
            + artist_score * 0.18
            + official_bonus
            + duration_bonus
        )
    else:
        total_score = (
            title_score * 0.66
            + artist_score * 0.24
            + official_bonus
            + duration_bonus
        )

    accepted = (
        title_score
        >= MIN_TITLE_SCORE[
            language
        ]
        and artist_score
        >= MIN_ARTIST_SCORE[
            language
        ]
        and total_score
        >= MIN_TOTAL_SCORE[
            language
        ]
    )

    return {
        "accepted": accepted,
        "score": round(
            total_score,
            4,
        ),
        "title_score": round(
            title_score,
            4,
        ),
        "artist_score": round(
            artist_score,
            4,
        ),
        "reason": (
            "accepted"
            if accepted
            else "below_threshold"
        ),
    }


# SEARCH REPLACEMENT USING yt-dlp

def search_replacement_with_ytdlp(
    title,
    artist,
    language,
):
    best = None
    best_score = -1.0
    errors = []

    for query in query_variants(
        title,
        artist,
        language,
    ):
        search_expression = (
            f"ytsearch{YTDLP_SEARCH_RESULTS}:"
            + query
        )

        try:
            with yt_dlp.YoutubeDL(
                SEARCH_YDL_OPTIONS
            ) as ydl:
                payload = ydl.extract_info(
                    search_expression,
                    download=False,
                )

        except Exception as error:
            errors.append(
                f"{query}: "
                f"{type(error).__name__}: "
                f"{str(error)[:250]}"
            )
            continue

        entries = []

        if isinstance(
            payload,
            dict,
        ):
            entries = [
                entry
                for entry in (
                    payload.get(
                        "entries",
                        [],
                    )
                    or []
                )
                if isinstance(
                    entry,
                    dict,
                )
            ]

        for result in entries:
            video_id = str(
                result.get(
                    "id",
                    "",
                )
                or ""
            ).strip()

            if not re.fullmatch(
                r"[A-Za-z0-9_-]{11}",
                video_id,
            ):
                continue

            scoring = (
                score_youtube_candidate(
                    title,
                    artist,
                    result,
                    language,
                )
            )

            if not scoring[
                "accepted"
            ]:
                continue

            if (
                scoring[
                    "score"
                ]
                > best_score
            ):
                best_score = scoring[
                    "score"
                ]

                best = {
                    "youtube_url": (
                        canonical_youtube_url(
                            video_id
                        )
                    ),
                    "youtube_video_id": (
                        video_id
                    ),
                    "youtube_match_status": (
                        "youtube_match_found_ytdlp"
                    ),
                    "youtube_match_score": (
                        scoring[
                            "score"
                        ]
                    ),
                    "youtube_title_score": (
                        scoring[
                            "title_score"
                        ]
                    ),
                    "youtube_artist_score": (
                        scoring[
                            "artist_score"
                        ]
                    ),
                    "youtube_matched_title": str(
                        result.get(
                            "title",
                            "",
                        )
                        or ""
                    ),
                    "youtube_matched_channel": str(
                        result.get(
                            "channel",
                            result.get(
                                "uploader",
                                "",
                            ),
                        )
                        or ""
                    ),
                    "youtube_query_used": (
                        query
                    ),
                    "youtube_search_error": "",
                }

        time.sleep(
            random.uniform(
                SEARCH_DELAY_MIN,
                SEARCH_DELAY_MAX,
            )
        )

    if best is not None:
        return best

    return {
        "youtube_url": "",
        "youtube_video_id": "",
        "youtube_match_status": (
            "ytdlp_no_acceptable_match"
            if not errors
            else "ytdlp_search_error"
        ),
        "youtube_match_score": "",
        "youtube_title_score": "",
        "youtube_artist_score": "",
        "youtube_matched_title": "",
        "youtube_matched_channel": "",
        "youtube_query_used": "",
        "youtube_search_error": (
            " | ".join(
                errors
            )[:1500]
        ),
    }



# SEARCH BILIBILI FALLBACK USING yt-dlp

def bilibili_query_variants(
    title,
    artist,
):
    title = str(title or "").strip()
    artist = str(artist or "").strip()
    first_artist = primary_artist(
        artist
    )

    queries = [
        f'"{title}" "{artist}" 官方 MV',
        f'"{title}" "{artist}" 官方音源',
        f'"{title}" "{first_artist}"',
        f'{title} {first_artist} 官方',
        f'{title} {first_artist}',
        f'"{title}"',
    ]

    output = []
    seen = set()

    for query in queries:
        query = re.sub(
            r"\s+",
            " ",
            query,
        ).strip()

        key = query.casefold()

        if query and key not in seen:
            seen.add(key)
            output.append(query)

    return output


def get_bilibili_result_url(result):
    for key in [
        "webpage_url",
        "original_url",
        "url",
    ]:
        value = str(
            result.get(
                key,
                "",
            )
            or ""
        ).strip()

        if "bilibili.com/video/" in value:
            return value

    result_id = str(
        result.get(
            "id",
            "",
        )
        or ""
    ).strip()

    if re.fullmatch(
        r"BV[A-Za-z0-9]{10}",
        result_id,
        flags=re.IGNORECASE,
    ):
        return canonical_bilibili_url(
            result_id
        )

    if result_id.isdigit():
        return canonical_bilibili_url(
            "av" + result_id
        )

    return ""


def search_bilibili_replacement_with_ytdlp(
    title,
    artist,
):
    """
    Chinese-only fallback.

    Search order in the main processor:
      YouTube first -> Bilibili second.
    """

    best = None
    best_score = -1.0
    errors = []

    for query in bilibili_query_variants(
        title,
        artist,
    ):
        search_expression = (
            f"bilisearch{YTDLP_SEARCH_RESULTS}:"
            + query
        )

        try:
            with yt_dlp.YoutubeDL(
                SEARCH_YDL_OPTIONS
            ) as ydl:
                payload = ydl.extract_info(
                    search_expression,
                    download=False,
                )

        except Exception as error:
            errors.append(
                f"{query}: "
                f"{type(error).__name__}: "
                f"{str(error)[:250]}"
            )
            continue

        entries = []

        if isinstance(
            payload,
            dict,
        ):
            entries = [
                entry
                for entry in (
                    payload.get(
                        "entries",
                        [],
                    )
                    or []
                )
                if isinstance(
                    entry,
                    dict,
                )
            ]

        for result in entries:
            result_url = get_bilibili_result_url(
                result
            )

            if not result_url:
                continue

            found_title = str(
                result.get(
                    "title",
                    "",
                )
                or ""
            )

            found_uploader = str(
                result.get(
                    "uploader",
                    result.get(
                        "channel",
                        "",
                    ),
                )
                or ""
            )

            found_description = str(
                result.get(
                    "description",
                    "",
                )
                or ""
            )

            if bilibili_result_has_wrong_language(
                found_title,
                found_uploader,
                found_description,
            ):
                continue

            scoring_result = {
                **result,
                "channel": found_uploader,
                "uploader": found_uploader,
            }

            scoring = score_youtube_candidate(
                title,
                artist,
                scoring_result,
                "Chinese",
            )

            if not scoring[
                "accepted"
            ]:
                continue

            if scoring[
                "score"
            ] > best_score:
                best_score = scoring[
                    "score"
                ]

                best = {
                    "bilibili_url": result_url,
                    "bilibili_video_id": (
                        extract_bilibili_video_id(
                            result_url
                        )
                    ),
                    "bilibili_match_status": (
                        "bilibili_match_found_ytdlp"
                    ),
                    "bilibili_match_score": scoring[
                        "score"
                    ],
                    "bilibili_title_score": scoring[
                        "title_score"
                    ],
                    "bilibili_artist_score": scoring[
                        "artist_score"
                    ],
                    "bilibili_matched_title": found_title,
                    "bilibili_matched_uploader": found_uploader,
                    "bilibili_query_used": query,
                    "bilibili_search_error": "",
                }

        time.sleep(
            random.uniform(
                SEARCH_DELAY_MIN,
                SEARCH_DELAY_MAX,
            )
        )

    if best is None:
        return {
            "bilibili_url": "",
            "bilibili_video_id": "",
            "bilibili_match_status": (
                "bilibili_no_acceptable_match"
                if not errors
                else "bilibili_search_error"
            ),
            "bilibili_match_score": "",
            "bilibili_title_score": "",
            "bilibili_artist_score": "",
            "bilibili_matched_title": "",
            "bilibili_matched_uploader": "",
            "bilibili_query_used": "",
            "bilibili_search_error": (
                " | ".join(
                    errors
                )[:1500]
            ),
        }

    # Verify the chosen Bilibili URL with full yt-dlp metadata.
    verification = check_bilibili_url_with_ytdlp(
        best[
            "bilibili_url"
        ]
    )

    if not verification[
        "ok"
    ]:
        return {
            **best,
            "bilibili_url": "",
            "bilibili_video_id": "",
            "bilibili_match_status": verification[
                "status"
            ],
            "bilibili_search_error": verification[
                "error"
            ],
        }

    best[
        "bilibili_video_id"
    ] = verification[
        "video_id"
    ]

    best[
        "bilibili_available_status"
    ] = verification[
        "status"
    ]

    if verification[
        "matched_title"
    ]:
        best[
            "bilibili_matched_title"
        ] = verification[
            "matched_title"
        ]

    if verification[
        "matched_uploader"
    ]:
        best[
            "bilibili_matched_uploader"
        ] = verification[
            "matched_uploader"
        ]

    return best


# MAIN FILE PROCESSOR

YOUTUBE_COLUMNS = {
    "youtube_url": "",
    "youtube_video_id": "",
    "youtube_available_status": "",
    "youtube_match_status": "",
    "youtube_match_score": "",
    "youtube_title_score": "",
    "youtube_artist_score": "",
    "youtube_matched_title": "",
    "youtube_matched_channel": "",
    "youtube_query_used": "",
    "youtube_search_error": "",

    # Chinese Bilibili fallback columns.
    "bilibili_url": "",
    "bilibili_video_id": "",
    "bilibili_available_status": "",
    "bilibili_match_status": "",
    "bilibili_match_score": "",
    "bilibili_title_score": "",
    "bilibili_artist_score": "",
    "bilibili_matched_title": "",
    "bilibili_matched_uploader": "",
    "bilibili_query_used": "",
    "bilibili_search_error": "",

    # Generic columns used by downstream code.
    "media_url": "",
    "media_platform": "",
    "media_id": "",
    "media_match_status": "",
    "download_url": "",

    "download_status": "",
    "download_retry_note": "",
}


def should_check_existing_row(
    row,
    language,
):
    if FORCE_RECHECK_ALL:
        return True

    download_status = str(
        row.get(
            "download_status",
            "",
        )
    ).strip().casefold()

    retry_note = str(
        row.get(
            "download_retry_note",
            "",
        )
    ).strip().casefold()

    has_failure_note = (
        "failed" in download_status
        or "error" in download_status
        or "manual" in retry_note
        or "replacement" in retry_note
    )

    youtube_good = (
        str(
            row.get(
                "youtube_available_status",
                "",
            )
        ).strip()
        in {
            "available_public",
            "available_public_ytdlp_search",
        }
        and not is_missing(
            row.get(
                "youtube_url",
                "",
            )
        )
        and not has_failure_note
    )

    if youtube_good:
        return False

    if language == "Chinese":
        bilibili_good = (
            str(
                row.get(
                    "bilibili_available_status",
                    "",
                )
            ).strip()
            in {
                "available_public_bilibili",
                "available_public_bilibili_search",
            }
            and not is_missing(
                row.get(
                    "bilibili_url",
                    "",
                )
            )
            and not has_failure_note
        )

        if bilibili_good:
            return False

    return True


def check_and_replace_bad_youtube_urls(
    csv_file,
    language,
):
    csv_path = (
        MANIFEST_DIR
        / csv_file
    )

    if not csv_path.exists():
        print(
            f"\nSkipping {language}: "
            f"file not found -> {csv_path}"
        )
        return None

    df = pd.read_csv(
        csv_path,
        low_memory=False,
    )

    missing_columns = {
        column: default
        for column, default in (
            YOUTUBE_COLUMNS.items()
        )
        if column not in df.columns
    }

    if missing_columns:
        df = pd.concat(
            [
                df,
                pd.DataFrame(
                    missing_columns,
                    index=df.index,
                ),
            ],
            axis=1,
        )

    df = df.astype(
        "object"
    ).copy()

    print(
        "\n"
        + "=" * 72
    )

    print(
        f"Checking YouTube URLs with yt-dlp: "
        f"{language}"
    )

    print(
        csv_path
    )

    print(
        "=" * 72
    )

    repair_indexes = []
    transient_indexes = []

    # PHASE 1: VERIFY EXISTING URLS
    indexes_to_check = [
        index
        for index, row in (
            df.iterrows()
        )
        if should_check_existing_row(
            row,
            language,
        )
    ]

    print(
        f"{language}: rows requiring availability check =",
        len(indexes_to_check),
    )

    for index in tqdm(
        indexes_to_check,
        desc=f"Checking {language} URLs",
        unit="track",
        dynamic_ncols=True,
    ):
        row = df.loc[
            index
        ]

        title = row.get(
            "title",
            "",
        )

        artist = row.get(
            "artist",
            "",
        )

        result = (
            check_youtube_url_with_ytdlp(
                row.get(
                    "youtube_url",
                    "",
                )
            )
        )

        df.at[
            index,
            "youtube_video_id",
        ] = result[
            "video_id"
        ]

        df.at[
            index,
            "youtube_available_status",
        ] = result[
            "status"
        ]

        df.at[
            index,
            "youtube_search_error",
        ] = result[
            "error"
        ]

        if result[
            "matched_title"
        ]:
            df.at[
                index,
                "youtube_matched_title",
            ] = result[
                "matched_title"
            ]

        if result[
            "matched_channel"
        ]:
            df.at[
                index,
                "youtube_matched_channel",
            ] = result[
                "matched_channel"
            ]

        if result[
            "ok"
        ]:
            df.at[
                index,
                "download_retry_note",
            ] = ""

            tqdm.write(
                f"OK | {index + 1}/{len(df)} | "
                f"{title} - {artist}"
            )

        elif result[
            "replaceable"
        ]:
            repair_indexes.append(
                index
            )

            tqdm.write(
                f"BAD | {index + 1}/{len(df)} | "
                f"{result['status']} | "
                f"{title} - {artist}"
            )

        else:
            transient_indexes.append(
                index
            )

            df.at[
                index,
                "download_retry_note",
            ] = result[
                "status"
            ]

            tqdm.write(
                f"RETRY LATER | "
                f"{index + 1}/{len(df)} | "
                f"{result['status']} | "
                f"{title} - {artist}"
            )

        df.to_csv(
            csv_path,
            index=False,
            encoding="utf-8-sig",
        )

        time.sleep(
            random.uniform(
                CHECK_DELAY_MIN,
                CHECK_DELAY_MAX,
            )
        )

    # Also repair rows already carrying a known bad/missing status.
    for index, row in df.iterrows():
        status = str(
            row.get(
                "youtube_available_status",
                "",
            )
        ).strip().casefold()

        if (
            status in RETRY_STATUSES
            and index not in repair_indexes
            and index not in transient_indexes
            and is_missing(
                row.get(
                    "youtube_url",
                    "",
                )
            )
        ):
            repair_indexes.append(
                index
            )

    repair_indexes = list(
        dict.fromkeys(
            repair_indexes
        )
    )

    print(
        f"\n{language}: rows eligible for URL replacement =",
        len(repair_indexes),
    )

    print(
        f"{language}: temporary verification errors retained =",
        len(transient_indexes),
    )

    
    # EXISTING MEDIA IDS FOR DUPLICATE PROTECTION
    existing_video_ids = {
        str(value).strip()
        for value in df[
            "youtube_video_id"
        ]
        if (
            not is_missing(value)
            and re.fullmatch(
                r"[A-Za-z0-9_-]{11}",
                str(value).strip(),
            )
        )
    }

    existing_bilibili_ids = {
        str(value).strip()
        for value in df[
            "bilibili_video_id"
        ]
        if not is_missing(value)
    }

    # Remove IDs belonging to rows being repaired, so their own old
    # media is not treated as a duplicate.
    for index in repair_indexes:
        old_youtube_id = str(
            df.at[
                index,
                "youtube_video_id",
            ]
        ).strip()

        old_bilibili_id = str(
            df.at[
                index,
                "bilibili_video_id",
            ]
        ).strip()

        existing_video_ids.discard(
            old_youtube_id
        )

        existing_bilibili_ids.discard(
            old_bilibili_id
        )

    # PHASE 2
    #   ALL LANGUAGES  YOUTUBE SEARCH
    #   CHINESE ONLY  - BILIBILI FALLBACK IF YOUTUBE FAILS
    for index in tqdm(
        repair_indexes,
        desc=f"Replacing bad URLs {language}",
        unit="track",
        dynamic_ncols=True,
    ):
        title = df.at[
            index,
            "title",
        ]

        artist = df.at[
            index,
            "artist",
        ]

        # STEP A: SEARCH YOUTUBE FIRST
        youtube_result = (
            search_replacement_with_ytdlp(
                title,
                artist,
                language,
            )
        )

        youtube_video_id = str(
            youtube_result.get(
                "youtube_video_id",
                "",
            )
        ).strip()

        if (
            youtube_result[
                "youtube_match_status"
            ]
            == "youtube_match_found_ytdlp"
            and youtube_video_id
            in existing_video_ids
        ):
            youtube_result = {
                **youtube_result,
                "youtube_url": "",
                "youtube_video_id": "",
                "youtube_match_status": (
                    "ytdlp_duplicate_video_rejected"
                ),
                "youtube_search_error": (
                    "candidate_video_id_already_used"
                ),
            }

            youtube_video_id = ""

        for column in [
            "youtube_url",
            "youtube_video_id",
            "youtube_match_status",
            "youtube_match_score",
            "youtube_title_score",
            "youtube_artist_score",
            "youtube_matched_title",
            "youtube_matched_channel",
            "youtube_query_used",
            "youtube_search_error",
        ]:
            df.at[
                index,
                column,
            ] = youtube_result.get(
                column,
                "",
            )

        youtube_found = (
            youtube_result[
                "youtube_match_status"
            ]
            == "youtube_match_found_ytdlp"
            and not is_missing(
                youtube_result.get(
                    "youtube_url",
                    "",
                )
            )
        )

        if youtube_found:
            youtube_url = youtube_result[
                "youtube_url"
            ]

            df.at[
                index,
                "youtube_available_status",
            ] = (
                "available_public_ytdlp_search"
            )

            # Clear stale Bilibili fallback values because YouTube
            # has priority.
            for column in [
                "bilibili_url",
                "bilibili_video_id",
                "bilibili_available_status",
                "bilibili_match_status",
                "bilibili_match_score",
                "bilibili_title_score",
                "bilibili_artist_score",
                "bilibili_matched_title",
                "bilibili_matched_uploader",
                "bilibili_query_used",
                "bilibili_search_error",
            ]:
                df.at[
                    index,
                    column,
                ] = ""

            df.at[
                index,
                "media_url",
            ] = youtube_url

            df.at[
                index,
                "download_url",
            ] = youtube_url

            df.at[
                index,
                "media_platform",
            ] = "YouTube"

            df.at[
                index,
                "media_id",
            ] = youtube_video_id

            df.at[
                index,
                "media_match_status",
            ] = youtube_result[
                "youtube_match_status"
            ]

            df.at[
                index,
                "download_status",
            ] = (
                "url_replaced_retry_needed"
            )

            df.at[
                index,
                "download_retry_note",
            ] = (
                "youtube_replacement_url_ready"
            )

            existing_video_ids.add(
                youtube_video_id
            )

            tqdm.write(
                f"YOUTUBE REPLACED | "
                f"{index + 1}/{len(df)} | "
                f"{title} - {artist} | "
                f"{youtube_url}"
            )

        # STEP B: CHINESE BILIBILI FALLBACK
        elif language == "Chinese":
            bilibili_result = (
                search_bilibili_replacement_with_ytdlp(
                    title,
                    artist,
                )
            )

            bilibili_video_id = str(
                bilibili_result.get(
                    "bilibili_video_id",
                    "",
                )
            ).strip()

            if (
                bilibili_result.get(
                    "bilibili_match_status",
                    "",
                )
                == "bilibili_match_found_ytdlp"
                and bilibili_video_id
                in existing_bilibili_ids
            ):
                bilibili_result = {
                    **bilibili_result,
                    "bilibili_url": "",
                    "bilibili_video_id": "",
                    "bilibili_match_status": (
                        "bilibili_duplicate_video_rejected"
                    ),
                    "bilibili_search_error": (
                        "candidate_bilibili_id_already_used"
                    ),
                }

                bilibili_video_id = ""

            for column in [
                "bilibili_url",
                "bilibili_video_id",
                "bilibili_available_status",
                "bilibili_match_status",
                "bilibili_match_score",
                "bilibili_title_score",
                "bilibili_artist_score",
                "bilibili_matched_title",
                "bilibili_matched_uploader",
                "bilibili_query_used",
                "bilibili_search_error",
            ]:
                df.at[
                    index,
                    column,
                ] = bilibili_result.get(
                    column,
                    "",
                )

            bilibili_found = (
                bilibili_result.get(
                    "bilibili_match_status",
                    "",
                )
                == "bilibili_match_found_ytdlp"
                and not is_missing(
                    bilibili_result.get(
                        "bilibili_url",
                        "",
                    )
                )
            )

            if bilibili_found:
                bilibili_url = bilibili_result[
                    "bilibili_url"
                ]

                df.at[
                    index,
                    "youtube_available_status",
                ] = (
                    "no_youtube_match_bilibili_used"
                )

                df.at[
                    index,
                    "bilibili_available_status",
                ] = bilibili_result.get(
                    "bilibili_available_status",
                    "available_public_bilibili_search",
                )

                df.at[
                    index,
                    "media_url",
                ] = bilibili_url

                df.at[
                    index,
                    "download_url",
                ] = bilibili_url

                df.at[
                    index,
                    "media_platform",
                ] = "Bilibili"

                df.at[
                    index,
                    "media_id",
                ] = bilibili_video_id

                df.at[
                    index,
                    "media_match_status",
                ] = bilibili_result[
                    "bilibili_match_status"
                ]

                df.at[
                    index,
                    "download_status",
                ] = (
                    "bilibili_url_ready"
                )

                df.at[
                    index,
                    "download_retry_note",
                ] = (
                    "youtube_not_found_bilibili_fallback_ready"
                )

                existing_bilibili_ids.add(
                    bilibili_video_id
                )

                tqdm.write(
                    f"BILIBILI FALLBACK | "
                    f"{index + 1}/{len(df)} | "
                    f"{title} - {artist} | "
                    f"{bilibili_url}"
                )

            else:
                df.at[
                    index,
                    "youtube_available_status",
                ] = (
                    "no_youtube_or_bilibili_url_found"
                )

                df.at[
                    index,
                    "bilibili_available_status",
                ] = (
                    "no_bilibili_replacement_url_found"
                )

                df.at[
                    index,
                    "media_url",
                ] = ""

                df.at[
                    index,
                    "download_url",
                ] = ""

                df.at[
                    index,
                    "media_platform",
                ] = ""

                df.at[
                    index,
                    "media_id",
                ] = ""

                df.at[
                    index,
                    "media_match_status",
                ] = (
                    str(
                        youtube_result[
                            "youtube_match_status"
                        ]
                    )
                    + " | "
                    + str(
                        bilibili_result.get(
                            "bilibili_match_status",
                            "",
                        )
                    )
                )

                df.at[
                    index,
                    "download_status",
                ] = (
                    "no_youtube_or_bilibili_url_found"
                )

                df.at[
                    index,
                    "download_retry_note",
                ] = (
                    "youtube_and_bilibili_search_failed"
                )

                tqdm.write(
                    f"NO YOUTUBE/BILIBILI | "
                    f"{index + 1}/{len(df)} | "
                    f"{title} - {artist}"
                )

        # STEP C: ENGLISH/HINDI REMAIN YOUTUBE-ONLY
        else:
            df.at[
                index,
                "youtube_available_status",
            ] = (
                "no_replacement_url_found"
            )

            df.at[
                index,
                "media_url",
            ] = ""

            df.at[
                index,
                "download_url",
            ] = ""

            df.at[
                index,
                "media_platform",
            ] = ""

            df.at[
                index,
                "media_id",
            ] = ""

            df.at[
                index,
                "media_match_status",
            ] = youtube_result[
                "youtube_match_status"
            ]

            df.at[
                index,
                "download_status",
            ] = (
                "no_replacement_url_found"
            )

            df.at[
                index,
                "download_retry_note",
            ] = youtube_result[
                "youtube_match_status"
            ]

            tqdm.write(
                f"NO ACCEPTABLE YOUTUBE URL | "
                f"{index + 1}/{len(df)} | "
                f"{title} - {artist}"
            )

        df.to_csv(
            csv_path,
            index=False,
            encoding="utf-8-sig",
        )

    
    # SUMMARY
    print(
        "\nSaved updated manifest:",
        csv_path,
    )

    print(
        "\nYouTube availability:"
    )

    print(
        df[
            "youtube_available_status"
        ].value_counts(
            dropna=False
        )
    )

    print(
        "\nYouTube match status:"
    )

    print(
        df[
            "youtube_match_status"
        ].value_counts(
            dropna=False
        )
    )

    if language == "Chinese":
        print(
            "\nBilibili availability:"
        )

        print(
            df[
                "bilibili_available_status"
            ].value_counts(
                dropna=False
            )
        )

        print(
            "\nBilibili match status:"
        )

        print(
            df[
                "bilibili_match_status"
            ].value_counts(
                dropna=False
            )
        )

    print(
        "\nSelected media platform:"
    )

    print(
        df[
            "media_platform"
        ].value_counts(
            dropna=False
        )
    )

    print(
        "\nDownload status:"
    )

    print(
        df[
            "download_status"
        ].value_counts(
            dropna=False
        )
    )

    print(
        "\nRetry note:"
    )

    print(
        df[
            "download_retry_note"
        ].value_counts(
            dropna=False
        )
    )

    return df





#
# Netscape-format cookies exported from a browser are supported.
# Metadata only. No audio/video download.

import html as _html
import os as _os
import random as _random
import re as _re
import time as _time
from http.cookiejar import MozillaCookieJar as _MozillaCookieJar
from http.cookies import SimpleCookie as _SimpleCookie
from pathlib import Path as _Path
from urllib.parse import quote_plus as _quote_plus

import requests as _requests


# BILIBILI API CONFIG


_BILIBILI_SEARCH_API = (
    "https://api.bilibili.com/x/web-interface/search/type"
)

_BILIBILI_VIEW_API = (
    "https://api.bilibili.com/x/web-interface/view"
)

_BILIBILI_HOME_URL = "https://www.bilibili.com/"

_BILIBILI_BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/150.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "application/json, text/plain, */*"
    ),
    "Accept-Language": (
        "zh-CN,zh;q=0.9,en-GB;q=0.8,en;q=0.7"
    ),
    "Origin": "https://search.bilibili.com",
    "Referer": "https://search.bilibili.com/",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-site",
}

_BILIBILI_MAX_RESULTS = 20

# Do not repeatedly hit Bilibili after the IP/session is blocked.
_BILIBILI_412_BLOCKED = False
_BILIBILI_412_MESSAGE_SHOWN = False
_BILIBILI_SESSION = None
_BILIBILI_SESSION_READY = False


def _bilibili_clean_text(value):
    value = _html.unescape(
        str(value or "")
    )

    value = _re.sub(
        r"<[^>]+>",
        " ",
        value,
    )

    return _re.sub(
        r"\s+",
        " ",
        value,
    ).strip()


def _bilibili_parse_duration(value):
    text = str(value or "").strip()

    if not text:
        return None

    if _re.fullmatch(
        r"\d+(?:\.\d+)?",
        text,
    ):
        return int(
            round(
                float(text)
            )
        )

    parts = text.split(":")

    try:
        numbers = [
            int(part)
            for part in parts
        ]
    except Exception:
        return None

    if len(numbers) == 2:
        return (
            numbers[0] * 60
            + numbers[1]
        )

    if len(numbers) == 3:
        return (
            numbers[0] * 3600
            + numbers[1] * 60
            + numbers[2]
        )

    return None


def _bilibili_load_netscape_cookie_file(
    session,
    path,
):
    path = _Path(path)

    if not path.exists():
        return 0

    jar = _MozillaCookieJar(
        str(path)
    )

    try:
        jar.load(
            ignore_discard=True,
            ignore_expires=True,
        )
    except Exception:
        return 0

    count = 0

    for cookie in jar:
        domain = cookie.domain or ".bilibili.com"

        session.cookies.set(
            cookie.name,
            cookie.value,
            domain=domain,
            path=cookie.path or "/",
        )

        count += 1

    return count


def _bilibili_load_raw_cookie_string(
    session,
    cookie_string,
):
    cookie_string = str(
        cookie_string or ""
    ).strip()

    if not cookie_string:
        return 0

    cookie = _SimpleCookie()

    try:
        cookie.load(
            cookie_string
        )
    except Exception:
        return 0

    count = 0

    for key, morsel in cookie.items():
        session.cookies.set(
            key,
            morsel.value,
            domain=".bilibili.com",
            path="/",
        )

        count += 1

    return count


def _get_bilibili_session():
    global _BILIBILI_SESSION
    global _BILIBILI_SESSION_READY

    if (
        _BILIBILI_SESSION is not None
        and _BILIBILI_SESSION_READY
    ):
        return _BILIBILI_SESSION

    session = _requests.Session()
    session.headers.update(
        _BILIBILI_BROWSER_HEADERS
    )

    cookie_count = 0

    candidate_cookie_files = [
        MANIFEST_DIR
        / "bilibili_cookies.txt",
        _Path("bilibili_cookies.txt"),
        _Path("cookies.txt"),
    ]

    for cookie_file in candidate_cookie_files:
        loaded = (
            _bilibili_load_netscape_cookie_file(
                session,
                cookie_file,
            )
        )

        if loaded:
            cookie_count += loaded

            print(
                "Bilibili cookies loaded:",
                cookie_file,
                "| cookies:",
                loaded,
            )

            break

    raw_cookie = (
        globals().get(
            "BILIBILI_COOKIE",
            "",
        )
        or _os.getenv(
            "BILIBILI_COOKIE",
            "",
        )
    )

    cookie_count += (
        _bilibili_load_raw_cookie_string(
            session,
            raw_cookie,
        )
    )

    # Browser bootstrap can provide buvid cookies. Failure is not
    # fatal because the API may still work.
    try:
        session.get(
            _BILIBILI_HOME_URL,
            timeout=15,
            allow_redirects=True,
        )
    except Exception:
        pass

    _BILIBILI_SESSION = session
    _BILIBILI_SESSION_READY = True

    print(
        "Bilibili API session ready | cookie count:",
        len(session.cookies),
    )

    if cookie_count == 0:
        print(
            "Bilibili login cookies not found. "
            "Public API will be tried without login."
        )

    return session


def _bilibili_api_get(
    url,
    params,
    referer,
):
    global _BILIBILI_412_BLOCKED
    global _BILIBILI_412_MESSAGE_SHOWN

    if _BILIBILI_412_BLOCKED:
        return {
            "ok": False,
            "blocked_412": True,
            "status": "bilibili_blocked_412_retry_later",
            "data": {},
            "error": (
                "Bilibili requests paused for this run "
                "after HTTP 412."
            ),
        }

    session = _get_bilibili_session()

    headers = {
        "Referer": referer,
    }

    try:
        response = session.get(
            url,
            params=params,
            headers=headers,
            timeout=20,
        )

    except Exception as error:
        return {
            "ok": False,
            "blocked_412": False,
            "status": (
                "bilibili_request_error_"
                + type(error).__name__
            ),
            "data": {},
            "error": str(error)[:500],
        }

    if response.status_code == 412:
        _BILIBILI_412_BLOCKED = True

        if not _BILIBILI_412_MESSAGE_SHOWN:
            print(
                "\nBILIBILI HTTP 412 DETECTED."
            )

            print(
                "Bilibili fallback is paused for the rest "
                "of this run to avoid repeated blocking."
            )

            print(
                "Rows are marked retry_later, not song_not_found."
            )

            _BILIBILI_412_MESSAGE_SHOWN = True

        return {
            "ok": False,
            "blocked_412": True,
            "status": "bilibili_blocked_412_retry_later",
            "data": {},
            "error": (
                "HTTP 412 Precondition Failed"
            ),
        }

    if response.status_code == 429:
        return {
            "ok": False,
            "blocked_412": False,
            "status": "bilibili_rate_limit_429_retry_later",
            "data": {},
            "error": (
                "HTTP 429 Too Many Requests"
            ),
        }

    if response.status_code != 200:
        return {
            "ok": False,
            "blocked_412": False,
            "status": (
                "bilibili_http_error_"
                + str(response.status_code)
            ),
            "data": {},
            "error": response.text[:500],
        }

    try:
        payload = response.json()
    except Exception as error:
        return {
            "ok": False,
            "blocked_412": False,
            "status": "bilibili_invalid_json",
            "data": {},
            "error": str(error)[:500],
        }

    api_code = payload.get(
        "code",
        0,
    )

    if api_code != 0:
        # -352 and -412 are also anti-bot/risk-control results.
        if api_code in {
            -352,
            -412,
        }:
            _BILIBILI_412_BLOCKED = True

            return {
                "ok": False,
                "blocked_412": True,
                "status": (
                    "bilibili_risk_control_"
                    + str(api_code)
                    + "_retry_later"
                ),
                "data": {},
                "error": str(
                    payload.get(
                        "message",
                        "",
                    )
                )[:500],
            }

        return {
            "ok": False,
            "blocked_412": False,
            "status": (
                "bilibili_api_code_"
                + str(api_code)
            ),
            "data": {},
            "error": str(
                payload.get(
                    "message",
                    "",
                )
            )[:500],
        }

    return {
        "ok": True,
        "blocked_412": False,
        "status": "bilibili_api_ok",
        "data": payload.get(
            "data",
            {},
        ),
        "error": "",
    }


def _bilibili_view_metadata(
    bvid,
):
    bvid = str(
        bvid or ""
    ).strip()

    if not bvid:
        return {
            "ok": False,
            "status": "missing_bvid",
            "data": {},
            "error": "",
        }

    referer = canonical_bilibili_url(
        bvid
    )

    return _bilibili_api_get(
        _BILIBILI_VIEW_API,
        {
            "bvid": bvid,
        },
        referer,
    )


def _empty_bilibili_result(
    status,
    error="",
):
    return {
        "bilibili_url": "",
        "bilibili_video_id": "",
        "bilibili_available_status": status,
        "bilibili_match_status": status,
        "bilibili_match_score": "",
        "bilibili_title_score": "",
        "bilibili_artist_score": "",
        "bilibili_matched_title": "",
        "bilibili_matched_uploader": "",
        "bilibili_query_used": "",
        "bilibili_search_error": str(
            error or ""
        )[:1500],
    }


def search_bilibili_replacement_with_ytdlp(
    title,
    artist,
):
    """
    Name retained so the existing Cell 8B processor does not need
    to change. Search itself now uses the Bilibili public API.

    The function returns "bilibili_match_found_ytdlp" on success
    because the existing main processor expects that exact value.
    The source is recorded in bilibili_available_status.
    """

    global _BILIBILI_412_BLOCKED

    if _BILIBILI_412_BLOCKED:
        return _empty_bilibili_result(
            "bilibili_blocked_412_retry_later",
            "Bilibili circuit breaker active.",
        )

    best = None
    best_score = -1.0
    collected_errors = []

    # Three strong queries instead of six repeated blocked calls.
    queries = bilibili_query_variants(
        title,
        artist,
    )[:3]

    for query in queries:
        referer = (
            "https://search.bilibili.com/all"
            "?keyword="
            + _quote_plus(query)
        )

        api_result = _bilibili_api_get(
            _BILIBILI_SEARCH_API,
            {
                "search_type": "video",
                "keyword": query,
                "order": "totalrank",
                "duration": 0,
                "tids": 0,
                "page": 1,
                "page_size": _BILIBILI_MAX_RESULTS,
            },
            referer,
        )

        if not api_result["ok"]:
            collected_errors.append(
                api_result["status"]
                + ":"
                + api_result["error"]
            )

            if api_result[
                "blocked_412"
            ]:
                return _empty_bilibili_result(
                    api_result["status"],
                    api_result["error"],
                )

            continue

        data = api_result.get(
            "data",
            {},
        )

        entries = (
            data.get(
                "result",
                [],
            )
            or []
        )

        for entry in entries:
            if not isinstance(
                entry,
                dict,
            ):
                continue

            bvid = str(
                entry.get(
                    "bvid",
                    "",
                )
                or ""
            ).strip()

            if not _re.fullmatch(
                r"BV[A-Za-z0-9]{10}",
                bvid,
                flags=_re.IGNORECASE,
            ):
                continue

            found_title = (
                _bilibili_clean_text(
                    entry.get(
                        "title",
                        "",
                    )
                )
            )

            found_uploader = (
                _bilibili_clean_text(
                    entry.get(
                        "author",
                        entry.get(
                            "up_name",
                            "",
                        ),
                    )
                )
            )

            found_description = (
                _bilibili_clean_text(
                    entry.get(
                        "description",
                        "",
                    )
                )
            )

            if bilibili_result_has_wrong_language(
                found_title,
                found_uploader,
                found_description,
            ):
                continue

            duration_seconds = (
                _bilibili_parse_duration(
                    entry.get(
                        "duration",
                        "",
                    )
                )
            )

            if (
                duration_seconds is not None
                and not (
                    90
                    <= duration_seconds
                    <= 1200
                )
            ):
                continue

            scoring_input = {
                "title": found_title,
                "channel": found_uploader,
                "uploader": found_uploader,
                "artist": found_uploader,
                "description": found_description,
                "duration": (
                    duration_seconds
                    if duration_seconds
                    is not None
                    else 0
                ),
            }

            scoring = score_youtube_candidate(
                title,
                artist,
                scoring_input,
                "Chinese",
            )

            if not scoring[
                "accepted"
            ]:
                continue

            result_score = float(
                scoring[
                    "score"
                ]
            )

            # Prefer popular, likely stable results when match scores
            # are otherwise similar.
            try:
                play_count = float(
                    entry.get(
                        "play",
                        0,
                    )
                    or 0
                )

                if play_count >= 100000:
                    result_score += 0.03
                elif play_count >= 10000:
                    result_score += 0.015
            except Exception:
                pass

            if result_score > best_score:
                best_score = result_score

                best = {
                    "bilibili_url": (
                        canonical_bilibili_url(
                            bvid
                        )
                    ),
                    "bilibili_video_id": bvid,
                    "bilibili_available_status": (
                        "available_public_bilibili_search_api"
                    ),
                    "bilibili_match_status": (
                        "bilibili_match_found_ytdlp"
                    ),
                    "bilibili_match_score": round(
                        result_score,
                        4,
                    ),
                    "bilibili_title_score": scoring[
                        "title_score"
                    ],
                    "bilibili_artist_score": scoring[
                        "artist_score"
                    ],
                    "bilibili_matched_title": found_title,
                    "bilibili_matched_uploader": found_uploader,
                    "bilibili_query_used": query,
                    "bilibili_search_error": "",
                }

        if best is not None:
            # One successful query is sufficient. This reduces
            # Bilibili traffic and lowers 412 risk.
            break

        _time.sleep(
            _random.uniform(
                1.2,
                2.0,
            )
        )

    if best is None:
        if _BILIBILI_412_BLOCKED:
            return _empty_bilibili_result(
                "bilibili_blocked_412_retry_later",
                "Bilibili circuit breaker active.",
            )

        return _empty_bilibili_result(
            (
                "bilibili_api_no_acceptable_match"
                if not collected_errors
                else "bilibili_api_search_error"
            ),
            " | ".join(
                collected_errors
            ),
        )

    # Lightweight public metadata verification. A 412 during this
    # verification does not erase a valid public search result.
    view_result = _bilibili_view_metadata(
        best[
            "bilibili_video_id"
        ]
    )

    if view_result["ok"]:
        metadata = view_result.get(
            "data",
            {},
        )

        verified_title = (
            _bilibili_clean_text(
                metadata.get(
                    "title",
                    "",
                )
            )
        )

        owner = metadata.get(
            "owner",
            {},
        ) or {}

        verified_uploader = (
            _bilibili_clean_text(
                owner.get(
                    "name",
                    "",
                )
            )
        )

        verified_description = (
            _bilibili_clean_text(
                metadata.get(
                    "desc",
                    "",
                )
            )
        )

        if bilibili_result_has_wrong_language(
            verified_title,
            verified_uploader,
            verified_description,
        ):
            return _empty_bilibili_result(
                "bilibili_wrong_language_japanese_or_korean",
                "Conflicting Kana or Hangul in verified metadata.",
            )

        verified_duration = metadata.get(
            "duration",
            None,
        )

        try:
            verified_duration = float(
                verified_duration
            )

            if not (
                90
                <= verified_duration
                <= 1200
            ):
                return _empty_bilibili_result(
                    "bilibili_bad_duration",
                    str(verified_duration),
                )
        except Exception:
            pass

        if verified_title:
            best[
                "bilibili_matched_title"
            ] = verified_title

        if verified_uploader:
            best[
                "bilibili_matched_uploader"
            ] = verified_uploader

        best[
            "bilibili_available_status"
        ] = "available_public_bilibili_api_verified"

    elif view_result[
        "blocked_412"
    ]:
        # Search result remains usable as metadata; do not falsely
        # call it unavailable.
        best[
            "bilibili_available_status"
        ] = (
            "available_in_search_but_view_check_blocked_412"
        )

        best[
            "bilibili_search_error"
        ] = view_result[
            "error"
        ]

    else:
        best[
            "bilibili_available_status"
        ] = (
            "available_in_search_view_check_"
            + view_result[
                "status"
            ]
        )

        best[
            "bilibili_search_error"
        ] = view_result[
            "error"
        ]

    return best


print(
    "\nBILIBILI 412 HOTFIX INSTALLED."
)

print(
    "Bilibili search now uses the public API, "
    "with one-call 412 circuit breaking."
)

print(
    "Interrupt the old run, then rerun Chinese only with:"
)

print(
    'checked_results["Chinese"] = '
    'check_and_replace_bad_youtube_urls('
    '"chinese_human_with_youtube_urls.csv", "Chinese")'
)



# BILIBILI MATCHING HOTFIX V2
#
# Run this AFTER interrupting the curren.
#
# This overrides only the Bilibili fallback search. Existing
# YouTube behaviour remains unchanged.


import html as _b2_html
import math as _b2_math
import random as _b2_random
import re as _b2_re
import time as _b2_time
from collections import Counter as _B2Counter
from urllib.parse import quote_plus as _b2_quote_plus


# COMMON TRADITIONAL -> SIMPLIFIED NORMALISATION

_B2_TRADITIONAL = (
    "體國語樂來電無傷風愛夢聽見間時後與為開關歲點聲裡麼還說誰過這個們會"
    "東門長萬葉張劉楊靜吳絲驚陽陰雲飛鳥魚馬車書畫學習實現發現頭髮臺灣"
    "灣廣場華話劇錄錄製製作網絡線經歷歷史應該讓給對從將將軍尋找愛戀"
    "戀戀寶貝貓龍鳳氣歡歡樂難離別歸回轉變變化遠近進愛覺覺得記憶憶"
    "願願意選擇擇鐘鍾鍾情終終於於麼樣樣子裊裊牆牆角燈燈光餘余溫"
    "傷傷心淚淚水雙雙手單單純純真夢夢想島島嶼嶼海邊邊緣緣份"
    "舊舊日親親愛愛著著你妳她們們"
)

_B2_SIMPLIFIED = (
    "体国语乐来电无伤风爱梦听见间时后与为开关岁点声里么还说谁过这个们会"
    "东门长万叶张刘杨静吴丝惊阳阴云飞鸟鱼马车书画学习实现发现头发台湾"
    "湾广场华话剧录录制制作网络线经历历史应该让给对从将将军寻找爱恋"
    "恋恋宝贝猫龙凤气欢欢乐难离别归回转变变化远近进爱觉觉得记忆忆"
    "愿愿意选择择钟钟钟情终终于于么样样子袅袅墙墙角灯灯光余余温"
    "伤伤心泪泪水双双手单单纯纯真梦梦想岛岛屿屿海边边缘缘份"
    "旧旧日亲亲爱爱着着你你她们们"
)

_B2_TRANSLATION = str.maketrans(
    {
        traditional: simplified
        for traditional, simplified in zip(
            _B2_TRADITIONAL,
            _B2_SIMPLIFIED,
        )
    }
)

# Additional common name/orthographic variants.
_B2_TRANSLATION.update(
    str.maketrans(
        {
            "峯": "峰",
            "妳": "你",
            "裏": "里",
            "着": "着",
            "著": "着",
            "祇": "只",
            "隻": "只",
            "佢": "他",
        }
    )
)


def _b2_to_simplified(value):
    return str(value or "").translate(
        _B2_TRANSLATION
    )


def _b2_clean_visible_text(value):
    text = _b2_html.unescape(
        str(value or "")
    )

    text = _b2_re.sub(
        r"<[^>]+>",
        " ",
        text,
    )

    return _b2_re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


_B2_NOISE_MARKERS = [
    "official music video",
    "official video",
    "official audio",
    "official mv",
    "music video",
    "lyrics video",
    "lyric video",
    "lyrics",
    "lyric",
    "官方完整版",
    "官方版",
    "官方mv",
    "官方 mv",
    "官方音源",
    "官方音频",
    "官方",
    "完整版",
    "高音质",
    "无损",
    "無損",
    "hi res",
    "hi-res",
    "4k",
    "1080p",
    "mv",
    "audio",
    "video",
]


def _b2_normalise(value):
    text = _b2_clean_visible_text(
        value
    ).casefold()

    text = _b2_to_simplified(
        text
    )

    for marker in _B2_NOISE_MARKERS:
        text = text.replace(
            marker,
            " ",
        )

    text = _b2_re.sub(
        r"[^a-z0-9"
        r"\u3400-\u4dbf\u4e00-\u9fff"
        r"\u3040-\u30ff\u31f0-\u31ff"
        r"\u1100-\u11ff\uac00-\ud7af]+",
        " ",
        text,
    )

    return _b2_re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def _b2_compact(value):
    return _b2_re.sub(
        r"\s+",
        "",
        _b2_normalise(value),
    )


def _b2_title_forms(value):
    raw = _b2_clean_visible_text(
        value
    )

    forms = {
        _b2_compact(raw),
    }

    # Base title without version/album notes.
    without_round = _b2_re.sub(
        r"\([^)]*\)",
        " ",
        raw,
    )

    forms.add(
        _b2_compact(
            without_round
        )
    )

    # Keep bracketed subtitle as a second form, but also try the
    # title before it.
    before_bracket = _b2_re.split(
        r"[\[【（(]",
        raw,
        maxsplit=1,
    )[0]

    forms.add(
        _b2_compact(
            before_bracket
        )
    )

    return {
        form
        for form in forms
        if form
    }


def _b2_multiset_recall(
    wanted,
    found,
):
    wanted = _b2_compact(
        wanted
    )

    found = _b2_compact(
        found
    )

    if not wanted or not found:
        return 0.0

    wanted_counter = _B2Counter(
        wanted
    )

    found_counter = _B2Counter(
        found
    )

    matched = sum(
        min(
            count,
            found_counter.get(
                character,
                0,
            ),
        )
        for character, count
        in wanted_counter.items()
    )

    return matched / max(
        1,
        sum(
            wanted_counter.values()
        ),
    )


def _b2_required_version_markers(
    source_title,
):
    text = _b2_normalise(
        source_title
    )

    required = set()

    groups = {
        "acoustic": [
            "acoustic",
            "unplugged",
            "不插电",
            "原声版",
            "木吉他版",
        ],
        "live": [
            "live",
            "现场",
            "演唱会",
        ],
        "remix": [
            "remix",
            "混音",
        ],
        "instrumental": [
            "instrumental",
            "伴奏",
            "纯音乐",
        ],
        "karaoke": [
            "karaoke",
            "卡拉ok",
            "ktv",
        ],
    }

    for group, markers in groups.items():
        if any(
            marker in text
            for marker in markers
        ):
            required.add(
                group
            )

    return required


def _b2_result_has_required_versions(
    source_title,
    result_text,
):
    required = _b2_required_version_markers(
        source_title
    )

    if not required:
        return True, ""

    result = _b2_normalise(
        result_text
    )

    alternatives = {
        "acoustic": [
            "acoustic",
            "unplugged",
            "不插电",
            "原声",
            "木吉他",
        ],
        "live": [
            "live",
            "现场",
            "演唱会",
        ],
        "remix": [
            "remix",
            "混音",
        ],
        "instrumental": [
            "instrumental",
            "伴奏",
            "纯音乐",
        ],
        "karaoke": [
            "karaoke",
            "卡拉ok",
            "ktv",
        ],
    }

    for group in required:
        if not any(
            marker in result
            for marker in alternatives[
                group
            ]
        ):
            return (
                False,
                "missing_required_version_"
                + group,
            )

    return True, ""


def _b2_artist_parts(value):
    raw = _b2_clean_visible_text(
        value
    )

    parts = _b2_re.split(
        r"\s*(?:,|&|＆|/|\+|、|"
        r"\bfeat\.?\b|\bft\.?\b|"
        r"\bfeaturing\b|\bx\b)\s*",
        raw,
        flags=_b2_re.I,
    )

    output = []

    for part in parts:
        compact = _b2_compact(
            part
        )

        if compact:
            output.append(
                compact
            )

    return output


def _b2_artist_evidence(
    source_artist,
    result_title,
    uploader,
    description,
):
    combined = _b2_compact(
        " ".join(
            [
                result_title,
                uploader,
                description,
            ]
        )
    )

    artist_parts = _b2_artist_parts(
        source_artist
    )

    if not artist_parts:
        return 0.0, 0, 0

    hits = 0
    best_recall = 0.0

    for artist_part in artist_parts:
        if artist_part in combined:
            hits += 1
            best_recall = max(
                best_recall,
                1.0,
            )
            continue

        recall = _b2_multiset_recall(
            artist_part,
            combined,
        )

        best_recall = max(
            best_recall,
            recall,
        )

        if (
            len(artist_part) >= 2
            and recall >= 0.80
        ):
            hits += 1

    score = max(
        best_recall,
        hits / max(
            1,
            len(
                artist_parts
            ),
        ),
    )

    return (
        score,
        hits,
        len(
            artist_parts
        ),
    )


def _b2_title_score(
    source_title,
    result_title,
):
    source_forms = _b2_title_forms(
        source_title
    )

    result_compact = _b2_compact(
        result_title
    )

    if not source_forms or not result_compact:
        return 0.0, "empty_title"

    best_score = 0.0
    best_reason = "low_title_similarity"

    for source_form in source_forms:
        if (
            source_form in result_compact
            or result_compact in source_form
        ):
            # Do not allow an extremely short generic title to pass
            # on containment alone; artist evidence will still be
            # required later.
            containment_score = (
                1.0
                if len(source_form) >= 3
                else 0.88
            )

            if containment_score > best_score:
                best_score = containment_score
                best_reason = "title_containment"

        recall = _b2_multiset_recall(
            source_form,
            result_compact,
        )

        if recall > best_score:
            best_score = recall
            best_reason = (
                "title_character_recall"
            )

        sequence = similarity(
            source_form,
            result_compact,
        )

        if sequence > best_score:
            best_score = sequence
            best_reason = (
                "title_sequence_similarity"
            )

    return (
        round(
            min(
                1.0,
                best_score,
            ),
            4,
        ),
        best_reason,
    )


def _b2_score_candidate(
    source_title,
    source_artist,
    result_title,
    uploader,
    description,
    duration_seconds,
):
    combined_result = " ".join(
        [
            result_title,
            uploader,
            description,
        ]
    )

    if bilibili_result_has_wrong_language(
        result_title,
        uploader,
        description,
    ):
        return {
            "accepted": False,
            "reason": "wrong_language_kana_or_hangul",
            "total_score": 0.0,
            "title_score": 0.0,
            "artist_score": 0.0,
        }

    bad_marker = contains_disallowed_version(
        source_title,
        combined_result,
    )

    if bad_marker:
        return {
            "accepted": False,
            "reason": (
                "bad_version_"
                + normalise_text(
                    bad_marker
                ).replace(
                    " ",
                    "_",
                )
            ),
            "total_score": 0.0,
            "title_score": 0.0,
            "artist_score": 0.0,
        }

    version_ok, version_reason = (
        _b2_result_has_required_versions(
            source_title,
            combined_result,
        )
    )

    if not version_ok:
        return {
            "accepted": False,
            "reason": version_reason,
            "total_score": 0.0,
            "title_score": 0.0,
            "artist_score": 0.0,
        }

    title_score, title_reason = (
        _b2_title_score(
            source_title,
            result_title,
        )
    )

    (
        artist_score,
        artist_hits,
        artist_count,
    ) = _b2_artist_evidence(
        source_artist,
        result_title,
        uploader,
        description,
    )

    source_base_length = max(
        [
            len(form)
            for form in _b2_title_forms(
                source_title
            )
        ]
        or [0]
    )

    # Strong title + at least one artist match is the main rule.
    strong_title_artist = (
        title_score >= 0.72
        and (
            artist_hits >= 1
            or artist_score >= 0.72
        )
    )

    # Exact/contained distinctive title can pass with weaker artist
    # evidence because Bilibili uploaders are often labels or fan
    # archival channels rather than the performing artist.
    distinctive_title = (
        title_score >= 0.94
        and source_base_length >= 4
        and artist_score >= 0.35
    )

    # Short/common titles always require explicit artist evidence.
    short_title_safe = (
        source_base_length <= 3
        and (
            artist_hits >= 1
            or artist_score >= 0.80
        )
        and title_score >= 0.80
    )

    accepted = (
        strong_title_artist
        or distinctive_title
        or short_title_safe
    )

    total_score = (
        title_score * 0.78
        + artist_score * 0.22
    )

    if duration_seconds is not None:
        if 150 <= duration_seconds <= 480:
            total_score += 0.03
        elif duration_seconds < 90 or duration_seconds > 1200:
            accepted = False

    return {
        "accepted": accepted,
        "reason": (
            "accepted_"
            + title_reason
            if accepted
            else (
                "below_safe_threshold_"
                + title_reason
            )
        ),
        "total_score": round(
            total_score,
            4,
        ),
        "title_score": title_score,
        "artist_score": round(
            artist_score,
            4,
        ),
        "artist_hits": artist_hits,
        "artist_count": artist_count,
    }


def _b2_unquoted_queries(
    title,
    artist,
):
    title = _b2_clean_visible_text(
        title
    )

    artist = _b2_clean_visible_text(
        artist
    )

    first_artist = primary_artist(
        artist
    )

    raw_queries = [
        f"{title} {artist}",
        f"{artist} {title}",
        f"{title} {first_artist}",
        f"{first_artist} {title}",
        title,
    ]

    output = []
    seen = set()

    for query in raw_queries:
        query = _b2_re.sub(
            r"\s+",
            " ",
            query,
        ).strip()

        key = _b2_normalise(
            query
        )

        if query and key not in seen:
            seen.add(
                key
            )
            output.append(
                query
            )

    return output


def search_bilibili_replacement_with_ytdlp(
    title,
    artist,
):
    """
    Compatibility name retained. Search uses the Bilibili public API.
    """

    global _BILIBILI_412_BLOCKED

    if _BILIBILI_412_BLOCKED:
        return _empty_bilibili_result(
            "bilibili_blocked_412_retry_later",
            "Bilibili circuit breaker active.",
        )

    best = None
    best_score = -1.0
    collected_errors = []
    diagnostics = []
    total_entries_seen = 0

    queries = _b2_unquoted_queries(
        title,
        artist,
    )

    for query in queries:
        referer = (
            "https://search.bilibili.com/all"
            "?keyword="
            + _b2_quote_plus(
                query
            )
        )

        api_result = _bilibili_api_get(
            _BILIBILI_SEARCH_API,
            {
                "search_type": "video",
                "keyword": query,
                "order": "totalrank",
                "duration": 0,
                "tids": 0,
                "page": 1,
                "page_size": 30,
            },
            referer,
        )

        if not api_result["ok"]:
            collected_errors.append(
                api_result["status"]
                + ":"
                + api_result["error"]
            )

            if api_result[
                "blocked_412"
            ]:
                return _empty_bilibili_result(
                    api_result["status"],
                    api_result["error"],
                )

            continue

        entries = (
            api_result.get(
                "data",
                {},
            ).get(
                "result",
                [],
            )
            or []
        )

        total_entries_seen += len(
            entries
        )

        for entry in entries:
            if not isinstance(
                entry,
                dict,
            ):
                continue

            bvid = str(
                entry.get(
                    "bvid",
                    "",
                )
                or ""
            ).strip()

            if not _b2_re.fullmatch(
                r"BV[A-Za-z0-9]{10}",
                bvid,
                flags=_b2_re.I,
            ):
                continue

            found_title = (
                _b2_clean_visible_text(
                    entry.get(
                        "title",
                        "",
                    )
                )
            )

            uploader = (
                _b2_clean_visible_text(
                    entry.get(
                        "author",
                        entry.get(
                            "up_name",
                            "",
                        ),
                    )
                )
            )

            description = (
                _b2_clean_visible_text(
                    entry.get(
                        "description",
                        "",
                    )
                )
            )

            duration_seconds = (
                _bilibili_parse_duration(
                    entry.get(
                        "duration",
                        "",
                    )
                )
            )

            scoring = _b2_score_candidate(
                title,
                artist,
                found_title,
                uploader,
                description,
                duration_seconds,
            )

            diagnostics.append(
                {
                    "query": query,
                    "bvid": bvid,
                    "title": found_title,
                    "uploader": uploader,
                    **scoring,
                }
            )

            if not scoring[
                "accepted"
            ]:
                continue

            result_score = float(
                scoring[
                    "total_score"
                ]
            )

            try:
                play_count = float(
                    entry.get(
                        "play",
                        0,
                    )
                    or 0
                )

                if play_count >= 100000:
                    result_score += 0.025
                elif play_count >= 10000:
                    result_score += 0.01
            except Exception:
                pass

            if result_score > best_score:
                best_score = result_score

                best = {
                    "bilibili_url": (
                        canonical_bilibili_url(
                            bvid
                        )
                    ),
                    "bilibili_video_id": bvid,
                    "bilibili_available_status": (
                        "available_public_bilibili_search_api_v2"
                    ),
                    "bilibili_match_status": (
                        "bilibili_match_found_ytdlp"
                    ),
                    "bilibili_match_score": round(
                        result_score,
                        4,
                    ),
                    "bilibili_title_score": scoring[
                        "title_score"
                    ],
                    "bilibili_artist_score": scoring[
                        "artist_score"
                    ],
                    "bilibili_matched_title": found_title,
                    "bilibili_matched_uploader": uploader,
                    "bilibili_query_used": query,
                    "bilibili_search_error": "",
                }

        if best is not None:
            break

        _b2_time.sleep(
            _b2_random.uniform(
                0.8,
                1.4,
            )
        )

    if best is None:
        ranked = sorted(
            diagnostics,
            key=lambda item: (
                item.get(
                    "title_score",
                    0,
                ),
                item.get(
                    "artist_score",
                    0,
                ),
            ),
            reverse=True,
        )[:3]

        debug_parts = []

        for item in ranked:
            debug_parts.append(
                (
                    f"{item['title'][:55]}"
                    f" | uploader={item['uploader'][:25]}"
                    f" | title={item['title_score']}"
                    f" | artist={item['artist_score']}"
                    f" | {item['reason']}"
                )
            )

        debug_text = " || ".join(
            debug_parts
        )

        print(
            f"BILIBILI DEBUG | {title} - {artist} | "
            f"API entries={total_entries_seen} | "
            f"{debug_text or 'no API result rows'}"
        )

        if _BILIBILI_412_BLOCKED:
            return _empty_bilibili_result(
                "bilibili_blocked_412_retry_later",
                "Bilibili circuit breaker active.",
            )

        return _empty_bilibili_result(
            (
                "bilibili_api_no_acceptable_match_v2"
                if not collected_errors
                else "bilibili_api_search_error_v2"
            ),
            (
                " | ".join(
                    collected_errors
                )
                + (
                    " | TOP_CANDIDATES: "
                    + debug_text
                    if debug_text
                    else ""
                )
            ),
        )

    # Verify public metadata through the lightweight view endpoint.
    view_result = _bilibili_view_metadata(
        best[
            "bilibili_video_id"
        ]
    )

    if view_result["ok"]:
        metadata = view_result.get(
            "data",
            {},
        )

        verified_title = (
            _b2_clean_visible_text(
                metadata.get(
                    "title",
                    "",
                )
            )
        )

        owner = (
            metadata.get(
                "owner",
                {},
            )
            or {}
        )

        verified_uploader = (
            _b2_clean_visible_text(
                owner.get(
                    "name",
                    "",
                )
            )
        )

        verified_description = (
            _b2_clean_visible_text(
                metadata.get(
                    "desc",
                    "",
                )
            )
        )

        verified_duration = (
            metadata.get(
                "duration",
                None,
            )
        )

        verification_score = (
            _b2_score_candidate(
                title,
                artist,
                verified_title
                or best[
                    "bilibili_matched_title"
                ],
                verified_uploader
                or best[
                    "bilibili_matched_uploader"
                ],
                verified_description,
                verified_duration,
            )
        )

        if not verification_score[
            "accepted"
        ]:
            return _empty_bilibili_result(
                "bilibili_view_metadata_match_rejected",
                (
                    verification_score[
                        "reason"
                    ]
                    + f" | title_score="
                    + str(
                        verification_score[
                            "title_score"
                        ]
                    )
                    + f" | artist_score="
                    + str(
                        verification_score[
                            "artist_score"
                        ]
                    )
                ),
            )

        if verified_title:
            best[
                "bilibili_matched_title"
            ] = verified_title

        if verified_uploader:
            best[
                "bilibili_matched_uploader"
            ] = verified_uploader

        best[
            "bilibili_available_status"
        ] = (
            "available_public_bilibili_api_verified_v2"
        )

    elif view_result[
        "blocked_412"
    ]:
        best[
            "bilibili_available_status"
        ] = (
            "available_in_search_view_check_blocked_412"
        )

        best[
            "bilibili_search_error"
        ] = view_result[
            "error"
        ]

    else:
        # A public search result is still retained, but the exact
        # verification failure is recorded.
        best[
            "bilibili_available_status"
        ] = (
            "available_in_search_view_check_"
            + view_result[
                "status"
            ]
        )

        best[
            "bilibili_search_error"
        ] = view_result[
            "error"
        ]

    print(
        f"BILIBILI MATCH | {title} - {artist} | "
        f"{best['bilibili_matched_title']} | "
        f"title={best['bilibili_title_score']} | "
        f"artist={best['bilibili_artist_score']} | "
        f"{best['bilibili_url']}"
    )

    return best


# Reset only the matching/search circuit state for the new run.
_BILIBILII_MATCHING_HOTFIX_V2 = True

print(
    "\nBILIBILI MATCHING HOTFIX V2 INSTALLED."
)

print(
    "Search now uses unquoted queries and "
    "Simplified/Traditional-aware title matching."
)

print(
    "Interrupt the current run, then rerun Chinese only:"
)

print(
    'checked_results["Chinese"] = '
    'check_and_replace_bad_youtube_urls('
    '"chinese_human_with_youtube_urls.csv", "Chinese")'
)



# RUN SELECTED LANGUAGES

MEDIA_LANGUAGES_TO_RUN = globals().get(
    "MEDIA_LANGUAGES_TO_RUN",
    ["English", "Hindi", "Chinese"],
)

checked_results = {}

for language, csv_file in LANGUAGE_YOUTUBE_FILES:
    if language not in MEDIA_LANGUAGES_TO_RUN:
        print(
            f"Skipping {language}: not selected in "
            "MEDIA_LANGUAGES_TO_RUN."
        )
        continue

    checked_df = check_and_replace_bad_youtube_urls(
        csv_file,
        language,
    )

    if checked_df is not None:
        checked_results[language] = checked_df

print("\nSELECTED MEDIA URL CHECKS COMPLETED.")


In [ ]:

# AUTOMATIC CHINESE FAILED ROW REPLACEMENT
# Same genre replacement + no duplicate + public YouTube URL check
# Metadata only, no download
# Does NOT need find_best_youtube_match()

import pandas as pd
import requests
import re
import time
from tqdm.auto import tqdm

YOUTUBE_SEARCH_URL = "https://www.googleapis.com/youtube/v3/search"
YOUTUBE_VIDEOS_URL = "https://www.googleapis.com/youtube/v3/videos"


# BASIC HELPERS

def norm_song_text(x):
    x = str(x).lower().strip()
    x = re.sub(r"\(.*?\)|\[.*?\]", " ", x)
    x = re.sub(
        r"official|video|audio|lyrics|lyric|remastered|hd|hq|topic|full song|mv",
        " ",
        x
    )
    x = re.sub(r"[^a-z0-9\u4e00-\u9fff\s]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def make_song_key(title, artist):
    return norm_song_text(title) + " | " + norm_song_text(artist)


def extract_video_id(url):
    url = str(url).strip()

    if url.lower() in ["", "nan", "none", "null"]:
        return ""

    if "watch?v=" in url:
        return url.split("watch?v=")[-1].split("&")[0]

    if "youtu.be/" in url:
        return url.split("youtu.be/")[-1].split("?")[0].split("&")[0]

    if "/shorts/" in url:
        return url.split("/shorts/")[-1].split("?")[0].split("&")[0]

    if "/embed/" in url:
        return url.split("/embed/")[-1].split("?")[0].split("&")[0]

    return ""


def is_empty_url(url):
    return str(url).strip().lower() in ["", "nan", "none", "null", ""]


def has_chinese_text(x):
    return bool(re.search(r"[\u4e00-\u9fff]", str(x)))


# CHECK PUBLIC YOUTUBE VIDEO

def check_youtube_video_available(video_id):
    if "YOUTUBE_API_KEY" not in globals() or not YOUTUBE_API_KEY:
        return False, "missing_api_key"

    if not video_id:
        return False, "missing_video_id"

    try:
        r = requests.get(
            YOUTUBE_VIDEOS_URL,
            params={
                "part": "snippet,status",
                "id": video_id,
                "key": YOUTUBE_API_KEY,
            },
            timeout=15
        )

        if r.status_code in [400, 401, 403, 429]:
            return False, f"stop_youtube_api_or_quota_error_{r.status_code}"

        if r.status_code != 200:
            return False, f"youtube_api_error_{r.status_code}"

        items = r.json().get("items", [])

        if len(items) == 0:
            return False, "video_unavailable_or_deleted"

        privacy_status = items[0].get("status", {}).get("privacyStatus", "")

        if privacy_status != "public":
            return False, f"not_public_{privacy_status}"

        return True, "available_public"

    except Exception as e:
        return False, f"request_failed_{type(e).__name__}"



# OWN YOUTUBE SEARCH FUNCTION

def youtube_search_public_url_for_song(title, artist):
    """
    Finds a public YouTube URL for title + artist.
    Metadata only. No download.
    """

    if "YOUTUBE_API_KEY" not in globals() or not YOUTUBE_API_KEY:
        return "", "", "missing_api_key"

    title_norm = norm_song_text(title)
    artist_norm = norm_song_text(artist)

    title_tokens = [t for t in title_norm.split() if len(t) > 1]
    artist_tokens = [t for t in artist_norm.split() if len(t) > 1]

    queries = [
        f'{artist} "{title}" official audio',
        f'{artist} "{title}" official video',
        f'{artist} "{title}" topic',
        f'{artist} "{title}" audio',
        f'{artist} "{title}"',
        f'"{title}" "{artist}"',
    ]

    best = None
    best_score = -999

    for q in queries:
        try:
            r = requests.get(
                YOUTUBE_SEARCH_URL,
                params={
                    "part": "snippet",
                    "q": q,
                    "type": "video",
                    "videoCategoryId": "10",
                    "maxResults": 10,
                    "key": YOUTUBE_API_KEY,
                },
                timeout=15
            )

            time.sleep(0.15)

            if r.status_code in [400, 401, 403, 429]:
                return "", "", f"stop_youtube_api_or_quota_error_{r.status_code}"

            if r.status_code != 200:
                continue

            for item in r.json().get("items", []):
                vid = item.get("id", {}).get("videoId", "")
                snip = item.get("snippet", {})

                yt_title = snip.get("title", "")
                channel = snip.get("channelTitle", "")

                combined = norm_song_text(yt_title + " " + channel)

                title_hits = sum(1 for t in title_tokens if t in combined)
                artist_hits = sum(1 for t in artist_tokens if t in combined)

                # At least title should match
                if title_tokens and title_hits < 1:
                    continue

                score = 0
                score += title_hits * 6
                score += artist_hits * 4

                if "official" in combined:
                    score += 5

                if "audio" in combined:
                    score += 4

                if "video" in combined:
                    score += 2

                if "topic" in combined:
                    score += 4

                # Avoid wrong versions
                bad_words = [
                    "cover",
                    "karaoke",
                    "instrumental",
                    "reaction",
                    "tutorial",
                    "piano",
                    "guitar",
                    "remix",
                ]

                for bad in bad_words:
                    if bad in combined:
                        score -= 20

                if "live" in combined:
                    score -= 5

                ok, available_status = check_youtube_video_available(vid)

                if str(available_status).startswith("stop_youtube_api_or_quota_error"):
                    return "", "", available_status

                if not ok:
                    continue

                if score > best_score:
                    best_score = score
                    best = {
                        "url": f"https://www.youtube.com/watch?v={vid}",
                        "video_id": vid,
                        "status": f"youtube_search_score_{score}",
                        "yt_title": yt_title,
                        "channel": channel,
                    }

        except Exception:
            continue

    if best:
        return best["url"], best["video_id"], best["status"]

    return "", "", "no_public_youtube_url_found"


# LOAD CHINESE REPLACEMENT POOL

def load_chinese_replacement_pool():
    possible_files = [
        MANIFEST_DIR / "chinese_strict_verified_live.csv",
        MANIFEST_DIR / "chinese_strict_verified_candidates.csv",
        MANIFEST_DIR / "chinese_itunes_search_plan_candidates_topped_up.csv",
        MANIFEST_DIR / "chinese_itunes_search_plan_candidates.csv",
        MANIFEST_DIR / "chinese_genius_300_balanced.csv",
    ]

    for f in possible_files:
        if f.exists():
            print("Using Chinese pool:", f)
            return pd.read_csv(f)

    try:
        print("Using chinese_itunes_candidates variable from memory")
        return chinese_itunes_candidates.copy()

    except NameError:
        raise FileNotFoundError(
            "No Chinese replacement pool found. Need one of these files:\n"
            "chinese_strict_verified_live.csv\n"
            "chinese_strict_verified_candidates.csv\n"
            "chinese_itunes_search_plan_candidates_topped_up.csv\n"
            "or chinese_itunes_candidates variable."
        )


# MAIN AUTO REPLACEMENT FUNCTION

def auto_replace_chinese_failed_rows():
    manifest_csv = MANIFEST_DIR / "chinese_human_with_youtube_urls.csv"

    if not manifest_csv.exists():
        raise FileNotFoundError(f"Chinese manifest not found: {manifest_csv}")

    df = pd.read_csv(manifest_csv)
    pool = load_chinese_replacement_pool()

    safe_cols = [
        "youtube_url",
        "youtube_video_id",
        "youtube_available_status",
        "youtube_match_status",
        "download_status",
        "download_retry_note",
    ]

    for col in safe_cols:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].astype("object")

    for col in safe_cols:
        if col not in pool.columns:
            pool[col] = ""
        pool[col] = pool[col].astype("object")

    for col in ["title", "artist", "final_genre"]:
        if col not in df.columns:
            raise ValueError(f"Manifest missing column: {col}")

        if col not in pool.columns:
            raise ValueError(f"Pool missing column: {col}")

    # Create duplicate keys
    df["match_key"] = df.apply(
        lambda r: make_song_key(r.get("title", ""), r.get("artist", "")),
        axis=1
    )

    pool["match_key"] = pool.apply(
        lambda r: make_song_key(r.get("title", ""), r.get("artist", "")),
        axis=1
    )

    # Find failed/manual rows
    bad_idx = df[
        df["download_status"].astype(str).str.lower().str.contains(
            "failed|manual_review_needed|error|downloaderror|403|forbidden|unavailable",
            na=False
        )
        | df["download_retry_note"].astype(str).str.lower().str.contains(
            "manual|no_same_genre|hard_failure|no_valid|failed",
            na=False
        )
        | df["youtube_url"].astype(str).apply(is_empty_url)
    ].index.tolist()

    print("Chinese rows selected for automatic replacement:", len(bad_idx))

    if len(bad_idx) > 0:
        display(
            df.loc[
                bad_idx,
                [
                    "song_id",
                    "title",
                    "artist",
                    "final_genre",
                    "youtube_url",
                    "youtube_available_status",
                    "download_status",
                    "download_retry_note",
                ]
            ]
        )

    # Existing good songs, so replacement will not duplicate
    existing_keys = set(df.drop(index=bad_idx)["match_key"].astype(str))

    # Clean replacement pool
    pool = pool.drop_duplicates(subset=["match_key"]).copy()
    pool = pool[~pool["match_key"].isin(existing_keys)].copy()

    # Remove failed/manual pool rows
    pool = pool[
        ~pool["download_status"].astype(str).str.lower().str.contains(
            "failed|manual|error|downloaderror|403|forbidden|unavailable",
            na=False
        )
    ].copy()

    # Remove bad English nursery/disney rows
    bad_title_words = [
        "head shoulders",
        "knees and toes",
        "itsy bitsy",
        "twinkle twinkle",
        "little star",
        "under the sea",
        "part of your world",
        "make a man out of you",
        "we wish you",
        "merry christmas",
    ]

    pool["_title_norm"] = pool["title"].astype(str).str.lower()

    for w in bad_title_words:
        pool = pool[~pool["_title_norm"].str.contains(w, na=False)].copy()

    pool = pool.drop(columns=["_title_norm"], errors="ignore")

    # Prefer Chinese title candidates
    pool["_has_chinese"] = pool["title"].astype(str).apply(
        lambda x: 1 if has_chinese_text(x) else 0
    )

    replacement_log = []

    for idx in tqdm(bad_idx, desc="Auto replacing Chinese failed rows"):
        old_row = df.loc[idx].copy()
        old_song_id = old_row["song_id"]
        genre = old_row["final_genre"]

        candidates = pool[pool["final_genre"] == genre].copy()
        candidates = candidates.sort_values("_has_chinese", ascending=False)

        replaced = False

        for cand_i, cand in candidates.iterrows():
            title = cand.get("title", "")
            artist = cand.get("artist", "")

            new_url, new_vid, match_status = youtube_search_public_url_for_song(
                title,
                artist
            )

            if (
                "429" in str(match_status)
                or "quota" in str(match_status).lower()
                or "stop_youtube_api" in str(match_status).lower()
            ):
                print("\nYouTube API quota/error happened.")
                print("Run this same cell later. It will resume.")
                df.to_csv(manifest_csv, index=False, encoding="utf-8-sig")
                return df

            if not new_url or is_empty_url(new_url):
                continue

            ok, available_status = check_youtube_video_available(new_vid)

            if str(available_status).startswith("stop_youtube_api_or_quota_error"):
                print("\nYouTube API quota/error happened during video check.")
                df.to_csv(manifest_csv, index=False, encoding="utf-8-sig")
                return df

            if not ok:
                continue

            # Replace metadata but keep old song_id
            for col in df.columns:
                if col in cand.index and col not in ["song_id", "match_key"]:
                    df.at[idx, col] = cand[col]

            df.at[idx, "song_id"] = old_song_id
            df.at[idx, "language"] = "Chinese"
            df.at[idx, "language_code"] = "ZH"
            df.at[idx, "label"] = "Human"
            df.at[idx, "model"] = "Human"
            df.at[idx, "final_genre"] = genre

            df.at[idx, "youtube_url"] = new_url
            df.at[idx, "youtube_video_id"] = new_vid
            df.at[idx, "youtube_available_status"] = available_status
            df.at[idx, "youtube_match_status"] = "auto_song_replaced_" + str(match_status)

            # Reset status for downloader retry
            df.at[idx, "download_status"] = "song_replaced_ready"
            df.at[idx, "download_retry_note"] = "auto_same_genre_replacement_ready"

            new_key = make_song_key(df.at[idx, "title"], df.at[idx, "artist"])
            existing_keys.add(new_key)

            pool = pool[pool["match_key"] != cand["match_key"]].copy()

            replacement_log.append({
                "song_id": old_song_id,
                "old_title": old_row["title"],
                "old_artist": old_row["artist"],
                "new_title": df.at[idx, "title"],
                "new_artist": df.at[idx, "artist"],
                "genre": genre,
                "new_youtube_url": new_url,
                "match_status": match_status,
                "youtube_available_status": available_status,
            })

            print(
                f"REPLACED {old_song_id}: "
                f"{old_row['title']} -> {df.at[idx, 'title']}"
            )

            replaced = True
            break

        if not replaced:
            df.at[idx, "download_status"] = "manual_review_needed"
            df.at[idx, "download_retry_note"] = "auto_replacement_not_found"

            print(
                f"WARNING no automatic replacement found for "
                f"{old_song_id} | {old_row['title']} | {genre}"
            )

        # Save after every row
        df.drop(columns=["match_key"], errors="ignore").to_csv(
            manifest_csv,
            index=False,
            encoding="utf-8-sig"
        )

        time.sleep(0.1)

    df = df.drop(columns=["match_key"], errors="ignore")

    log_csv = MANIFEST_DIR / "chinese_auto_replacement_log.csv"
    pd.DataFrame(replacement_log).to_csv(log_csv, index=False, encoding="utf-8-sig")

    df.to_csv(manifest_csv, index=False, encoding="utf-8-sig")

    print("\nSaved updated Chinese manifest:", manifest_csv)
    print("Saved replacement log:", log_csv)

    print("\nYouTube availability:")
    print(df["youtube_available_status"].value_counts(dropna=False))

    print("\nDownload status:")
    print(df["download_status"].value_counts(dropna=False))

    print("\nDownload retry note:")
    print(df["download_retry_note"].value_counts(dropna=False))

    print("\nGenre counts:")
    print(df["final_genre"].value_counts(dropna=False))

    print("\nDuplicate title+artist:")
    print(df.duplicated(subset=["title", "artist"]).sum())

    return df


# RUN

chinese_fixed = auto_replace_chinese_failed_rows()

display(
    chinese_fixed[
        [
            "song_id",
            "title",
            "artist",
            "final_genre",
            "youtube_url",
            "youtube_available_status",
            "download_status",
            "download_retry_note",
            "youtube_match_status",
        ]
    ].tail(50)
)

## Automated Audio Acquisition

> **Notice:** This replaces the old placeholder. It uses `yt-dlp` to iterate through the datasets, extract the audio streams from the verified YouTube URLs, and convert them to standard `.mp3` files matching the `song_id`.

In [ ]:

# English + Hindi + Chinese
# Fixes: too many values to unpack
# Saves failed review CSV for each language
# Bilibili fallback for Chinese via yt-dlp
# v2: format-fallback retry + ytsearch 403 retry

import yt_dlp
import os
import time
import pandas as pd
from tqdm.auto import tqdm


# 1. ERROR CLASSIFIER

def classify_ytdlp_error(error_text):
    e = str(error_text).lower()

    page_level_403 = (
        "403" in e or "forbidden" in e
    ) and any(kw in e for kw in (
        "sign in", "private", "age", "restricted", "login", "account"
    ))
    if page_level_403:
        return "Failed: 403 Forbidden"

    if "unable to download video data" in e and "403" in e:
        return "Failed: DownloadError"

    if ("403" in e or "forbidden" in e) and "unable to download" in e:
        return "Failed: DownloadError"

    if "video unavailable" in e or "this video is unavailable" in e:
        return "Failed: Video unavailable"

    if "private video" in e:
        return "Failed: Private video"

    if "age-restricted" in e or "confirm your age" in e or "sign in" in e:
        return "Failed: Restricted video"

    if "copyright" in e:
        return "Failed: Copyright restricted"

    if "requested format is not available" in e or "format is not available" in e:
        return "Failed: Format unavailable"

    if "unable to download webpage" in e:
        return "Failed: Unable to download webpage"

    if "timed out" in e or "timeout" in e:
        return "Failed: Network timeout"

    if "412" in e:
        return "Failed: Bilibili 412 Risk Control"

    return "Failed: DownloadError"


# 2. CORE DOWNLOADER FUNCTION

def is_bilibili_url(url):
    url = str(url).strip()
    return "bilibili.com/video/" in url or url.startswith("https://b23.tv/")


def is_valid_url(url):
    """Accept YouTube and Bilibili URLs."""
    url = str(url).strip()
    return (
        url.startswith("https://www.youtube.com/watch?v=")
        or url.startswith("https://youtu.be/")
        or url.startswith("http://www.youtube.com/watch?v=")
        or url.startswith("http://youtu.be/")
        or url.startswith("https://music.youtube.com/watch?v=")
        or is_bilibili_url(url)
    )


is_valid_youtube_url = is_valid_url


# Format strings tried in order. The first is the normal preference.
# The second is a broader fallback that catches videos where the usual
# bestaudio stream ID is missing/blocked but other audio streams exist.
FORMAT_ATTEMPTS = [
    "bestaudio/best",
    "bestaudio*/best*",
]


def _download_audio_single_attempt(url, output_path, format_string, bilibili):
    ydl_opts = {
        "format": format_string,
        "outtmpl": str(output_path.with_suffix("")),
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "mp3",
                "preferredquality": "192",
            }
        ],
        "quiet": True,
        "noprogress": True,
        "no_warnings": True,
        "ignoreerrors": False,
        "retries": 2,
        "fragment_retries": 2,
        "socket_timeout": 20,
        "http_headers": {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/119.0.0.0 Safari/537.36"
            )
        },
        "geo_bypass": True,
    }

    if bilibili:
        bilibili_cookies = MANIFEST_DIR.parent / "bilibili_cookies.txt"
        if bilibili_cookies.exists():
            ydl_opts["cookiefile"] = str(bilibili_cookies)

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])

        if output_path.exists() and output_path.stat().st_size > 10_000:
            return "Downloaded"

        return "Failed: File not created"

    except Exception as e:
        return classify_ytdlp_error(str(e))


def download_audio(url, output_path):
    if not url or str(url).strip().lower() in ["nan", "none", "null", ""]:
        return "Failed: Missing URL"

    output_path.parent.mkdir(parents=True, exist_ok=True)

    bilibili = is_bilibili_url(str(url))

    last_status = "Failed: DownloadError"

    for attempt_index, format_string in enumerate(FORMAT_ATTEMPTS):
        status = _download_audio_single_attempt(
            url, output_path, format_string, bilibili
        )

        if status == "Downloaded":
            return status

        last_status = status

        # Only worth retrying with a different format string when the
        # failure was specifically a format-availability issue. Hard
        # failures (private, restricted, copyright, etc.) won't be fixed
        # by trying another format, so stop immediately.
        if status != "Failed: Format unavailable":
            return status

        if output_path.exists():
            try:
                os.remove(output_path)
            except Exception:
                pass

    return last_status


# 3. URL + RETRY HELPERS

def _yt_text_norm(t):
    import re as _re
    t = str(t).lower()
    t = _re.sub(r"[\(\[\{].*?[\)\]\}]", " ", t)
    t = _re.sub(r"[^a-z0-9 ]", " ", t)
    return " ".join(t.split())


def _yt_similarity(a, b):
    from difflib import SequenceMatcher
    a, b = _yt_text_norm(a), _yt_text_norm(b)
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


_BAD_MARKERS = (
    "live", "concert", "tour", "karaoke", "instrumental",
    "cover", "tribute", "parody", "remix", "extended", "piano version",
    "acoustic version", "nightcore", "slowed", "sped up",
)


def ytsearch_best_url(title, artist, n=5):
    """
    Search YouTube using yt-dlp's built-in ytsearch extractor.
    Retries once on a 403 from the search API itself, since this is
    usually a transient block rather than a permanent failure.
    """
    title  = str(title  or "").strip()
    artist = str(artist or "").strip()

    if not title or not artist:
        return "", "missing_title_or_artist"

    query = f"{title} {artist}"
    search_url = f"ytsearch{n}:{query}"

    ydl_opts = {
        "quiet": True,
        "noprogress": True,
        "no_warnings": True,
        "extract_flat": True,
        "skip_download": True,
        "socket_timeout": 15,
    }

    info = None
    last_error = ""

    for attempt in range(2):
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(search_url, download=False)
            last_error = ""
            break
        except Exception as e:
            last_error = str(e)
            if "403" in last_error and attempt == 0:
                time.sleep(5)
                continue
            return "", f"ytsearch_error_{type(e).__name__}"

    if last_error:
        return "", "ytsearch_error_after_retry"

    entries = (info or {}).get("entries") or []
    if not entries:
        return "", "ytsearch_no_entries"

    best_url   = ""
    best_score = -1.0

    for entry in entries:
        vid_id    = entry.get("id", "")
        vid_title = entry.get("title", "")
        channel   = entry.get("channel", "") or entry.get("uploader", "")

        if not vid_id:
            continue

        title_sim  = _yt_similarity(title,  vid_title)
        artist_sim = max(
            _yt_similarity(artist, vid_title),
            _yt_similarity(artist, channel),
        )

        score = title_sim * 0.72 + artist_sim * 0.28

        combined = _yt_text_norm(vid_title + " " + channel)
        if "official audio" in combined:
            score += 0.10
        if "official video" in combined:
            score += 0.07
        if "topic" in _yt_text_norm(channel):
            score += 0.06

        norm_title = _yt_text_norm(vid_title)
        norm_src   = _yt_text_norm(title)
        bad_marker = any(
            m in norm_title and m not in norm_src
            for m in _BAD_MARKERS
        )
        if bad_marker:
            score -= 0.35

        acceptable = (
            title_sim >= 0.48
            and (artist_sim >= 0.22 or score >= 0.70)
            and not bad_marker
        )

        if acceptable and score > best_score:
            best_score = score
            best_url   = f"https://www.youtube.com/watch?v={vid_id}"

    if not best_url:
        return "", "ytsearch_no_acceptable_match"

    return best_url, "ok"


def parse_youtube_match_result(result):
    if isinstance(result, tuple):
        url = result[0] if len(result) > 0 else ""
        status = result[2] if len(result) > 2 else "youtube_match_found"
        return str(url).strip(), str(status).strip()
    return str(result).strip(), "youtube_match_found"


def get_status_and_url(row, output_path, language=""):
    """
    ALWAYS returns 3 values:
      status, final_url, retry_note
    """

    old_url = str(row.get("youtube_url", "")).strip()

    status = download_audio(old_url, output_path)

    if status == "Downloaded":
        return status, old_url, "original_url_ok"

    hard_fail_words = [
        "403 forbidden",
        "private",
        "restricted",
        "copyright",
        "sign in",
        "video unavailable",
        "not available",
        "has been removed",
        "deleted",
    ]

    if any(w in status.lower() for w in hard_fail_words):
        if language == "Chinese":
            bilibili_fallback = str(row.get("bilibili_url", "")).strip()
            if bilibili_fallback and is_bilibili_url(bilibili_fallback):
                if output_path.exists():
                    try:
                        os.remove(output_path)
                    except Exception:
                        pass
                bili_status = download_audio(bilibili_fallback, output_path)
                if bili_status == "Downloaded":
                    return "Downloaded via Bilibili", bilibili_fallback, "bilibili_fallback_ok"
                return bili_status, bilibili_fallback, "bilibili_fallback_failed"

        return status, old_url, "hard_failure_manual_review"

    if language == "Chinese":
        bilibili_fallback = str(row.get("bilibili_url", "")).strip()
        if bilibili_fallback and is_bilibili_url(bilibili_fallback) and bilibili_fallback != old_url:
            if output_path.exists():
                try:
                    os.remove(output_path)
                except Exception:
                    pass
            bili_status = download_audio(bilibili_fallback, output_path)
            if bili_status == "Downloaded":
                return "Downloaded via Bilibili", bilibili_fallback, "bilibili_fallback_ok"

    new_url, search_note = ytsearch_best_url(
        row.get("title", ""),
        row.get("artist", ""),
    )

    if not new_url:
        return status, old_url, f"ytsearch_no_match_{search_note}"

    if new_url == old_url:
        return status, old_url, "replacement_same_as_old"

    if output_path.exists():
        try:
            os.remove(output_path)
        except Exception:
            pass

    retry_status = download_audio(new_url, output_path)

    if retry_status == "Downloaded":
        return "Downloaded after ytsearch", new_url, "ytsearch_ok"

    return retry_status, new_url, f"ytsearch_download_failed"


# 4. MAIN EXECUTION LOOP

LANGUAGE_FILES = [
    ("English", "english_human_with_youtube_urls.csv"),
    ("Hindi", "hindi_human_with_youtube_urls.csv"),
    ("Chinese", "chinese_human_with_youtube_urls.csv"),
]

for language, csv_file in LANGUAGE_FILES:

    csv_path = MANIFEST_DIR / csv_file

    if not csv_path.exists():
        print(f"\nSkipping {language}: file not found -> {csv_path}")
        continue

    print(f"\n--- Starting {language} Audio Collection ---")

    df = pd.read_csv(csv_path)

    for col in ("download_status", "download_retry_note", "youtube_url"):
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].astype(object)

    out_dir = AUDIO_DIRS[language]
    out_dir.mkdir(parents=True, exist_ok=True)

    stop_language = False

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Downloading {language}"):

        song_id = row.get("song_id", f"{language}_{idx+1:03d}")
        title = row.get("title", "")
        artist = row.get("artist", "")
        output_file = out_dir / f"{song_id}.mp3"
        row_no = idx + 1

        if output_file.exists() and output_file.stat().st_size > 10_000:
            df.at[idx, "download_status"] = "Already exists"
            df.at[idx, "download_retry_note"] = "file_already_exists"

            tqdm.write(
                f"{row_no}/300 | Already exists | {song_id} | {title} - {artist}"
            )

            continue

        try:
            status, final_url, retry_note = get_status_and_url(
                row, output_file, language=language
            )

            df.at[idx, "download_status"] = status
            df.at[idx, "download_retry_note"] = retry_note
            df.at[idx, "youtube_url"] = final_url

            df.to_csv(csv_path, index=False, encoding="utf-8-sig")

            tqdm.write(
                f"{row_no}/300 | {status} | {song_id} | {title} - {artist} | {retry_note}"
            )

            if "429" in str(retry_note) or "quota" in str(retry_note).lower():
                print("\nYouTube API quota/rate-limit hit. Stop now and run later.")
                df.to_csv(csv_path, index=False, encoding="utf-8-sig")
                stop_language = True
                break

            time.sleep(2.0)

        except Exception as e:
            error_msg = f"Failed: {str(e)}"

            tqdm.write(
                f"{row_no}/300 | Error: {str(e)} | {song_id} | {title} - {artist}"
            )

            df.at[idx, "download_status"] = error_msg
            df.at[idx, "download_retry_note"] = "exception_in_main_loop"

            df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    failed_df = df[
        ~df["download_status"].astype(str).str.contains(
            "Downloaded|Already exists",
            case=False,
            na=False,
        )
    ].copy()

    failed_csv = MANIFEST_DIR / f"{language.lower()}_download_failed_review.csv"
    failed_df.to_csv(failed_csv, index=False, encoding="utf-8-sig")

    print(f"\nManifest saved for {language}: {csv_path}")
    print(f"Failed review saved for {language}: {failed_csv}")

    print("\nDownload status:")
    print(df["download_status"].value_counts(dropna=False))

    print("\nRetry note:")
    print(df["download_retry_note"].value_counts(dropna=False))

    if stop_language:
        print(f"\nStopped during {language}. Run same cell later to resume.")
        break

print("\nAUDIO COLLECTION CELL FINISHED.")

## Audio Validation & Duration Analysis
Binds the successfully downloaded `.mp3` file paths back into the pandas dataframe and utilizes `Mutagen` to read hardware-level duration metrics.

In [ ]:
# AUDIO VALIDATION + PATHS + DURATIONS
# Loads manifests from CSV, so no variable NameError
# English + Hindi + Chinese

import pandas as pd
from pathlib import Path

try:
    from mutagen import File as MutagenFile
    MUTAGEN_AVAILABLE = True
except ImportError:
    MUTAGEN_AVAILABLE = False
    print("mutagen not installed. Run: !pip install mutagen")


# SETTINGS

LANGUAGE_FILES = [
    ("English", "english_human_with_youtube_urls.csv", "english_human_final_with_audio.csv"),
    ("Hindi", "hindi_human_with_youtube_urls.csv", "hindi_human_final_with_audio.csv"),
    ("Chinese", "chinese_human_with_youtube_urls.csv", "chinese_human_final_with_audio.csv"),
]


# FUNCTION: ATTACH PATHS + DURATIONS

def attach_paths_and_durations(df, audio_folder):
    df = df.copy()

    paths = []
    exists_list = []
    sizes = []
    durations = []
    readable_list = []

    for _, row in df.iterrows():
        song_id = str(row["song_id"]).strip()
        audio_path = audio_folder / f"{song_id}.mp3"

        paths.append(str(audio_path))

        if audio_path.exists():
            exists_list.append(True)
            sizes.append(audio_path.stat().st_size)

            if MUTAGEN_AVAILABLE:
                try:
                    audio = MutagenFile(str(audio_path))

                    if audio is not None and hasattr(audio, "info"):
                        durations.append(round(float(audio.info.length), 2))
                        readable_list.append(True)
                    else:
                        durations.append(None)
                        readable_list.append(False)

                except Exception:
                    durations.append(None)
                    readable_list.append(False)
            else:
                durations.append(None)
                readable_list.append(None)

        else:
            exists_list.append(False)
            sizes.append(0)
            durations.append(None)
            readable_list.append(False)

    df["audio_path"] = paths
    df["audio_exists"] = exists_list
    df["audio_file_size_bytes"] = sizes
    df["duration_seconds"] = durations
    df["audio_readable"] = readable_list

    # Review flag
    df["audio_validation_status"] = "PASS"

    df.loc[df["audio_exists"] == False, "audio_validation_status"] = "MISSING_FILE"
    df.loc[df["audio_file_size_bytes"] < 50000, "audio_validation_status"] = "FILE_TOO_SMALL"

    if MUTAGEN_AVAILABLE:
        df.loc[df["audio_readable"] == False, "audio_validation_status"] = "NOT_READABLE"

        df.loc[
            df["duration_seconds"].notna() & (df["duration_seconds"] < 30),
            "audio_validation_status"
        ] = "DURATION_TOO_SHORT"

        df.loc[
            df["duration_seconds"].notna() & (df["duration_seconds"] > 900),
            "audio_validation_status"
        ] = "DURATION_TOO_LONG"

    return df


# RUN FOR ALL LANGUAGES

final_dfs = {}

for language, input_csv, output_csv in LANGUAGE_FILES:
    manifest_path = MANIFEST_DIR / input_csv

    if not manifest_path.exists():
        print(f"\nSkipping {language}: manifest not found -> {manifest_path}")
        continue

    print("\n" + "=" * 80)
    print(f"VALIDATING {language}")
    print("=" * 80)

    df = pd.read_csv(manifest_path)
    audio_folder = AUDIO_DIRS[language]

    final_df = attach_paths_and_durations(df, audio_folder)

    final_dfs[language] = final_df

    output_path = MANIFEST_DIR / output_csv
    final_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    review_df = final_df[final_df["audio_validation_status"] != "PASS"].copy()
    review_path = MANIFEST_DIR / f"{language.lower()}_audio_validation_review.csv"
    review_df.to_csv(review_path, index=False, encoding="utf-8-sig")

    print("Rows:", len(final_df))
    print("Audio exists:", final_df["audio_exists"].sum(), "/", len(final_df))
    print("Validation:")
    print(final_df["audio_validation_status"].value_counts(dropna=False))

    print("\nGenre counts:")
    print(final_df["final_genre"].value_counts(dropna=False))

    print("\nDuration summary:")
    print(final_df["duration_seconds"].describe())

    print("\nSaved final CSV:", output_path)
    print("Saved review CSV:", review_path)

    if len(review_df) > 0:
        display(
            review_df[
                [
                    "song_id",
                    "title",
                    "artist",
                    "final_genre",
                    "audio_path",
                    "audio_exists",
                    "audio_file_size_bytes",
                    "duration_seconds",
                    "audio_validation_status",
                    "download_status",
                    "download_retry_note",
                ]
            ].head(50)
        )


# COMBINE ALL LANGUAGES

if len(final_dfs) > 0:
    all_final = pd.concat(final_dfs.values(), ignore_index=True)

    all_output_path = MANIFEST_DIR / "all_human_final_with_audio.csv"
    all_final.to_csv(all_output_path, index=False, encoding="utf-8-sig")

    all_review = all_final[all_final["audio_validation_status"] != "PASS"].copy()
    all_review_path = MANIFEST_DIR / "all_human_audio_validation_review.csv"
    all_review.to_csv(all_review_path, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print("ALL LANGUAGE FINAL SUMMARY")
    print("=" * 80)

    print("Total rows:", len(all_final))

    print("\nBy language:")
    print(all_final["language"].value_counts(dropna=False))

    print("\nValidation:")
    print(all_final["audio_validation_status"].value_counts(dropna=False))

    print("\nLanguage + Genre counts:")
    print(
        all_final.groupby(["language", "final_genre"])
        .size()
        .unstack(fill_value=0)
    )

    print("\nSaved combined final CSV:", all_output_path)
    print("Saved combined review CSV:", all_review_path)

    display(all_final.head())

In [ ]:

# Loading and preprocessing of the final human-song corpus


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

try:
    from mutagen import File as MutagenFile
    MUTAGEN_AVAILABLE = True
except ImportError:
    MUTAGEN_AVAILABLE = False
    print("mutagen not installed - duration will not be computed from audio files")


# Load the consolidated manifest if it exists, otherwise
# reconstruct it from the per-language manifests and attach
# durations by reading each audio file's metadata.

final_csv = MANIFEST_DIR / "all_human_final_with_audio.csv"

if final_csv.exists():
    print(f"Loading consolidated manifest: {final_csv}")
    df = pd.read_csv(final_csv)
else:
    print("Consolidated manifest not found - reconstructing from per-language manifests")

    language_files = [
        ("English", "english_human_with_youtube_urls.csv"),
        ("Hindi", "hindi_human_with_youtube_urls.csv"),
        ("Chinese", "chinese_human_with_youtube_urls.csv"),
    ]

    all_rows = []

    for language, csv_file in language_files:
        csv_path = MANIFEST_DIR / csv_file

        if not csv_path.exists():
            print(f"  Missing manifest for {language}: {csv_path}")
            continue

        temp = pd.read_csv(csv_path)
        temp["language"] = language

        audio_folder = AUDIO_DIRS[language]

        audio_paths = []
        durations = []

        for _, row in temp.iterrows():
            song_id = str(row["song_id"]).strip()
            audio_path = audio_folder / f"{song_id}.mp3"
            audio_paths.append(str(audio_path))

            if audio_path.exists() and MUTAGEN_AVAILABLE:
                try:
                    audio = MutagenFile(str(audio_path))
                    duration = (
                        round(float(audio.info.length), 2)
                        if audio and hasattr(audio, "info")
                        else np.nan
                    )
                except Exception:
                    duration = np.nan
            else:
                duration = np.nan

            durations.append(duration)

        temp["audio_path"] = audio_paths
        temp["duration_seconds"] = durations

        all_rows.append(temp)
        print(f"  {language}: {len(temp)} rows loaded")

    df = pd.concat(all_rows, ignore_index=True)


# Standardise the language and genre fields, and derive
# duration in minutes/hours for the descriptive statistics
# reported in the dissertation.

df["language"] = df["language"].replace({
    "English": "English",
    "Hindi": "Hindi",
    "Chinese": "Chinese",
    "EN": "English",
    "HI": "Hindi",
    "ZH": "Chinese",
})

df["final_genre"] = df["final_genre"].astype(str)
df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")
df["duration_minutes"] = df["duration_seconds"] / 60
df["duration_hours"] = df["duration_seconds"] / 3600

# Retain only rows belonging to the three target languages defined
# in the dissertation scope.
df = df[df["language"].isin(["English", "Hindi", "Chinese"])].copy()

GENRE_ORDER = ["Pop", "Rock", "Hip-Hop/Rap", "R&B/Soul", "Folk/Acoustic"]
LANG_ORDER = ["English", "Hindi", "Chinese"]

df["final_genre"] = pd.Categorical(df["final_genre"], categories=GENRE_ORDER, ordered=True)
df["language"] = pd.Categorical(df["language"], categories=LANG_ORDER, ordered=True)

print(f"\nRows in final corpus: {len(df)}")
df.head()

In [ ]:

# Descriptive summary: song count and total audio duration
# by language and genre
#
# These tables form the basis of the dataset composition figures
# reported in the Data Collection chapter.

count_table = (
    df.groupby(["language", "final_genre"], observed=False)
    .size()
    .reset_index(name="song_count")
)

hours_table = (
    df.groupby(["language", "final_genre"], observed=False)["duration_hours"]
    .sum()
    .reset_index(name="total_hours")
)

summary = count_table.merge(hours_table, on=["language", "final_genre"])
summary["total_hours"] = summary["total_hours"].round(2)

print("Song count and duration summary by language and genre:")
display(summary)

summary_csv = MANIFEST_DIR / "genre_song_count_and_hours_summary.csv"
summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
print(f"Summary table exported to: {summary_csv}")

# Pivoted views used by the visualisations in the following cells
pivot_counts = (
    summary.pivot(index="final_genre", columns="language", values="song_count")
    .reindex(GENRE_ORDER)
    .reindex(columns=LANG_ORDER)
)

pivot_hours = (
    summary.pivot(index="final_genre", columns="language", values="total_hours")
    .reindex(GENRE_ORDER)
    .reindex(columns=LANG_ORDER)
)

language_hours = (
    df.groupby("language", observed=False)["duration_hours"]
    .sum()
    .reindex(LANG_ORDER)
    .reset_index()
)
language_hours["duration_hours"] = language_hours["duration_hours"].round(2)

In [ ]:
# ============================================================
# CELL 10C
# Figure 1: song count by genre and language
# Figure 2: total audio duration by genre and language
#
# These two figures characterise the balance of the corpus across
# the five target genres and three languages, supporting the
# discussion of dataset composition and potential class imbalance.
# ============================================================

plt.rcParams.update({
    "figure.figsize": (12, 7),
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

LANG_COLORS = {
    "English": "#4C78A8",
    "Hindi": "#F58518",
    "Chinese": "#54A24B",
}


def add_bar_labels(ax, fmt="{:.0f}", padding=0.02):
    """Annotate each bar with its value, positioned just above the bar."""
    ymax = ax.get_ylim()[1]

    for bar in ax.patches:
        height = bar.get_height()

        if pd.isna(height) or height == 0:
            continue

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + ymax * padding,
            fmt.format(height),
            ha="center",
            va="bottom",
            fontsize=10,
        )


# ------------------------------------------------------------
# Figure 1: Song count by genre and language
# ------------------------------------------------------------

ax = pivot_counts.plot(
    kind="bar",
    width=0.78,
    color=[LANG_COLORS[l] for l in LANG_ORDER],
    edgecolor="white",
    linewidth=1,
)

ax.set_title("Figure 1: Number of Songs by Genre and Language", pad=16)
ax.set_xlabel("Genre")
ax.set_ylabel("Number of songs")
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right")
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.legend(title="Language", frameon=False)

add_bar_labels(ax, fmt="{:.0f}", padding=0.01)

plt.tight_layout()

count_png = MANIFEST_DIR / "visual_song_count_by_genre_language.png"
plt.savefig(count_png, bbox_inches="tight")
plt.show()

print(f"Figure saved: {count_png}")


# ------------------------------------------------------------
# Figure 2: Total audio duration (hours) by genre and language
# ------------------------------------------------------------

ax = pivot_hours.plot(
    kind="bar",
    width=0.78,
    color=[LANG_COLORS[l] for l in LANG_ORDER],
    edgecolor="white",
    linewidth=1,
)

ax.set_title("Figure 2: Total Audio Duration by Genre and Language", pad=16)
ax.set_xlabel("Genre")
ax.set_ylabel("Total duration (hours)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right")
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.legend(title="Language", frameon=False)

add_bar_labels(ax, fmt="{:.2f}", padding=0.01)

plt.tight_layout()

hours_png = MANIFEST_DIR / "visual_audio_hours_by_genre_language.png"
plt.savefig(hours_png, bbox_inches="tight")
plt.show()

print(f"Figure saved: {hours_png}")

In [ ]:
# ============================================================
# CELL 10D
# Figure 3: total audio duration by language
# Figure 4: stacked audio duration by genre, decomposed by language
#
# These figures give an overall view of corpus scale per language
# and confirm that the collected duration is broadly comparable
# across the three languages, which is a precondition for a fair
# cross-lingual comparison in the dissertation's evaluation stage.
# ============================================================

# ------------------------------------------------------------
# Figure 3: Total duration by language
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(
    language_hours["language"].astype(str),
    language_hours["duration_hours"],
    color=[LANG_COLORS[l] for l in language_hours["language"].astype(str)],
    edgecolor="white",
    linewidth=0.8,
    width=0.55,
)

ax.set_title("Figure 3: Total Audio Duration by Language", pad=12)
ax.set_xlabel("Language")
ax.set_ylabel("Total duration (hours)")
ax.grid(axis="y", linestyle="--", alpha=0.3)

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + max(language_hours["duration_hours"]) * 0.02,
        f"{height:.2f} h",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()

language_hours_png = MANIFEST_DIR / "visual_total_hours_by_language.png"
plt.savefig(language_hours_png, bbox_inches="tight")
plt.show()

print(f"Figure saved: {language_hours_png}")


# ------------------------------------------------------------
# Figure 4: Stacked duration by genre, decomposed by language
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 4.3))

pivot_hours.plot(
    kind="bar",
    stacked=True,
    width=0.65,
    color=[LANG_COLORS[l] for l in LANG_ORDER],
    edgecolor="white",
    linewidth=0.8,
    ax=ax,
)

ax.set_title("Figure 4: Total Duration per Genre, Decomposed by Language", pad=12)
ax.set_xlabel("Genre")
ax.set_ylabel("Total duration (hours)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.legend(title="Language", frameon=False, fontsize=9)

totals = pivot_hours.sum(axis=1)

for i, total in enumerate(totals):
    ax.text(
        i,
        total + totals.max() * 0.02,
        f"{total:.2f} h",
        ha="center",
        va="bottom",
        fontsize=8.5,
    )

plt.tight_layout()

stacked_png = MANIFEST_DIR / "visual_stacked_hours_by_genre.png"
plt.savefig(stacked_png, bbox_inches="tight")
plt.show()

print(f"Figure saved: {stacked_png}")


# ------------------------------------------------------------
# Corpus-level statistics reported in the dissertation text
# ------------------------------------------------------------

print("\nCorpus summary statistics")
print("-" * 40)
print(f"Total songs: {len(df)}")
print(f"Total duration: {round(df['duration_hours'].sum(), 2)} hours")

print("\nSongs by language:")
print(df["language"].value_counts().reindex(LANG_ORDER))

print("\nDuration by language (hours):")
print(language_hours.to_string(index=False))

print("\nFiles exported for the dissertation appendix:")
print(f"  {count_png}")
print(f"  {hours_png}")
print(f"  {language_hours_png}")
print(f"  {stacked_png}")
print(f"  {summary_csv}")

# AI Songs Collection

### English, Chinese and Hindi AI-generated Song Dataset

This notebook collects and generates AI songs across three languages:

| Language | Method | Target |
|---|---|---:|
| English | Muse dataset (HuggingFace `bolshyC/Muse`) | 300 songs |
| Chinese | Muse dataset + Suno v5 gap-fill prompts | 300 songs |
| Hindi | Suno v5 prompt generation (all 300) | 300 songs |

Each language targets **5 genres x 60 songs = 300 songs**.

| Genres |
|---|
| Pop, Rock, Hip-Hop/Rap, R&B/Soul, Folk/Acoustic |

## Workflow

```text
Setup
  -> [English + Chinese] Muse metadata download + tar extraction
  -> [Chinese]           Suno v5 gap-fill prompt generation
  -> [Hindi]             Suno v5 full 300-prompt generation
  -> Verification
```

> Manifests are saved as CSV after every stage. Rerun an interrupted cell safely.


## 0. Install dependencies

Run once in a new environment.


In [ ]:
%pip install -q huggingface_hub pandas tqdm


## 1. Environment setup

Imports, folder structure and shared configuration.


In [ ]:
# ONE-TIME MIGRATION: move old root-level AI outputs into
# dataset_collection_clean/ai/  (delete after running once)
import shutil
from pathlib import Path

_AI_ROOT = Path('dataset_collection_clean/ai')

_MOVES = [
    # Muse English
    (Path('muse_english_ai_audio'),         _AI_ROOT / 'english' / 'audio'),
    (Path('muse_english_ai_manifest.csv'),  _AI_ROOT / 'english' / 'english_ai_manifest.csv'),
    (Path('muse_english_missing_plan.csv'), _AI_ROOT / 'english' / 'english_missing_plan.csv'),
    # Muse Chinese
    (Path('muse_chinese_ai_audio'),         _AI_ROOT / 'chinese' / 'audio'),
    (Path('muse_chinese_ai_manifest.csv'),  _AI_ROOT / 'chinese' / 'chinese_ai_manifest.csv'),
    (Path('muse_chinese_missing_plan.csv'), _AI_ROOT / 'chinese' / 'chinese_missing_plan.csv'),
    (Path('chinese_missing_suno_prompts.csv'), _AI_ROOT / 'chinese' / 'chinese_missing_suno_prompts.csv'),
    # Hindi
    (Path('hindi_ai_300_suno_prompts.csv'), _AI_ROOT / 'hindi' / 'hindi_ai_300_suno_prompts.csv'),
    (Path('hindi_ai_audio'),                _AI_ROOT / 'hindi' / 'audio'),
    # Combined missing plan + raw downloads
    (Path('muse_missing_generation_plan.csv'), _AI_ROOT / 'muse_missing_generation_plan.csv'),
    (Path('muse_raw'),                      _AI_ROOT / 'raw_sources'),
    (Path('suno_chinese_generated_audio'),  _AI_ROOT / 'chinese' / 'suno_generated_audio'),
]

print('AI OUTPUT MIGRATION')
print('=' * 50)
_moved, _skipped, _missing = 0, 0, 0
for _src, _dst in _MOVES:
    if not _src.exists():
        print(f'  NOT FOUND   {_src}')
        _missing += 1
    elif _dst.exists():
        print(f'  ALREADY OK  {_src}')
        _skipped += 1
    else:
        _dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(_src), str(_dst))
        print(f'  MOVED       {_src}  ->  {_dst}')
        _moved += 1

print()
print(f'Done. Moved: {_moved}  |  Already at destination: {_skipped}  |  Not found: {_missing}')
print('You can delete this cell and the markdown cell above it now.')


In [ ]:
import json
import random
import tarfile
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
from tqdm.notebook import tqdm
from huggingface_hub import hf_hub_download, HfApi

# SHARED SETTINGS

REPO_ID = "bolshyC/Muse"

TARGET_GENRES = [
    "Pop",
    "Rock",
    "Hip-Hop/Rap",
    "R&B/Soul",
    "Folk/Acoustic",
]

SONGS_PER_GENRE = 60

# Languages fetched from Muse
LANGUAGES_TO_FETCH = [
    {
        "code": "EN",
        "name": "English",
        "jsonl": "train_en.jsonl",
        "prefix": "en_part",
    },
    {
        "code": "CN",
        "name": "Chinese",
        "jsonl": "train_cn.jsonl",
        "prefix": "cn_part",
    },
]

# Note: Chinese may not reach 60 for Hip-Hop/Rap and R&B/Soul from Muse.
# Missing songs are planned for Suno v5 generation in Cell 4.
TARGET_COUNTS_BY_LANGUAGE = {
    "EN": {g: SONGS_PER_GENRE for g in TARGET_GENRES},
    "CN": {g: SONGS_PER_GENRE for g in TARGET_GENRES},
}

AI_ROOT = Path("dataset_collection_clean/ai")
RAW_DIR = AI_ROOT / "raw_sources"
RAW_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)

print("Environment ready.")
print("RAW_DIR:", RAW_DIR.resolve())


## 2. Genre mapping helpers

Maps Muse style tags to the five target genres.


In [ ]:
def clean_text(x):
    return str(x).lower().replace("_", " ").replace("-", " ").strip()


def genre_from_style(style):
    s = clean_text(style)

    hiphop_terms = [
        "hip hop", "hiphop", "rap", "trap", "drill", "boom bap",
        "gangsta rap", "conscious rap"
    ]
    rnb_terms = [
        "r&b", "rnb", "soul", "neo soul", "funk", "rhythm and blues"
    ]
    folk_terms = [
        "folk", "folk pop", "indie folk", "acoustic",
        "singer songwriter", "country", "bluegrass"
    ]
    rock_terms = [
        "rock", "soft rock", "pop rock", "indie rock",
        "alternative rock", "hard rock", "punk", "metal"
    ]
    pop_terms = [
        "pop", "c pop", "mandopop", "dance pop",
        "synth pop", "electropop", "ballad"
    ]

    if any(t in s for t in hiphop_terms):
        return "Hip-Hop/Rap"
    if any(t in s for t in rnb_terms):
        return "R&B/Soul"
    if any(t in s for t in folk_terms):
        return "Folk/Acoustic"
    if any(t in s for t in rock_terms):
        return "Rock"
    if any(t in s for t in pop_terms):
        return "Pop"
    return None


def safe_extract_one(tar, member, output_path):
    f = tar.extractfile(member)
    if f is None:
        return False
    with open(output_path, "wb") as out:
        shutil.copyfileobj(f, out)
    return True


print("Genre helpers loaded.")


## 3. English and Chinese extraction from Muse

Downloads JSONL metadata + tar archives from `bolshyC/Muse`, extracts up to 60 MP3s per genre per language, and saves:
- `muse_english_ai_audio/` / `muse_chinese_ai_audio/` — MP3 files
- `muse_english_ai_manifest.csv` / `muse_chinese_ai_manifest.csv` — manifests
- `muse_<language>_missing_plan.csv` genres below 60-song target



In [ ]:
def load_metadata(lang_config):
    lang_code = lang_config["code"]
    lang_name = lang_config["name"]

    print(f"\nDownloading {lang_name} metadata...")

    metadata_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=lang_config["jsonl"],
        repo_type="dataset",
        local_dir=RAW_DIR,
    )

    rows = []

    with open(metadata_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Reading {lang_name} metadata"):
            try:
                item = json.loads(line)
            except Exception:
                continue

            audio_path = item.get("audio_path")
            style = item.get("style")

            if not audio_path or not style:
                continue

            final_genre = genre_from_style(style)
            if final_genre not in TARGET_GENRES:
                continue

            rows.append({
                "song_id_original": item.get("song_id"),
                "track_index": item.get("track_index"),
                "style": style,
                "style_sim": item.get("style_sim"),
                "audio_path_original": audio_path,
                "audio_name": Path(audio_path).name,
                "final_genre": final_genre,
                "language": lang_name,
                "language_code": lang_code,
                "model": "SunoV5",
                "source_dataset": "Muse",
            })

    meta = pd.DataFrame(rows)
    print(f"\nAvailable by mapped genre for {lang_name}:")
    print(meta["final_genre"].value_counts())
    return meta


def extract_language(lang_config):
    lang_code = lang_config["code"]
    lang_name = lang_config["name"]
    target_counts = TARGET_COUNTS_BY_LANGUAGE[lang_code]

    print("\n=========================================================")
    print(f"STARTING {lang_name.upper()} EXTRACTION")
    print("=========================================================")

    # ── Early-exit: skip entirely if manifest is already complete ──
    _out_manifest_check = AI_ROOT / lang_name.lower() / f"{lang_name.lower()}_ai_manifest.csv"
    if _out_manifest_check.exists():
        _check_df = pd.read_csv(_out_manifest_check)
        _check_counts = _check_df["final_genre"].value_counts().to_dict()
        if all(_check_counts.get(g, 0) >= target_counts[g] for g in TARGET_GENRES):
            print(f"[SKIP] {lang_name} manifest already complete at {_out_manifest_check}")
            print(_check_df["final_genre"].value_counts().to_string())
            _missing_rows = []
            for g in TARGET_GENRES:
                _missing_rows.append({"language": lang_name, "genre": g,
                    "selected_from_muse": _check_counts.get(g, 0),
                    "target": SONGS_PER_GENRE, "missing_to_generate_with_suno_v5": 0})
            return _check_df, pd.DataFrame(_missing_rows)

    meta = load_metadata(lang_config)
    available_counts = meta["final_genre"].value_counts().to_dict()

    print("\nTarget counts:")
    for g in TARGET_GENRES:
        available = available_counts.get(g, 0)
        target = target_counts[g]
        print(f"{g}: target {target}, available {available}")
        if available < target:
            print(f"WARNING: {lang_name} has only {available} available for {g}")

    out_audio_dir = AI_ROOT / lang_name.lower() / "audio"
    out_manifest = AI_ROOT / lang_name.lower() / f"{lang_name.lower()}_ai_manifest.csv"
    out_audio_dir.mkdir(parents=True, exist_ok=True)

    by_audio_name = defaultdict(list)
    for row in meta.to_dict("records"):
        by_audio_name[row["audio_name"]].append(row)

    selected = []
    selected_counts = Counter()

    api = HfApi()
    all_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
    tar_files = sorted([
        f for f in all_files
        if f.startswith(lang_config["prefix"]) and f.endswith(".tar")
    ])

    print(f"\nFound {len(tar_files)} tar files for {lang_name}.")

    for tar_name in tar_files:
        if all(selected_counts[g] >= target_counts[g] for g in TARGET_GENRES):
            break

        print(f"\nDownloading/checking {tar_name}...")
        tar_path = hf_hub_download(
            repo_id=REPO_ID,
            filename=tar_name,
            repo_type="dataset",
            local_dir=RAW_DIR,
        )

        with tarfile.open(tar_path, "r") as tar:
            members = [
                m for m in tar.getmembers()
                if m.isfile() and m.name.lower().endswith(".mp3")
            ]
            random.shuffle(members)

            for member in tqdm(members, desc=f"Scanning {tar_name}", leave=False):
                if all(selected_counts[g] >= target_counts[g] for g in TARGET_GENRES):
                    break

                audio_name = Path(member.name).name
                if audio_name not in by_audio_name:
                    continue

                candidate = by_audio_name[audio_name][0]
                genre = candidate["final_genre"]

                if selected_counts[genre] >= target_counts[genre]:
                    continue

                selected_counts[genre] += 1
                genre_short = (
                    genre.replace("/", "_")
                         .replace("-", "_")
                         .replace(" ", "_")
                         .upper()
                )
                new_song_id = f"AI_{lang_code}_{genre_short}_{selected_counts[genre]:03d}"
                new_filename = f"{new_song_id}.mp3"
                out_path = out_audio_dir / new_filename

                ok = safe_extract_one(tar, member, out_path)
                if not ok:
                    selected_counts[genre] -= 1
                    continue

                candidate = candidate.copy()
                candidate["new_song_id"] = new_song_id
                candidate["new_filename"] = new_filename
                candidate["local_audio_path"] = str(out_path)
                candidate["tar_source"] = tar_name
                selected.append(candidate)

        pd.DataFrame(selected).to_csv(out_manifest, index=False, encoding="utf-8-sig")

        print("\nCurrent selected counts:")
        for g in TARGET_GENRES:
            print(f"{g}: {selected_counts[g]}/{target_counts[g]}")

        try:
            Path(tar_path).unlink()
            print(f"Deleted {tar_name} to save disk.")
        except Exception:
            print(f"Could not delete {tar_name}. Delete manually if needed.")

    final_df = pd.DataFrame(selected)
    print(f"\nFinal selected counts for {lang_name}:")
    if len(final_df) > 0:
        print(final_df["final_genre"].value_counts())
    else:
        print("No songs selected.")

    final_df.to_csv(out_manifest, index=False, encoding="utf-8-sig")

    # Build missing plan
    missing_rows = []
    for g in TARGET_GENRES:
        selected_n = selected_counts[g]
        missing_n = max(0, SONGS_PER_GENRE - selected_n)
        missing_rows.append({
            "language": lang_name,
            "genre": g,
            "selected_from_muse": selected_n,
            "target": SONGS_PER_GENRE,
            "missing_to_generate_with_suno_v5": missing_n,
        })

    missing_df = pd.DataFrame(missing_rows)
    missing_path = AI_ROOT / lang_name.lower() / f"{lang_name.lower()}_missing_plan.csv"
    missing_df.to_csv(missing_path, index=False, encoding="utf-8-sig")

    print(f"\nSaved manifest:     {out_manifest}")
    print(f"Saved missing plan: {missing_path}")
    return final_df, missing_df


# RUN 

all_missing = []

for lang in LANGUAGES_TO_FETCH:
    final_df, missing_df = extract_language(lang)
    all_missing.append(missing_df)

all_missing_df = pd.concat(all_missing, ignore_index=True)
all_missing_df.to_csv(AI_ROOT / "muse_missing_generation_plan.csv", index=False, encoding="utf-8-sig")

print("\nDONE.")
print(f"Check: {AI_ROOT / 'muse_missing_generation_plan.csv'}")


## 4. Chinese missing songs: Suno v5 prompt generator

If Chinese Hip-Hop/Rap or R&B/Soul counts fell below 60 in the Muse extraction, this cell generates Suno v5 prompts for the missing tracks.

> **After generating audio in Suno v5:** save each MP3 to `suno_chinese_generated_audio/` named as the `new_song_id` (e.g. `AI_CN_R_B_SOUL_004.mp3`), then merge with the Muse manifest.


In [ ]:
INPUT_CSV = str(AI_ROOT / "chinese" / "chinese_ai_manifest.csv")
OUTPUT_CSV = str(AI_ROOT / "chinese" / "chinese_missing_suno_prompts.csv")

if not Path(INPUT_CSV).exists():
    print(f"[Skip] {INPUT_CSV} not found. Run Cell 3 first.")
else:
    df = pd.read_csv(INPUT_CSV)
    counts = Counter(df["final_genre"])
    missing_rows = []

    # R&B/Soul scenario bank
    #
    # Each entry is a full commercial song concept with an explicit
    # emotional arc (starting feeling -> turning point -> ending
    # feeling), not a single mood adjective. 24 unique scenarios
    # so large missing counts do not cycle back to the same concept
    # within one generation batch.

    RNB_SCENARIOS = [
        {
            "scene": "sitting in a parked car outside her old apartment at 2am, engine off, "
                     "not ready to go up or drive away",
            "emotion_arc": "starts in numb hesitation, slides into aching nostalgia, "
                            "ends in reluctant acceptance that the door won't open tonight",
            "hook_direction": "a repeated line about not knowing which door to knock on anymore",
            "vocal": "breathy verses building into a full-voiced, slightly cracked chorus",
        },
        {
            "scene": "finding an old voice message from an ex on a phone that still hasn't been changed",
            "emotion_arc": "opens in quiet shock, deepens into raw grief on the second listen, "
                            "closes in bittersweet warmth at the memory",
            "hook_direction": "a hook built around replaying the same three seconds of a voice memo",
            "vocal": "soft falsetto verses, chest-voice belt on the final chorus",
        },
        {
            "scene": "watching rain hit the window of a 30th-floor apartment in Shanghai, "
                     "phone face-down on the table, choosing not to text first",
            "emotion_arc": "begins in restless loneliness, tightens into wounded pride, "
                            "settles into a fragile, self-protective calm",
            "hook_direction": "a hook about silence being louder than any reply",
            "vocal": "smooth, restrained verses with soulful ad-libs in the outro",
        },
        {
            "scene": "meeting an ex by accident at a wedding banquet, both pretending to be fine "
                     "in front of relatives",
            "emotion_arc": "starts in forced composure, cracks into sudden heartbreak beneath the smile, "
                            "ends in quiet dignity walking away",
            "hook_direction": "a hook contrasting the polite smile with what's actually being felt",
            "vocal": "controlled, understated delivery that opens up only in the bridge",
        },
        {
            "scene": "packing the last box out of a shared apartment, keeping one small item "
                     "that doesn't belong to either person anymore",
            "emotion_arc": "moves from detached practicality into sudden overwhelming grief, "
                            "then into gentle, sentimental closure",
            "hook_direction": "a hook about a single object carrying an entire relationship",
            "vocal": "intimate, close-mic verses, layered harmonies on the hook",
        },
        {
            "scene": "a late-night rooftop conversation with a friend, finally admitting out loud "
                     "that the relationship is over",
            "emotion_arc": "starts in guarded denial, breaks into tearful honesty, "
                            "ends in unexpected relief and lightness",
            "hook_direction": "a hook that turns confession into quiet relief rather than sadness",
            "vocal": "conversational verses, warm and open chorus",
        },
        {
            "scene": "riding the last subway line home after a breakup, watching strangers' "
                     "reflections in the dark window instead of the city outside",
            "emotion_arc": "begins in hollow exhaustion, drifts into aching self-reflection, "
                            "ends in a small, hard-won sense of self-recognition",
            "hook_direction": "a hook using the subway window reflection as a metaphor for self-recognition",
            "vocal": "airy verses, emotional swell into the final hook repeat",
        },
        {
            "scene": "cooking a meal for one for the first time in years, still setting out two bowls out of habit",
            "emotion_arc": "starts in muscle-memory denial, aches into sudden loneliness, "
                            "settles into tender, forgiving acceptance",
            "hook_direction": "a hook about muscle memory outlasting the relationship itself",
            "vocal": "gentle, almost spoken verses, full soulful belt on the last chorus",
        },
        {
            "scene": "scrolling through old photos at 3am after seeing an ex's engagement announcement",
            "emotion_arc": "opens in disbelief and quiet jealousy, spirals into private sorrow, "
                            "ends in a hard-earned wish of genuine goodwill",
            "hook_direction": "a hook admitting jealousy honestly before turning it into peace",
            "vocal": "hushed, trembling verses opening into a soaring, cathartic chorus",
        },
        {
            "scene": "receiving a wedding invitation addressed only in your own name, "
                     "the plus-one line left blank",
            "emotion_arc": "starts in stinging embarrassment, deepens into aching solitude, "
                            "resolves into stubborn self-respect",
            "hook_direction": "a hook turning the blank plus-one line into a quiet declaration of self-worth",
            "vocal": "restrained verses, defiant power in the final hook",
        },
        {
            "scene": "hearing a song on the radio that used to be 'our song', alone in the kitchen at night",
            "emotion_arc": "opens in involuntary tenderness, aches into fresh grief, "
                            "settles into gratitude for having had it at all",
            "hook_direction": "a hook about a song refusing to let a memory fade",
            "vocal": "vulnerable low verses, full emotional release on the bridge",
        },
        {
            "scene": "deleting a shared photo album at midnight, hesitating on the very last photo",
            "emotion_arc": "starts in cold resolve, falters into sudden hesitation and doubt, "
                            "ends in a slow, deliberate letting go",
            "hook_direction": "a hook built around the pause before pressing delete",
            "vocal": "sparse verses with long held notes, layered harmony on the final chorus",
        },
        {
            "scene": "answering a friend's question 'are you okay?' with 'I'm fine' for the fifth time that week",
            "emotion_arc": "opens in practiced denial, cracks into private exhaustion, "
                            "ends in the first honest admission of not being okay",
            "hook_direction": "a hook that repeats 'I'm fine' until it finally breaks into truth",
            "vocal": "clipped, guarded verses opening into a raw, unguarded chorus",
        },
        {
            "scene": "watching an old couple dance at a night market, remembering what that used to feel like",
            "emotion_arc": "starts in wistful envy, deepens into longing for what was lost, "
                            "ends in quiet hope that it can still happen again",
            "hook_direction": "a hook contrasting someone else's forever with your own uncertain future",
            "vocal": "warm, nostalgic verses, hopeful lift in the final chorus",
        },
        {
            "scene": "moving all your things into a new, smaller apartment alone for the first time",
            "emotion_arc": "opens in quiet fear of being alone, moves through grief for the old life, "
                            "ends in tentative pride in starting over",
            "hook_direction": "a hook about a small space becoming, slowly, a home again",
            "vocal": "soft, breath-forward verses building to a resilient, uplifted chorus",
        },
        {
            "scene": "texting 'I miss you' and deleting it three times before sending nothing at all",
            "emotion_arc": "starts in aching temptation, spirals into wounded self-restraint, "
                            "ends in bruised but intact self-respect",
            "hook_direction": "a hook built entirely around the unsent message",
            "vocal": "hesitant, half-whispered verses, aching held notes in the chorus",
        },
        {
            "scene": "sitting through a mutual friend's birthday dinner, carefully avoiding the same side of the table",
            "emotion_arc": "opens in tense civility, aches into buried longing beneath small talk, "
                            "ends in reluctant, mature distance",
            "hook_direction": "a hook about pretending the distance is easy when it isn't",
            "vocal": "cool, composed verses, quietly devastating bridge",
        },
        {
            "scene": "waking up reaching for someone who isn't there anymore, weeks after the breakup",
            "emotion_arc": "starts in disoriented half-sleep, aches into the fresh sting of remembering, "
                            "settles into slow, patient healing",
            "hook_direction": "a hook about the body forgetting what the mind already knows",
            "vocal": "drowsy, intimate verses, cathartic release in the final chorus",
        },
        {
            "scene": "standing in the rain outside a café where the first date happened years ago",
            "emotion_arc": "opens in sentimental warmth, deepens into sharp grief for lost time, "
                            "ends in grateful closure",
            "hook_direction": "a hook comparing the same rain now and then",
            "vocal": "smooth, controlled verses, emotional swell on the final hook",
        },
        {
            "scene": "helping a friend move out of the apartment you two used to share, boxing up the last memories",
            "emotion_arc": "starts in practical numbness, breaks into sudden grief mid-task, "
                            "ends in bittersweet gratitude for the years shared",
            "hook_direction": "a hook about boxes holding more than just belongings",
            "vocal": "steady verses cracking slightly on the bridge, full harmony on the outro",
        },
        {
            "scene": "seeing your ex's name flash on caller ID after a year of no contact, letting it ring out",
            "emotion_arc": "opens in startled panic, spikes into old longing, "
                            "resolves into calm, hard-won self-control",
            "hook_direction": "a hook about choosing silence as its own kind of answer",
            "vocal": "tense, controlled verses, release into a soaring final chorus",
        },
        {
            "scene": "writing a letter that will never be sent, just to finally say everything out loud",
            "emotion_arc": "starts in careful restraint, spills into unfiltered honesty, "
                            "ends in a lighter, unburdened calm",
            "hook_direction": "a hook framed as the opening line of the unsent letter",
            "vocal": "spoken-word-adjacent verses building into a full sung chorus",
        },
        {
            "scene": "hearing a friend say 'you deserve better' and finally starting to believe it",
            "emotion_arc": "opens in defensive doubt, softens into cautious hope, "
                            "ends in quiet, growing self-belief",
            "hook_direction": "a hook that repeats 'you deserve better' until it becomes self-spoken",
            "vocal": "gentle verses, warm and affirming chorus",
        },
        {
            "scene": "dancing alone in the living room at midnight, choosing joy on purpose after months of sadness",
            "emotion_arc": "starts in tired heaviness, softens into playful defiance, "
                            "ends in genuine, hard-won lightness",
            "hook_direction": "a hook about choosing to feel good again, on your own terms",
            "vocal": "loose, playful verses, joyful, full-bodied chorus",
        },
    ]

    # Hip-Hop/Rap scenario bank
    # 24 unique commercial song concepts, each with an explicit
    # emotional arc alongside the hustle narrative — real C-hip-hop
    # singles carry emotional depth (fear, pride, doubt, defiance),
    # not just a flat "confident" mood tag.

    HIPHOP_SCENARIOS = [
        {
            "scene": "grinding through a part-time delivery job between classes, headphones in, "
                     "writing bars on the back of a receipt at a red light",
            "emotion_arc": "starts in quiet frustration at the grind, builds into stubborn hope, "
                            "ends in defiant self-belief",
            "hook_direction": "a hook contrasting the delivery bag on one shoulder with the dream on the other",
            "flow": "laid-back verses with a triplet flow switch into a shouted hook",
        },
        {
            "scene": "recording a first demo in a bedroom studio built from foam mattress panels, "
                     "parents asleep in the next room unaware of the late-night sessions",
            "emotion_arc": "opens in nervous secrecy, grows into quiet pride, "
                            "ends in fierce determination to prove it wasn't wasted time",
            "hook_direction": "a hook about turning a small room into the whole world",
            "flow": "tight, percussive verses building to a gang-vocal style hook",
        },
        {
            "scene": "standing outside a job interview in a suit that isn't quite the right size, "
                     "still mentally rehearsing bars from the night before",
            "emotion_arc": "starts in anxious self-doubt, splits into internal conflict between two futures, "
                            "resolves into resolve to chase both without apology",
            "hook_direction": "a hook that plays two identities against each other — the suit and the stage",
            "flow": "confident mid-tempo verses, melodic sung hook",
        },
        {
            "scene": "getting rejected by a label A&R after years of grinding, deciding to self-release instead",
            "emotion_arc": "opens in raw humiliation, hardens into anger, "
                            "ends in cold, focused determination",
            "hook_direction": "a hook about rejection becoming the reason to go independent",
            "flow": "aggressive double-time verses, anthemic hook",
        },
        {
            "scene": "a group of friends from the same neighbourhood pooling money to book one studio session",
            "emotion_arc": "starts in nervous excitement, deepens into fierce loyalty, "
                            "ends in shared, triumphant joy",
            "hook_direction": "a hook celebrating loyalty and collective hustle over individual fame",
            "flow": "back-and-forth trade verses, unison shouted hook",
        },
        {
            "scene": "moving to a first-tier city with one suitcase, sending money home every month, "
                     "never telling family how hard it actually is",
            "emotion_arc": "opens in homesick loneliness, carries quiet guilt and pressure, "
                            "ends in proud, protective resolve",
            "hook_direction": "a hook about carrying a hometown's expectations silently",
            "flow": "reflective verses over a slower beat, melodic hook with a key change",
        },
        {
            "scene": "getting a beat rejected by three different producers before one finally clicks at 4am",
            "emotion_arc": "starts in mounting frustration, dips into real self-doubt, "
                            "breaks into sudden euphoric relief when it finally works",
            "hook_direction": "a hook about persistence paying off in the smallest, quietest moment",
            "flow": "sparse, confident verses building into a full-band hook",
        },
        {
            "scene": "performing a first small show to a half-empty room, treating it like a sold-out arena anyway",
            "emotion_arc": "opens in embarrassed disappointment, flips into stubborn pride, "
                            "ends in genuine, unshakeable belief",
            "hook_direction": "a hook about belief mattering more than the size of the crowd",
            "flow": "energetic verses, crowd-chant style hook",
        },
        {
            "scene": "scrolling past an old classmate's success post at 1am while still working on an unreleased track",
            "emotion_arc": "starts in bitter comparison, aches into private insecurity, "
                            "ends in refocused, competitive drive",
            "hook_direction": "a hook that turns comparison into fuel instead of doubt",
            "flow": "moody, half-sung verses into a defiant, driving hook",
        },
        {
            "scene": "walking through a night market with childhood friends, realising how far everyone has come "
                     "since school",
            "emotion_arc": "opens in nostalgic warmth, touches on quiet grief for lost time, "
                            "ends in grateful, hopeful pride",
            "hook_direction": "a hook about growth measured against where the group started",
            "flow": "nostalgic verses over a boom-bap-influenced beat, warm melodic hook",
        },
        {
            "scene": "sitting in a hospital waiting room at 5am after a family member falls ill, "
                     "phone full of unfinished lyrics",
            "emotion_arc": "starts in helpless fear, deepens into raw vulnerability, "
                            "ends in quiet strength and renewed purpose",
            "hook_direction": "a hook about music becoming the only way to hold it together",
            "flow": "slow, weighted verses, restrained emotional hook",
        },
        {
            "scene": "counting the last bit of savings before rent is due, still refusing to take a 9-to-5 that kills the dream",
            "emotion_arc": "opens in genuine financial anxiety, hardens into stubborn refusal to give up, "
                            "ends in gritty, unglamorous resolve",
            "hook_direction": "a hook about choosing hunger over comfort",
            "flow": "tense, clipped verses, hard-hitting repetitive hook",
        },
        {
            "scene": "an old friend from the block turning to a life of crime while you chose music instead",
            "emotion_arc": "starts in conflicted loyalty, aches with sadness for the path not shared, "
                            "ends in resolve to make it out and look back for others",
            "hook_direction": "a hook about two roads from the same starting point",
            "flow": "heavy, deliberate verses, somber but driving hook",
        },
        {
            "scene": "finally affording to send your mother on her first flight, watching her face at the airport",
            "emotion_arc": "opens in nervous anticipation, swells into overwhelming pride and love, "
                            "ends in tearful gratitude",
            "hook_direction": "a hook framed as a direct thank-you to a parent",
            "flow": "warm, melodic verses, uplifting sung hook",
        },
        {
            "scene": "getting doubted by an old teacher who said this dream was a waste of time, "
                     "running into them years later",
            "emotion_arc": "starts in old resentment, flares into vindicated anger, "
                            "settles into calm, quiet satisfaction",
            "hook_direction": "a hook that repeats the old doubt before flipping it into proof",
            "flow": "sharp, cutting verses, triumphant hook",
        },
        {
            "scene": "losing a close collaborator or group member to a different path, finishing the album alone",
            "emotion_arc": "opens in grief and disorientation, moves through frustration, "
                            "ends in solitary but proud resolve to finish what was started",
            "hook_direction": "a hook about carrying a shared dream alone now",
            "flow": "introspective verses, restrained emotional hook with space to breathe",
        },
        {
            "scene": "posting a song online and refreshing the view count every few minutes, terrified of silence",
            "emotion_arc": "starts in anxious hope, spirals into private fear of failure, "
                            "ends in defiant commitment regardless of the numbers",
            "hook_direction": "a hook about making art matter regardless of who's watching",
            "flow": "restless, anxious verses, confident release in the hook",
        },
        {
            "scene": "visiting a childhood neighbourhood that's been torn down for new development, "
                     "nothing left but memory",
            "emotion_arc": "opens in disorienting loss, deepens into aching nostalgia, "
                            "ends in determination to keep the old story alive through music",
            "hook_direction": "a hook about carrying a demolished home in the music itself",
            "flow": "slow, weighted verses, emotionally rich melodic hook",
        },
        {
            "scene": "being the first in the family to go to university, then dropping out to chase music, "
                     "facing a difficult conversation at the dinner table",
            "emotion_arc": "starts in guilt and fear of disappointing family, hardens into defensive conviction, "
                            "ends in vulnerable hope for their eventual understanding",
            "hook_direction": "a hook framed as a promise to make the risk worth it",
            "flow": "tense, deliberate verses, pleading but strong melodic hook",
        },
        {
            "scene": "hearing your song played by a stranger's speaker on the street for the first time, unnoticed",
            "emotion_arc": "opens in disbelief, swells into overwhelming private joy, "
                            "ends in quiet, humbled gratitude",
            "hook_direction": "a hook about a small, anonymous moment meaning everything",
            "flow": "bright, energetic verses, joyful, wide-open hook",
        },
        {
            "scene": "burning out after months of nonstop hustle, forced to finally rest and question everything",
            "emotion_arc": "starts in exhausted numbness, breaks into honest doubt about the whole path, "
                            "ends in renewed, more grounded purpose",
            "hook_direction": "a hook about rebuilding motivation from empty, not from hype",
            "flow": "slow, tired verses, restrained hook that builds gradually",
        },
        {
            "scene": "seeing a younger cousin start rapping too, being asked for advice for the first time",
            "emotion_arc": "opens in surprised pride, warms into protective responsibility, "
                            "ends in reflective wisdom about the road ahead",
            "hook_direction": "a hook framed as advice passed down, not bragging",
            "flow": "measured, mentor-like verses, warm and grounded hook",
        },
        {
            "scene": "finally quitting a soul-crushing office job the day after the first real payout from music",
            "emotion_arc": "starts in disbelieving shock, rises into explosive relief and joy, "
                            "ends in fierce, hard-won validation",
            "hook_direction": "a hook about the exact moment the risk finally pays off",
            "flow": "explosive, high-energy verses, celebratory anthemic hook",
        },
        {
            "scene": "standing backstage before the biggest show yet, hands shaking, remembering the empty rooms it took to get here",
            "emotion_arc": "opens in genuine fear, steadies into focused resolve, "
                            "ends in full-hearted, earned confidence",
            "hook_direction": "a hook that connects this exact moment to every hard night before it",
            "flow": "building, tense verses, massive, victorious hook",
        },
    ]


    def make_cn_prompt(genre, number):
        """
        Build a full commercial-style Suno generation prompt for a
        Mandarin Chinese track: specific scene, explicit emotional arc,
        song structure, instrumentation, vocal delivery, and hook
        direction — matching the level of detail used in real
        streaming-release prompts rather than a single mood adjective.
        """

        if genre == "R&B/Soul":
            scenario = RNB_SCENARIOS[(number - 1) % len(RNB_SCENARIOS)]

            return (
                f"Write and produce an original Mandarin Chinese R&B/Soul song, "
                f"simplified Chinese lyrics only, no other language mixed in. "
                f"\n\nSong concept: {scenario['scene']}. "
                f"\n\nEmotional arc: {scenario['emotion_arc']}. The lyrics and vocal "
                f"performance should trace this emotional journey clearly from verse 1 "
                f"through to the final chorus, not stay flat on one feeling throughout. "
                f"\n\nStructure: short intro, verse 1, pre-chorus, chorus, verse 2, "
                f"pre-chorus, chorus, bridge, final chorus with ad-libs, short outro. "
                f"\n\nHook: {scenario['hook_direction']}, repeated as the chorus's core line. "
                f"\n\nVocal delivery: {scenario['vocal']}. "
                f"\n\nProduction: 75-85 BPM, warm Rhodes/electric piano chords, "
                f"soft trap-influenced drums, deep sub-bass, subtle vinyl-style texture, "
                f"layered background harmonies on the chorus. "
                f"\n\nMood: intimate, emotionally honest, contemporary Mandopop R&B, "
                f"radio-ready but not overproduced. "
                f"\n\nDo not reference any real artist, song title, or existing melody. "
                f"Original composition only — no cover, no remix, no sample."
            )

        if genre == "Hip-Hop/Rap":
            scenario = HIPHOP_SCENARIOS[(number - 1) % len(HIPHOP_SCENARIOS)]

            return (
                f"Write and produce an original Mandarin Chinese Hip-Hop/Rap song, "
                f"simplified Chinese lyrics only, no other language mixed in. "
                f"\n\nSong concept: {scenario['scene']}. "
                f"\n\nEmotional arc: {scenario['emotion_arc']}. The verses should carry "
                f"this emotional progression honestly — real doubt, fear, or grief where "
                f"it belongs, not just constant bravado — before landing on the resolution "
                f"described. "
                f"\n\nStructure: intro tag, verse 1, hook, verse 2, hook, short bridge "
                f"or breakdown, final hook (double-length with ad-libs), outro tag. "
                f"\n\nHook: {scenario['hook_direction']}, delivered as a repeatable, "
                f"chantable line. "
                f"\n\nFlow: {scenario['flow']}. "
                f"\n\nProduction: 85-100 BPM, hard-hitting 808 bass, crisp trap hi-hats, "
                f"dark minor-key synths, occasional guzheng or dizi flourish for a "
                f"regional Chinese texture, punchy mix with clear vocal presence. "
                f"\n\nMood: emotionally grounded and realistic, rooted in everyday hustle "
                f"and its real costs rather than generic bravado — contemporary Chinese "
                f"hip-hop, radio- and short-video-platform ready. "
                f"\n\nDo not reference any real artist, song title, or existing melody. "
                f"Original composition only — no cover, no remix, no sample."
            )

        return ""


    for genre in ["R&B/Soul", "Hip-Hop/Rap"]:
        current = counts.get(genre, 0)
        missing = SONGS_PER_GENRE - current

        if missing <= 0:
            print(f"{genre}: already at {current}/{SONGS_PER_GENRE}, nothing to generate.")
            continue

        genre_short = genre.replace("/", "_").replace("-", "_").replace(" ", "_").upper()

        for i in range(1, missing + 1):
            song_number = current + i
            song_id = f"AI_CN_{genre_short}_{song_number:03d}"
            filename = f"{song_id}.mp3"

            missing_rows.append({
                "new_song_id": song_id,
                "language": "Chinese",
                "final_genre": genre,
                "model": "Suno v5",
                "source_dataset": "Self-generated",
                "status": "to_generate_in_suno",
                "new_filename": filename,
                "local_audio_path": str(AI_ROOT / "chinese" / "audio" / filename),
                "generation_prompt": make_cn_prompt(genre, i),
            })

    if missing_rows:
        missing_df = pd.DataFrame(missing_rows)
        missing_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

        print("Missing songs to generate:")
        print(missing_df["final_genre"].value_counts())
        print("Saved:", OUTPUT_CSV)

        display(missing_df.head(10))
    else:
        print("No missing Chinese songs — both genres are at or above target.")

## 5. Hindi: Suno v5 prompt generation (all 300 songs)

Hindi AI songs are fully generated via Suno v5 (no Muse dataset available for Hindi). This cell generates **300 unique high-fidelity prompts** — 60 per genre — with language-specific lyrical constraints to ensure pure Hindi lyrics with no Punjabi crossover.

**Output:** `hindi_ai_300_suno_prompts.csv`

| Column | Description |
|---|---|
| `song_id` | Unique ID e.g. `AI_HIN_POP_001` |
| `final_genre` | One of 5 target genres |
| `prompt_used` | Full Suno v5 generation prompt |
| `audio_link` | Fill in after Suno generation |

> **After generating audio in Suno v5:** paste each audio link into `audio_link` in the saved CSV, then save the MP3 files as `AI_HIN_<GENRE>_<NNN>.mp3` in `hindi_ai_audio/`.


In [ ]:
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME       = "Suno v5"
TRACKS_PER_GENRE = 60
OUTPUT_CSV       = str(AI_ROOT / "hindi" / "hindi_ai_300_suno_prompts.csv")
OUTPUT_AUDIO_DIR = str(AI_ROOT / "hindi" / "audio")

# Language constraint per genre — enforces pure Hindi, no Punjabi
HINDI_CONSTRAINTS = {
    "Pop":          "pure Hindi lyrics, strictly no Punjabi",
    "Hip-Hop/Rap":  "pure Hindi rap flow, no Punjabi trap slang",
    "Rock":         "authentic Hindi songwriting, no Punjabi phrasing",
    "R&B/Soul":     "classic Hindi lyricism, free of Punjabi crossover terms",
    "Folk/Acoustic":"traditional pure Hindi dialect, no Punjabi folk inflections",
}


# SCENARIO BANKS
#
# Each entry is a full commercial song concept rooted in everyday
# Indian/Hindi cultural settings, with an explicit emotional arc
# (starting feeling -> turning point -> ending feeling) rather than
# a single mood adjective. 15 scenarios per genre so all 60 tracks
# in a genre cycle through real variety (4 passes) instead of
# combinator repetition.


POP_SCENARIOS = [
    {
        "scene": "sharing one umbrella with a stranger during the season's first monsoon "
                 "downpour outside a Mumbai college gate, both laughing at how soaked they still got",
        "emotion_arc": "starts in shy awkwardness, warms into playful comfort, ends in the "
                        "unmistakable spark of new attraction",
        "hook_direction": "a hook about one umbrella somehow being just the right size for two",
        "vocal": "bright, playful verses lifting into a big, sing-along chorus",
    },
    {
        "scene": "a long-distance couple on a late-night video call, one in Mumbai and one abroad, "
                 "the time difference making 'goodnight' always feel too early",
        "emotion_arc": "opens in tired longing, softens into tender reassurance, ends in hopeful "
                        "anticipation of reuniting",
        "hook_direction": "a hook built around counting the hours until the next hello",
        "vocal": "warm, intimate verses opening into an uplifting, wide chorus",
    },
    {
        "scene": "the night before a childhood best friend's wedding, staying up late finally "
                 "admitting a decade-long crush that arrived too late",
        "emotion_arc": "starts in nervous hesitation, aches into bittersweet regret, ends in "
                        "gentle, mature acceptance",
        "hook_direction": "a hook about saying it out loud even though the timing is wrong",
        "vocal": "soft, conversational verses building to an emotionally open chorus",
    },
    {
        "scene": "reconnecting with a school sweetheart at an airport arrivals gate after years "
                 "living in different countries",
        "emotion_arc": "opens in disbelieving nervousness, rushes into overwhelming joy, ends in "
                        "certainty that this reunion is meant to last",
        "hook_direction": "a hook about a decade disappearing the moment their eyes meet",
        "vocal": "energetic verses rising into a triumphant, celebratory chorus",
    },
    {
        "scene": "a Diwali night terrace conversation lit only by string lights and diyas, finally "
                 "confessing feelings kept quiet for months",
        "emotion_arc": "starts in quiet nervous energy, warms into vulnerable honesty, ends in "
                        "glowing, mutual happiness",
        "hook_direction": "a hook comparing the confession itself to the diyas lighting up the dark",
        "vocal": "gentle verses building into a warm, radiant chorus",
    },
    {
        "scene": "windows down on the Mumbai-Pune expressway at sunset, a road-trip playlist on "
                 "shuffle, realising feelings for a friend in the passenger seat",
        "emotion_arc": "opens in carefree lightness, deepens into sudden nervous realisation, ends "
                        "in giddy, hopeful excitement",
        "hook_direction": "a hook about a friendship quietly turning into something more, somewhere "
                           "on the highway",
        "vocal": "breezy, upbeat verses into an infectious, driving chorus",
    },
    {
        "scene": "rekindling an old flame through late-night Instagram messages a decade after "
                 "losing touch",
        "emotion_arc": "starts in cautious curiosity, grows into nostalgic warmth, ends in genuine, "
                        "present-day hope",
        "hook_direction": "a hook about old feelings finding a new way back",
        "vocal": "soft, textured verses opening into a bright, optimistic chorus",
    },
    {
        "scene": "the season's first rain hitting a Mumbai balcony, remembering an old relationship "
                 "fondly without any lingering pain",
        "emotion_arc": "opens in wistful nostalgia, stays gentle throughout, ends in peaceful, "
                        "grateful closure",
        "hook_direction": "a hook about rain washing away hurt but keeping the good memories",
        "vocal": "smooth, easy verses into a warm, unhurried chorus",
    },
    {
        "scene": "a proposal at a hill-station café overlooking a misty valley at golden hour",
        "emotion_arc": "starts in quiet nervous anticipation, swells into overwhelming love, ends "
                        "in pure, radiant joy",
        "hook_direction": "a hook framed as the exact words spoken in that moment",
        "vocal": "tender verses building into a soaring, celebratory chorus",
    },
    {
        "scene": "two strangers exchanging a fleeting smile across a crowded Mumbai local train "
                 "during the evening rush",
        "emotion_arc": "opens in curious surprise, lingers in playful imagination, ends in hopeful "
                        "wondering about what if",
        "hook_direction": "a hook about a single glance lasting the whole train ride home",
        "vocal": "light, rhythmic verses into a catchy, bouncy chorus",
    },
    {
        "scene": "a bride getting ready on her wedding morning, her mother tying the dupatta with "
                 "tears of pride and quiet emotion",
        "emotion_arc": "starts in nervous excitement, deepens into tender gratitude, ends in "
                        "joyful, tearful blessing",
        "hook_direction": "a hook framed as a mother's blessing to her daughter",
        "vocal": "soft, heartfelt verses building into a warm, emotional chorus",
    },
    {
        "scene": "two childhood best friends slowly realising, over one ordinary evening, that "
                 "their bond has quietly become love",
        "emotion_arc": "opens in comfortable familiarity, shifts into nervous new awareness, ends "
                        "in warm, certain affection",
        "hook_direction": "a hook about a friendship crossing a line neither one planned",
        "vocal": "easy, familiar verses opening into a glowing, heartfelt chorus",
    },
    {
        "scene": "moving to a big city alone for a new job, missing home during the first Diwali "
                 "away from family, but proud of the leap taken",
        "emotion_arc": "starts in homesick loneliness, moves through quiet pride, ends in hopeful, "
                        "independent confidence",
        "hook_direction": "a hook about home living inside you even when you're far away",
        "vocal": "warm, grounded verses into an uplifting, anthemic chorus",
    },
    {
        "scene": "a Holi festival scene, colors and water balloons flying, catching the eye of "
                 "someone new across the chaos",
        "emotion_arc": "opens in playful excitement, sparks into sudden attraction, ends in joyful, "
                        "colorful elation",
        "hook_direction": "a hook about falling for someone in the middle of complete chaos and color",
        "vocal": "energetic, playful verses into a huge, festive chorus",
    },
    {
        "scene": "a New Year's Eve rooftop party in Delhi, fireworks overhead, finally deciding to "
                 "say what's been unsaid all year",
        "emotion_arc": "starts in nervous buildup, releases into brave honesty, ends in "
                        "exhilarated, hopeful joy",
        "hook_direction": "a hook about starting the new year by finally telling the truth",
        "vocal": "building, anticipatory verses into an explosive, celebratory chorus",
    },
]

ROCK_SCENARIOS = [
    {
        "scene": "standing at the front of a peaceful protest march, disillusioned with a broken "
                 "system but refusing to stay silent any longer",
        "emotion_arc": "starts in simmering frustration, builds into open defiance, ends in "
                        "unified, collective strength",
        "hook_direction": "a hook that turns silence into a shout",
        "vocal": "gritty, driving verses erupting into an anthemic, chant-like chorus",
    },
    {
        "scene": "a young musician being told by family that music will never be a 'real career', "
                 "choosing to walk on stage anyway",
        "emotion_arc": "opens in wounded self-doubt, hardens into stubborn resolve, ends in "
                        "fierce self-belief",
        "hook_direction": "a hook about proving a doubted dream right, one show at a time",
        "vocal": "raw, urgent verses building to a powerful, defiant chorus",
    },
    {
        "scene": "returning to a childhood neighbourhood to find it unrecognisable after years of "
                 "unchecked construction and development",
        "emotion_arc": "starts in disoriented grief, deepens into quiet anger, ends in determined "
                        "resolve to remember what was lost",
        "hook_direction": "a hook about a hometown existing now only in memory",
        "vocal": "heavy, weighted verses into a driving, emotionally charged chorus",
    },
    {
        "scene": "a soldier writing a letter home from the border, weighing duty against the ache "
                 "of missing family",
        "emotion_arc": "opens in quiet homesickness, deepens into internal conflict, ends in "
                        "solemn, resolute duty",
        "hook_direction": "a hook framed as the soldier's own private vow",
        "vocal": "restrained, weighted verses building into a powerful, anthemic chorus",
    },
    {
        "scene": "losing a childhood friend to addiction, channeling grief and helpless anger into "
                 "the only outlet that makes sense",
        "emotion_arc": "starts in numb shock, spirals into raw anger, ends in cathartic, aching "
                        "release",
        "hook_direction": "a hook about screaming what couldn't be said in time",
        "vocal": "intense, cracking verses into an explosive, cathartic chorus",
    },
    {
        "scene": "a small-town teenager packing one bag and leaving for the city against every "
                 "expectation placed on them",
        "emotion_arc": "opens in restless frustration, surges into reckless courage, ends in "
                        "defiant hope",
        "hook_direction": "a hook about outrunning everyone else's plan for your life",
        "vocal": "driving, urgent verses into a fast, rebellious chorus",
    },
    {
        "scene": "finally performing on a real stage after years of empty rooms and rejection, "
                 "the crowd actually singing back",
        "emotion_arc": "starts in disbelieving nervousness, rises into overwhelming validation, "
                        "ends in triumphant joy",
        "hook_direction": "a hook about finally, truly being heard",
        "vocal": "building, energetic verses into a massive, victorious chorus",
    },
    {
        "scene": "a city-wide power outage during a monsoon storm, neighbours coming together with "
                 "candles and shared food against the chaos",
        "emotion_arc": "opens in tense uncertainty, warms into unexpected unity, ends in grounded, "
                        "hopeful solidarity",
        "hook_direction": "a hook about light coming from people, not electricity",
        "vocal": "steady, building verses into a warm but powerful chorus",
    },
    {
        "scene": "confronting a controlling authority figure at home or work after years of "
                 "staying quiet",
        "emotion_arc": "starts in suppressed resentment, breaks into open confrontation, ends in "
                        "liberated self-respect",
        "hook_direction": "a hook about refusing to shrink any longer",
        "vocal": "tense, controlled verses erupting into a fierce, unrestrained chorus",
    },
    {
        "scene": "lying awake at 3am questioning whether the years spent chasing success actually "
                 "meant anything",
        "emotion_arc": "opens in restless doubt, spirals into existential unease, ends in "
                        "hard-won, grounded clarity",
        "hook_direction": "a hook that asks the same question until it finally answers itself",
        "vocal": "moody, introspective verses building into a resolute, driving chorus",
    },
    {
        "scene": "watching a once-clean childhood river turn polluted and lifeless, anger at the "
                 "indifference that let it happen",
        "emotion_arc": "starts in aching nostalgia, deepens into righteous anger, ends in a call "
                        "for accountability",
        "hook_direction": "a hook demanding someone finally answer for what was lost",
        "vocal": "gritty, urgent verses into a hard-hitting, anthemic chorus",
    },
    {
        "scene": "walking away from a toxic relationship for good, reclaiming a sense of self that "
                 "had quietly disappeared",
        "emotion_arc": "opens in exhausted resignation, hardens into clear-eyed resolve, ends in "
                        "liberated strength",
        "hook_direction": "a hook about finally choosing yourself",
        "vocal": "controlled, simmering verses breaking into a powerful, freeing chorus",
    },
    {
        "scene": "standing at a cremation ground after losing a mentor, processing grief while the "
                 "world keeps moving around you",
        "emotion_arc": "starts in numb disbelief, deepens into raw grief, ends in quiet, resolved "
                        "tribute",
        "hook_direction": "a hook framed as a promise to carry the mentor's lessons forward",
        "vocal": "restrained, aching verses building into an emotional, powerful chorus",
    },
    {
        "scene": "a band's final gig before going separate ways, playing like it's both a goodbye "
                 "and a celebration",
        "emotion_arc": "opens in bittersweet nostalgia, swells into shared pride, ends in defiant, "
                        "grateful closure",
        "hook_direction": "a hook about one last night mattering more than any ending",
        "vocal": "warm but driving verses into a huge, celebratory final chorus",
    },
    {
        "scene": "facing family and social judgment for an inter-caste or inter-religion "
                 "relationship, choosing love anyway",
        "emotion_arc": "starts in anxious pressure, hardens into quiet defiance, ends in "
                        "unshaken, committed resolve",
        "hook_direction": "a hook about choosing each other over everyone else's approval",
        "vocal": "tense, determined verses into a bold, unwavering chorus",
    },
]

HIPHOP_SCENARIOS = [
    {
        "scene": "growing up in a one-room chawl in Mumbai, writing rhymes in a school notebook "
                 "by the light of a single bulb",
        "emotion_arc": "starts in quiet frustration at cramped circumstances, builds into "
                        "stubborn hope, ends in fierce self-belief",
        "hook_direction": "a hook about a small room holding a very big dream",
        "flow": "measured, grounded verses building into a defiant, chanted hook",
    },
    {
        "scene": "working a warehouse night shift, scribbling bars on a break in the stairwell, "
                 "exhausted but unwilling to quit",
        "emotion_arc": "opens in weary frustration, deepens into quiet determination, ends in "
                        "stubborn pride",
        "hook_direction": "a hook contrasting the uniform on the body with the fire underneath it",
        "flow": "laid-back verses with a triplet flow switch into a shouted hook",
    },
    {
        "scene": "being mocked growing up for background or accent, turning that same mockery "
                 "into confident material on the mic",
        "emotion_arc": "starts in old, buried shame, hardens into righteous anger, ends in "
                        "reclaimed dignity and pride",
        "hook_direction": "a hook that repeats an old insult before flipping its meaning completely",
        "flow": "sharp, cutting verses into a triumphant, assertive hook",
    },
    {
        "scene": "a late-night cypher in a hostel room, trading bars with friends, proving skill "
                 "among people who actually know the craft",
        "emotion_arc": "opens in competitive nervous energy, rises into focused confidence, "
                        "ends in shared, respected pride",
        "hook_direction": "a hook celebrating being recognised by the people who matter most",
        "flow": "back-and-forth trade verses into a unison, chanted hook",
    },
    {
        "scene": "selling an old phone to afford one real studio session, betting everything on "
                 "a single track",
        "emotion_arc": "starts in anxious risk, deepens into total commitment, ends in cautious, "
                        "hard-earned hope",
        "hook_direction": "a hook about betting it all on one shot and meaning it",
        "flow": "tense, deliberate verses building into a driving, confident hook",
    },
    {
        "scene": "watching parents struggle to make ends meet, feeling the quiet weight of being "
                 "the eldest child expected to provide",
        "emotion_arc": "opens in heavy responsibility, aches with private fear, ends in "
                        "protective, determined resolve",
        "hook_direction": "a hook framed as a quiet promise to make the sacrifice worth it",
        "flow": "slow, weighted verses over a reflective beat into a melodic, sincere hook",
    },
    {
        "scene": "being asked to leave home after choosing music over a 'respectable' degree, "
                 "facing a hard conversation at the door",
        "emotion_arc": "starts in guilt and fear, hardens into painful conviction, ends in "
                        "vulnerable hope for eventual understanding",
        "hook_direction": "a hook framed as a promise that the risk will be proven right",
        "flow": "tense, deliberate verses into a pleading but strong hook",
    },
    {
        "scene": "busking at a local market with a borrowed speaker, imagining the crowd as a "
                 "sold-out arena",
        "emotion_arc": "opens in humble uncertainty, grows into stubborn pride, ends in "
                        "unshaken self-belief",
        "hook_direction": "a hook about belief mattering more than the size of the crowd",
        "flow": "energetic street-style verses into a chantable, crowd-style hook",
    },
    {
        "scene": "losing badly in a local rap battle, going home to rewrite every weak verse "
                 "before the next one",
        "emotion_arc": "starts in raw embarrassment, sharpens into hungry focus, ends in "
                        "sharper, more dangerous confidence",
        "hook_direction": "a hook about turning a loss into fuel",
        "flow": "aggressive double-time verses into a hard, punchy hook",
    },
    {
        "scene": "juggling a delivery job by day and writing verses on the ride between "
                 "drop-offs, running on almost no sleep",
        "emotion_arc": "opens in exhausted frustration, holds onto quiet ambition, ends in "
                        "gritty, unglamorous resolve",
        "hook_direction": "a hook contrasting the delivery bag with the mic waiting at home",
        "flow": "laid-back, tired verses building into an energised, driving hook",
    },
    {
        "scene": "a track finally going viral on social media after years of being ignored by "
                 "everyone",
        "emotion_arc": "starts in disbelief, rushes into overwhelming validation, ends in "
                        "humbled, grateful joy",
        "hook_direction": "a hook about the exact moment years of silence finally broke",
        "flow": "bright, energetic verses into a joyful, wide-open hook",
    },
    {
        "scene": "watching an old friend from the same neighbourhood choose a life of crime "
                 "while the narrator chose the mic instead",
        "emotion_arc": "opens in conflicted loyalty, aches with sadness for a path not shared, "
                        "ends in resolve to make it out and look back for others",
        "hook_direction": "a hook about two roads starting from the exact same corner",
        "flow": "heavy, deliberate verses into a somber but driving hook",
    },
    {
        "scene": "moving to Mumbai or Delhi from a small town with one bag and a notebook full "
                 "of unfinished verses",
        "emotion_arc": "starts in nervous uncertainty, deepens into lonely determination, ends "
                        "in proud, hard-won belonging",
        "hook_direction": "a hook about turning a notebook into a name people know",
        "flow": "reflective verses over a slower beat into a melodic hook with a key change",
    },
    {
        "scene": "an old teacher who once called the dream a waste of time, running into them "
                 "years later after real success",
        "emotion_arc": "starts in old resentment, flares briefly into vindication, settles into "
                        "calm, quiet satisfaction",
        "hook_direction": "a hook that repeats the old doubt before flipping it into proof",
        "flow": "sharp, cutting verses into a triumphant hook",
    },
    {
        "scene": "finally affording to take a struggling parent on their first proper holiday, "
                 "watching their face light up",
        "emotion_arc": "opens in nervous anticipation, swells into overwhelming pride and love, "
                        "ends in tearful gratitude",
        "hook_direction": "a hook framed as a direct thank-you to a parent who sacrificed everything",
        "flow": "warm, melodic verses into an uplifting, sung hook",
    },
]

RNB_SCENARIOS = [
    {
        "scene": "sitting on a Mumbai balcony at 2am, missing someone who moved to another city "
                 "for work months ago",
        "emotion_arc": "starts in quiet loneliness, deepens into aching longing, ends in tender "
                        "hope for a return",
        "hook_direction": "a hook about distance not being able to touch the way you feel",
        "vocal": "breathy, intimate verses building into a warm, soulful chorus",
    },
    {
        "scene": "finding an old WhatsApp chat with an ex still saved on the phone, scrolling "
                 "through it alone at night",
        "emotion_arc": "opens in quiet nostalgia, deepens into raw ache, ends in bittersweet "
                        "acceptance",
        "hook_direction": "a hook built around rereading the same old messages",
        "vocal": "soft falsetto verses into a full-voiced, slightly cracked chorus",
    },
    {
        "scene": "a monsoon evening, rain streaking down the window, deliberately choosing not "
                 "to call first",
        "emotion_arc": "starts in restless loneliness, tightens into wounded pride, settles into "
                        "fragile self-protective calm",
        "hook_direction": "a hook about silence saying more than any phone call could",
        "vocal": "smooth, restrained verses with soulful ad-libs in the outro",
    },
    {
        "scene": "running into an ex at a mutual friend's wedding, both keeping up polite smiles "
                 "in front of family",
        "emotion_arc": "starts in forced composure, cracks into sudden heartbreak beneath the "
                        "smile, ends in quiet dignity",
        "hook_direction": "a hook contrasting the smile on the surface with what's underneath",
        "vocal": "controlled, understated verses opening up only in the bridge",
    },
    {
        "scene": "packing the last box out of a shared apartment, finding one small item neither "
                 "person wants to claim",
        "emotion_arc": "moves from detached practicality into sudden grief, then gentle, "
                        "sentimental closure",
        "hook_direction": "a hook about one small object carrying an entire relationship",
        "vocal": "intimate, close verses with layered harmonies on the hook",
    },
    {
        "scene": "a late-night rooftop conversation with a close friend, finally admitting out "
                 "loud that it's really over",
        "emotion_arc": "starts in guarded denial, breaks into tearful honesty, ends in unexpected "
                        "lightness",
        "hook_direction": "a hook that turns confession into quiet relief",
        "vocal": "conversational verses opening into a warm chorus",
    },
    {
        "scene": "riding the last local train home after a breakup, watching reflections in the "
                 "dark window instead of the city outside",
        "emotion_arc": "begins in hollow exhaustion, drifts into aching self-reflection, ends in "
                        "quiet self-recognition",
        "hook_direction": "a hook using the train window's reflection as a mirror to the self",
        "vocal": "airy verses swelling into the final hook repeat",
    },
    {
        "scene": "cooking dinner for one for the first time in years, still instinctively "
                 "reaching for two plates",
        "emotion_arc": "starts in muscle-memory denial, aches into sudden loneliness, settles "
                        "into tender acceptance",
        "hook_direction": "a hook about habit outlasting the relationship itself",
        "vocal": "gentle, almost spoken verses into a full soulful belt on the last chorus",
    },
    {
        "scene": "seeing an ex's engagement announcement online at 3am, scrolling through old "
                 "photos afterward",
        "emotion_arc": "opens in disbelief and quiet jealousy, spirals into private sorrow, ends "
                        "in a hard-earned wish of goodwill",
        "hook_direction": "a hook admitting jealousy honestly before turning it into peace",
        "vocal": "hushed, trembling verses opening into a soaring, cathartic chorus",
    },
    {
        "scene": "receiving a wedding invitation with the '+1' line left blank, feeling it more "
                 "than expected",
        "emotion_arc": "starts in stinging embarrassment, deepens into aching solitude, resolves "
                        "into stubborn self-respect",
        "hook_direction": "a hook turning a blank line into a quiet declaration of self-worth",
        "vocal": "restrained verses with defiant strength in the final hook",
    },
    {
        "scene": "hearing an old 'our song' play unexpectedly in a café, alone at a corner table",
        "emotion_arc": "opens in involuntary tenderness, aches into fresh grief, settles into "
                        "gratitude for having had it at all",
        "hook_direction": "a hook about a song refusing to let a memory fade",
        "vocal": "vulnerable low verses with full emotional release on the bridge",
    },
    {
        "scene": "deleting a shared photo album at midnight, hesitating on the very last photo "
                 "before letting go",
        "emotion_arc": "starts in cold resolve, falters into sudden doubt, ends in slow, "
                        "deliberate release",
        "hook_direction": "a hook built around the pause before pressing delete",
        "vocal": "sparse verses with long held notes and layered harmony on the final chorus",
    },
    {
        "scene": "telling a worried friend 'I'm fine' for the fifth time in a week, finally "
                 "breaking down and admitting the truth",
        "emotion_arc": "opens in practiced denial, cracks into private exhaustion, ends in the "
                        "first honest admission of pain",
        "hook_direction": "a hook that repeats 'I'm fine' until it finally breaks into truth",
        "vocal": "clipped, guarded verses opening into a raw, unguarded chorus",
    },
    {
        "scene": "watching grandparents dance together at a family wedding, quietly wistful for "
                 "a love that lasted decades",
        "emotion_arc": "starts in wistful envy, deepens into longing, ends in quiet hope that "
                        "lasting love is still possible",
        "hook_direction": "a hook contrasting someone else's forever with your own uncertain future",
        "vocal": "warm, nostalgic verses lifting into a hopeful final chorus",
    },
    {
        "scene": "typing 'I miss you' late at night and deleting it three times before sending "
                 "nothing at all",
        "emotion_arc": "starts in aching temptation, spirals into wounded restraint, ends in "
                        "bruised but intact self-respect",
        "hook_direction": "a hook built entirely around the message that was never sent",
        "vocal": "hesitant, half-whispered verses with aching held notes in the chorus",
    },
]

FOLK_SCENARIOS = [
    {
        "scene": "a grandmother humming an old lullaby passed down through generations, rocking "
                 "a grandchild to sleep",
        "emotion_arc": "opens in tender warmth, deepens into quiet nostalgia for her own "
                        "childhood, ends in peaceful, timeless love",
        "hook_direction": "a hook framed as the lullaby itself, simple and repeating",
        "vocal": "soft, unhurried verses with a gentle, repeating refrain",
    },
    {
        "scene": "a farmer standing in a cracked, drought-stricken field, praying quietly for the "
                 "rains to finally come",
        "emotion_arc": "starts in weary desperation, deepens into humble faith, ends in fragile, "
                        "hopeful surrender",
        "hook_direction": "a hook framed as a direct, simple prayer for rain",
        "vocal": "raw, weathered verses building into an aching, devotional refrain",
    },
    {
        "scene": "a village wedding procession moving through narrow lanes at dusk, dhol beats "
                 "and traditional songs echoing through the streets",
        "emotion_arc": "opens in joyful anticipation, swells into communal celebration, ends in "
                        "warm, shared happiness",
        "hook_direction": "a hook that the whole procession could sing together",
        "vocal": "bright, communal-feeling verses with a rousing, chant-like refrain",
    },
    {
        "scene": "a Sufi seeker's longing for the divine beloved, using the language of earthly "
                 "love to describe a spiritual devotion",
        "emotion_arc": "starts in restless yearning, deepens into surrendered devotion, ends in "
                        "quiet, transcendent peace",
        "hook_direction": "a hook repeating the beloved's name as both prayer and love song",
        "vocal": "soaring, emotionally open verses building into an ecstatic, whirling refrain",
    },
    {
        "scene": "an old riverside village slowly emptying out as younger generations leave for "
                 "the city, the elders left behind",
        "emotion_arc": "opens in quiet loss, deepens into aching nostalgia, ends in resigned but "
                        "loving acceptance",
        "hook_direction": "a hook about a river that keeps flowing even as the village empties",
        "vocal": "weathered, storytelling verses with a mournful, lingering refrain",
    },
    {
        "scene": "a soldier's mother waiting by the door every evening for a letter that hasn't "
                 "arrived in weeks",
        "emotion_arc": "starts in anxious hope, deepens into aching worry, ends in unwavering, "
                        "patient faith",
        "hook_direction": "a hook framed as a mother's quiet daily prayer",
        "vocal": "restrained, tender verses with a gently repeating, prayerful refrain",
    },
    {
        "scene": "two lovers separated by distance, writing letters that take weeks to arrive, "
                 "treasuring every line",
        "emotion_arc": "opens in aching separation, warms into devoted patience, ends in hopeful "
                        "anticipation of reunion",
        "hook_direction": "a hook framed as a line from one of the letters",
        "vocal": "intimate, storytelling verses with a warm, yearning refrain",
    },
    {
        "scene": "a wandering minstrel telling old stories at a village fair, passing down "
                 "history through song to children gathered around",
        "emotion_arc": "opens in playful wonder, deepens into reverence for the old stories, "
                        "ends in joyful continuity",
        "hook_direction": "a hook that functions like the opening line of an old, familiar tale",
        "vocal": "storytelling verses with a bright, communal refrain",
    },
    {
        "scene": "a daughter leaving her childhood village for the city for the first time, her "
                 "mother's tearful blessing at the doorstep",
        "emotion_arc": "starts in nervous excitement, deepens into aching separation, ends in "
                        "tearful, loving blessing",
        "hook_direction": "a hook framed as a mother's blessing for the road ahead",
        "vocal": "tender, trembling verses with an emotional, swelling refrain",
    },
    {
        "scene": "a pilgrim walking barefoot toward a distant shrine, carrying a private sorrow "
                 "the whole journey",
        "emotion_arc": "opens in heavy grief, deepens into humble surrender, ends in quiet, "
                        "healing peace",
        "hook_direction": "a hook framed as a plea offered at journey's end",
        "vocal": "weathered, weary verses building into a devotional, releasing refrain",
    },
    {
        "scene": "a harvest festival by the riverside after the monsoon, the whole village giving "
                 "thanks for the land and the rains",
        "emotion_arc": "opens in relieved gratitude, swells into communal joy, ends in warm, "
                        "shared celebration",
        "hook_direction": "a hook of thanks that the whole gathering sings together",
        "vocal": "joyful, communal verses with a rousing, celebratory refrain",
    },
    {
        "scene": "an elderly craftsman teaching a dying traditional art to a reluctant "
                 "grandchild who would rather be elsewhere",
        "emotion_arc": "starts in quiet patience, deepens into gentle insistence, ends in a "
                        "small, hopeful moment of connection",
        "hook_direction": "a hook framed as wisdom passed from one generation to the next",
        "vocal": "measured, wise verses with a steady, teaching refrain",
    },
    {
        "scene": "a widow reflecting alone on decades of marriage, grief and gratitude "
                 "intertwined as she looks through old photographs",
        "emotion_arc": "opens in quiet grief, deepens into warm remembrance, ends in peaceful "
                        "gratitude for a life shared",
        "hook_direction": "a hook that thanks a lifetime rather than mourns its ending",
        "vocal": "soft, reflective verses with a gentle, aching refrain",
    },
    {
        "scene": "a shepherd under an open night sky, singing alone to pass the long hours of "
                 "solitary watch",
        "emotion_arc": "opens in quiet solitude, deepens into peaceful communion with the land, "
                        "ends in contented stillness",
        "hook_direction": "a hook that feels like it's sung to the stars themselves",
        "vocal": "unhurried, open verses with a spacious, echoing refrain",
    },
    {
        "scene": "childhood friends reuniting after years scattered across different cities, "
                 "sitting together late into the night reminiscing",
        "emotion_arc": "opens in nostalgic warmth, deepens into bittersweet awareness of time "
                        "passed, ends in grateful, renewed closeness",
        "hook_direction": "a hook about old friendships picking up exactly where they left off",
        "vocal": "warm, easy verses with a nostalgic, heartfelt refrain",
    },
]

HINDI_SCENARIOS = {
    "Pop": POP_SCENARIOS,
    "Rock": ROCK_SCENARIOS,
    "Hip-Hop/Rap": HIPHOP_SCENARIOS,
    "R&B/Soul": RNB_SCENARIOS,
    "Folk/Acoustic": FOLK_SCENARIOS,
}


# Per-genre structure, production, and mood, referenced by the
# prompt builder below.


GENRE_STRUCTURE = {
    "Pop": "short intro, verse 1, pre-chorus, chorus, verse 2, pre-chorus, chorus, "
           "bridge, final chorus, outro",
    "Rock": "intro riff, verse 1, pre-chorus, chorus, verse 2, chorus, guitar break, "
            "bridge, final chorus, outro",
    "Hip-Hop/Rap": "intro tag, verse 1, hook, verse 2, hook, short bridge or breakdown, "
                   "final hook (extended with ad-libs), outro tag",
    "R&B/Soul": "short intro, verse 1, pre-chorus, chorus, verse 2, pre-chorus, chorus, "
                "bridge, final chorus with ad-libs, outro",
    "Folk/Acoustic": "short instrumental intro, verse 1, refrain, verse 2, refrain, "
                      "instrumental interlude, final verse, refrain, gentle outro",
}

GENRE_PRODUCTION = {
    "Pop": "95-115 BPM, contemporary Hindi pop instrumentation with acoustic and electric "
           "guitar hooks, warm piano chords, modern pop-leaning drums, lush string swells "
           "on the choruses, polished radio-ready mix",
    "Rock": "120-145 BPM, driving distorted electric guitars, punchy live-feel drums, "
             "melodic bassline, occasional guitar lead, anthemic group vocals on the chorus",
    "Hip-Hop/Rap": "85-100 BPM, hard-hitting 808 bass, crisp trap hi-hats, dark minor-key "
                   "synths, occasional tabla or dhol flourish for regional texture, punchy "
                   "mix with clear vocal presence",
    "R&B/Soul": "70-85 BPM, warm Rhodes or electric piano chords, soft contemporary drums, "
                "deep bass, subtle string pads, layered background harmonies on the chorus",
    "Folk/Acoustic": "70-95 BPM or free tempo, acoustic harmonium, dholak or tabla, bansuri "
                      "flute accents, acoustic guitar or sarangi, minimal, authentic "
                      "folk/devotional texture with no electronic elements",
}

GENRE_MOOD = {
    "Pop": "warm, contemporary Hindi pop, radio-ready and emotionally sincere",
    "Rock": "raw, powerful contemporary Hindi rock, emotionally direct and unafraid of intensity",
    "Hip-Hop/Rap": "emotionally grounded and realistic, rooted in everyday hustle and its "
                   "real costs rather than generic bravado",
    "R&B/Soul": "intimate, emotionally honest, contemporary Hindi soul/R&B, understated but "
                "deeply felt",
    "Folk/Acoustic": "authentic, rooted traditional Hindi folk, reverent and emotionally "
                      "sincere without modern production gloss",
}


# PROMPT BUILDER

def make_hindi_prompt(genre, song_number):
    """
    Build a full commercial-style Suno generation prompt for a Hindi
    track: specific scene, explicit emotional arc, song structure,
    instrumentation, vocal/flow delivery, and hook direction.

    Scenarios are selected deterministically by song_number so the
    full scenario bank is cycled through evenly across all tracks
    in a genre, rather than relying on random draws that can repeat
    combinations early or cluster similar moods together.
    """
    scenarios = HINDI_SCENARIOS[genre]
    scenario = scenarios[(song_number - 1) % len(scenarios)]

    constraint = HINDI_CONSTRAINTS[genre]
    structure = GENRE_STRUCTURE[genre]
    production = GENRE_PRODUCTION[genre]
    mood = GENRE_MOOD[genre]

    is_rap = genre == "Hip-Hop/Rap"
    delivery_label = "Flow" if is_rap else "Vocal delivery"
    delivery_value = scenario["flow"] if is_rap else scenario["vocal"]

    return (
        f"Write and produce an original Hindi {genre} song, {constraint}. "
        f"\n\nSong concept: {scenario['scene']}. "
        f"\n\nEmotional arc: {scenario['emotion_arc']}. The verses should carry this "
        f"emotional progression honestly before landing on the resolution described, "
        f"rather than staying flat on one feeling throughout. "
        f"\n\nStructure: {structure}. "
        f"\n\nHook: {scenario['hook_direction']}, repeated as the song's core memorable line. "
        f"\n\n{delivery_label}: {delivery_value}. "
        f"\n\nProduction: {production}. "
        f"\n\nMood: {mood}. "
        f"\n\nDo not reference any real artist, song title, or existing melody. "
        f"Original composition only — no cover, no remix, no sample."
    )


# MAIN: GENERATE 300 ROWS (5 GENRES x 60)

dataset = []

for genre in TARGET_GENRES:
    genre_code = genre[:3].upper()
    if "Hip-Hop" in genre:
        genre_code = "HIP"
    if "Folk" in genre:
        genre_code = "FLK"
    if "R&B" in genre:
        genre_code = "RNB"   # keeps IDs consistent with HI_HUMAN_RNB_xxx used elsewhere

    for i in range(1, TRACKS_PER_GENRE + 1):
        song_id = f"AI_HIN_{genre_code}_{i:03d}"
        dataset.append({
            "song_id":        song_id,
            "title":          song_id,
            "artist":         MODEL_NAME,
            "language":       "Hindi",
            "final_genre":    genre,
            "human_ai_label": "ai",
            "target":         1,
            "release_year":   2026,
            "song_number":    i,
            "source_dataset": "Self-generated (Suno v5)",
            "audio_link":     "",
            "local_audio_path": f"{OUTPUT_AUDIO_DIR}/{song_id}.mp3",
            "production_tier": "High-Fidelity",
            "prompt_used":    make_hindi_prompt(genre, i),
        })

df_hindi = pd.DataFrame(dataset)
df_hindi.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("---------------------------------------------------------")
print("  HINDI SUNO v5 PROMPT GENERATION COMPLETE")
print("---------------------------------------------------------")
print(f"Total rows: {len(df_hindi)}")
print("\nGenre counts:")
print(df_hindi["final_genre"].value_counts())
print(f"\nSaved: {OUTPUT_CSV}")
print("\nSample prompts:")
display(df_hindi[["song_id", "final_genre", "prompt_used"]].head(10))

## 6. Verification: check all output files

Confirms that all three languages produced the expected manifests and audio.


In [ ]:
ai_root = Path("dataset_collection_clean/ai")
check_paths = [
    # English
    (str(ai_root / "english" / "audio"),                           "dir"),
    (str(ai_root / "english" / "english_ai_manifest.csv"),         "csv"),
    (str(ai_root / "english" / "english_missing_plan.csv"),        "csv"),
    # Chinese
    (str(ai_root / "chinese" / "audio"),                           "dir"),
    (str(ai_root / "chinese" / "chinese_ai_manifest.csv"),         "csv"),
    (str(ai_root / "chinese" / "chinese_missing_plan.csv"),        "csv"),
    (str(ai_root / "chinese" / "chinese_missing_suno_prompts.csv"),"csv"),
    # Hindi
    (str(ai_root / "hindi"   / "hindi_ai_300_suno_prompts.csv"),   "csv"),
    (str(ai_root / "hindi"   / "audio"),                           "dir"),
    # Combined missing plan
    (str(ai_root / "muse_missing_generation_plan.csv"),            "csv"),
]

print("=" * 60)
print("OUTPUT FILE VERIFICATION")
print("=" * 60)

for p, kind in check_paths:
    path = Path(p)
    if not path.exists():
        print(f"  MISSING  {p}")
        continue

    if kind == "csv":
        df_v = pd.read_csv(path)
        genre_info = ""
        if "final_genre" in df_v.columns:
            counts = df_v["final_genre"].value_counts()
            genre_info = " | " + ", ".join(f"{g}:{n}" for g, n in counts.items())
        print(f"  OK  {p}  ({len(df_v)} rows{genre_info})")
    elif kind == "dir":
        mp3s = list(path.glob("*.mp3"))
        print(f"  OK  {p}/  ({len(mp3s)} MP3 files)")

print("\nDone.")


## 7. Transfer manually downloaded Suno songs

Run this cell to move any manually downloaded MP3s from common drop folders
into the correct `dataset_collection_clean/ai/` sub-directories.



In [ ]:
# TRANSFER MANUALLY DOWNLOADED SUNO SONGS
# Reads genre from the parent folder name, numbers files
# sequentially, renames to AI_{LANG}_{GENRE}_{NNN}.mp3
# and moves into the correct audio/ folder.
import shutil, re
from pathlib import Path

_AI_ROOT = Path('dataset_collection_clean/ai')

# Map folder name keywords -> (lang_code, genre_tag, final_genre)
_FOLDER_MAP = {
    # Hindi genre folders
    'folk':    ('HIN', 'FOLK_ACOUSTIC', 'Folk/Acoustic'),
    'hip hop': ('HIN', 'HIP_HOP_RAP',   'Hip-Hop/Rap'),
    'hiphop':  ('HIN', 'HIP_HOP_RAP',   'Hip-Hop/Rap'),
    'pop':     ('HIN', 'POP',            'Pop'),
    'rnb':     ('HIN', 'R_B_SOUL',       'R&B/Soul'),
    'rock':    ('HIN', 'ROCK',           'Rock'),
    # Chinese genre folders
    'hiphop_cn':  ('CN', 'HIP_HOP_RAP', 'Hip-Hop/Rap'),
    'rnb_cn':     ('CN', 'R_B_SOUL',    'R&B/Soul'),
}

# Source genre folders to scan -> language they belong to
# (folder_path, language_code, audio_dest)
_GENRE_FOLDERS = [
    # Hindi
    (_AI_ROOT / 'hindi' / 'Folk',          'HIN', _AI_ROOT / 'hindi'   / 'audio'),
    (_AI_ROOT / 'hindi' / 'Hip hop hindi',  'HIN', _AI_ROOT / 'hindi'   / 'audio'),
    (_AI_ROOT / 'hindi' / 'pop hindi',      'HIN', _AI_ROOT / 'hindi'   / 'audio'),
    (_AI_ROOT / 'hindi' / 'RNB hindi',      'HIN', _AI_ROOT / 'hindi'   / 'audio'),
    (_AI_ROOT / 'hindi' / 'rock hindi',     'HIN', _AI_ROOT / 'hindi'   / 'audio'),
    # Chinese (Suno gap-fill)
    (_AI_ROOT / 'chinese' / 'HipHop',       'CN',  _AI_ROOT / 'chinese' / 'audio'),
    (_AI_ROOT / 'chinese' / 'RnB',          'CN',  _AI_ROOT / 'chinese' / 'audio'),
]

# Map folder name  genre tag + final_genre label
_FOLDER_NAME_TO_GENRE = {
    'Folk':         ('FOLK_ACOUSTIC', 'Folk/Acoustic'),
    'Hip hop hindi':('HIP_HOP_RAP',   'Hip-Hop/Rap'),
    'pop hindi':    ('POP',           'Pop'),
    'RNB hindi':    ('R_B_SOUL',      'R&B/Soul'),
    'rock hindi':   ('ROCK',          'Rock'),
    'HipHop':       ('HIP_HOP_RAP',   'Hip-Hop/Rap'),
    'RnB':          ('R_B_SOUL',      'R&B/Soul'),
}

def _next_index(audio_dir, lang_code, genre_tag):
    """Return the next available sequential index for this lang+genre."""
    pattern = re.compile(
        rf'^AI_{lang_code}_{genre_tag}_(\d+)\.mp3$', re.IGNORECASE
    )
    existing = []
    if audio_dir.exists():
        for f in audio_dir.glob('*.mp3'):
            m = pattern.match(f.name)
            if m:
                existing.append(int(m.group(1)))
    return max(existing, default=0) + 1

print('SUNO SONG TRANSFER')
print('=' * 55)
_total_moved = _total_skipped = _total_unrouted = 0
_manifest_rows = []

for _src_folder, _lang, _audio_dst in _GENRE_FOLDERS:
    if not _src_folder.exists():
        print(f'  [not found]  {_src_folder}')
        continue

    _genre_info = _FOLDER_NAME_TO_GENRE.get(_src_folder.name)
    if _genre_info is None:
        print(f'  [unrecognised folder name]  {_src_folder.name}')
        _total_unrouted += 1
        continue

    _genre_tag, _final_genre = _genre_info
    _mp3s = sorted(_src_folder.glob('*.mp3'))

    if not _mp3s:
        print(f'  [empty]  {_src_folder}')
        continue

    _audio_dst.mkdir(parents=True, exist_ok=True)
    _idx = _next_index(_audio_dst, _lang, _genre_tag)
    print(f'\n  {_src_folder.name}  ({len(_mp3s)} MP3s)  ->  {_audio_dst}')
    print(f'  Starting index: {_idx:03d}  |  genre tag: {_genre_tag}  |  lang: {_lang}')

    for _mp3 in _mp3s:
        _new_name = f'AI_{_lang}_{_genre_tag}_{_idx:03d}.mp3'
        _dst = _audio_dst / _new_name
        if _dst.exists():
            print(f'    SKIP (exists)  {_new_name}')
            _total_skipped += 1
        else:
            shutil.copy2(str(_mp3), str(_dst))  # copy first, delete after verify
            if _dst.exists() and _dst.stat().st_size > 0:
                _mp3.unlink()  # remove original only after confirmed copy
                print(f'    MOVED  {_mp3.name}  ->  {_new_name}')
                _total_moved += 1
                _manifest_rows.append({
                    'song_id':        _new_name.replace('.mp3',''),
                    'language':       'Hindi' if _lang == 'HIN' else 'Chinese',
                    'language_code':  'hi' if _lang == 'HIN' else 'zh',
                    'final_genre':    _final_genre,
                    'human_ai_label': 'ai',
                    'model':          'Suno v5',
                    'source_dataset': 'Self-generated',
                    'local_audio_path': str(_dst),
                })
            else:
                print(f'    ERROR  copy failed for {_mp3.name}')
        _idx += 1

print()
print(f'Done.  Moved: {_total_moved}  |  Already exists: {_total_skipped}  |  Unrouted: {_total_unrouted}')
print()

# Show final counts in each audio folder
print('Audio folder counts after transfer:')
for _lang_name, _afolder in [('Hindi',   _AI_ROOT/'hindi'/'audio'),
                               ('Chinese', _AI_ROOT/'chinese'/'audio'),
                               ('English', _AI_ROOT/'english'/'audio')]:
    _by_genre = {}
    if _afolder.exists():
        for _f in _afolder.glob('*.mp3'):
            # extract genre tag from filename AI_LANG_GENRE_NNN.mp3
            _parts = _f.stem.split('_')
            _gtag = '_'.join(_parts[2:-1]) if len(_parts) > 3 else 'UNKNOWN'
            _by_genre[_gtag] = _by_genre.get(_gtag, 0) + 1
    _total = sum(_by_genre.values())
    print(f'  {_lang_name}: {_total} total  |  {_by_genre}')

# Save a transfer log CSV
if _manifest_rows:
    import pandas as pd
    _log_path = _AI_ROOT / 'suno_transfer_log.csv'
    pd.DataFrame(_manifest_rows).to_csv(_log_path, index=False, encoding='utf-8-sig')
    print(f'\nTransfer log saved: {_log_path}')


### 7b. Rebuild AI manifests from audio folders

Run this immediately after the transfer cell above.
Scans each `audio/` folder, reads duration from every MP3,
and writes/updates the per-language manifest CSVs so they
reflect all files including the newly transferred Suno songs.

In [ ]:
# REBUILD AI MANIFESTS FROM AUDIO FOLDERS
# Run after the Suno transfer cell to sync manifests.
import re
import pandas as pd
import numpy as np
from pathlib import Path

try:
    from mutagen import File as MutagenFile
    MUTAGEN_AVAILABLE = True
except ImportError:
    MUTAGEN_AVAILABLE = False
    print('mutagen not installed — duration will be NaN')

_AI_ROOT = Path('dataset_collection_clean/ai')

# genre tag in filename -> final_genre label
_GENRE_TAG_MAP = {
    'FOLK_ACOUSTIC': 'Folk/Acoustic',
    'HIP_HOP_RAP':   'Hip-Hop/Rap',
    'HIP':           'Hip-Hop/Rap',   # Muse short tag
    'POP':           'Pop',
    'R_B_SOUL':      'R&B/Soul',      # Suno underscore variant
    'R&B_SOUL':      'R&B/Soul',      # Muse ampersand variant
    'R&B':           'R&B/Soul',      # short variant
    'ROCK':          'Rock',
    'FOLK':          'Folk/Acoustic', # short variant
}

_LANG_CONFIG = [
    ('Hindi',   'HIN', 'hi', _AI_ROOT / 'hindi'   / 'audio', _AI_ROOT / 'hindi'   / 'hindi_ai_300_suno_prompts.csv'),
    ('Chinese', 'CN',  'zh', _AI_ROOT / 'chinese' / 'audio', _AI_ROOT / 'chinese' / 'chinese_ai_manifest.csv'),
    ('English', 'EN',  'en', _AI_ROOT / 'english' / 'audio', _AI_ROOT / 'english' / 'english_ai_manifest.csv'),
]

# filename pattern: AI_{LANG}_{GENRE_TAG}_{NNN}.mp3
_FNAME_RE = re.compile(
    r'^AI_(?P<lang>[A-Z]+)_(?P<genre>.+)_(?P<num>\d{3})\.mp3$', re.IGNORECASE
)

def _parse_mp3(path):
    """Return (song_id, genre_tag, duration_seconds) from an MP3 path."""
    m = _FNAME_RE.match(path.name)
    if m:
        # Normalise: R&B_SOUL and R_B_SOUL both -> R&B_SOUL
        _raw = m.group('genre').upper()
        # Normalise all R&B tag variants -> R&B_SOUL
        genre_tag = re.sub(r'R[_&]B[_]?SOUL|R&B$', 'R&B_SOUL', _raw)
    else:
        genre_tag = 'UNKNOWN'
    song_id   = path.stem
    if MUTAGEN_AVAILABLE:
        try:
            audio = MutagenFile(str(path))
            dur = round(float(audio.info.length), 2) if audio and hasattr(audio, 'info') else np.nan
        except Exception:
            dur = np.nan
    else:
        dur = np.nan
    return song_id, genre_tag, dur

print('REBUILDING AI MANIFESTS')
print('=' * 55)

for _lang_name, _lang_code, _lang_iso, _audio_dir, _manifest_path in _LANG_CONFIG:
    if not _audio_dir.exists():
        print(f'  [{_lang_name}] audio folder not found: {_audio_dir}')
        continue

    _mp3s = sorted(_audio_dir.glob('*.mp3'))
    if not _mp3s:
        print(f'  [{_lang_name}] no MP3s found in {_audio_dir}')
        continue

    # Build rows for every MP3 currently in the audio folder
    # Audio folder is the source of truth — rebuild clean from disk.
    _rows = []
    for _mp3 in _mp3s:
        _song_id, _genre_tag, _dur = _parse_mp3(_mp3)
        _final_genre = _GENRE_TAG_MAP.get(_genre_tag, _genre_tag)
        _rows.append({
            'song_id':          _song_id,
            'language':         _lang_name,
            'language_code':    _lang_iso,
            'final_genre':      _final_genre,
            'human_ai_label':   'ai',
            'model':            'Suno v5',
            'source_dataset':   'Self-generated',
            'local_audio_path': str(_mp3),
            'duration_seconds': _dur,
        })

    _out = pd.DataFrame(_rows)

    _manifest_path.parent.mkdir(parents=True, exist_ok=True)
    _out.to_csv(_manifest_path, index=False, encoding='utf-8-sig')

    # Summary
    _genre_counts = _out['final_genre'].value_counts()
    print(f'\n  [{_lang_name}] {len(_out)} MP3s  ->  {_manifest_path.name}')
    for _g, _n in _genre_counts.items():
        _status = '' if _n >= 60 else f'⚠ need {60-_n} more'
        print(f'    {_g:<20} {_n:>3}  {_status}')

print('\nManifests updated.')


## 8. AI dataset Exploratory Data Analysis

Mirrors the human EDA (Cells 10A–10D) for the AI-generated corpus.
Loads all three language manifests, attaches measured audio duration,
and produces the same descriptive tables and figures.

In [ ]:
# Load and preprocess the AI song corpus

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

try:
    from mutagen import File as MutagenFile
    MUTAGEN_AVAILABLE = True
except ImportError:
    MUTAGEN_AVAILABLE = False
    print('mutagen not installed - duration will not be computed from audio files')

_AI_ROOT = Path('dataset_collection_clean/ai')

AI_MANIFEST_FILES = [
    ('English', _AI_ROOT / 'english' / 'english_ai_manifest.csv',  _AI_ROOT / 'english' / 'audio'),
    ('Chinese', _AI_ROOT / 'chinese' / 'chinese_ai_manifest.csv',  _AI_ROOT / 'chinese' / 'audio'),
    ('Hindi',   _AI_ROOT / 'hindi'   / 'hindi_ai_300_suno_prompts.csv', _AI_ROOT / 'hindi' / 'audio'),
]

ai_rows = []

for _lang, _csv, _audio_dir in AI_MANIFEST_FILES:
    if not _csv.exists():
        print(f'  Missing: {_csv}')
        continue

    _df = pd.read_csv(_csv)
    _df['language'] = _lang
    _df['human_ai_label'] = 'ai'

    # Use new_song_id (Muse) or song_id (Suno prompts) as the canonical id
    if 'new_song_id' in _df.columns:
        _df['song_id'] = _df['new_song_id']

    _audio_paths, _durations = [], []
    for _, _row in _df.iterrows():
        _sid = str(_row.get('song_id', _row.get('new_song_id', ''))).strip()
        _path = _audio_dir / f'{_sid}.mp3'
        _audio_paths.append(str(_path))
        if _path.exists() and MUTAGEN_AVAILABLE:
            try:
                _audio = MutagenFile(str(_path))
                _dur = round(float(_audio.info.length), 2) if _audio and hasattr(_audio, 'info') else np.nan
            except Exception:
                _dur = np.nan
        else:
            _dur = np.nan
        _durations.append(_dur)

    _df['audio_path'] = _audio_paths
    _df['duration_seconds'] = _durations
    ai_rows.append(_df)
    print(f'  {_lang}: {len(_df)} rows loaded')

df_ai = pd.concat(ai_rows, ignore_index=True)

df_ai['language'] = df_ai['language'].astype(str)
df_ai['final_genre'] = df_ai['final_genre'].astype(str)
df_ai['duration_seconds'] = pd.to_numeric(df_ai['duration_seconds'], errors='coerce')
df_ai['duration_minutes'] = df_ai['duration_seconds'] / 60
df_ai['duration_hours']   = df_ai['duration_seconds'] / 3600

GENRE_ORDER = ['Pop', 'Rock', 'Hip-Hop/Rap', 'R&B/Soul', 'Folk/Acoustic']
LANG_ORDER  = ['English', 'Hindi', 'Chinese']

df_ai['final_genre'] = pd.Categorical(df_ai['final_genre'], categories=GENRE_ORDER, ordered=True)
df_ai['language']    = pd.Categorical(df_ai['language'],    categories=LANG_ORDER,  ordered=True)

print(f'\nAI corpus total rows: {len(df_ai)}')
df_ai[['song_id','language','final_genre','duration_seconds']].head()


In [ ]:
# AI corpus: song count and total duration by language/genre

ai_count_table = (
    df_ai.groupby(['language', 'final_genre'], observed=False)
    .size().reset_index(name='song_count')
)

ai_hours_table = (
    df_ai.groupby(['language', 'final_genre'], observed=False)['duration_hours']
    .sum().reset_index(name='total_hours')
)

ai_summary = ai_count_table.merge(ai_hours_table, on=['language', 'final_genre'])
ai_summary['total_hours'] = ai_summary['total_hours'].round(2)

print('AI corpus — song count and duration summary:')
display(ai_summary)

ai_summary_csv = _AI_ROOT / 'ai_genre_song_count_and_hours_summary.csv'
ai_summary.to_csv(ai_summary_csv, index=False, encoding='utf-8-sig')
print(f'Saved: {ai_summary_csv}')

ai_pivot_counts = (
    ai_summary.pivot(index='final_genre', columns='language', values='song_count')
    .reindex(GENRE_ORDER).reindex(columns=LANG_ORDER)
)
ai_pivot_hours = (
    ai_summary.pivot(index='final_genre', columns='language', values='total_hours')
    .reindex(GENRE_ORDER).reindex(columns=LANG_ORDER)
)
ai_language_hours = (
    df_ai.groupby('language', observed=False)['duration_hours']
    .sum().reindex(LANG_ORDER).reset_index()
)
ai_language_hours['duration_hours'] = ai_language_hours['duration_hours'].round(2)


In [ ]:
#  AI corpus: Figure 5 (count) and Figure 6 (duration by genre)

plt.rcParams.update({
    'figure.figsize': (12, 7), 'axes.titlesize': 15, 'axes.labelsize': 12,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
})

LANG_COLORS = {'English': '#4C78A8', 'Hindi': '#F58518', 'Chinese': '#54A24B'}

def _bar_labels(ax, fmt='{:.0f}', padding=0.02):
    ymax = ax.get_ylim()[1]
    for bar in ax.patches:
        h = bar.get_height()
        if pd.isna(h) or h == 0: continue
        ax.text(bar.get_x() + bar.get_width()/2, h + ymax*padding,
                fmt.format(h), ha='center', va='bottom', fontsize=10)

# Figure 5: song count
ax = ai_pivot_counts.plot(kind='bar', width=0.78,
    color=[LANG_COLORS[l] for l in LANG_ORDER], edgecolor='white', linewidth=1)
ax.set_title('Figure 5: AI Songs — Number of Songs by Genre and Language', pad=16)
ax.set_xlabel('Genre'); ax.set_ylabel('Number of songs')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(title='Language', frameon=False)
_bar_labels(ax)
plt.tight_layout()
_f5 = _AI_ROOT / 'visual_ai_song_count_by_genre_language.png'
plt.savefig(_f5, bbox_inches='tight'); plt.show()
print(f'Figure saved: {_f5}')

# Figure 6: duration
ax = ai_pivot_hours.plot(kind='bar', width=0.78,
    color=[LANG_COLORS[l] for l in LANG_ORDER], edgecolor='white', linewidth=1)
ax.set_title('Figure 6: AI Songs — Total Audio Duration by Genre and Language', pad=16)
ax.set_xlabel('Genre'); ax.set_ylabel('Total duration (hours)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(title='Language', frameon=False)
_bar_labels(ax, fmt='{:.2f}')
plt.tight_layout()
_f6 = _AI_ROOT / 'visual_ai_audio_hours_by_genre_language.png'
plt.savefig(_f6, bbox_inches='tight'); plt.show()
print(f'Figure saved: {_f6}')


In [ ]:
# AI corpus: Figure 7 (by language) and Figure 8 (stacked)

# Figure 7: total duration by language
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(ai_language_hours['language'].astype(str), ai_language_hours['duration_hours'],
    color=[LANG_COLORS[l] for l in ai_language_hours['language'].astype(str)],
    edgecolor='white', linewidth=0.8, width=0.55)
ax.set_title('Figure 7: AI Songs — Total Audio Duration by Language', pad=12)
ax.set_xlabel('Language'); ax.set_ylabel('Total duration (hours)')
ax.grid(axis='y', linestyle='--', alpha=0.3)
_max_h = ai_language_hours['duration_hours'].max()
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+_max_h*0.02,
            f'{h:.2f} h', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
_f7 = _AI_ROOT / 'visual_ai_total_hours_by_language.png'
plt.savefig(_f7, bbox_inches='tight'); plt.show()
print(f'Figure saved: {_f7}')

# Figure 8: stacked by genre
fig, ax = plt.subplots(figsize=(7, 4.3))
ai_pivot_hours.plot(kind='bar', stacked=True, width=0.65,
    color=[LANG_COLORS[l] for l in LANG_ORDER], edgecolor='white', linewidth=0.8, ax=ax)
ax.set_title('Figure 8: AI Songs — Total Duration per Genre, Decomposed by Language', pad=12)
ax.set_xlabel('Genre'); ax.set_ylabel('Total duration (hours)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(title='Language', frameon=False, fontsize=9)
_totals = ai_pivot_hours.sum(axis=1)
for i, t in enumerate(_totals):
    ax.text(i, t+_totals.max()*0.02, f'{t:.2f} h', ha='center', va='bottom', fontsize=8.5)
plt.tight_layout()
_f8 = _AI_ROOT / 'visual_ai_stacked_hours_by_genre.png'
plt.savefig(_f8, bbox_inches='tight'); plt.show()
print(f'Figure saved: {_f8}')

print('\nAI Corpus summary statistics')
print('-' * 40)
print(f'Total AI songs  : {len(df_ai)}')
print(f'Total duration  : {round(df_ai["duration_hours"].sum(), 2)} hours')
print('\nSongs by language:')
print(df_ai['language'].value_counts().reindex(LANG_ORDER))
print('\nDuration by language (hours):')
print(ai_language_hours.to_string(index=False))


## 9. Combined corpus Human + AI

Merges the human and AI datasets into a single dataframe and produces
combined corpus statistics and figures for the dissertation.

In [ ]:
# Combined Human + AI corpus EDA
# Self-contained: loads human manifests fresh from disk so
# it works even if Cell 10A was not run beforehand.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

_MANIFEST_DIR  = Path('dataset_collection_clean/manifests')
_HUMAN_AUDIO_ROOT = Path('dataset_collection_clean/human')
_AI_ROOT       = Path('dataset_collection_clean/ai')
_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

GENRE_ORDER  = ['Pop', 'Rock', 'Hip-Hop/Rap', 'R&B/Soul', 'Folk/Acoustic']
LANG_ORDER   = ['English', 'Hindi', 'Chinese']
LABEL_ORDER  = ['human', 'ai']
LANG_COLORS  = {'English': '#4C78A8', 'Hindi': '#F58518', 'Chinese': '#54A24B'}
LABEL_COLORS = {'human': '#4C78A8', 'ai': '#E45756'}

# Load human manifests
_HUMAN_MANIFESTS = [
    ('English', _MANIFEST_DIR / 'english_human_with_youtube_urls.csv', _HUMAN_AUDIO_ROOT / 'english' / 'audio'),
    ('Hindi',   _MANIFEST_DIR / 'hindi_human_with_youtube_urls.csv',   _HUMAN_AUDIO_ROOT / 'hindi'   / 'audio'),
    ('Chinese', _MANIFEST_DIR / 'chinese_human_with_youtube_urls.csv', _HUMAN_AUDIO_ROOT / 'chinese' / 'audio'),
]

try:
    from mutagen import File as _MutagenFile
    _MUTAGEN_OK = True
except ImportError:
    _MUTAGEN_OK = False

_human_rows = []
for _lang, _csv, _audio_dir in _HUMAN_MANIFESTS:
    if not _csv.exists():
        print(f'  [human] Missing: {_csv}')
        continue
    _df = pd.read_csv(_csv)
    _df['language'] = _lang
    _df['human_ai_label'] = 'human'
    # Compute duration from local audio files
    _df['duration_seconds'] = pd.to_numeric(_df.get('duration_seconds', np.nan), errors='coerce')
    _durs = {}
    if _audio_dir.exists() and _MUTAGEN_OK:
        for _mp3 in _audio_dir.glob('*.mp3'):
            try:
                _a = _MutagenFile(str(_mp3))
                _d = round(float(_a.info.length), 2) if _a and hasattr(_a, 'info') else np.nan
            except Exception:
                _d = np.nan
            # Index by stem as-is AND by swapping HUMAN_EN_ <-> EN_HUMAN_ prefix
            _stem = _mp3.stem
            _durs[_stem] = _d
            # Handle HUMAN_EN_* <-> EN_HUMAN_* naming mismatch
            import re as _re
            _alt = _re.sub(r'^(HUMAN)_([A-Z]+)_', r'\2_\1_', _stem)
            if _alt != _stem:
                _durs[_alt] = _d
            _alt2 = _re.sub(r'^([A-Z]+)_(HUMAN)_', r'\2_\1_', _stem)
            if _alt2 != _stem:
                _durs[_alt2] = _d
    if _durs:
        _df['duration_seconds'] = _df['song_id'].map(_durs).combine_first(_df['duration_seconds'])
    _mp3_count = len(list(_audio_dir.glob('*.mp3'))) if _audio_dir.exists() else 0
    _with_dur = _df['duration_seconds'].notna().sum()
    _human_rows.append(_df)
    print(f'  [human] {_lang}: {len(_df)} rows | {_mp3_count} MP3s found | {_with_dur} with duration')

df_human = pd.concat(_human_rows, ignore_index=True) if _human_rows else pd.DataFrame()

# Load AI manifests
_AI_MANIFESTS = [
    ('English', _AI_ROOT / 'english' / 'english_ai_manifest.csv'),
    ('Chinese', _AI_ROOT / 'chinese' / 'chinese_ai_manifest.csv'),
    ('Hindi',   _AI_ROOT / 'hindi'   / 'hindi_ai_300_suno_prompts.csv'),
]

_ai_rows = []
for _lang, _csv in _AI_MANIFESTS:
    if not _csv.exists():
        print(f'  [ai] Missing: {_csv}')
        continue
    _df = pd.read_csv(_csv)
    _df['language'] = _lang
    _df['human_ai_label'] = 'ai'
    if 'new_song_id' in _df.columns and 'song_id' not in _df.columns:
        _df['song_id'] = _df['new_song_id']
    if 'duration_seconds' not in _df.columns:
        _df['duration_seconds'] = np.nan
    _ai_rows.append(_df)
    print(f'  [ai]    {_lang}: {len(_df)} rows')

df_ai_combined = pd.concat(_ai_rows, ignore_index=True) if _ai_rows else pd.DataFrame()

# Merge 
KEEP_COLS = ['song_id', 'language', 'final_genre', 'duration_seconds', 'human_ai_label']

def _safe_select(frame, cols):
    return frame[[c for c in cols if c in frame.columns]].copy()

df_combined = pd.concat(
    [_safe_select(df_human, KEEP_COLS), _safe_select(df_ai_combined, KEEP_COLS)],
    ignore_index=True
)

df_combined['duration_seconds'] = pd.to_numeric(df_combined['duration_seconds'], errors='coerce')
df_combined['duration_hours']   = df_combined['duration_seconds'] / 3600
df_combined['final_genre']      = pd.Categorical(df_combined['final_genre'],    GENRE_ORDER, ordered=True)
df_combined['language']         = pd.Categorical(df_combined['language'],       LANG_ORDER,  ordered=True)
df_combined['human_ai_label']   = pd.Categorical(df_combined['human_ai_label'], LABEL_ORDER, ordered=True)

#  Save combined manifest
_combined_csv = _MANIFEST_DIR / 'all_combined_human_ai.csv'
df_combined.to_csv(_combined_csv, index=False, encoding='utf-8-sig')
print(f'\nCombined manifest saved: {_combined_csv}')

# Summary table 
_comb_count = (
    df_combined.groupby(['human_ai_label', 'language', 'final_genre'], observed=False)
    .size().reset_index(name='song_count')
)
_comb_hours = (
    df_combined.groupby(['human_ai_label', 'language', 'final_genre'], observed=False)['duration_hours']
    .sum().reset_index(name='total_hours')
)
_comb_summary = _comb_count.merge(_comb_hours, on=['human_ai_label', 'language', 'final_genre'])
_comb_summary['total_hours'] = _comb_summary['total_hours'].round(2)
print('\nCombined corpus summary:')
display(_comb_summary)

#  Plotting helpers
plt.rcParams.update({
    'figure.figsize': (13, 7), 'axes.titlesize': 15, 'axes.labelsize': 12,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
})

def _bar_labels(ax, fmt='{:.0f}', padding=0.02):
    ymax = ax.get_ylim()[1]
    for bar in ax.patches:
        h = bar.get_height()
        if pd.isna(h) or h == 0: continue
        ax.text(bar.get_x()+bar.get_width()/2, h+ymax*padding,
                fmt.format(h), ha='center', va='bottom', fontsize=9)

# Figure 9: Human vs AI song count by language
_pivot_label_lang = (
    df_combined.groupby(['human_ai_label', 'language'], observed=False)
    .size().unstack('language').reindex(LABEL_ORDER).reindex(columns=LANG_ORDER)
)
fig, ax = plt.subplots()
_pivot_label_lang.plot(kind='bar', width=0.65,
    color=[LANG_COLORS[l] for l in LANG_ORDER], edgecolor='white', linewidth=1, ax=ax)
ax.set_title('Figure 9: Song Count — Human vs AI by Language', pad=14)
ax.set_xlabel('Corpus type'); ax.set_ylabel('Number of songs')
ax.set_xticklabels(['Human', 'AI'], rotation=0)
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(title='Language', frameon=False)
_bar_labels(ax)
plt.tight_layout()
_f9 = _MANIFEST_DIR / 'visual_combined_count_human_vs_ai_by_language.png'
plt.savefig(_f9, bbox_inches='tight'); plt.show()
print(f'Figure saved: {_f9}')

# Figure 10: song count by genre, human vs ai
_pivot_genre_label = (
    df_combined.groupby(['final_genre', 'human_ai_label'], observed=False)
    .size().unstack('human_ai_label').reindex(GENRE_ORDER).reindex(columns=LABEL_ORDER)
)
fig, ax = plt.subplots()
_pivot_genre_label.plot(kind='bar', width=0.65,
    color=[LABEL_COLORS[l] for l in LABEL_ORDER], edgecolor='white', linewidth=1, ax=ax)
ax.set_title('Figure 10: Combined Corpus — Song Count by Genre (Human vs AI)', pad=14)
ax.set_xlabel('Genre'); ax.set_ylabel('Number of songs')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(title='Type', frameon=False)
_bar_labels(ax)
plt.tight_layout()
_f10 = _MANIFEST_DIR / 'visual_combined_count_by_genre_human_vs_ai.png'
plt.savefig(_f10, bbox_inches='tight'); plt.show()
print(f'Figure saved: {_f10}')

# Figure 11: stacked total by language
_pivot_lang_label = (
    df_combined.groupby(['language', 'human_ai_label'], observed=False)
    .size().unstack('human_ai_label').reindex(LANG_ORDER).reindex(columns=LABEL_ORDER)
)
fig, ax = plt.subplots(figsize=(6, 4.5))
_pivot_lang_label.plot(kind='bar', stacked=True, width=0.55,
    color=[LABEL_COLORS[l] for l in LABEL_ORDER], edgecolor='white', linewidth=0.8, ax=ax)
ax.set_title('Figure 11: Total Songs per Language (Human + AI stacked)', pad=12)
ax.set_xlabel('Language'); ax.set_ylabel('Number of songs')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(title='Type', frameon=False)
_totals_lang = _pivot_lang_label.sum(axis=1)
for i, t in enumerate(_totals_lang):
    ax.text(i, t+_totals_lang.max()*0.02, str(int(t)), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
_f11 = _MANIFEST_DIR / 'visual_combined_stacked_by_language.png'
plt.savefig(_f11, bbox_inches='tight'); plt.show()
print(f'Figure saved: {_f11}')

# Overall stats 
print('\n' + '=' * 50)
print('FULL CORPUS SUMMARY')
print('=' * 50)
print(f'Total songs            : {len(df_combined)}')
print(f'  Human                : {(df_combined["human_ai_label"]=="human").sum()}')
print(f'  AI                   : {(df_combined["human_ai_label"]=="ai").sum()}')
print(f'Total duration         : {round(df_combined["duration_hours"].sum(), 2)} hours')
print(f'  Human                : {round(df_combined.loc[df_combined["human_ai_label"]=="human","duration_hours"].sum(), 2)} hours')
print(f'  AI                   : {round(df_combined.loc[df_combined["human_ai_label"]=="ai",  "duration_hours"].sum(), 2)} hours')
print('\nSongs by language:')
print(df_combined.groupby(['language','human_ai_label'], observed=False).size().unstack(fill_value=0).reindex(LANG_ORDER))
print('\nSongs by genre:')
print(df_combined.groupby(['final_genre','human_ai_label'], observed=False).size().unstack(fill_value=0).reindex(GENRE_ORDER))


# MP3 Bitrate Normalization

Scans every MP3 in the dataset, reports the bitrate distribution, and re-encodes any file whose bitrate is **above** the dataset minimum down to the common target.

.


## Imports & path configuration

In [ ]:
import os
import json
import shutil
import subprocess
import tempfile
from pathlib import Path
from collections import Counter

import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────────────
BASE = Path("dataset_collection_clean")

AUDIO_DIRS = [
    BASE / "human" / "english" / "audio",
    BASE / "human" / "hindi"   / "audio",
    BASE / "human" / "chinese" / "audio",
    BASE / "ai"    / "english" / "audio",
    BASE / "ai"    / "chinese" / "audio",
]

# Standard MP3 bitrates (kbps) — valid snap-points for re-encoding
STANDARD_BITRATES = [32, 40, 48, 56, 64, 80, 96, 112, 128, 160, 192, 224, 256, 320]

# Safety: never re-encode below this floor (avoid destroying quality)
MIN_FLOOR_KBPS = 64

print("Audio directories to scan:")
for d in AUDIO_DIRS:
    exists = "-" if d.exists() else "✗ MISSING"
    count  = len(list(d.glob("*.mp3"))) if d.exists() else 0
    print(f"  {exists}  {d}  ({count} mp3s)")

## Probe helper (ffprobe)

In [ ]:
def probe_bitrate(mp3_path: Path) -> int | None:
    """
    Returns the bitrate in kbps (rounded to nearest integer) for an MP3 file.
    Returns None if ffprobe fails or the stream has no bit_rate field.
    """
    cmd = [
        "ffprobe", "-v", "quiet",
        "-select_streams", "a:0",
        "-show_entries", "stream=bit_rate",
        "-of", "json",
        str(mp3_path),
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        data   = json.loads(result.stdout)
        streams = data.get("streams", [])
        if streams and streams[0].get("bit_rate") not in (None, "N/A", ""):
            return round(int(streams[0]["bit_rate"]) / 1000)
    except Exception:
        pass
    # Fallback: derive from file size ÷ duration
    cmd2 = [
        "ffprobe", "-v", "quiet",
        "-show_entries", "format=duration,bit_rate",
        "-of", "json",
        str(mp3_path),
    ]
    try:
        result = subprocess.run(cmd2, capture_output=True, text=True, timeout=15)
        data   = json.loads(result.stdout)
        fmt    = data.get("format", {})
        if fmt.get("bit_rate") not in (None, "N/A", ""):
            return round(int(fmt["bit_rate"]) / 1000)
    except Exception:
        pass
    return None


def snap_to_standard(kbps: int) -> int:
    """Snap a raw kbps value to the nearest standard bitrate."""
    return min(STANDARD_BITRATES, key=lambda b: abs(b - kbps))


print("Helper functions defined.")

## Audit: probe all MP3s and collect bitrates

In [ ]:
records = []

for audio_dir in AUDIO_DIRS:
    if not audio_dir.exists():
        print(f"  SKIP (missing): {audio_dir}")
        continue
    mp3_files = sorted(audio_dir.glob("*.mp3"))
    print(f"Probing {len(mp3_files):>3} files in {audio_dir} …", flush=True)
    for mp3 in mp3_files:
        raw_kbps    = probe_bitrate(mp3)
        snapped     = snap_to_standard(raw_kbps) if raw_kbps is not None else None
        records.append({
            "path":      str(mp3),
            "folder":    audio_dir.parts[-3] + "/" + audio_dir.parts[-2],  # e.g. human/english
            "filename":  mp3.name,
            "raw_kbps":  raw_kbps,
            "kbps":      snapped,
        })

df = pd.DataFrame(records)
print(f"\nTotal files probed: {len(df)}")
print(f"Probe failures:     {df['raw_kbps'].isna().sum()}")

## Bitrate distribution report

In [ ]:
valid = df.dropna(subset=["kbps"])

print("═" * 55)
print(" Bitrate distribution (snapped to standard kbps)")
print("═" * 55)
dist = valid["kbps"].value_counts().sort_index()
for kbps, count in dist.items():
    bar = "█" * (count // max(1, len(valid) // 40))
    print(f"  {kbps:>4} kbps  {count:>5} files  {bar}")

print()
print("Per-folder breakdown:")
print(valid.groupby(["folder", "kbps"]).size().rename("count").to_string())

print()
min_kbps = int(valid["kbps"].min())
max_kbps = int(valid["kbps"].max())
uniform  = min_kbps == max_kbps

print(f"  Min bitrate : {min_kbps} kbps")
print(f"  Max bitrate : {max_kbps} kbps")
print(f"  Uniform     : {'YES' if uniform else 'NO — normalisation needed'}")